# The bee's knees: reading MRI with an anatomical mirror

![An artistic knee joint with mirrored MRI planes](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/knee-anatomical-mirror-hero-v1.png)

*AI-generated artistic illustration, not measured MRI. [Prompt and artwork provenance](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/ARTWORK_PROVENANCE.json).*

A knee MRI is a collection of acquisitions, not one picture. This notebook shows how a strong public method turns that collection into a finding vector: choose the slices, preserve their physical geometry, read adjacent triplets, pool the evidence, and average a plain and an anatomically mirrored view. The implementation is deliberately readable enough to fork.

The checkpoint and mirror recipe come from [nartaa / Danial Zakaria's scored V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), building on dreaddevelopment's Raptor and timm's CoAtNet. **The upstream notebook displayed 0.949. That is nartaa's score, not ours.**

## Results so far

| Entry | Score | Basis |
|---|---|---|
| nartaa V2: upstream recipe and checkpoint | 0.949 | Their score on their notebook. Attributed here, not reproduced |
| This public notebook (card) | 0.941 | Official rows 56759401 (V31), 56779708 and 56779710 (V32), all 0.941 |
| Account's verified best | 0.943 | Private notebook, row 56859861 (V8) |
| Public frontier, observed 2026-10-08 | Six notebooks at 0.950 (blends) | Public leaderboard. Not ours |
| Private validation bench, V14 (mirror arm) | 0.949 | Reported by the maintainer; not part of this public notebook |
| This notebook, version 34 (mirror arm) | 0.949 | Official row 56949940, complete |
| Private bench, V18 (mirror plus ConvNeXt blend, 0.70 / 0.30) | 0.948 | Official row 56955785. One tick below the mirror arm alone; not adopted under the predeclared rule |
| Public version 35 (same blend) | 0.948 | Official row 56957785, complete. Same result as the bench twin; not adopted |
| This notebook, version 36 (mirror arm restored) | pending | Official row not yet submitted; no score is claimed |

Earlier versions and their displayed results are recorded in the preserved history at the end. Version 36 restores the version 34 active code path byte for byte. Version 35, which added a second reader, is preserved below as inactive cells.

## Why the second reader is not in the active path

Version 35 added a released three-fold ConvNeXt 2.5D reader from goodpjw2008 and blended it with the mirror arm at a predeclared 0.70 / 0.30 weight. The private bench run of that blend (bench version 18) scored 0.948 in official row 56955785. The mirror arm alone scored 0.949. Under our predeclared rule, a blend is adopted only if it beats the mirror arm. This one does not. A difference of 0.001 is one tick, within noise, but it is not an improvement, and we do not describe it as one. Public version 35 with the same blend scored 0.948 in official row 56957785, the same as its bench twin, which settles the question for this blend.

We deliberately did not tune per-label or blend weights on the labelled data. Our reading of the six public 0.950 notebooks is that they use per-label hand-set weights or weights tuned on a gold set of 58 studies. We have not reproduced them and make no claim about their internals. Our single weight is fixed in advance and is the same for every label.

## What changed in version 34

Version 34 starts its clock in the first statement of the notebook, before any import, staging or CUDA work, so the whole time budget includes setup. Inference runs as the leader of its own process group under the same owned-process watchdog our private campaign uses. The watchdog cleans up the whole group on success as well as on failure. The output is staged and checked for coverage, schema, row order, finiteness, source and checkpoint identity, and file digest. Only then is `/kaggle/working/submission.csv` written, with exclusive creation.

Version 34 also replaces the strict reader with a tolerant one. Section 3 explains why that matters.

[Scientific contract](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/SCIENTIFIC_CONTRACT.md) · [Licenses and model-use terms](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The active reader source is the cell in section 3. Its sha256 is pinned in the launcher.


## A tiny DICOM reader contract before the CPU controls

The original [Synthetic MRI DICOM Fixtures + Golden Arrays](https://www.kaggle.com/datasets/prvsiyan/rsna-knee-synthetic-dicom-fixtures) dataset contains invented intensity patterns: 48 small DICOM files, 43 analytic goldens and five intentional refusals. [Versioned source and provenance](https://github.com/cubres/rsna-knee-abnormality-detection/tree/36bd0508ed2dbbd395d07212fc7fddd3ade10899/datasets/synthetic-dicom-fixtures-v1).

![Six independent analytical golden arrays, plotted from the released NumPy files](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/36bd0508ed2dbbd395d07212fc7fddd3ade10899/datasets/synthetic-dicom-fixtures-v1/golden-preview.png)


```text
Invented pixels → DICOM encoding → bounded single-frame reader
                     ↓                      ↓
              physical file hashes    float32 intensity + geometry
                                             ↓
                                  independent analytic goldens
```

The cell below checks all physical DICOM/golden hashes, reconstructs every analytic truth, compares reader outputs, preserves row/column order and orientation metadata, and checks all five refusal codes. It then prints the shape and endpoint intensities of a non-square phantom so an accidental transpose is visible. Passing establishes this deliberately narrow synthetic pixel contract; it adds no clinical or competition result.

The reference dependencies are NumPy 2.2.6 and pydicom 3.0.1. The demo checks dependency availability before the original CPU canary starts; it uses attached files and downloads nothing. The existing artwork and all prior cells are preserved below. The following process-ownership canary keeps its own original time pair and controls; its earlier "only active cell" description records that preserved version.


## A tiny CPU check before the next reader

The only active cell in this prospective version tests process ownership, using
two invented sleeping fixtures. It does not read the attached competition data,
load a checkpoint, install code, predict MRI findings, train, or submit a file.
The current artwork and every original Version 36 cell remain intact below.

```text
Original time pair → stage two small sources → own fixture group
  1. leader exits zero → TERM-resistant child → TERM then KILL → closed group
  2. leader + child ignore TERM → tiny timeout → TERM then KILL → closed HOLD
                                      ↓
                       durable receipt → final seal → stdout
```

The timeout is a successful **negative control** only when the unchanged owner
closes its group and the scientific wrapper still refuses qualification. A
synthetic control pass is never evidence of MRI correctness or an official score.
The native run must independently finish COMPLETE, and its source, image,
CPU metadata, receipt and final seal must all match before these two observations
can be accepted. The hard budget is 120 seconds from the first active time pair;
staging, both cases, exports and stdout have an earlier code-checked 25-second deadline.
The exported clock scalar is sampled before receipt export, not after final stdout;
acceptance still requires exact native COMPLETE and final receipt/seal/stdout joins.
Only the minimal RAM repeat guard precedes that time pair. The clock does not
cover interpreter/container initialization before the cell begins.

The MIT process owner is preserved byte for byte (sha256 `675cb35e…`), derived
from the independently reviewed Enveda ownership pattern. It covers processes
remaining in its newly owned group, and reaps the leader after closure. These
fixtures never create a new session. They are isolated from the future R7
scientific invocation, so their records cannot enter its three-stage evidence.


## Preserved Version 36 source — inactive history

The original opening artwork above and the 139 original cells below preserve all
140 retained Version 36 cell identifiers, source values and cell metadata. Its
five code cells have become inactive raw cells; their prior execution counts
were null and their output lists were empty. No original file or notebook is
changed by this source proposal. The original top-level notebook metadata is
preserved unchanged as historical metadata, including its former active indices.
The only active code cell in this proposal is the CPU ownership control above.


## Read the experiment before running it

![MRI volume, windows, two views, attention and ranks](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/anatomical_mirror_pipeline_v2.png)

| Choice | This baseline | Why it matters |
|---|---|---|
| Checkpoint | One selected public SWA checkpoint, exact V1 hash | Reproducible identity; this is not a new independent model |
| Physical field of view | 140 mm center crop | Pixels alone do not fix the anatomical field of view |
| Source volume | 96 slices at 384 × 384 | Preserve the pretrained geometry and five acquisition slots |
| Model input | 94 adjacent triplets, center crop to 320 × 320 | Keep neighboring-slice information without resizing again |
| Mirror | Acquisition channels for sagittal centers; image width otherwise | A reflection must follow the acquisition plane |
| Pooling | Finding-specific attention over windows | Each finding can attend to different evidence |
| Combine views | Mean sigmoid probabilities, then per-finding ordinal ranks | This is not logit averaging or clinical calibration |

There is no training step and no model download. Attach [the publisher's original weights dataset V1](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007/versions/1), use competition inputs you are authorized to access, and choose one or more **T4 GPUs**. The qualified environment used two. The qualified container has Torch `2.11.0+cu128` and timm `1.0.29`; library checks fail before any model is loaded.

The baseline runs all current test rows in `inference` mode. Set `MODE="canary"` for a three-study smoke test. That writes a separately named file and is not an accuracy measurement. A notebook run only creates predictions. It does not submit to the competition.


## 1. A small configuration you can actually use

The first code cell starts the clock. Its first statement reads the monotonic time before any import, staging or CUDA work, so the budget covers setup as well as inference. A competition rerun (`KAGGLE_IS_COMPETITION_RERUN` set to true or 1) uses the published 32,400-second GPU limit. A commit uses the 1,800-second notebook limit. Each budget keeps a 50-second margin, so the whole-run deadlines are 32,350 s and 1,750 s. An unknown flag value stops the run instead of guessing.

Change `MODE` and `PREP_WORKERS` for a bounded smoke test or a preparation-speed experiment. Change paths only if your attached data layout differs. The baseline geometry and equal view weights live in the reader in section 3. If you change them, update `READER_SHA256` and document the change before treating a fork as reproducible.


## 2. Respect the acquisition geometry

| Slot | Source slice indices | Count |
|---|---|---:|
| Sagittal, fluid-sensitive | 0–25 | 26 |
| Sagittal, non-fluid-sensitive | 26–47 | 22 |
| Coronal, fluid-sensitive | 48–65 | 18 |
| Coronal, non-fluid-sensitive | 66–77 | 12 |
| Axial, any sequence | 78–95 | 18 |

We prefer the first suitable unused series in metadata order, falling back within the requested plane. DICOM position and orientation order the slices, with `InstanceNumber` as a fallback. Apply the modality LUT and MONOCHROME1 inversion before robust per-series intensity scaling. Sample the 2–98% index span, use the selected images' 2nd/98th pixel percentiles, crop a 140 mm field of view, then area-resize and quantize to uint8.

The volume layout survives missing acquisition slots. A study with no usable acquisition fails instead of receiving model priors. Window selection preserves the upstream span and can cross slot boundaries; the center determines the mirror operation.

![Acquisition-channel reversal and width reflection before normalization](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/mirror_before_normalization_v2.png)

Mirror the **uint8** triplet first. Then crop the central 320 × 320 region (32 pixels from each edge of 384 × 384), and only then apply the three channel-specific ImageNet means and standard deviations. Reversing channels after normalization would alter the recipe because those statistics differ.


## 3. The reader, and why tolerance matters

The next cell writes the exact version-34 reader (sha256 `e9ec648e…`). The launcher verifies those bytes before it starts. The same tolerant reader, its diff from the strict version, the offline test harness and the visible-run receipts are published in the [official-results folder](https://github.com/cubres/rsna-knee-abnormality-detection/tree/b8e1d6e89c391e27821a41455d5f42336f70420d/official-results-2026-10-08) of the [repository](https://github.com/cubres/rsna-knee-abnormality-detection). The architecture and the happy-path arithmetic are the same as before. What changed is how the reader treats bad input.

An earlier private version of this pipeline ran the same architecture under a strict reader. That reader stopped the whole job on an unreadable DICOM header, an unreadable or non-2D pixel array, a study with no usable slice, any per-study model error, or a device count other than two. Its visible three-study run passed. Its rerun on the hidden test cohort, about 1,300 studies, ended with an unhandled error and no score. We cannot see the hidden log, so this is our most likely explanation rather than a proven one. A cohort that large is very likely to contain at least one file that a strict reader rejects, and one such file should not zero the submission.

The public reference handles these cases deliberately. An unreadable header keeps its file as a neutral record. An unreadable slice is skipped and its slot stays zero. A study whose volume cannot be built is predicted from a zero volume and mask. A failed model call gives that study 0.5 probabilities. Every study still receives a prediction. In offline tests with synthetic DICOM, this reader matched the reference on truncated, garbage, multi-frame, NaN and missing-series studies. Its output was identical to the strict reader's on healthy studies. Those tests do not measure accuracy, and the GPU path still needs its own native run.

The general lesson is that a competition reader is part of the scoring pipeline. Its failure policy should be stated, tested on bad inputs, and chosen so that every study stays in the output. The functions to read are `build_volume` (the fault policy), `prepare_windows` and `predict_pair` (the model path), and `run` (coverage and ranks).


## 4. The owned process group

A timeout on one child is not enough. A reader that starts workers can leave descendants behind. A notebook that exports a file before those processes are cleaned up can hide a problem. The next cell writes the owned-process watchdog (sha256 `54636411…`), the same module our private campaign uses. It runs the reader as the leader of a fresh process group, enforces an absolute deadline, and on success terminates the group and verifies that teardown has resolved before the launcher accepts any output. It is Apache-2.0, like the reader.


## 4. Let each finding choose its windows

CoAtNet encodes the triplets in chunks of 16. LayerNorm and a small attention head give each finding its own softmax over the windows. The weighted sum of window features produces one vector per finding; a finding-specific classifier turns it into a logit. Dropout is disabled in evaluation mode.

```text
attention[k, c] = softmax over k (attention_head(feature[k]))
pooled[c]       = sum over k (attention[k, c] * feature[k])
logit[c]        = classifier[c](pooled[c])
probability[c]  = (sigmoid(plain_logit[c]) + sigmoid(mirror_logit[c])) / 2
```

The final CSV uses the upstream per-column `argsort().argsort() / max(1, N-1)` transform. These ordinal ranks are useful for ranking; they are not calibrated probabilities. Preserve the source's NumPy tie behavior. Canary ranks depend on the canary's N and therefore cannot establish full-test prediction equivalence.


## 5. Run in stages and export last

The launcher checks the pinned runtime, the reader and watchdog bytes, and the competition mount before any checkpoint is loaded. It starts the reader under the owned watchdog with the absolute deadline set in the first cell. The reader writes to a fresh stage directory, `/kaggle/working/public_v34_stage_<run>/`, and never to the submission path. After the child's process group is cleaned up and its receipt is read, the staged file is validated: sample columns and row order, finite values in [0, 1], full coverage, and the SHA-256 bound in the native receipt. Only then is `/kaggle/working/submission.csv` written, with exclusive creation. If any check fails, no submission file appears.

The checkpoint requires the publisher's weights dataset attachment. Source SHA-256 and the checkpoint digest are distinct pins. The reader hashes the actual checkpoint file before deserialization. No code license grants rights to redistribute checkpoint bytes or controlled-access records.


## 6. Verify the file without displaying study identifiers

A successful run is not sufficient by itself. The launcher has already checked the sample columns and row order, finite values in [0, 1], full coverage in inference mode, and the file digest bound into the native receipt. The cell below re-reads the exported file and prints only aggregate counts and hashes.

The staged directory contains `probabilities.npz`, the reader's log and the receipt. Some of these carry study identifiers. Use them only in your authorized workspace. Do not publish them as demonstration data or as raw notebook output. A canary file must never be submitted as a full-test prediction.


## What to change in a useful fork

Begin with this exact baseline and save its receipt. For a new experiment, change one mechanism, write down the hypothesis before running, and compare against the same baseline on a suitable independent validation split.

| Question | Small, interpretable experiment | Main caution |
|---|---|---|
| Does the mirrored view contribute independent information? | Compare plain, mirror and their equal probability mean | Keep the checkpoint, studies and preprocessing identical |
| Is acquisition geometry responsible for the gain? | Compare the documented plane-aware mirror with an explicitly defined alternative | Channel statistics and reflection order must stay explicit |
| Do two trained readers add diversity? | Test a preregistered blend with an independently trained, permitted checkpoint | Shared training data or ancestry can reduce diversity |
| Can preparation use fewer CPU threads? | Change `PREP_WORKERS` and measure elapsed time/peak memory | A source check does not measure speed |

Do not tune weights repeatedly on a reused exposed panel or infer causality from a single public leaderboard change. The publisher's Gold58 set was reused for selection, so it is not independent cross-validation. The older plain-versus-affine experiment belongs to a separate training question; it is not evidence that this mirror recipe improved or failed.


## Credits, changes and required acknowledgement

## Upstream lineage

- **nartaa / Danial Zakaria**: selected learned checkpoint and plain/anatomical-mirror inference recipe in [RSNA Knee 0949 Anatomical Mirror, V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), Apache-2.0. The displayed 0.949 score belongs to that upstream notebook.
- **dreaddevelopment**: [Raptor CoAtNet/MIL recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). The selected nartaa weights are the publisher's retrained checkpoints, not these original baseline weights.
- **goodpjw2008**: the three-fold ConvNeXt 2.5D reader and its released sources, Apache-2.0, dataset [goodpjw2008 / rsna-knee-2-5d-convnext-reader](https://www.kaggle.com/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader), version 1. The dataset stays attached to this notebook. Its sources are preserved in the inactive version 35 cells below. Nothing in the active path imports them.
- **timm / Ross Wightman and contributors**: [CoAtNet implementation and pretrained component lineage](https://github.com/huggingface/pytorch-image-models). Third-party components retain their own licenses.
- The checkpoint publisher credits weak-label contributions from [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), together with the publisher's report-derived labels. These tables and source reports are not included.
- **RSNA and competition data contributors**: data use remains subject to the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

## Original implementation and changes

`public_mirror_reader.py` (version 34, sha256 `e9ec648e…`) is an original readable implementation of the visible upstream recipe, not a copy of private campaign code or an independently trained model. Relative to the strict reader of earlier versions, it adopts the public reference's fault semantics: unreadable headers keep a neutral record, unreadable slices are skipped, unbuildable studies are predicted from zero inputs, failed model calls give 0.5, and no study is dropped. It adds checkpoint and source identity constants, schema and coverage gates, tensor-only deserialization, bounded thread queues without multiprocessing, separate canary output names and create-only outputs. It does not execute the upstream fast-reader payload. The owned-process watchdog (`owned_watchdog.py`, sha256 `54636411…`) is original Apache-2.0 code. The selected checkpoint bytes, geometry and aggregation formulas remain attributed to the public references. These differences require native validation; this version makes no reproduced-score or bit-equivalence claim.

## Publisher's required OAI / NDA acknowledgement

The following paragraph is preserved verbatim from the upstream publisher's credits and dataset description. It acknowledges the upstream research lineage; distributing this reader does not assert that this repository obtained or redistributed controlled-access OAI records.

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study 3407](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

The upstream weights package grants only its stated model-use permission. OAI access and redistribution remain governed by NDA. This notebook contains no OAI or competition MRI, reports, source label tables, participant identifiers or learned checkpoint payloads.


### Source and model licenses stay separate

- Active original reader and upstream source lineage: **Apache-2.0**. [Complete Apache license](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/Apache-2.0.txt).
- Newly written notebook explanations/configuration/diagrams: **MIT**, copyright 2026 cubres. [MIT text](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/MIT.txt).
- Selected checkpoint: publisher's **Other / research and educational model-use terms**, linked as original V1; no checkpoint redistribution or sublicensing grant is assumed.
- Historical inactive source below retains all original notices, including its DINOv3 license. This refresh does not relicense that archive.

The publisher's model-use statement is:

> These learned weights are provided for research and educational use, including the RSNA Knee competition. This model-use permission grants no rights to obtain, publish, redistribute or re-identify OAI participant data, source MRI images, reports or other controlled-access records. Access to OAI data remains governed by NDA's terms. Third-party source code and pretrained components retain their respective licences. No CC0 or other open-data licence is applied to OAI data or raw corpora by this model release.

[Versioned model-use explanation](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/MODEL_USAGE.md) · [License boundaries](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The version-34 reader is the cell in section 3. No weights, source MRI, reports, label tables or participant records are embedded in this notebook source.


<details><summary>Complete Apache-2.0 license for the active reader</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "{}"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2019 Ross Wightman

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>


## Preserved historical source

Every original Version 33 cell remains below as an inactive raw cell with its original cell identifier. The exact original notebook bytes (203,753 bytes, SHA-256 `df02bfdc…`) are also preserved as a SHA-256-bound gzip/base64 snapshot in the notebook metadata, so the original JSON, outputs, cell types and metadata can be reconstructed exactly. The historical launcher and reader in those cells do not run in version 34.

Version 35 cells follow the version 34 cells, preserved verbatim after a header cell. A cell identifier carries a `v35-` prefix only where the bare identifier would repeat one already used in this notebook. The cell sources are unchanged.


## Preserved version 35 cells (inactive)

The 81 cells below are the complete version 35 notebook as published, in order. Code cells are inactive raw cells whose original source is verbatim. Their metadata records the original type, index, identifier and a source digest. Nothing below runs. Markdown is kept as published.


# The bee's knees: reading MRI with an anatomical mirror, and a second reader

![An artistic knee joint with mirrored MRI planes](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/knee-anatomical-mirror-hero-v1.png)

*AI-generated artistic illustration, not measured MRI. [Prompt and artwork provenance](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/ARTWORK_PROVENANCE.json).*

A knee MRI is a collection of acquisitions, not one picture. This notebook turns that collection into a finding vector in two ways, and blends them. The first arm reads adjacent slice triplets with a public checkpoint and averages a plain and an anatomically mirrored view. The second arm is a released three-fold ConvNeXt reader from another team. The blend is a rank average at one predeclared weight, 0.70 for the mirror arm and 0.30 for the ConvNeXt arm. The implementation is readable enough to fork.

## Results so far

| Entry | Score | Basis |
|---|---|---|
| nartaa V2: upstream recipe and checkpoint | 0.949 | Their score on their notebook. Attributed here, not reproduced |
| This public notebook (card) | 0.941 | Official rows 56759401 (V31), 56779708 and 56779710 (V32), all 0.941 |
| Account's verified best | 0.943 | Private notebook, row 56859861 (V8) |
| Public frontier, observed 2026-10-08 | Six notebooks at 0.950 (blends) | Public leaderboard. Not ours |
| Private validation bench, V14 (mirror arm) | 0.949 | Reported by the maintainer; not part of this public notebook |
| Private validation bench, V17 (E3 r224 blend) | 0.949 | Reported by the maintainer; ties V14 |
| Private validation bench, V18 (ConvNeXt arm, decoder fix) | no score yet | Arm ran on all 3 visible studies; oracle parity max abs diff 2.98e-08 against tolerance 2e-4 |
| This notebook, version 34 | 0.949 | Official row 56949940, complete |
| This notebook, version 35 | pending | Adds the ConvNeXt arm. No score is claimed until a completed official submission exists |

Earlier versions appear in the preserved history below, with their cells unchanged. Version 35 changes the active path, so its own native run and official submission are still to come.

## What changed in version 35

The clock starts in the first statement of the notebook, before any import, staging or CUDA work, so the time budget covers setup as well as inference. Both arms run inside the same owned process-group watchdog, which cleans up the whole process tree on success as well as on failure. The mirror arm is validated before anything else. The ConvNeXt arm runs only if the remaining budget clears a time gate. Each decoder wheel is installed on its own, best effort. The blend is written, and only then does the notebook export `/kaggle/working/submission.csv`, exclusively and last. If the ConvNeXt arm fails for any reason, the file is the mirror arm alone, and the receipt says so.

The blend in this notebook is predeclared: 0.70 for the mirror arm (nartaa's recipe and checkpoints, Apache-2.0) and 0.30 for goodpjw2008's ConvNeXt 2.5D reader (Apache-2.0). We have not reproduced the public 0.950 notebooks and make no claim about their internals. Credits are in the notices cell.


## Read the experiment before running it

![MRI volume, windows, two views, attention and ranks](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/anatomical_mirror_pipeline_v2.png)

| Choice | This baseline | Why it matters |
|---|---|---|
| Mirror arm: checkpoint | One selected public SWA checkpoint, exact V1 hash | Reproducible identity; this is not a new independent model |
| Mirror arm: field of view | 140 mm center crop | Pixels alone do not fix the anatomical field of view |
| Mirror arm: input | 94 adjacent triplets, center crop to 320 × 320 | Keep neighboring-slice information without resizing again |
| Mirror arm: mirror | Acquisition channels for sagittal centers; image width otherwise | A reflection must follow the acquisition plane |
| ConvNeXt arm: reader | goodpjw2008's three-fold ConvNeXt 2.5D reader, released sources, DICOM preprocessing as published | A second, independently trained reader with its own preprocessing |
| Blend | 0.70 × rank(mirror) + 0.30 × rank(ConvNeXt), then the rank of that score | One global weight, fixed before any run; no per-finding weights |
| Time gate | ConvNeXt runs only if the remaining budget clears 875 s (commit) or 8,100 s (rerun) | A slow arm can never delay the mirror-only fallback |

The baseline needs one or more **T4 GPUs** (the qualified environment used two), the publisher's original weights dataset V1, the goodpjw ConvNeXt dataset V1, and competition inputs you are authorized to access. The qualified container has Torch `2.11.0+cu128` and timm `1.0.29`; library checks fail before any model is loaded. Set `MODE="canary"` for a three-study smoke test of the mirror arm; it writes a separate file and is not an accuracy measurement. A notebook run only creates predictions. It never submits to the competition.


## 1. A small configuration you can actually use

The first code cell starts the clock with its first statement, before any import or staging, so the budget includes setup. A competition rerun (`KAGGLE_IS_COMPETITION_RERUN` set to true or 1) uses the published 32,400-second GPU limit. A commit uses the 1,800-second notebook limit. Each budget keeps a 50-second margin, so the whole-run deadlines are 32,350 s and 1,750 s. An unknown flag value stops the run.

The ConvNeXt time gate threshold is the smaller of the full estimate plus margin (7,200 s + 900 s) and half the whole budget. On a commit that is 875 s; on a rerun it is 8,100 s. Inside the arm, the child process gets the whole budget minus a 120-second reserve, so the blend and export always keep their time. Change `MODE`, `PREP_WORKERS` or the gate constants for a bounded experiment; paths change only if your attached data layout differs. If you change the geometry, the reader or the blend, update the pins and document the change before treating a fork as reproducible.


## 2. Respect the acquisition geometry

| Slot | Source slice indices | Count |
|---|---|---:|
| Sagittal, fluid-sensitive | 0–25 | 26 |
| Sagittal, non-fluid-sensitive | 26–47 | 22 |
| Coronal, fluid-sensitive | 48–65 | 18 |
| Coronal, non-fluid-sensitive | 66–77 | 12 |
| Axial, any sequence | 78–95 | 18 |

We prefer the first suitable unused series in metadata order, falling back within the requested plane. DICOM position and orientation order the slices, with `InstanceNumber` as a fallback. Apply the modality LUT and MONOCHROME1 inversion before robust per-series intensity scaling. Sample the 2–98% index span, use the selected images' 2nd/98th pixel percentiles, crop a 140 mm field of view, then area-resize and quantize to uint8.

The volume layout survives missing acquisition slots. A study with no usable acquisition fails instead of receiving model priors. Window selection preserves the upstream span and can cross slot boundaries; the center determines the mirror operation.

![Acquisition-channel reversal and width reflection before normalization](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/mirror_before_normalization_v2.png)

Mirror the **uint8** triplet first. Then crop the central 320 × 320 region (32 pixels from each edge of 384 × 384), and only then apply the three channel-specific ImageNet means and standard deviations. Reversing channels after normalization would alter the recipe because those statistics differ.


## 3. The reader, and why tolerance matters

The next cell writes the exact version-34 reader (sha256 `e9ec648e…`). The launcher verifies those bytes before it starts. The same tolerant reader, its diff from the strict version, the offline test harness and the visible-run receipts are published in the [official-results folder](https://github.com/cubres/rsna-knee-abnormality-detection/tree/b8e1d6e89c391e27821a41455d5f42336f70420d/official-results-2026-10-08) of the [repository](https://github.com/cubres/rsna-knee-abnormality-detection). The architecture and the happy-path arithmetic are the same as before. What changed is how the reader treats bad input.

An earlier private version of this pipeline ran the same architecture under a strict reader. That reader stopped the whole job on an unreadable DICOM header, an unreadable or non-2D pixel array, a study with no usable slice, any per-study model error, or a device count other than two. Its visible three-study run passed. Its rerun on the hidden test cohort, about 1,300 studies, ended with an unhandled error and no score. We cannot see the hidden log, so this is our most likely explanation rather than a proven one. A cohort that large is very likely to contain at least one file that a strict reader rejects, and one such file should not zero the submission.

The public reference handles these cases deliberately. An unreadable header keeps its file as a neutral record. An unreadable slice is skipped and its slot stays zero. A study whose volume cannot be built is predicted from a zero volume and mask. A failed model call gives that study 0.5 probabilities. Every study still receives a prediction. In offline tests with synthetic DICOM, this reader matched the reference on truncated, garbage, multi-frame, NaN and missing-series studies. Its output was identical to the strict reader's on healthy studies. Those tests do not measure accuracy, and the GPU path still needs its own native run.

The general lesson is that a competition reader is part of the scoring pipeline. Its failure policy should be stated, tested on bad inputs, and chosen so that every study stays in the output. The functions to read are `build_volume` (the fault policy), `prepare_windows` and `predict_pair` (the model path), and `run` (coverage and ranks).


## 4. The owned process group

A timeout on one child is not enough. A reader that starts workers can leave descendants behind. A notebook that exports a file before those processes are cleaned up can hide a problem. The next cell writes the owned-process watchdog (sha256 `54636411…`), the same module our private campaign uses. It runs the reader as the leader of a fresh process group, enforces an absolute deadline, and on success terminates the group and verifies that teardown has resolved before the launcher accepts any output. It is Apache-2.0, like the reader.


## 5. The second arm: goodpjw2008's ConvNeXt 2.5D reader

goodpjw2008 published a three-fold ConvNeXt reader (dataset [goodpjw2008 / rsna-knee-2-5d-convnext-reader](https://www.kaggle.com/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader), version 1, Apache-2.0). Its released sources are `preprocess.py`, `knee.py` and `infer.py`. The next four cells write them to disk byte for byte, and the launcher checks each file against its published hash. The reader turns each DICOM series into a canonical volume, takes twelve three-slice windows per slot over six slots, runs the three checkpoints of `convnext_tiny.fb_in22k_ft_in1k`, and averages their sigmoid outputs. The ConvNeXt glue cell adds an output directory, a per-study fault flag and a decode report. It never reads labels and never trains.

**Why a 0.30 weight.** The predeclared weight is 0.30. It is one global weight for all twelve findings, chosen before any run, and it is not tuned on the public board or on the gold studies. A public notebook that blends this reader at about the same weight reports a similar small change on its own base.

**Failures.** A study whose input or model call fails, or whose selected series has a slice or series that cannot be decoded, is left at 0.5 for this arm and counted; it is never fatal. If more than half the studies fail, or the child process fails or times out, the arm is marked failed and the notebook exports the mirror arm alone. The decoder step installs each pylibjpeg wheel on its own. The pure-Python frontend alone decodes nothing; a binary plugin is needed for compressed transfer syntaxes. The receipt records each wheel's outcome, the Python version and which transfer syntaxes this runtime can decode.

**Credits and licences.** The released reader, its sources and the dataset are goodpjw2008's, Apache-2.0. The backbone is `convnext_tiny.fb_in22k_ft_in1k` (ImageNet-22k, then ImageNet-1k). The decoder packages keep their own terms: pylibjpeg MIT; pylibjpeg-openjpeg MIT with bundled notices; pylibjpeg-libjpeg GPLv3 with a proprietary alternative. The blend, the receipts and this notebook's orchestration are original and released under MIT, copyright 2026 cubres.


## 4. Let each finding choose its windows

CoAtNet encodes the triplets in chunks of 16. LayerNorm and a small attention head give each finding its own softmax over the windows. The weighted sum of window features produces one vector per finding; a finding-specific classifier turns it into a logit. Dropout is disabled in evaluation mode.

```text
attention[k, c] = softmax over k (attention_head(feature[k]))
pooled[c]       = sum over k (attention[k, c] * feature[k])
logit[c]        = classifier[c](pooled[c])
probability[c]  = (sigmoid(plain_logit[c]) + sigmoid(mirror_logit[c])) / 2
```

The final CSV uses the upstream per-column `argsort().argsort() / max(1, N-1)` transform. These ordinal ranks are useful for ranking; they are not calibrated probabilities. Preserve the source's NumPy tie behavior. Canary ranks depend on the canary's N and therefore cannot establish full-test prediction equivalence.


## 6. Run in stages and export last

The launcher checks the pinned runtime, the reader, the watchdog, the ConvNeXt glue and each released module against its hash before any model is loaded. It runs the mirror arm first, under the owned watchdog with the absolute deadline set in the first cell. The mirror output is validated: receipt, coverage, schema, row order, finiteness, the checkpoint hash, and the recomputed ranks. Then the ConvNeXt arm runs if the time gate allows. Its child process has its own receipt, and the blend is computed from both arms' probabilities. The staged blend, the mirror-only file and the arm receipts go to a fresh stage directory, not to the submission path. Only after every check does the notebook write `/kaggle/working/submission.csv`, with exclusive creation.

The checkpoints require the publisher's weights dataset attachment, and the ConvNeXt arm requires the goodpjw dataset attachment. The reader hashes the actual checkpoint file before deserialization. No code license grants rights to redistribute checkpoint bytes or controlled-access records.


## 7. Verify the run without displaying study identifiers

A successful run is not sufficient by itself. The launcher has already checked the sample columns and row order, finite values in [0, 1], full coverage in inference mode, the file digests in the receipts, and the blend arithmetic. The cell below re-reads the exported file and prints only aggregate counts, hashes and arm statuses.

The stage directory contains the mirror and ConvNeXt outputs, the receipts and the logs. Some carry study identifiers. Use them only in your authorized workspace; do not publish them as demonstration data or raw notebook output. A canary file must never be submitted as a full-test prediction.


## What to change in a useful fork

Begin with this exact baseline and save its receipt. For a new experiment, change one mechanism, write down the hypothesis before running, and compare against the same baseline on a suitable independent validation split.

| Question | Small, interpretable experiment | Main caution |
|---|---|---|
| Does the mirrored view contribute independent information? | Compare plain, mirror and their equal probability mean | Keep the checkpoint, studies and preprocessing identical |
| Is acquisition geometry responsible for the gain? | Compare the documented plane-aware mirror with an explicitly defined alternative | Channel statistics and reflection order must stay explicit |
| Do two trained readers add diversity? | Test a preregistered blend with an independently trained, permitted checkpoint | Shared training data or ancestry can reduce diversity |
| Can preparation use fewer CPU threads? | Change `PREP_WORKERS` and measure elapsed time/peak memory | A source check does not measure speed |

Do not tune weights repeatedly on a reused exposed panel or infer causality from a single public leaderboard change. The publisher's Gold58 set was reused for selection, so it is not independent cross-validation. The older plain-versus-affine experiment belongs to a separate training question; it is not evidence that this mirror recipe improved or failed.


## Credits, changes and required acknowledgement

## Upstream lineage

- **nartaa / Danial Zakaria**: selected learned checkpoint and plain/anatomical-mirror inference recipe in [RSNA Knee 0949 Anatomical Mirror, V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), Apache-2.0. The displayed 0.949 score belongs to that upstream notebook.
- **dreaddevelopment**: [Raptor CoAtNet/MIL recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). The selected nartaa weights are the publisher's retrained checkpoints, not these original baseline weights.
- **goodpjw2008**: the three-fold ConvNeXt 2.5D reader and its released sources (`preprocess.py`, `knee.py`, `infer.py`), Apache-2.0, dataset [goodpjw2008 / rsna-knee-2-5d-convnext-reader](https://www.kaggle.com/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader), version 1. The sources are embedded byte for byte and checked against their published hashes.
- **timm / Ross Wightman and contributors**: [CoAtNet implementation and pretrained component lineage](https://github.com/huggingface/pytorch-image-models). Third-party components retain their own licenses.
- The checkpoint publisher credits weak-label contributions from [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), together with the publisher's report-derived labels. These tables and source reports are not included.
- **RSNA and competition data contributors**: data use remains subject to the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

## Original implementation and changes

The ConvNeXt glue, the blend, the time gate, the decoder probe and this orchestration are original to this notebook and are released under MIT, copyright 2026 cubres. The owned-process watchdog (`owned_watchdog.py`) is Apache-2.0 original code.

`public_mirror_reader.py` (version 34, sha256 `e9ec648e…`) is an original readable implementation of the visible upstream recipe, not a copy of private campaign code or an independently trained model. Relative to the strict reader of earlier versions, it adopts the public reference's fault semantics: unreadable headers keep a neutral record, unreadable slices are skipped, unbuildable studies are predicted from zero inputs, failed model calls give 0.5, and no study is dropped. It adds checkpoint and source identity constants, schema and coverage gates, tensor-only deserialization, bounded thread queues without multiprocessing, separate canary output names and create-only outputs. It does not execute the upstream fast-reader payload. The owned-process watchdog (`owned_watchdog.py`, sha256 `54636411…`) is original Apache-2.0 code. The selected checkpoint bytes, geometry and aggregation formulas remain attributed to the public references. These differences require native validation; this version makes no reproduced-score or bit-equivalence claim.

## Publisher's required OAI / NDA acknowledgement

The following paragraph is preserved verbatim from the upstream publisher's credits and dataset description. It acknowledges the upstream research lineage; distributing this reader does not assert that this repository obtained or redistributed controlled-access OAI records.

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study 3407](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

The upstream weights package grants only its stated model-use permission. OAI access and redistribution remain governed by NDA. This notebook contains no OAI or competition MRI, reports, source label tables, participant identifiers or learned checkpoint payloads.


### Source and model licenses stay separate

- Active original reader and upstream source lineage: **Apache-2.0**. [Complete Apache license](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/Apache-2.0.txt).
- Newly written notebook explanations/configuration/diagrams: **MIT**, copyright 2026 cubres. [MIT text](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/MIT.txt).
- Selected checkpoint: publisher's **Other / research and educational model-use terms**, linked as original V1; no checkpoint redistribution or sublicensing grant is assumed.
- Historical inactive source below retains all original notices, including its DINOv3 license. This refresh does not relicense that archive.

The publisher's model-use statement is:

> These learned weights are provided for research and educational use, including the RSNA Knee competition. This model-use permission grants no rights to obtain, publish, redistribute or re-identify OAI participant data, source MRI images, reports or other controlled-access records. Access to OAI data remains governed by NDA's terms. Third-party source code and pretrained components retain their respective licences. No CC0 or other open-data licence is applied to OAI data or raw corpora by this model release.

[Versioned model-use explanation](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/MODEL_USAGE.md) · [License boundaries](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The version-34 reader is the cell in section 3. No weights, source MRI, reports, label tables or participant records are embedded in this notebook source.


<details><summary>Complete Apache-2.0 license for the active reader</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "{}"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2019 Ross Wightman

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>


## Preserved history

Every Version 34 cell appears below as an inactive cell, in its original order. Code cells are raw, so nothing runs; their sources are verbatim, and their metadata holds only small keys (cell type, original index, original identifier and source hashes). The earlier Version 33 source is preserved inside Version 34 in the same way. Nothing in the preserved history is duplicated in metadata.


# The bee's knees: reading MRI with an anatomical mirror

![An artistic knee joint with mirrored MRI planes](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/knee-anatomical-mirror-hero-v1.png)

*AI-generated artistic illustration, not measured MRI. [Prompt and artwork provenance](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/ARTWORK_PROVENANCE.json).*

A knee MRI is a collection of acquisitions, not one picture. This notebook shows how a strong public method turns that collection into a finding vector: choose the slices, preserve their physical geometry, read adjacent triplets, pool the evidence, and average a plain and an anatomically mirrored view. The implementation is deliberately readable enough to fork.

The checkpoint and mirror recipe come from [nartaa / Danial Zakaria's scored V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), building on dreaddevelopment's Raptor and timm's CoAtNet. **The upstream notebook displayed 0.949. That is nartaa's score, not ours.**

## Results so far

| Entry | Score | Basis |
|---|---|---|
| nartaa V2: upstream recipe and checkpoint | 0.949 | Their score on their notebook. Attributed here, not reproduced |
| This public notebook (card) | 0.941 | Official rows 56759401 (V31), 56779708 and 56779710 (V32), all 0.941 |
| Account's verified best | 0.943 | Private notebook, row 56859861 (V8) |
| Public frontier, observed 2026-10-08 | Six notebooks at 0.950 (blends) | Public leaderboard. Not ours |
| Private V14 | pending | Row 56948803; no score |
| This version, V34 | pending | No score is claimed until a completed official submission exists |

Earlier versions and their displayed results are recorded in the preserved history at the end. Version 34 changes the active code path. Its own native GPU run and official submission are still to come.

## What changed in version 34

Version 34 starts its clock in the first statement of the notebook, before any import, staging or CUDA work, so the whole time budget includes setup. Inference runs as the leader of its own process group under the same owned-process watchdog our private campaign uses. The watchdog cleans up the whole group on success as well as on failure. The output is staged and checked for coverage, schema, row order, finiteness, source and checkpoint identity, and file digest. Only then is `/kaggle/working/submission.csv` written, with exclusive creation.

Version 34 also replaces the strict reader with a tolerant one. Section 3 explains why that matters.

[Scientific contract](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/SCIENTIFIC_CONTRACT.md) · [Licenses and model-use terms](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The active reader source is the cell in section 3. Its sha256 is pinned in the launcher.


## Read the experiment before running it

![MRI volume, windows, two views, attention and ranks](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/anatomical_mirror_pipeline_v2.png)

| Choice | This baseline | Why it matters |
|---|---|---|
| Checkpoint | One selected public SWA checkpoint, exact V1 hash | Reproducible identity; this is not a new independent model |
| Physical field of view | 140 mm center crop | Pixels alone do not fix the anatomical field of view |
| Source volume | 96 slices at 384 × 384 | Preserve the pretrained geometry and five acquisition slots |
| Model input | 94 adjacent triplets, center crop to 320 × 320 | Keep neighboring-slice information without resizing again |
| Mirror | Acquisition channels for sagittal centers; image width otherwise | A reflection must follow the acquisition plane |
| Pooling | Finding-specific attention over windows | Each finding can attend to different evidence |
| Combine views | Mean sigmoid probabilities, then per-finding ordinal ranks | This is not logit averaging or clinical calibration |

There is no training step and no model download. Attach [the publisher's original weights dataset V1](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007/versions/1), use competition inputs you are authorized to access, and choose one or more **T4 GPUs**. The qualified environment used two. The qualified container has Torch `2.11.0+cu128` and timm `1.0.29`; library checks fail before any model is loaded.

The baseline runs all current test rows in `inference` mode. Set `MODE="canary"` for a three-study smoke test. That writes a separately named file and is not an accuracy measurement. A notebook run only creates predictions. It does not submit to the competition.


## 1. A small configuration you can actually use

The first code cell starts the clock. Its first statement reads the monotonic time before any import, staging or CUDA work, so the budget covers setup as well as inference. A competition rerun (`KAGGLE_IS_COMPETITION_RERUN` set to true or 1) uses the published 32,400-second GPU limit. A commit uses the 1,800-second notebook limit. Each budget keeps a 50-second margin, so the whole-run deadlines are 32,350 s and 1,750 s. An unknown flag value stops the run instead of guessing.

Change `MODE` and `PREP_WORKERS` for a bounded smoke test or a preparation-speed experiment. Change paths only if your attached data layout differs. The baseline geometry and equal view weights live in the reader in section 3. If you change them, update `READER_SHA256` and document the change before treating a fork as reproducible.


## 2. Respect the acquisition geometry

| Slot | Source slice indices | Count |
|---|---|---:|
| Sagittal, fluid-sensitive | 0–25 | 26 |
| Sagittal, non-fluid-sensitive | 26–47 | 22 |
| Coronal, fluid-sensitive | 48–65 | 18 |
| Coronal, non-fluid-sensitive | 66–77 | 12 |
| Axial, any sequence | 78–95 | 18 |

We prefer the first suitable unused series in metadata order, falling back within the requested plane. DICOM position and orientation order the slices, with `InstanceNumber` as a fallback. Apply the modality LUT and MONOCHROME1 inversion before robust per-series intensity scaling. Sample the 2–98% index span, use the selected images' 2nd/98th pixel percentiles, crop a 140 mm field of view, then area-resize and quantize to uint8.

The volume layout survives missing acquisition slots. A study with no usable acquisition fails instead of receiving model priors. Window selection preserves the upstream span and can cross slot boundaries; the center determines the mirror operation.

![Acquisition-channel reversal and width reflection before normalization](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/mirror_before_normalization_v2.png)

Mirror the **uint8** triplet first. Then crop the central 320 × 320 region (32 pixels from each edge of 384 × 384), and only then apply the three channel-specific ImageNet means and standard deviations. Reversing channels after normalization would alter the recipe because those statistics differ.


## 3. The reader, and why tolerance matters

The next cell writes the exact version-34 reader (sha256 `e9ec648e…`). The launcher verifies those bytes before it starts. The same tolerant reader, its diff from the strict version, the offline test harness and the visible-run receipts are published in the [official-results folder](https://github.com/cubres/rsna-knee-abnormality-detection/tree/b8e1d6e89c391e27821a41455d5f42336f70420d/official-results-2026-10-08) of the [repository](https://github.com/cubres/rsna-knee-abnormality-detection). The architecture and the happy-path arithmetic are the same as before. What changed is how the reader treats bad input.

An earlier private version of this pipeline ran the same architecture under a strict reader. That reader stopped the whole job on an unreadable DICOM header, an unreadable or non-2D pixel array, a study with no usable slice, any per-study model error, or a device count other than two. Its visible three-study run passed. Its rerun on the hidden test cohort, about 1,300 studies, ended with an unhandled error and no score. We cannot see the hidden log, so this is our most likely explanation rather than a proven one. A cohort that large is very likely to contain at least one file that a strict reader rejects, and one such file should not zero the submission.

The public reference handles these cases deliberately. An unreadable header keeps its file as a neutral record. An unreadable slice is skipped and its slot stays zero. A study whose volume cannot be built is predicted from a zero volume and mask. A failed model call gives that study 0.5 probabilities. Every study still receives a prediction. In offline tests with synthetic DICOM, this reader matched the reference on truncated, garbage, multi-frame, NaN and missing-series studies. Its output was identical to the strict reader's on healthy studies. Those tests do not measure accuracy, and the GPU path still needs its own native run.

The general lesson is that a competition reader is part of the scoring pipeline. Its failure policy should be stated, tested on bad inputs, and chosen so that every study stays in the output. The functions to read are `build_volume` (the fault policy), `prepare_windows` and `predict_pair` (the model path), and `run` (coverage and ranks).


## 4. The owned process group

A timeout on one child is not enough. A reader that starts workers can leave descendants behind. A notebook that exports a file before those processes are cleaned up can hide a problem. The next cell writes the owned-process watchdog (sha256 `54636411…`), the same module our private campaign uses. It runs the reader as the leader of a fresh process group, enforces an absolute deadline, and on success terminates the group and verifies that teardown has resolved before the launcher accepts any output. It is Apache-2.0, like the reader.


## 4. Let each finding choose its windows

CoAtNet encodes the triplets in chunks of 16. LayerNorm and a small attention head give each finding its own softmax over the windows. The weighted sum of window features produces one vector per finding; a finding-specific classifier turns it into a logit. Dropout is disabled in evaluation mode.

```text
attention[k, c] = softmax over k (attention_head(feature[k]))
pooled[c]       = sum over k (attention[k, c] * feature[k])
logit[c]        = classifier[c](pooled[c])
probability[c]  = (sigmoid(plain_logit[c]) + sigmoid(mirror_logit[c])) / 2
```

The final CSV uses the upstream per-column `argsort().argsort() / max(1, N-1)` transform. These ordinal ranks are useful for ranking; they are not calibrated probabilities. Preserve the source's NumPy tie behavior. Canary ranks depend on the canary's N and therefore cannot establish full-test prediction equivalence.


## 5. Run in stages and export last

The launcher checks the pinned runtime, the reader and watchdog bytes, and the competition mount before any checkpoint is loaded. It starts the reader under the owned watchdog with the absolute deadline set in the first cell. The reader writes to a fresh stage directory, `/kaggle/working/public_v34_stage_<run>/`, and never to the submission path. After the child's process group is cleaned up and its receipt is read, the staged file is validated: sample columns and row order, finite values in [0, 1], full coverage, and the SHA-256 bound in the native receipt. Only then is `/kaggle/working/submission.csv` written, with exclusive creation. If any check fails, no submission file appears.

The checkpoint requires the publisher's weights dataset attachment. Source SHA-256 and the checkpoint digest are distinct pins. The reader hashes the actual checkpoint file before deserialization. No code license grants rights to redistribute checkpoint bytes or controlled-access records.


## 6. Verify the file without displaying study identifiers

A successful run is not sufficient by itself. The launcher has already checked the sample columns and row order, finite values in [0, 1], full coverage in inference mode, and the file digest bound into the native receipt. The cell below re-reads the exported file and prints only aggregate counts and hashes.

The staged directory contains `probabilities.npz`, the reader's log and the receipt. Some of these carry study identifiers. Use them only in your authorized workspace. Do not publish them as demonstration data or as raw notebook output. A canary file must never be submitted as a full-test prediction.


## What to change in a useful fork

Begin with this exact baseline and save its receipt. For a new experiment, change one mechanism, write down the hypothesis before running, and compare against the same baseline on a suitable independent validation split.

| Question | Small, interpretable experiment | Main caution |
|---|---|---|
| Does the mirrored view contribute independent information? | Compare plain, mirror and their equal probability mean | Keep the checkpoint, studies and preprocessing identical |
| Is acquisition geometry responsible for the gain? | Compare the documented plane-aware mirror with an explicitly defined alternative | Channel statistics and reflection order must stay explicit |
| Do two trained readers add diversity? | Test a preregistered blend with an independently trained, permitted checkpoint | Shared training data or ancestry can reduce diversity |
| Can preparation use fewer CPU threads? | Change `PREP_WORKERS` and measure elapsed time/peak memory | A source check does not measure speed |

Do not tune weights repeatedly on a reused exposed panel or infer causality from a single public leaderboard change. The publisher's Gold58 set was reused for selection, so it is not independent cross-validation. The older plain-versus-affine experiment belongs to a separate training question; it is not evidence that this mirror recipe improved or failed.


## Credits, changes and required acknowledgement

## Upstream lineage

- **nartaa / Danial Zakaria**: selected learned checkpoint and plain/anatomical-mirror inference recipe in [RSNA Knee 0949 Anatomical Mirror, V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), Apache-2.0. The displayed 0.949 score belongs to that upstream notebook.
- **dreaddevelopment**: [Raptor CoAtNet/MIL recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). The selected nartaa weights are the publisher's retrained checkpoints, not these original baseline weights.
- **timm / Ross Wightman and contributors**: [CoAtNet implementation and pretrained component lineage](https://github.com/huggingface/pytorch-image-models). Third-party components retain their own licenses.
- The checkpoint publisher credits weak-label contributions from [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), together with the publisher's report-derived labels. These tables and source reports are not included.
- **RSNA and competition data contributors**: data use remains subject to the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

## Original implementation and changes

`public_mirror_reader.py` (version 34, sha256 `e9ec648e…`) is an original readable implementation of the visible upstream recipe, not a copy of private campaign code or an independently trained model. Relative to the strict reader of earlier versions, it adopts the public reference's fault semantics: unreadable headers keep a neutral record, unreadable slices are skipped, unbuildable studies are predicted from zero inputs, failed model calls give 0.5, and no study is dropped. It adds checkpoint and source identity constants, schema and coverage gates, tensor-only deserialization, bounded thread queues without multiprocessing, separate canary output names and create-only outputs. It does not execute the upstream fast-reader payload. The owned-process watchdog (`owned_watchdog.py`, sha256 `54636411…`) is original Apache-2.0 code. The selected checkpoint bytes, geometry and aggregation formulas remain attributed to the public references. These differences require native validation; this version makes no reproduced-score or bit-equivalence claim.

## Publisher's required OAI / NDA acknowledgement

The following paragraph is preserved verbatim from the upstream publisher's credits and dataset description. It acknowledges the upstream research lineage; distributing this reader does not assert that this repository obtained or redistributed controlled-access OAI records.

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study 3407](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

The upstream weights package grants only its stated model-use permission. OAI access and redistribution remain governed by NDA. This notebook contains no OAI or competition MRI, reports, source label tables, participant identifiers or learned checkpoint payloads.


### Source and model licenses stay separate

- Active original reader and upstream source lineage: **Apache-2.0**. [Complete Apache license](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/Apache-2.0.txt).
- Newly written notebook explanations/configuration/diagrams: **MIT**, copyright 2026 cubres. [MIT text](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/MIT.txt).
- Selected checkpoint: publisher's **Other / research and educational model-use terms**, linked as original V1; no checkpoint redistribution or sublicensing grant is assumed.
- Historical inactive source below retains all original notices, including its DINOv3 license. This refresh does not relicense that archive.

The publisher's model-use statement is:

> These learned weights are provided for research and educational use, including the RSNA Knee competition. This model-use permission grants no rights to obtain, publish, redistribute or re-identify OAI participant data, source MRI images, reports or other controlled-access records. Access to OAI data remains governed by NDA's terms. Third-party source code and pretrained components retain their respective licences. No CC0 or other open-data licence is applied to OAI data or raw corpora by this model release.

[Versioned model-use explanation](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/MODEL_USAGE.md) · [License boundaries](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The version-34 reader is the cell in section 3. No weights, source MRI, reports, label tables or participant records are embedded in this notebook source.


<details><summary>Complete Apache-2.0 license for the active reader</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "{}"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2019 Ross Wightman

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>


## Preserved historical source

Every original Version 33 cell remains below as an inactive raw cell with its original cell identifier. The exact original notebook bytes (203,753 bytes, SHA-256 `df02bfdc…`) are also preserved as a SHA-256-bound gzip/base64 snapshot in the notebook metadata, so the original JSON, outputs, cell types and metadata can be reconstructed exactly. The historical launcher and reader in those cells do not run in version 34.


## Reproducing the public 0.957 recipe

[Heliosli Version 1](https://www.kaggle.com/code/heliosli/this-is-new-version-0-9576-in-one-click?scriptVersionId=358332966), native **358332966**, has a verified **0.957** badge. **We have no new official score.** All historical sources and artwork remain; earlier CPU demonstrations are inactive.

R6 keeps four folds, original flip TTA and stackers. Raptor's “384” checkpoint keeps **320-pixel paired inference**; its 224 checkpoint keeps the **0.8 crop and single view**. Prediction transforms and successful-row percentile ranks stay unchanged.

```mermaid
flowchart LR
    A[Test MRI] --> B[R6: four folds + flip TTA]
    A --> C[Raptor384: input320]
    A --> D[Raptor224: crop0.8]
    B -->|rank × 0.500| E[Fixed blend]
    C -->|rank × 0.325| E
    D -->|rank × 0.175| E
    E --> F[Validate submission.csv]
```

No training or tuning. The three-study public commit has a **900-second** budget. Before a separate native-menu full rerun (**29,880 seconds**), the coordinator compares its CSV with the exact upstream output: identical ordered columns/IDs, all 36 predictions within absolute `1e-7`, and reported byte equality. A mismatch stops progression.

The next two cells retain upstream acknowledgements. Lossless compression preserves every pinned runtime byte. The [full reproduction guide and tools](https://github.com/cubres/rsna-knee-abnormality-detection/tree/54418b537cdebf274e97d4d0b2985333d5655539/tools/public0957-reproduction/2026-10-15) explain the fixed recipe and remaining gates.


# RSNA Knee: rank blend of the public Raptor models (384 + 224) and our R6 ConvNeXt model with flip TTA

Private.  Stage 1 runs the public notebook [RSNA Knee 0949 Anatomical Mirror](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror) as published (checkpoint check and scored source, SHA-256 `bcd864cb...`) with two marked `[blend]` changes (output file name, raw-probability dump).  Stage 1b runs the public notebook [RSNA Knee 0945 Efficient 224crop](https://www.kaggle.com/code/nartaa/rsna-knee-0945-efficient-224crop) (the second checkpoint of the same weights dataset, `raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt`: a 0.8 centre crop of the 384-pixel field resized to 224, one view) as published (checkpoint check and scored source, SHA-256 `e85d994f...`) with the same two `[blend]` changes.  Stage 2 decodes every test study once and scores it with our compact ConvNeXt-small model R6 (bundle `fe055406fad6a12f`, 4 fold models, mean logit, its own Effusion fluid and Baker's state stackers; the same worker and code path as submissions 57067822 and 57068142; flip TTA: every fold model also scores the mirrored windows (medial / lateral outputs swapped back, the two passes averaged in logit space, MCL from the plain pass)).  Stage 3: per-finding percentile-rank blend, ours 0.5 / Raptor half 0.5, the Raptor half = 0.65 rank(384) + 0.35 rank(224); all weights fixed before any run, not fitted on any labels.

## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) 路 [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).


The 224-pixel stage and its share of the Raptor half follow [varshithakolupuri's notebook](https://www.kaggle.com/code/varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956) (35 % of the Raptor slot); both checkpoints and both inference scripts are nartaa's public work.

In [ ]:
_BEE0957_START_MONOTONIC=__import__('time').monotonic()
if globals().get('_BEE0957_NATIVE_ATTEMPTED'):
 raise RuntimeError('BEE0957_ATTEMPT_ALREADY_CONSUMED')
_BEE0957_NATIVE_ATTEMPTED=True
BEE0957_PACK='Ogg-Tc!!DP-afHG!6:%P#!!!>h^lmi58&{[(;[%Vdu<[##^x3^eKpwj?[fL+m14|k-fl&<g=hU<Y_vltm_a0yt]J:QwREcc--U#xgc/@v~1<k%#AAq,by%sZk.Uv`EpBOPsVejzv<f^53@j&)1J+_P{gInD1Xsww|q%@*X].Q0EHnFkip69pxRGkW,s&_f97W1&sb*uaU`!A4HQ9#ERG(&,j`LePw,6&DJKgD:i^P(&ETFzZvi6JPnly7`S|iYg-@%uKW)u/FK$j*zc/p,<)mhNbBR$Kmd1L07R$aC:1n<TmUg8U,Vg2sH40}opC+C_h[A=8jk#[U)b)xuNPPmD`lRMvh8qtHSg>d,TK]I/3okW-9ilmbqMWj5HvnKpo^QNN!wZ.MHb|X~i8y2O$9#ce>YVI.#1=X[ius%@RIA}6mbWT!GL5POuh[#cOZ8]G:?v(D.Lh%6y`3(&d!R}CfK6f0*k(=goLxE3B#(xIhInt*311W4Q6>!|;~)X,E^jfh5Y]1hyyRC{=Fr@Sqietj)G3SFci_oGoV2@iPE`B|w`9%mj<D#Qsy0O)Q&bKR^{%IYX{)RGHsZS*1#S<,vw;=JwPG7*u{~Fe]4e3J$/-d?d.QiPNw8z74=<WHhtgL5,>Q:3BAG]u*}[E0:M#9ZA(3P46*wIvE<N3LF3/Z;?MLO&+dJ~^Hl<qk&h:~))~[`DzN[)bZtfxo#2SG3#FCmru!@8D]dZYg4;w<.q|3JC>u*`?}WhwUhv+3Zc%5*Wh&#GKO{HEs7m`|,7jm)J_)k|Zd:}4ed]C:7`=j*7[am3I6_Ly1TUI,hk})MnL*RD~j&H%O9/3oaB@~!RE%J1h[6Wzz6dxCS;6}RT$e$%hxUSui8h,=aX?HJS$UOETU7_o?mWGr3%i2|Wgn|Q-HbH-cUYE-]_!/$7V+i4$CX%@}]Qmkr;>S`8:XjGj4B>xT>~kHGNlQPjws?*R*-#<$lJeq2R_*QWF[qPVo#6VH+#pBBf/kVT+_N+=3eSBg7+Pd&>g};DCnx@M~d=E&ua:H-xMV2>o^&/ifS`YgS1NrfD.nc[0w&:jsd.(k1A0f@L!`/sup2H1G~d;zi$uX]AW`H6ZYsf(v1;,8pbhtY@LRj$G%]%~96@@(2?;LO0/Pp2V^1wKN,?X?F*{.Rm8Ml7=fZal*e1QyNv9E{g=W*DW^tAwWNNscF2A~j=S>c2s.VnJP0:g^jMo^WA8vB#g#*HvGX+J_{Rxcry#g6jw/vrBM}(P|rhB21{B<Tto?1oYi0rh5y_i!S)UzR]10~RB~Wz4u1f9G[EPV#xGxR+m2b89]GX>:RiX*[Y/}&eDRS.Hc=2Ix0^gHO%mkgz{[MbJRd`=<i}|1TYrLIw&}Ly]6_mPYvn^EGZcI68cyTueR8[(W[)zx6)KtvJ4`zV1;):aX/5`lBGuj3,S-$cIqjGEK;x&W_tRvSA@w*WH;Ha&bhPuP0Jm(;(DiFJdCSg.oR6>FXvfnA?|DpE=>P>Vz|Gm.T*QdiZPGucsC)T1Cm+@3q_>+O}2u=ze~#]bxR&9P%2w;yj{*ngDQ}}nMT+vV-tE6ci`BjmQsp%1`@CM5LdE1Ne*BeE-gxkV|>p@v;mp2m&Co9/SaoI*3=/?4H%SOb!@2dlx~[7AM<0-06ZkV8^L|TX&M@CU#,&?^TNYyW&Jz!+_,FtI08,fzdaF,<O179)RSAE6(Bf6R<F!~YROD))58z,wk{L6BPB)0e*b/;9k!K]qZGXgui5P%|.QFk(xLt.3QY08hoP5z)>BM6!)baOCxlm(?L0QAqaC5rA6:?=2^:y0J0t2V=+UeJZv*h2$OHtU`++&F21}uN0o:U8I6eV(v#Q[=8k~Ea`3hOYJFK]s@X@R|,J%FE:[,-Tgl:#Sae|:%nl]jbACDlr3V6mW8&S+GZ5,^-9t:rLw<|>JC,o&=daQ>0kzE@l9F6R>h0;zb`UO=5lD3`C%oTluB~APaS$pcN_C5=1cx^[8RhggP~xk%2(Q;wsP`Ovi+h|x4KGi55hU[{kj#]a,lpoSR|=k(S_r`(mh62.W1%V{vHVLLdN_)#f]DtB3@X;`CFo3[_})#i+4@BF3xuG0IdU7P+zyxb];8+LD:q}>JGfRRRTJ::@8e5t&<#@+6^%`[[e);&R$A]b-nGhb-HS$X.$y{JX&h8L+L^2APQ1s(X>l*s@^EO(pBBY}qqn6F>rlsm~.]<B&LS,bs<TCjVh+~6|QF@>gHowBYv1*c/AKrtd{&;@s-/b+#3&P/hZ|T_`Z$/eQHR|k,H1+f*`.y~Li@hOEk$sSab4[QE.!nSVT.9OE0{^O,$T`|%9V3ZWv.i{$Y+7<S.c<)|JCEYl>odk,3<,,u0HbS4W*_FO33BL;kXU^.}mYm[Kz|L@/]QT#++U&ypn[{Ke>@{:-7h#:+_+<.z27CrmFd]{6t1D^n4mJEr2%?keYuv{,/:ryTs2,34}G8Oum/0==a^W94G2vD2GL{DJ.jeC~Px4vmevRhE/,hCE|?h8o)ndfQ0J]D1x577Z6lj.6&jb7KW,$C?,JjNKed`K#A+n{_9cDf[bF.UoxZj2i24pU(qD#rkN33}{adW]3m=Y^$:z*[T#So|]w<>Om]yOv{Zw&EX+>FCTa5e8nfwxM9.ZRC2?]Y5^G-EL`E{!2>6Q@idk(g9,mF#&sn3$reAZERNWvODgp<5aQTt.Tr,hSVQa0/myF4taf2dQ0vC+iu[_c1g^#j@_e;@ok=Kiyx.l|8&XEkHh/eTjh&6/D2tvh+*!u-*mmDOt/G*3e|Csw5Lr6Ddk=P>}.=b5GG6{R^jn:*[?k{kYU`Hl&&E(-j=fJAA+{*0>y7u!v)BA+0(F}`kiQp`[Mz*k9h}<!Y@M^p|x-j}w,H`}4&uibu;tr(1Md#U!}>m44Z@G@{PhZ@!;0@nLq0eFYUX*=I4csan!{kUZziAH7G+IC?L,7)0#(W74>Gma1.-p<Ah+w9^gMB51xy@:c}+>J(YPI|[65r6W((`iIy^!GVj-uWV5Tof=6(VL%53ET+uX|(Fl3q1Z6)a!x|@bJ$Wg_rB9}T&RTSQ}1N{5bby~qUmV,mqVw0Ml72vu%n*r8trho~1~?Ky7YkGlY0?ej/bmCNXFLr+GSq/$4<s$hx55@JBRd&Js;mY#z+tZyLMn(n/1$VB$X$(3gjG:I|y-S)bX5RS`U%:_b$kL(uhhxYp36cy/@*z0oI6e/UCM%Z_Aoq]._*T(]b{xk(1GVlS7Xc8IP|YurpDNb`<huk!u%u@`=E&T[3**d8rxK6>UyxSQ+8SM{UFyS}4A.ivB`QW}jbR}O:}fR-/Yz_y:p048gd31nXp1H,N/@F1H&~N[}jQ/{@73f<%6,%yO`hl%HN$AMJq?5@H<_+(GRs{&K9N}qqEIOWE+:jbsfkAn3`66pfxjR47GXga(?a_;>c8Wvo#(Z3j1x4$(UOBE[qz4>XOOFhR2ccP:pa6@`6e`JeHOR-|27X^[2N51n1UHMfSLp|k]ypH]u?}Ki`hcMbh<++EqcYWfr]iZZR|{c%$9|OgG5J)$;>rj$D^^.n5emK*`{Bxd[X/4J@_S{]T3@=x.bn`*QfMTn`;{AQcQ}xrhCG<2/Z6U0(TVK{YAOEcNtR:6hVa@);m4ip{^;#8=53v=r{=WhH;U&kp@_l%++)y]VRTYS6clz+?a);]6e#}6]1guqwrj828j.-TUv61BMFx[>l|%Pw<OP$JM-Wh,o!~3Nf9.$<+gjxpe*!9a~!U1(lgPkkRp=KumE_ThJ20Pue,<*SxuY9u&k5YU3BYmCqQJw,DRbKw6&Vj}jt&WXrhVMDbCykP}S5_YHM8RKxkKBKbblzk|z:H,f~.&a9ysR*pa<x-$jEf5t:?94i|ISv4%KErroVGTfO&FzYeRK~#2rvz:nT6:-IBA6vfV9hlzc*cNiq0:1@[P@mFda5XJI*K5I$]Ce8$Y6E49j#%rAR=y{_G]D-]*Y7H,<Rjq?>oI3.YalUF3C^1BuGX,L0|UJ.Q;li]fRw+@DRDjNO7GL_D#G]PN({.$n%4G3D2nNoH96v_O*<K&wHn8GD%BCbE3a<-/].#r{E7LM]r;#jZ!`5J`%yezt/qA>x4Nkp-{Jn=lcO-F*K31l^phAB:dzB!4nt^1voE]=-~wWWa8zLPMHdwUtp#(}xH/($Hw)_]bT}y)&ON%;3CMdAsF21Zk=FxHX/p&f^qmD1yxyS4W{&D$hYpTQVecF~w3K@W>;.h0:s.Xrj0.IEeW<v,+aCA#&kp0lg$JmlV0DAvE@X.}k^t%C/{H..:h42/l&X|>=9uyR.6Hu+8@(mpJ>,?KZho#.UFc#FhDJkwl23@Wep<oArC,~7e`:;N#:8Y;zva(hQ;wyl0k;iL_Hw8rr{+13Ss}<L`O89=1,t6N8g@{|nY)MPyTe>^KF;$Z|`#om16iNLa}nOOfq4E,oZB]K9GLb]3)<LrF8inh]/]4U@xC9ifI+k?tiXJi<u(1YG6Vt5Lg8=fkdes09cszeY%Vl?vu(`!gUhpXO4[>5%KYAa%blM@epwu>pX)wy^$T!/3(i^EgPBJ`/&}%v~cO=*+6H$;m(.aCP;<eT:&knNv7QP`@_O]o/</!kS^{cmaOIqs9O%+b0!q)RJ@14)cfIf?$-zLtk[{IyTR*VL9hcEysD07,o>}E{}m=LX^FOwll%<ePs_p;Sl-Cn16aN2Sz0T9C0gc..22!c~v~71IPVk6MN+TI!5[<`93)Mb=&j(Lolym($=bm1wwg+C)J2x^m?Nm2>k|Ar,;Wd*;|Cw`nOYUx>P,Z@o>K,;s8=(7y>D/:vG+wjR|&aeznz6y@<C_Y^zZ{yo5H8K8CX0:18A565wrKlY@NuQcCJc#vG;LX=WCh`G717-vDdPE=Q~QnkH=0nH$P>t8u+)9Ru.dDa_>L]0|@R(^[TAccWTW<.n_tQ8}X&Z3h&THRKUz=*Guz^bShMYE(BhPn#Ha&%U(ji^emLaWySr2ADDIxr5G2#@X=/kWepYWbm=a}@z!OP?0Y-|&EO3ci!/uhIH;zqKGb/WfT7}&{Ea1dMma4>/`lZRiLSNCB4}z+1<6l$bN{Wbf|}e|G$p^#d7Y683I[oJyP>vtKOpJuFoPT`lV}S`f?wNrZYBr9`hwAVis&(E{fnK(]ddS1is;p]a.Cak$*<2[s?<L7v&[_HIh/3?o;5FD0?*V]C|^+*B-8Scju[XG,P;^v{Uhoz@ZQFo>W.9$r5%B8RUiVT66=J5$0DW96-7YC)019v;T*MsIVX;=!0.cJ)*q3p^*R&q&p6Bkz`+iPblk)4#6cf@dy3JT*[VvdVZ_5FNB_]Z?jz~i<xaM<3.eTY-;thRG+zq|`[:Z>KEOh]amU!-5;Yd^aZ*oM,~PJxABi|c7399+rxQ1W;-XTS)c-9GA6R-Z4{y>D1ZAtKu.:5PrCy:]}LQaArYSPHo%fA>6{pG_CIX4$3PB5nMl6.,pq(#ETzqLGV}g-MoY_Qgg*UsEO<DomxkFJOT*Smlp;DhQ~s%O#u5DzPT3Fq{/hJ]8N|5e0C2Euq/s[@*kO2FQA6^&~Ot)2S3lB<{d:]Y8-;W]n>MZJs`t[3~7{.sX,(oP3QHTM%s<Rjq8Ox%4L[#2m6Fs`zkg>6^{1dH[txQxV=1uL{d#[&|8F74~QLB4=1/Uu2!;8Xz*^rSVLKhw-gzL@476mn}==iw5)N%4`_q!@;~&;Jw1|xA1lz+H4ZrRX{V-`COgrB3sKj%l&/:;7Zgt^q,kHtM0k<U_WJU;?o;K=*?-DFePvBQg6}f$`622ugQ;}AAq8ui`Q.@?Fd;#(MVs``bQRPGh,}:L=&4dZ;+d:{)Z*xT4*bwE[FxG.Z:kdIoUPDpx&%RVEua[iCw*Iz<@KK//{6j9-Ko^juXTLvSe!n:Bo+8Wz</xF?s%K{{Wd64TbFMI>.oh*2x6LL~>Pc}o*2(*?==>80X=f{*:Hxrh&7[iZNl(oMfLk^M[-Tk+r3%j:=_4M]%{ED_I2=`=v1pXjzE(fMV<~Rd$AAW&gy/s)v,*I{<jAp+F3t/[LmXdg>Dj#0{XLzoEsIcwYKQ=Cb)4^d1x-;MN_AH#vSv$}mWH-U4lT3s@#?XU]ahkc4%?R(B,bFC>3MgxLr=/*.Q0RpR|Nl]v^4Wwvj$q-b})i+M)O?qVE&e:ix2+;$banKlDN0v7bkg=Q@=K-8/V~)SO^D;}.7dG5:28a=GNJDFSpHX+y},1EdCqw%D7q?DJiZM/G<2P*^8pLSL62[jQs}PkK/B3kc<[#.@ISl-H$7DRnXu$dPt}#0Q@D<]@|MQ1`2iaq@8kUq%0IC&5PV(xu8}m/D];TO$,,C36o&]`dh,Enb@gv0^jc`O)j:H?}0Y|NI$RirF8lZn@b]M$}+R*Fvo-f9A<2J,r>V&%9(B}S]*_C(>M>uq&~^X+lV776M@S;|7Zr$>W;z[7NK>{U5*OdEd_U&?CpI%6jIXf[^/i-WSMSW7`g82Q@y}N|3|r$y!2apig9Ye&d4cCMkvUKG-p#*ayjV4yU#Etxl4c#j=vgu?UX@Rs`yw]($;=V{B7IX9kz{S4gj8B[9vIUsofg^2/T+0CCvUSxDb:*O5:V)M|Hj&[O=7YCWy0H)C]s=aXbB-3dRaI/}?ovPtCQ>7?Evx.2_^6^|6sQiuVu01qcm+IFLaIyqgBm>s=xv!(E|w4L-^Z_VQAe7WSFibnOMMOEQ=R@%JL89G];=8xQD-oZ,ER[;sp1:/I*[rD(Ze{f=^6Bsc9|&aScCq>Bx>&T9yNzT5`M_XCG+M*#XJdBM4}M&2rlur);x@xIVv}?6$(>D#/BjK0InW[jpc`O#H2w+?c]HN%%shSa_1,LQh:We8RN|-F~:9+/%1A](fJ;v$H=Ok9OK=36Y_H(C2v$-2tBb4f;3(.1>r4Y-Yqg?RKUind7|_/jA77jtUUAe4]^]WSg@/lwf(.[NoVh`@^:uh!;YQ%LFD{1AT(o]6XqA4sVvPTBVDZ}IVEtnz0JDL*Oi=cY_:+LQC:sB-Nu2]Mc4`}Iv&ecZ2v/UEbnht72]yHr&`X3c+T5IH$OUk&A9)bBb1iDQ]u1J|Q)M:p@J4-MzCZxedGmxi6v4d15j%G*(;0QWzR&|xaPjk/yj&@QWn5&Q7F#5x&SRf&eWj.gez*ZGW]_[K/x*rietAPY!V3E36k8_KYFM!~Kuq.?gqpl8`yfb|,-qmletdYqNhF1PYVg;E)9YDUD/=kQ~|!~rv;tney7YD6myK:5>}v;8-D[TIK~NW#Su4#ZwQyziv)KLw=~Dxh#DF;ie#ie/*fW3wf?U%;>iHLftB){qv,8:ykoA6kF41]9fO|5fC873{u^K|RIOYGpbiWBvW}+aO9CK{D]+&<sE|Nt_y9e,pp0dpmE]9mWf.C}_4UReqV`qD`<w3)-U>aIQ7h_#<{1.C,YISa#3x`xoV_<l9TK`..zRVvjrVfGwqrUib|<SMbIgL;}4=?2,);AUuN9O5.<_a`1ui;;I/V|q^}?b]kqi0+:<o2$wq{%Mp}y8|;Q`iqzLl%A[>M}%]CXQy0VhmGVHqB{d~=Y{Kf)tXuzE8}Ka7>22|CHAG7gm$b/#~;WGk#HgMy,CH)beo((OR7w8KWR9U|RXGz|rnSO&/N*SVlIox4%NVq#f)#(n!LStiO^4e=^5ExFK0RhIa_]*WZYD!z|5avaL[=DwEm%fFc[a.dg6g:HX4UQOpMv9&L2#{/t~TdKZ7}VP{M6q>%+WX6nzt.n?,w*qDF~KJpf8,N.a:s`Hs+.Kx$nw`OK9%B1KHd~fzQG72h<ETv#H.?e)G`=$7>wDoBN3Wy-$z%2tOS{o]yF}?yyQX`T$gI(Q$$E3qL%E,02-_E_yf#u[iyhrUX<o4F8}]B?mjNexx0y+]@--o-cqANcbu}/@lnJ,+{q@6<60hag~hs8Dvs+xq>ya4jjo;#XRiAh_(&9.+uQH)<wmGH~fSz<#Vhhd3Ti_WR0+0zV[9X/2358,EmL~E;lL}JaLBD8]9d]/d5e!Tg[vGWgEoFcrsL7CF3|IiXE:&`vxeR!b[Ppo_y?zXOqzU!tj_zC(-RNBVetya{vC$H]9-PMMf1U){#I=19S^W:}WO5-,Ha)rI5:MNxkY|#U7Wbgc9/yJG)+9e0vMgi7ZS%ZTMRW]|4UScq|!g>&}e2>[yv6f4o3@xR0f_&wju`j0%vfY-Fc3|zgwh|3v$[%j$XzqJCWRDYi#4;-aDQiJMdwFih!H[X8onm<Z4KAcbp**D@%E;>2k95|mge8SHX-*&IV&xVYMx;4.F(:e^fY,5yOg}X-e5$Wn46W)0Yjj*YCoh1^0([.}Zy~=C0-<bO*s$}_Ki72k+^d+:2nN`KzKw<xHWFt6M7w%!dQsNpw&uyNy4I=w80:_ZeWVt_n;m{d{.4#PC4=[_]hr`xz$9(rzp(xV]0~TqSHI%SZV2z.EapBU4>9/dH{NR]SHuscT+GQtQp>:u=uF|:lV5N>q`aJm~q/[n9eP@t[^(txeGx_X(c@kLQq2#a.BjOe)5v3K1Nl2,P-`[O2ts%_;*}~6$jU3FKBg)lN]k:IZi^%Z.<*.{U;bd~Is$o,Xt9WE3Md,5<=2tt0+@U8u6PzH--1$&hlX7m>`<i)@ddwqqpPMxN`iDGTz~is5QPr+V(|Q_uYoMub7f#qGw/V`y4GSMV$_Qk!ZDZ2Gm`]L9R%5wJHbFJ?41qC8WOjwx^P!SC58/6iZ`6A=*dfr-TA/ISpCV.u6p>|C$?8N{,ae|nm0~E0Z<t@9T8yt-:l:!YRFmUn)^4~D/-}YeG`}X>AYYkO8z{)5zpWRxg#9tq6b.u<]&4Zf}ef51w^ow<y5?vFn/AbaAVV6X|s:|m.iGS<6E4E|vwaI^q4!!0XIW#iu;8yZ`[6a(UcR}{Wzd[hkKYqB{Z;T`0}=Y>L6/IX~e{ZwM**{i_<*|CUH|!F-~pK4k=niDXSf~nO!!C?r47-Q](zz4WC4I%U!{~**TVLc!>bY=7c0^162ipgh2P?96Hw=g,k^i-FyJzp=$5w4HAs#M!lf8JI%ZiTK[w1?It~FFJKO/Dz:|:REDj&6H)k#2tI1?/]AOGHk+?LY!d)(_20P8FaM]}]J|3.JT!`yEu#m5O`dr:M&Z8NO:SS<cJ!]9O`is[xLILm7fmry|&zIKhe,hq1BA)JR]>@0G`FC81m6:Uf6Mb:j(qQ:lFc!S&lzaI8rCPUG<18X#9/uRVTN{j+s;=&:A*WcTBWnnQ1p`5+A@Rc:$&*X~s3JUht59y`}G~<6Cc<.K16dxm{.L@<[Zo5h-%IjzqvKt9+Yj,DvueXax)#mA)tx<{|})3JeTd/.j>MBpK3;;#.y&ythtgf@!TD8jQTY&<utZk%;~v?y)UE{SmXFsReXBnPFP`klK:l/~6@b+-%!<#0jt8yKubc9D+<J0tRu)h6D1gIdUApI}zOYn^g?8bo~uWZ_-,;9@EaLPWv{C)zThmiTvl@dVO2~Gd$`$E23w;6l2|C?9f5FGRk1Y%.&=Gpyx%sxJAo+chwG.rB.7nl;VVj_wD#0f2tli3x6[l4p=]n{H=^JB(1UezM;Q*jTZ6Lc<,P==$)TSX=,7TC=i{W1i(7I?E}9HEdG@_[|ZJB,y$$ALkw`<P4M;G+dV+tc1a0^JT_auWp&#*#YH!*!|fE8QprCiZAH3wFLE$%x,^E|}F/ZIz?GD[y?(2dv|ZMt)@4T#L4Ffa+{Y!&(UwdY-Do1WYe}fYwcGNsAFzE{k.f7]bU1[ge${3%p/ChPY`m%Hget#22hf|!}Lcllx$z(<sL1^{uWHVO^Dq:tXc2/],kV|~dk$M=fi>~`*YL-0s$**s8Mz=F5,G^eK5kV!cZs!u2I+%_nUvUsm.g[xghBrFYyZSPwzvfff{42gj!;9l[An4|9yA,25_6RZSzOp<7P6$(+zO8|X6V]<.tI|8s5AD(Z$vlZG4FV?2|LBK]&W`?_{qmLv,1>~sw>m:zBDiQIl2+yrv&U<V60nJ.-{unce$:5@Cn-{%_vm0EJ.;Xfc`+~q0&&hySV6;8PpS9?gH.|4=Z(@^jRJ=j8>h_RN/>H+bR$Zfv9CKDKIFvyK$cS8^W+rJO}GIyck5|$@JH&Iv0OGt>;+nJsrd^ss=uE;k>E_B82In%Lzj-^E8Mkx!MzJ:k@`dq+w$q;I2fihxg5ex}k/u0n6Lr<AQ8c[%rE<31L<b`5El8tAyEpPLK]7fye@uH%%f~`Xs%dX}K_1ci%G1h&+4<2wJPpB&CnZGt~g)HARNZ?&6L5|~Zwf18QkBO8TvBp[/#ODKlx=H61H[G_8<Mb?I0Mx,3D=}geA?|*G)Yks&J)oOZY!dX,,&6D.2<7pbqV?XjL%Hz*PY};tM.=Mf4V>IrsF+=`FOQK8kAjavik>1?yuQQ&Is_QsC(t5^,oG=y1<1!q^>e^1(zUW1,ge;A8K9#:Dvym9b_2zGVzva=|nht%.3L<^>:dZTdHGfCgoprfU%]0gU3}[a3l$RX?vKaR86GQ{:[@RN2X~lNyi6!y.G}@V%YXNE*O.h}7p}A]_=lv6Pd-$YYw]&U+hS_T;t%Q8CO(n*E0?NsqQ0EFVfDA<kR{m<rc,ax?m0x(8g(`2T0@T{!Z_WEy^Kr;Fe16jN+/!>TV+cIOQ_m;P3N)g<x)1Y5ns8?+w$m$SZ:=QN_9c<DkT70^~`U=8#Gv^|p1h|BuG(w945@9LKq]6NFecp6VTGg;dNJF$3]<?6hq*)oUA0|lDRVY8o8s.bXGJY_.[K|KXizA@Dek{Vfi{z+!-qZ[;j.B#?KQhv9t/%3X`9#UB]+Uxf1#w<9e3XiV<xmu4GuWg}VYv[$#>`~]{S!L=ELtm81-ekl8oUW+xF+pHCn{uPqE2?V=.SfrhDz;$ftA).X0ScFv%Qu;9v#,UKkwW@_}JJMw(C%t%IOe)H;=_p)_VS0>LsDV:-)1-Ukahv-$K3zWySP=0>fz4#|S):I*,JJXD]}`k>NE%%v8=VRecfy-/OTi*+&bYk.*`prPpMKA}>~i`cC0?vrh?b>v[$w1:C2i8aHwk`Vjft~_aDY#`Q+{~946sO)M*4_46KV3I5_bk`]1B+cT82kLqfd}9IYT~rNo`I7nl`B+p$kw<x.eOkPtXd^0E1wyq=GS@|iMCH*f)}#0ho7Z(L?C?&G3EmjKai;Z|!xsDkVgm5,}9gsGN^e$bIZ>::%sAT`&5V.Pcbe-FR:B`oC0[2i>T:2:<XpZ]0D]`scit$NqtxA-]rl?KfK:CR?eWlO+;&El77/,SM|z+HH4:`An7:m0M(:e1fcl%q}YKkSBhbAk,P>plymK5h-oC`5iC|h^AFo6xogn*<w1HdgtnhXfHscI[,2IJMihFT`{<9Da)&=jn8|U4%Ihiy<%SpdrGrooHY:Fl?wspv|(eoy{g*5^zSXAY?gjR?WC.NbM[#6fiyp<6MYtdqLC<]6y1GYJ&()q*K&JM_kT%[@}A7{_9c9>_KW=7#yT]_T.,#Qp29_F-:2pgdHO_kUf<^!l&]1hFbNsG`;QND$?,X/Q5.e&z!2Ay&+OI9]/WI`?e5|}QVQau>Mhl/f$scm[LzHN#d)bSG)m7#PC/W7%|Y7[(H&6PY=*h-uy1984HmcL`cZNFD#}C%}s>;#Zy*P8ppmnuu8b)(@yHv_%:O>V@HGTu,Qx%`TXZ3N0<e1(TS%k^.UP>]lN|hkA@+v3W<<HAzii&hHifNyR0Sx*P&2Wxou2>]1W1E$7fL71},1un!W=xI4dm[D;;Z<b.b5d&Av_u%%XIQeV84.Ol@;K[KUH&L51zB;~fXU175i7Lw{cTW])WVcqt<-}/a7)<1*v79Wkui.<z,VA__wOlx~s=0n^Ywv@52|>U<!-8l2iDFjz`+Zz!Cv9PQ4JteMoOu+_)y~V%-igyX9CJ7=/v}SwzR`ZIk*A(af2QHeBT:{wV#]n$Ut]b)@@[nW@W4lG!w$paWLo`(och?!VSM!n~|^Cox_Mr0T%+tNK7|-^^3:=uDEH0K%odK4/4tb:]7}zDDxwcq>0]pN:<tsd}2:EbOC9ZzugZDi&JFHi}=I<.=|,/=pb%pBI0p{,q;V$nswD!Y:+`]PfOOMSYT.%Y4%)]VqQ8JrQgWzN2lY]D>!/_^N8xdbFi^9i:WIK{@f-_]1Kxx%RA@S;=+dZ-lz:^.M?g`T|8*>r.|1Sknn94s0fz=0pnId{e{u`6]BCO_r0;}0KG7MTsm),2eYn,Uwjpy8P,Ic`QApqUf+/2fa~Ek=8]a=uYf9_TW%/ZO0ASrKn-GyMpe%5cCgtkr>7&n5V@ENt_.+4rOY*:`o&kv_z29SI&&atv@fe}PHc]viN5?|&C{vjMb4Sw{;A>Bt78ia_IB%J(I6srsEJKlP(;t.y4edMTjhyq5U^1N@C}(-Ra{KM8GSUhO|I;4%)|i6*Vo_AWO<45DytWyvy*8/)3iR2}CmnnmU54N$P}<}rUj;<VX]nW|4eW$m&=KOMdgCv{=I_PB;:-j#2yzEH6kP|*eerN)G=Ac#by8g7mg)ADt:jA/srPp~/<Daag(6Mr5B@k!0L4hhOd;U06jtRg&yV?Mrx}{2[@+.s5w8:z|!X7hin;D],S<f/Pwx0=2YwmP6t*:R}}oJ3?hQBE1t|sr8[-{d5bw#wdn(e66zHS|6J5kox{yIO>4lkqm3ZG3M=%cIR5-0|l{#(|1{|WtP^p17E{+ZXz/?@s_JHHw/td)4,#zqU>-spLWKB_~?nB/UR_D5s~XG>o4D#j#)@ezKK|.=px^VqrM)U[6jFUIEcljeKUn^y&Px?9de;[FrSi2ezrOvk[m>ZZ}e5!){`VqT<R{H;2_kM+WEA!817BUa^Lq)l*.hoVT@Bou:57S}=hq>+,v+x)*BZAP`//=6{W[yy?TNy/N7sOcnYT=jz$p4TYJjiP?_o)HmEBv&OO!($7Inr=b291#|DFWaw2?aph{{)EjjSpbDF;ANuXD8oc=;l6hOi!kiI4h!VNQy<z6ka{IYusp?o#V9IhE+D=^}:wE`u*Y-yM-X6#)<o}F[2R=`hLB3N=(DWBnYxaHV&Xb.&UaYPG#n$s6#iFTkn/jUPTatNg%lgJOL>uGLgbInQ&Ia[~iR/SR=`,MvQAD1CXEZnIdn?$+!`21ljZ<[%I5,jpQ}sxYnz>Y&(^<ia!a//jgcxovFC5q?I6}yn>TdnP&j5K_|}n%=(+cqrHv{fRjz`tksX*uMyj=H{l,|;)X@lt|S#nektRIr:[:P>ogMNMCK6Vb:/k&Ul+/D&|N-&#Fs!V9#0n<ef$R&`%VZUx$b4$S/sE:JD5I4m^am>o/y^APggH)i9qHLOS,Ya<^3XT2`Yi5h$6?7We|5<5w)RVLz9M/?}2#4r+cX=BBeDq&cnIg9LI8OWW-|}-m!&LhmfRj(?c9nT%Ae2a4y:v1,({+@G/1a4+[eCo-Ply>.^]0VV/2>.HDteyHx)s76Ycb>Aj@mz$woIS)8ZiL2,yeV,MIpo^;$#ODj<(MpiKRjG3KuYig&U6V-M%#`fcdM;cccZ;F-XG3;IYd/N^iM`nmibGGtv/#X=fy0[vQr]f7*gU<;IC[LyeXHACLRa}Awr`256|{jd>%2z@&fAtCzp$ny<#UTR!};)q1A^tvhBeWLaK]tIkga,D@ahg1[^sUh,8l1jK>AP[cFGgo6.ZjFv|m{{IX.m$v_X!pjh?A]PYpDhL3,~)[~){Qf,SoO2my>SS`*N{#_rd1BI`hYCdhd`aL*:!]`pMywocwBV7GaZQ]SApq[]uf.d)Cl-CS[qyIUrFCUbB.+ET,Z{/*w/1sYE:KuQK5~GP]ahIP:J->toHy5*QWYF9FgquLq>Kj]=&O2i}x?x%Wz<iYjK0Q$#R/p/LA#S(UH@;qFM&N*J$JSf1X}zZrMVqGB9E$6J31LxfpzCTU+vCaNGODvHOR#$Ik*Dl6Cl>Sn5Id0-38qo+1,rP%<$T0gAyL/%7}D%q30wYEu8T(|rUnU`7Is+1rdZi6KkAcq5myM&wH]=RRON_GNPON:D!ONET!.b.n:^Ngu?&ywt}VG^LE.pqd@`@HlByXC.hxV6x`b00,sGMBkQhQ4(/OYy[i!G)&m]&f/-OmnLdo=dj@(9NRl9:>KUwbiu2l`^U*TA8MqeDoffnjhB?Y8w-tL=%bB+<YF;5]LnM&!@j0DG^C<Rln?[j?i!fCRmc@H9^Z8)A>9M(:{U{A{H[V**1pDng@Dqs^uf%6i@(5&S-j/@g|u@Ds]-u,ON~&$}7G@jr1WAM+NL2+SEnLeaD7g(8Y9vfxLzqN&RztUF%A0yO>[o@$cDosmBJS>ZqL9dup[gZoRv?/kbM|qtERE$&yy([h5/y]h!W1LWa2#E<hW5)<1>J2gXcX~#NE%jvEJv#0>0}W#HI@8_35Qw=@j+eeWNHo4|@Vva8-,@s??m~MdF)m%zGdIa4NTiHPM>Si[$*Z^<kx>Z72<2Re4ow7lU07,dHnkE)_gkTXF$42.uW(W^q!x/CVhUuLc)bjD8-Xw*x{UkQD3#M6A6jv|=$o5Y?IDNvY=S`l-e%)R)j{.B<WGA}-eMLdNo@b=I%;z7y@?Ugp~>}J7n]cRYjV;aGO6&:{*47{*81O<%<fozk8PUV(NX-XT5CQg/eTSq~HUV6115Da>aVYbLn*}TOYBMAll)t;o2SmNG3U+#u%+q$Dp8c@EBK26g*X:-(`xg~v(LqQV,0KKIdR6M]8i[~a0w+R]MzWQ%$*S/g;5{$S!#)+z}o^[D<Az;e|FLeI#2(cs[(7;&&)MZ{Zk8E#HR^bWgkO~l2jS0|2@@*x$4F`8{mFG#S6(|7R_?LU*I`&0]y,H8fNE{,!;pk^#hR@_u<.^f0<HZm)^.UxzRY%V54jnWG-ut80&Dp*17c5w#x1+>,Y8IE[$/*mMwRj6KEVs7[Y)SB,@N]X8S[)DIwn/pJpd)o}f3ueb7Utw:]eEIIderyC>nXC=pQj}u|8%eZCdYHaS232VbRo&KqKUUpIn)~@@h{{</d(tgohQxx}1X]:P#vuL.ruFg)k!`.pwA:I:%,50IpC[4_eSW-MSUwFq63qhD5j;D|2ZH!sD<X6(uj8Y5}v~g<3-Ct/-2w9=3hG[{>Xolf[#PU|{|nD]Dh{/{qapLfdxtd=F:F;p<rZPLnUY.0k3cps5=8sdXiXy#We91N7Qv)?3**kLy.t<1cIicN%/`%u<vO<?@W=QU(vETrngZ9/0H!er5woWPttxE1siFl>gc49Ce,Hq#o1[_Tt&B!)/i&{0?m)/s?q3Th<4fqzD]I:I*(^bvN+#RS)hF>{+&D&fFoTuXH)-Fq}aqq_2rX7#Q=@}sHmNsIddoLG!Fpi>Nbs~}a?/kZp]}G=F.Ef2NeEUWGxCE0T/NFGe`A0>kWHa}E*][PL}c>B3WA#UId|]DqZ&IS^8y8ucn=vAr/A-.KZ~$y[CGi^^;dP,Eny~.0/Imo&zgT+!;4$$vGxUJ)ab)b-%rnJ<xIU3dcvNO3rRTP(G_{KK]oc@Wv-<^NV>m01-Lab5v|(}}LPFT7(/,l%rqS0aq<RO<Ozs3YwMo7?6f2EU.4Tzpek+U]+;pWd9$_v)Cpa5GSfT^nOIYkSGOpYqEYDa0}CZyLcLhzY%D!5E`_52k|-i(h<le$&uQ}<Ld^XXqab!0fJw:$kRWh%~=Hd3Zs{8/jZ3OJP`4n<08hSWUDMBCo{6n!T-,?S03KhC;5TX(8N^DvhTE}|o7%p,j4OUuQdAQ*L/A4P/E`*6IvhSn57s3cYs&%BVMhcjR$(8OCAD1H;-[(ERnG@#|Vwt2*&wc*R.|.mjFhy7mFwv0Lh+U(9{@r?)j3>)%J:)%R,3PctOi@CDJb)s:},&mDEpk>Q_R>2{nQdofL[WO?VD+u-u6tKEp3.lDjYub&qn<?*jL+*(Ro07[rK4NZ`*mVpXlh6}TjkWzNT.Nb9BmxCWGW2LRYx&>!ih^m<D-<%`(]l[(B8JhQnltLbM>~VL%{9RA!_U?Q=K02xgYE/Y|aC5ton$rp%Dxad;H1l91~p,7Zgqr43g3V4Y}>_zQ4dFlDCqrK01~yad4%]TDZh6c$3^M.tu|+Fd/WJ3408sej3cP)l?Mwm0rrcrX9_![m]baw[KM|v?X8z489WS1XrWZ5WX&l``YAAgkn&3/:eWqqo1ty|~QNH6Z*hFa:Jq2A^gRdxnE&#?|{M?,LO>ZRo7_d{#f8#08yabG0vw]fHGgw2dA:2_?w$3m%{~jL,:Edd%OB]Z99uFAyKUAV`k8mrQkzd&O2A_z2qh!4+!P?k&SZ/VY?#0J[eG|!Siq0mYF5dj*nJVYGo+.r|^86KHa+)26ci{R*2=P-{qK`^IU;.9?]0E.U5;xk_UN|=jV0JKnwixq)QwLe.!_n5/b^(.aT2u[GE0/SC4nH6e-(^dQ.K:s2AZ+yNsSAY3%[J$<Zlm6xm}P!p&E_--iT2kK]9%x=Y~R@P),MluOY8LfZ>}*g&.rN`[=Xhx6vAQK2dQW[g[?v#ZE|%rW|#dl<9vgL_?OEa3|y3<$W&(/:&(wXb*8:8*N6#q:LN6,ya#j;6~]v&Kq_YH.D(&W>0;ts4K*A-VYPF<bayH.ChUnVhh<=I59Kj]O.;}Jqt60}aw@W,?wF;^l6jUyTj*md<}G`-?W9Gz_^.G?A+yfQ8hop9a&NZT&>Bv-G-cX-S.iH1TKS6~ftKgJsa[3js?cGFJ:97S7b/hWpMs7R{)b2E9<FdUg-{(*#B,NQs{o.#e91#qMp1KT,em%6jl4D1q4^l9o%Tiro^Jo+zF*yV[eS3>o^bL>09@AX}(~D:`(JIWhx1;`doc5O&I<CX[qYlvuaQg,#59sv%CK6HlAkIPKEx*@gCM|tZ5:7>v;ktvv;[kb2,)9s5@F3oO<+-ucG9yJBQ<oNpeod:lSjTE:)C/y/l=f$L,EQmzT6uo^.?:&jgsphl~oeev:u7n3+2`g|/S;WhmH4AVk0MZ%KE_t6Po0=3;(ts64tPIrmUgl-$-.9q1Ia@.}Br,P!?6zMCB1|]FMGEm>3Uu?fnzt3yM`Au}^Ne/QBYK}U~vgMn?shc*a8t=_][FXKpZU-,`9`{&kcTFNa?CKGtf}LW]`X{X3X<6uZ$cf}7ux,NnS=>}=uQkocq~3e)@)X)(e{NHPO,R[_?j;kYd7XaB$0;)(+_ggb/CZK/Lp;Zm@KU%>w+aC;h*I`G@<jvr_SvNa}IJjb_}cF*eJp?Ay9Uf~BG>&[C,cIg>v%H,1oHB@=4~H1(6j~*TV{:A@h=82uzshV}70ljwc44JN!Ec$[HIr.15pVM)Iji(R[MVqmV|,8<UW{%&29r>_7)@KJRUvt=iJSHLMjO12BI08@MTG$w5;!W[jU?6U>q#Id?gyMX%u7^%E-X:s_llo|3Bp2M:=k/e572{yF_+9xfnOuQh8[S5M`VSC&OFQ|k>U0mQh.h.=8a[o9i@Dnu;6BU]AtO]Ggkv]I)U{ITx,din]Q7Cp,x_<5:]#G/q}ArlFs28s9m`17@:_Je~Ta|Y|b.A`*g0ORzm||Qj-$iMx*|vau$%RFd71lLG}yb/z$g!fZJ,5G<.fx?AD~,d7p:=G)VEa8wmn{MF.xge%xhC$%WX0*fww7nrE/#=<5Ag:c2?[!5BbF`xIlYmOB)zSAaZ(;2y$J<@m6^%dj18b9^fulE|IrH!0TXVsa8-c&HiM}Ce>UbMDLNXn7y?^m#pLKXs0}5LOz$qy,?uj<.q5&1^s,%-w{1(.y22C,`9t=kBZY%Oo?<Fy{:`^WhBz#)Wz>#-aCXa5X<.Zl/o=)akmb5k@Zhi0cPB7Fc<2O]N@Dx{ckiIgc8FFoMRY`vGO>TEpME[,X90gD}^K-T9D&4=?oYJO`)?!w1tTD[,plz^m1s(Q/@h#s[ww0)vWVy:2()X7J#dkv_+<,wJMCUb1*4jF%|16W5E/o[O;NnC#EPu=QM=H6BD`JB[n<WWwmz)Y8uU}_;^^jSD,vAxrZT|wKgG%_+_Tabo>WmA+|9&u:XisUrWnI|v(3_Q|@Ltau(:hJ~8}h{}fy9%I/-xI1^CS8sl0H~U+_P&74Dm?X`7)8I,~{P8y[v_jT&zE<sn^75B6B3.B>|pjScFgbzor!nC3(&Nt*|$(.4zo*?TBppApAhdvnEk55v-J-@4Ku4{e<rL=*qPI^orEww2!B$@M]hohd.e4j>UHQ`s@_SKx5[2,NK{W+@s8vg/xw;.Db+ic>WV0dAT-:MW0^~EHex#+5Tjr>H;[rR=q10ALV~O9Uw9DDa-|>t!]XTAwKDpzOvYA#nm>=4.06^1Ch_wHC[-PNM}SQaraFNA=Mg}Hw%?6td0Z)jKyd`9,cPE^`^XTlwE/hPwnmC)^WvjkcNlM?`9T]u7I[su-#%60R%1&U{iF$ok@-McLrvNyjYaXoMgz514_(>*&S[vTZDH_?A;>C=Ig~bT/lLzEsdzw8dtKnS5*29S=WLlQr0us8YIC}qCq;8b8pXc-+]d^=.cp>ly`zVvWJV?%64=4u#aKqgC]*5*N<a?/U1/e)8i.&-SB?9]T#;A-%xl+Z@<WMamz;.NU.Jia&8E$A5{}>.{b%F>,g(y<Jy/DD`cNQ%mGj/monrbnp(%AQ>`vAz:l!QO%H$(ll(X3ZO.+TLf~6TvmcrYpF3C%JNp;FPkT@D%W.s?w9>V|ZBwSn~TPywGcm}gau#>Yx6EN=t8*`?FmrlGK-agL1`BS|L1CX|o>j__B~d&i(EoWj7<,E#B=Bne{pnatg1kx.SDQEq$6KT`4V`RrE4~M~0(E9:PcomO/ZlMq/<.C&XnQvE{/OD6N$|@[e3hv*5~Vxp[VhdC_@|C<m(Z4X+h{SpsxQ+%&X8}FzMk4=8vArJ}vL39HN>XZ|sig^]sZhgXnLpcyGwQgRMfb!tD[W(QhiQIC#8pXxZzT3|8Geo2zT>z#MSA+x.3_L6g^}{zFD{LF;^>{P*^zj0qJdSr(fg&!qGZBoaDVe18g0p?}B_:[od_>iu|9d=b<Y9bzwA?jcZ+ORkhFWo73],R(aTpcz5Du5i;sKJ/HRkkszczzMk%&.ixRJV&9kH%6Zr*v(KJM>Zr{[VzqtS2$^JS5#48)3bS-%GRrPyjG+EoKMtX6tpfW`Y!`>HK<sv2MafT8{Xb=HF{02y7w^p7/*2*^o0?gk]dP`q~Mz$~Lx:`DNI>T*Y>=^5Fz_.?&cz0v~1(n@MA$)2}aW]zfe|.b{9I^cTz(R|p<3FaAp)dX1EwZ*#}+x8TBG}^~l>Sow=_KTundE~7gk1C,?|]%tp<501lB#?T:!H%1D+3J6_(^Exu]SU({BY$:aAbE21p;jA/GT3ZH#Ej:C[n14,1QyOa&vCK{INO$@D.mW-I_PWoig:H_5|#m>3e}=?g)R7,NIS,JAEow)c$Y@ua=CEyVVsPuS^c4e|fDP|~6rxTAMot>,%xDb)_nSAO6f~cl0RbygZfAhHigDxCCIZnHG#7;]b%c3Xc9!xSHIr7Zq6RZrC?V!nkm`m_)|TaFvkyBicN%qz49G}V&(iOnZ>$<OQ%b>mM8Y?Ar#b9{nQhRNC$itVTfwPuh2%adlti^%I{J6}+-(JUkR2^e58k`eWd@V@@R1>qq@|dm)P!U54n;qNu249/<%)t^n6n}g6X4lkzC^l(Qw%:7/`(DuffKgn)*`lf9r8b[TtAI:oIdkN&{Y*>*j:;Q/xP]X3#cB)F6SUPS<8}%[ndDZ<+Lx.WPvBsC^5Xl=SkTa1xp<$QMT=wTR{Z(8I?,#rp<ES_e|F2lrKU3QmH}a`4BN*KWL|M?:QPc%yu5f}3}M[CjbkpKB2,SoA%jj/ThL05LI%[yO^sSeYmqJMuV}r*ewlyhS<@o7q>QC#exd%Q6@$[{c8m|0K9n,t;xJ@4f84<5F>~q?JyJ=g:L%c:1:~3e;^@s[*kB7Ncln2rT>eY+k.1^L-}7C-]$8fUVLy,^L_N;ovzlF%Da3DwS+#,5jlTZe_>f}r?`^(g+E@ABAUXp7!~@2(ibuxSb,CR7E}CJdZV6?iFp=-q_D`NIJd-*/^{NiSMH/[wZwNnzY$ss0EnXw0m;n^2wjOP}d%_x%{?7Dt;l*11CkW{o]-R;`VLi05]:~kY3s8LnBT_3d5c[fi,PON~rU@#J0i)TT6Sz!UvPKYtyIrq%d/$):.(l(7iX]yx1D%&YPk=+6Tj3fbrX#sw*EViZ0omv{bNZ<+?{NQ6<D2T(y+?19sQ4]rE?SRsCe|tNERb$N1DvqSgZ4VhJ%GbAU$i^s1bQpnT^7bIm[wZpjHWhb5QDTch#8q-s,jA7Q||42>:D;7!0Gd%*~4U4m?)MXt}rw7bf@/Kp8#2LnnZw8r/uJIQ/A-U=7:$KO=@YkX@Z`&3IA=]9LvqmB5X=Hn2_CIGwA)aR6Ox-0A1K|o#K]fX^aHRj#73qM|%vLbe9@^(Myi!4c/:J;oi:ma.TvJF385W:_@T*S+y/]qQnh[sUw{LE?XEG+[4Y[S0LXTWhkt|c?sx{HPS>F)wTavD)0oSnwGr{?TGiu%Ipx%_cGt^nMeyb%WRU^0,k$P$x$;ld9li`pU3z@wTd;o+/sxCpa$gh5e/_dt^lOtKDVQ[~7(.,~mIm9x)hH0h>F1JM!58sHk;ZIne1|EM09>WNZs$#*41*T%07owxKb>!_08isA>b[g)<&D&[jzE>!srg{7)2!@^c(j[xaf`Oi>!w>h4m[|X5]~0N>f0K>[f1*6@vDnItyQ7DWSQF*6,vX,ntp-QwGxJ@x!Lx6cI@.Q=Dw,W.`K]Jlu-+mbqq[%O_Wu6D)D&B<(Os`Wb*^R929Jli^g>Y|b_E!0vN66x|fe:Hf!,_)DOA3E){oH&miod}Ee?}$TcA7?&4x+rI<n,W>>a?DHx1=Ancst3(1jnRae/2L@-;qF*_Sb7S%&`+@,(F*)xv|O;N7I$jG4Z<,bYae)RV$):K`Wz|z$Au0~fvx{1@.b/RFXWaE{[>sT`ad1gn/TnRWx7ca+%#-xk^H^0uI)gJ**}%EP:2p;7zAkW^B=rs<OBbe9:zMxp%6,;%MIOtoxYxArJri2iC<Pk,vb/OD/MpQ@b3VYv0gt*r3iuj(oeG#@y*)3x=9>q:Npd=QG8`4TPqEAbq]Z[<S|$vQ-.]O--s9>tqS0MrH~qoxb.;1E2(;v^T*4->05&}K%BP[F]_LYn#fwY0^}8=ei{mSz)0dh^D~!~]A7Ce>evi]N7I*y+@0wwubpd,p)iHW,if<Jnz<$BnqSeZaqRX`m|l<y5,*co}$]9ya02Lg8G3.sli5j,Q7)+-bAk*z:(R.qs}|ONB|o`BVwk_]9CAZt#mk>!Nu0G}i$<<37TRc4P-5e9ic}!|h,cD+[rPvS05K?]H1}ACls>RSx<TOqE;6sKsecTgi<QLztXKZo*ercK5C-i%ar33su^Q+(t|L>$yIa9KK#p=M%OvPoh[dmDf%JwRTJ;&^-sJLh4[xVSZwc:_/}+)AXXrBH<f].lIOaM8bSs$;m6`+;W6Xrx{0Z65i@@a#[4)=_ADz?TAF-:JmipZ1/6[42cdC#0(R$,)UmN`lCgb):{s&YTp`paVt)dI(E0)?7.NX-AB=*>lI5fpoDoPrtUxaVF[2,*Db|jb9$@$)W.ol:6!+*X3?o>xkGsE^+@nksrl0Fq*h}=L]z)MVWW*O;gN)r?u.].++-|8GHf+D#84Lu:g?&E55m0W|=cIX=Q@(B#O.lx=K_.u2hKd.+<Ke1r[g^A:21U7v)QIQtBYR@by`6Z<@Y)XyaC<rp)d;X$G#[lYJm7}0mdta}>j7!t.FohT~SI,%hO*PGIT0hU)eJ_W#U01lYa;jx>P`cN+%ip]<uxMF80n/pve95>qHYzP)%;a2uoM&N,MB?x[HE=n:Fbf>*id_&@S;uVMIjCq84-^Qho!{wvK[%(Xm2p!%j!?s}a_jVnJFFTXKgys^4U6DJ@q#eEYLf[7o|wS8~f$3f)dI`kb%#3yO4nhqayW`YwQD}tZ*[`&fx&dgD6R;Na?<?{Y]a?8=R5X#992BC)BR)G&)$3uaCjA$&T0luYWLkx!mC9S@7k3xhp-mP+K3qN}@iTv2l[*&sHZUEDAFO86Vk#`}/8c/N8S__RjhQ{iH&c~ght(S:AEjRWkBrglj;tyr-/fm=;**U!hZvM}Ecmw4P%st9>fkc^h6>.<,IBh+<=mE~X8Q+A|-<5|)ooG&5MvP/5=:Z$KC@w3DSdAo$Q*sep]G7t1S>umDd;Z0{}G*U*YI9Jvy=?;ZJT;:8E$8ac{}?Qig<0IWs72lRaj}2ZDN{smd/,(EK{3N:|@r;<:}C:seBjPqYPwX[}j^n$X_IDbU~UTM;!&JYb*Hr_@j<qSQu`!,sbDOKEjlxBsxeLkLAch{vo=@7R||,lV.t!HoJ9??Z4&J;3!O,=q!{J^X(S,On=IzjP?V)Ich[Qy6w[@$sZe!/m8gM+42hcR5#xx/1Dqp2Ugzba(gd3LfBc|9rb/Z-+[L%2~9LH@O8z=?[mXlZS7.foXc;@*)|l#qDvemqw{TO`v.aT6N$*p/3{!WAg`w=zUHcEIQA3X4^&G&-c{VQg:J=W_zY*rPJ)kJvKAAKtnYj0dYZrj8zcX;q>RQgA{-02,Q=AzGA22&Y7}8oWqH&eiR$=bUFkS#xQjQfIE.YJ[GsVKz5:)gh!a+]P8iGC$0F@[:jLG&p1ASG*U|)kH1Gy1q>QY,!t)d`[P+q;CZ%Q$:cFp&E5i/]gE=fHv}2`^sJC?e;sSn2T)fVMR4sXAU,pY#fhXFq;gi8V2oglB`v<@a$pkyV$jS@-/V-TIMJC|.$0Hx(C-So4F)4L|iq!)-:x,Q}K!`8&$x`B&R&g^`W}6VA}-YDk]|I@We^?Vmi3Qsod?7_.b=Qd=>P{eV[#SAQlC]@.tzIHdcWf$?~COoq}R<E0A-1H?h,OK%+[7@;g)l{W]u}f.eKze_?A3iv;*>zwYpD>rUSZ{]B)RcF&%|N1L,.lbBcB=G{QN-uee;z^g,@~YVqOEG7cF@V>jB#lQ(AO@Q.jCE0a%5Gkqzoo&qk]5@l_p?zr?(EH~patdGP4.2*_1KhNcLQyW4/|!5F6QtAc:8rT%/po9<@UTqtMHoO2@SHL3+im48{TMekVT9q(&##|MNAmF)FqT/m%hho}mIOvD9,*FQg2+/2.Cvp/LU<Lw$:2{r0Bnf9&[D*.N!G$x`0`r{WAtioBev|(Lb-H`456D/.a{i?z~zjo[WsQDp4]u}bQ@|C1%@@:Z{tcGE9<;}DYY{o<+FgH[wV1G<W?e-8q&X4.LTn*2SUH2+wAsR-rI!}]-j9SX:V2Qp?Jtr^H];_|!I5$09*kG*gIml>%7@B]vf0j@S;Z!Jsx;^9H&3IZ}4H_)GcbqSW.aw#~~Rt/Ben8EYRN)ZA=R5(R@0r(&lOm;$nmDNX-v.(vR@:jt,Elo8g@b^^>MzD#%[zR)E9K2u)O+(.Dr@G+o}Gz+NUPfgR<:}]5C_i@?^N2,ymje{Dv$%~1WMcCS_koah>2wPm@C.::6|?Q`Q6I6}j8eae1t,[P3hb6~y(H8nf>C={Kmi])`9C&LdzQ#lj.D]$_I$9?r.bU=!sC>&B43|8U^0ixBJivePnq[c7w]4>jr-6nSB5]}Uq,r}00a]1qnQ4AgC<isc7|QLJ0w8lW=.iib;yH}MGBz>hQW|JY^LLb-qnz`UeDoTV5s3-<BVk=s+{t@Eg>4L22<|yw4l?eC]?r-@VZrJ#K|4?`X=c|1syM{<aDp_IdE3m*{j4KYXK@urT/jF`~Ev0O`V.~1n*1ve;g3nq1k%|R#DGI]aBTJW}Dr-?5fMn^VXsh@=lrL`~{i5pH3`ypl^}I/rr56ioDZDWOar*:$Hx[7]{tLcpvBE251j3S{m+fGZ)c,@9-v#:p.SrrFe|-Y+*7YKEC7dN>S;Vg2KqI$KV[,-Q2_r}$r/RQ#[k.lb2qy/Zt.*:G)<@7xsfe*i*09_Ly:%fE%o|N]?R6w1-neqC/N8)z[zg6@D17&&cJr@8?~eiW)87Fx7IRO4+Eu8iCXNB:jFe.XK[(m+@^+7j$8CtQf3!f!zQf13LrAD*]FTO<PKeN:txXJ@gOrEnXQ~TEr:],n@wHetDC-sCo~FZrOx%|K3b#lAA}W6gzOk)unZQ7T*23U/6x.UpH&@Ki&k;Pld,d2-!;z3j<2u.1lz`?sY|,j#N(k)KT8<c1(m.&?Q9wwZQD%7l8kVKa0mD`,^Xt{>3d[>JfEfSR1X%`6l+^?9-3Nr2SBk-bO1Xn#FHB$ESd_avUGLXF.uEe1aJr/0O{UPpe.2PB^=(yIpWagwRooLlV3%B+9u(4NQ359TDN]:2SwPeEIw^7_RWjm(ceQ4ps{fDsx@R@(NSXZ$^h>]a6OvA(cYRPKt5^{n8+ZS3hFPyphB=m`cV{y1v6Ky4ro8+s);X=}]@fn*041.`mr@DP`N:h6^^]$}w*rCyDv4QuPC:&m[:}xDvqYnv+=MZ^r%Vzm[U-B+R9d|oS-ma+-^8kwGu16)vSgX^cx&B38ocI]2IC<]bNFr,Ff;kR$|U%sQlz&<A<Q-=tdtM:TfP)HL|3Y{<fXC3w);.eA}LBWfvZbtf4GRzB=wU(?vLPw*KEUNTOxq6.rEt]5o`i(AThe/jk?^xZDshQ}8]m}I=GFWGG#Q},^=UadZ%H-FI$nyU<Swf(Jlc<bWEwroA>N_Fc6Y/At},+|G:X13Vk>F/lGqpf[z~$WoH$i6VNNuzXohIw8q`bWX?yiM5%^$7xfBW?H_<e+24n@@kR!Je<d[JS/|sELe_UVl9~M~HRu4yZ&8oII}2@$b!b@ngg+&d9Y0Fzq@!urucij.4W8JLr?tyo]u&IY%g;Ds0F%mJ3p]I]1SE<JxtY#T6ZE+9qVEV?8hrZi64<JW.XGBnx3qG`Fc)@ixo2,bV5x#$NsCm|z?)w_SMLREZ84+0hVi)mcg{Nn]w,7uXuT,>&#y43Lz,C2eF:Ej]2)wA&wbNopryku.h9&+OdjdHGF6Rv7}(_ii[!NAX4ReW1fJO+TJ,;{U}sl3Oq}u$0*gElsj`^+cDy)fw,HIs/(XYrB.cA[AZoM==%nMHp/F$hKh^;*)wt/:--uc5pWE5p[gXE((BQVJ!Y$S]Udeut2^scHlS`./$2ip8]NJR[f{7>2V#ak`I2ZVmu&|pT/p1[zye,=xHO6f*F]2I0:2m5!r@hWU&6Pr!Gu}-^0Q+C~vD0ZVqsHUj@/o/t1WQWIW9l~K8y55Dg.Zsx.t_[E.Uh/IDXCxDc_Ato8~F+{:wp*jF|%8-xDXJAv20U09]W(<)PI!G8|28&Dh_bWB%0~g0+o.oa3BRuGK@.Q}yY$QsWU:3@$dkV/BA/xw#]aOz6&Gf1brh+9Tc+#epnxPy;tDez~J;u]U_+]M3p@DrWwu[oaLoclvTLy^,Or2|8solj.4[`zP9?l-8K&85VQ1c_BiH^?3^_jYAp`D0{8`mx!0v_!>:LgC:LT=PcE?bkrSkNOx[1z/wP!F_IM/r@Cb8?IT3w@`dsgILV^J%11xZZ`Ez1D!M30iGFsbJVKrZ1gBU;QJTnM**h!{GES9<c>+i4H$0-6Vk.i/3v3V_!]BtuP_a9!LJY$W|}-4w_bGT&:Ax:IQT&0n;,`E_41;J?X/njk+U<~+i9.T&F2a}vn?hmu[CL!r(w<C(24%}e5G&Jqp(dDPdy6[iZLnl93RcUzv40$lNu3E.(h|IWzK:IQ6|ty]X$Eq1W}mg<{>;&7`XoRf8Q6)iQ3v$1EFL}WWPs_[%fDcI}Rf$~i?:u9ZW8lScYGYa%QiX#HUe~<G2ju#haIXZJ%Zd;E9V3oX8:#RtH#9.O*f>IoD@.H7b2~iK%k0-cyTMtwCNQ}u!|]O1jtsh+t^65{h?LPJzze9Q#dp<Jh`.:6g(-JH9;-dC[bZvNKbxFEc!1.e.2d4}Vf_)z~<BE3nmiN1w332x;!`.XOo7bL.*PM|A_!>Ai.,=zUg~O}VTIdH<y=&?dk2_R+ovE%EWt0M:)n1i@8!3~%a:zY1a|o70}>[p:BI%3T]#gqq0h`$n[h):tH#y>#:46L%ldm<x^;5qIMi=MJ@6cBYuadQ[`ID[]iaR4+>cV7@md0PLru7vIp=G5#n.6wfHI>]r%$jU$]FLD:&(5FX,wCH?-x:NiM73`GMGkbnTFhLwW^Jg8K^NVNx+Nnr65Nza~DMbn2tB+1F9%yhJy}2@,zxOsd#!}2PE>bVEteiK(4r%4>W.Zpgv<`wdeN,roZG~>rR`_F8*WG^5!j,<<wnUl,Gk<>VZY7AsD+tVA-h.$768O_6dlla<]sIZ7Ln/O=kw0Y^{ptV}?`.-@0W2[=^IpzDb*?=3R</&jymXC~0V!`xH$gkd9U)GH:b%wia@F!hB]0jnfjghtF/j<ut~vd1oV`o@j2N%fH/$Fp}raOs#)jemJZYoR_AO/tcTY6]]aGshp,J>wkBcl[S_VU;.`^kk/Nmx/b;@)aMEE)<@V]410W$_<:4C|T,l[f1Ol7?7G_Up!3Eg>kZ{23m#k,@]v;Fj.v.EtJc8#[L%PXgB!A(DD@jjj>133Y+n+`Kwb:i>p855+Uc5tN=vw}!)Ta+8JF2TZ.wxJt%7J_2(3q4w`!-t<Vnhiwz-D]eWN$bpB2%vcbdioH&{3%Z-K[DxhbX~(bWDSIYZADjRZqeblpu-*D5oS[dKCe;9]I7GHb>D%H{nLh>Fo2^|WE*H%hJmNC_]3}HBtSc{r~Jv10^UkW]]G,oNY7S6jQt>^b-tlq~j/s_Bwkb}nq+}AY?bvz,CcaP*z,Mgqs!%7476h=|^9ODcOZ<X2ySnFmf)2|;F=PBj5HgGDObI=3s.kuJ=Na;%Xj8rA7<_v<?(|VQ#UIA)&aF0;6w-7L(%xZ6AB)#nH{wB@y2hY&tl?@]0_BMw{(p#2?]n3ow(3;d,-PY8HMa6}+c*$fz2D@V?08=uiYm_E],.?ARWe[`=#,2hEI%!62!zE~Kc7?@Q?Co!ml[.fMxr{j]s-<tH>z}=Bh~2Q3hA|CE3N{h}K%BnxS6PLyg0ry/L}R{EC9v<0BTa_P9E4^+HM/llj=zQY}@_HCtgleWrW,+5d:Fed>C$JK*aE/ihSeY:w{-ModJ>K@R-AVWsbcY:)!+!<|#P?QRu#:T=zrSmnJ0%Qr__,N`2H-)5lsSQjJ5UhZj~0J~e8|F/D3c7MFF8]Xj!b+j`ETJM3xMNhM!_Ws-Z{}<ZU5FrRF^s/b0%pM8NT_D.C#|dPImF^[Joi#T^C4Pv1p8ua90&]&[=Q--F/6]Dss1Gv3$yqs24]oPX<-pi(&B$]-:UwLG<oJ4?&(}fG3WYOf5L9ClB6J;|JYvng*p5n5KajOEv`|t-G1gP#xq+ZaT}C$ykMORXq{7)H.JIW$[vA4-xd]Vd<]]2[G1F.,8jcrQDpU1=Si.aoSn}^dN3c1z2uwpcBTX^cL5gE~l};vXlW;FF72a6-#+).sgyj>YBw9yl2knq5u)mu7=r85)b[.`z=zMl>~dn;4VLRN;TmTTC2K`CY@4[}a7m[gakjc4&yu@k0f#_qbcWi*%a?+8WNWpaRnVxH36pC&UMiN-C1O,%gZR`8.PHs*[i?6z%rgqrIgM<6+&S2Ky,#-aR5WszBQ,2Wh5YLb:s:D`emx?/uDEC*d2swY},6[K7,[k*bn1.?9Pm+sBkQ;o@E(Fu!vQ=vEev+f+8QBrEoH^*5yz99eT?A-~o97WX?BKvT]IT~k=oC1bA4T5BIoYlCb_^<cbwQ^WEkhmV(Ko2vZ7XX79+_jF,Asd,FvTJL&2Oq%HZQ+9H1h%e.Sw94W;,bBR>rh<3z@M-,w*rmSa|q`oh)#$%8z0**v}eBEjU&Q>Bltb9804^H`r2l3!U?5=/2uaB/VD]z1X!63Jz>gtnU:5p+@I.H5`J{2WKPl9~<|U8}UT*[1JYcy%.R@0O2]9IMatU_G,kY:kY:YA,kys-HL&t>IAiw:JZp/n|BiYZ3ptr<HI<y^k[[/}YvfxR4r^f2Mk){$6aM#;qs0xd(}!{r`NKSjh[zf2}0sRu<jv0tFExS`SMLBdHnh(v>,?u#}P)ZzvJB%ugZgOzgd[|.y:3A:#l<6ttYF|7AC|{1F%~1%j{7PsXN+*y.gu[nmW~-~N^7#_,5gU~LfL_lO._W40F$Zgoskzy%^3MOK.u|<fkI`D56x$R<PFf*6,*m3):eh-Rryp@*PC|y73!F7Qt.R[=H9E7(|EJ<?r[X]i2lmFr/19X0J,Y:sU3IP.6rm^K*!}_W]W$($+D3iofH2@yq7;!(c&}bgm=+Ni#c*VHEn1GNp:5nuGE_<3JBoJ{F^9l`7zt^<%N(m@Xfw@q$%9wQ,5+d@ocV+rr|y}Rnsg)BQ7NvG@FFNe0)J&o|#)nVO$@h{d`?M`MKzfx&gzee7zA4&T{Sahy;g<i}}N<*,MyZ6YpK6$7.(>mJmR7dZcBG=kO1PsDUMjw1FBa<T9F$-V-h-3wxX>lk*[6]>Pj?S5]A2i44Uc~##qt>`]g#{=y[${oop9a6fPn`SouE@&MT3h#CmRtC[%`#pN[*snrwH+>CwrFwFLcG_}C=a:Y|ZZXy<b(Q6k_(xf*B+mRj`tsNai&=3oPdOeR;mz{pN2d6%w)1zpk%fq^Lw[?tu@d17%)~/way_(2ca2)X!4)caqL9;]X~Q9)O?[1)I,`F3+5f^A8uM~EbF5h-Iu~/<;J+Ba8:V=k%f`+I61|FH|~1]TAS&xcy^*hZ24U)B`D:@Rgy:QW,4)}0k8,pp9PBjT^fDy#h7gQntH_h&9cR=?MLOZ/$=S:pj]9UMt`|Nsi9#C9g~c2P)<BH~)5qV2Y$}6kUah+f2SI=JBF^c,TvGng@.N}ds@$-,m]FbYG7.eLNt!dh%hGGTtU`;A^LAWwBNiXPhLA(L}D9pJC5gK45TR,MS2!N99aML(*d|V31TI|QT!qD$(/|u4LD:x2mdBpc4L^3iERTSaDRHl%D%Yp=J(lm5XCwF^bm^XOJUnkGWS!2Za4gmTLLshxVc/{=g:B2RItSakN+mpr;8xf-S#PB:^pq-&pj*wh,;)0nVJL2hpaD$8aICq#FKBQ%[o]TV~|6h.JGWKY.I>dbO5/E|/{]O=6s|$%Rn^LvxAiFNc-H77j;.n07)RfZY9RT-q`3L`B#TTPl.D:nc&?_PWC.ad&,L#pBO#T8g*0~UeEYpo72NyeC%-sy[RW]-?KXtX/{va;}/`pURSUha[vt*R=H%_4MCUl9Vx?7>J_o|oJIl05vpYE(m#g#DWMCjMn9OBe#lr0Lm[0_Ru^v|PDR}V_eN4,e(7Y37^^[|BBmJ*P^Y?WNa#Vs&ZMe5Cv5Gsk-Cl,A|S(x@9=s$j=L3+-2AZ[oX.E@_Vqm)jN8:%TzMIVb^hW}BRh~9dr6P;O%N.mkLv-*9-GyotqKo9=6Y^I<UXF6:ggmdpV;d),4}-eJ@Jb,Ed{l7G&qH@[w8}96DO>m#`G3D&Rb7e4KE0;5LQ1xN@m&8n7A#R*(e$?:&Fn9:+u.lDRHBK&5ECNN)RM<8`BvnLc)A&hm&ER;k4{<pN]^SMqRuA|PmE2X-Z^WQG2mR_Ui7tMJq*@6/DO:~%=JP(_t*FOD??$!7=nQ-DDYEjUw`k?9Ux:=Q<awrc]01|B3TWM[Ebzh?Y,Djyzli|j7I:!^aX:blIvh)F/$%pU$t{GM_e>i_(i>VNeL@7Oq]S9#aS*|z-07$,/Rq}t?:O6lH3{%3qH8gpo8m=W$v1F>m4,-r0hQL5[IcGDvzs=$b@&KPQQoGG6%>,NE?*MSPe$iG}GrHci@l?zC/,[XEa9]X2=^qlAh|)hThPzVJT|;KsLCM4cUQRd@KN]RHr]n4xHirkEQfUzD!^B1rs?u&PE{3dwCV}7m;}*}t*ml&pC%Hw/oYRe(JE8.9%<+V(Z!58)jhL+0xeclYB-O5z`q^H{j8C|svQH&mYt:W2uK6z,zeD14pw&[U|i[2]ny|(-`UA,Lf65>CP=iqIhnF$dpPP<{$)^Rk_V1=@ke8fe?}^e[tOqF<!MLbR;6#]jAG#_<OWOV^_@5`#/aAQE(EoaSgO;R#3F6#n?HGuzPk54IS)nhvY8&V:!0>W01[/b6zlq[nZ+mjWg|BV:Fj0%=!1Jd?}V/-1i8qdvSPYoq!X7bMC4qgME9B#&t`x6[*D=#+-D{mIRGd15F3v4S{GwH^ZX;brLlT|YOe;p01tyCW;>b;]a[Z}U,~8mW:E2iRx%e68}8CZ7|mZ7]J9FDGyR|k]U$CK8`KQY*,&>b@i:6Ot)49wb#loyJYIzC(W^`hxY_ki:.U?Ki=ACJP3$abu({9Z)w^o9Ufe.!HypFzGoJm@cY{MDJ[6S7!1<2-E1pT[G`;(9D2TN4PCzQK0OU25M:Ulxw-9X4_d:Q:Ma;K=WF7z4gE@8l{ZiR}C`8Tv!)XcE`JWq^[UQ#V}l)q-b_O%ZnFGMulxDD9Dn4+;9&BQ$;h2K#`x/10DHqr5zhatq-DRq6;uMksuc9zG!U9i*(71J?vyN{mKDcr4<QBG|dCQ7DfLNuA1n9QDYim=z,{i?0+TQB;R0,Xm:Yl~PGMWN.Np%Qn.hPrRQ&p,9.OA{EIQ5{s^h4E;,I1:Y5_,2r<3J#a1B~3NTp|Dl}<W31xe9b}SJ:{def3HJx=,JL{b(Z;<rS%>;Bd]q&5II3cB-&Xep]ikeQR2wuYg=[^%;+o}qEtU.9tkHF6uF|qu^OW*JW!U_-gm$k._>`5ux}@(k;imE+)Fq3-5Z0HiQ@E,S5!H=p2k/Ix/L[GGP<T,+YmE)9}CxhOx_CylGxKn):Ma|q<F{-P;[*ji`i^e7az$o5-^t>{art4iCNcc!O.d$sD|vRm*.&mc5r`G2qbaZv[b<B%4C<[1bosV26EL1ALjxm|#qsCX3TES.PDfn?zr35v9}qtBzZ]+8b>%f@gd[/h~/sp.:V2)D%1Vy]]*PIl?s_*aR[Mx,M[ynROqsNAcr})QE)hS4D[(*S)+gaPQUZ>@!UN`BM_h)_JifSKICQ:>IW;{qR=O#+|1j^n%Sb~=zjDfJ,mXd(T,<#Cs?DX6iFVN1?rRX67U:ZOfvv`[H[W6#`npr%&{w*MED|II>Z|r]PIvEk:c7#EqO(d=355Onrp[a`niM`q0k@dVxU+eHaat8,hy;wo9/9nYKOnkqNAy[uYo4#[>H|ZaHxd{:tm#W4KsmdE_S-k5n<^;`pJ7V22b`>;7&TzoF`2x?D]D=iR,WyFesxCluC)u+aoJn>ulbnqp[3B>E},HE@[,(Gy$r`M;evgfF1SVg;?zGml?O(`0UV8M40)nFG9<R}nUJ8}iqT!2xefLBR#g>M2z:iT!2fXjK$j4|}*|6FQn!$=bbJhpC4Lx:cQO0hrs}eeXr>i,T|KZW%q>]VF@oBX0(GA]jc1U*YK9bUpYX>fsLQD^Ph.(<_S~MJ:Dpt4{mkqO+>#@S9@e=Rn2%XIZ^m+$<0smh;sA=qf,^w&n9@JqN`:)r?`1Tm,O/E]QuNEQ>2tVD<;e*]w9rx)Ze@K?hRI7A+UHiz(@j-{[qyrj-dWU(e/D@f>1r@#|e#&+`A5r.dS,Kmi$_by^WAZ9qi#ERKYw(Os:m/o-W1eel>|>LZGT>P7#LLQl3Cn@i-I*aS[/?8P&qj|,YxCMVmDDq;I/<?mNogN(]6f7&T^3q==~_AOO]]QCf:b^$(aJDY1y^uzo`B5AMdKbcJ}S`pb#Cfaj>(Jhk?gKgEaFED&/clO-KxK1&=H`fa,;`IkQ:cjPg|+o{/7@M2v.~KqfLyqw5DsNRZ^^zQdMeej5k.by8UHY~y`<DT*,hNYmv53491Xm`7]XcxAP7z5lJ&[eYZM%(Wg2Vo6y[;>W[%uG5-3]aw!v49@Sf>KDQIBW`vC]EFg%B|a?,AyQt~9Gu-DLw*DE#`9U2C#!8Wiw`6:B<!U^4Hv{E#0ns*0)1KXt1#OPbeHzjAHO8NkLAbuz9M^w.V<tit|kRJ2Phi&K>ESE#0Gz$Daz%/>1Qi%H4mtrqgoQwU?[8ii09nhniwaMZtUHz2JG{vI5ssn#?s:LvHi`|bp>@C~EvgPDTs/t!!>yg97nohP2E1*YF--49^S0B:.g%4R|CKvew,7PpWD):m}|W@Agh.?#ji[4C;-X5J>y%t.)-K<Z>[Po1K9&wP>1ur{d|#EZAt27nRGBz<Z0vid+/HkCaa>8R6uRgL!M`~FQ?r?9n:3k$~TKtcz#9NH;@P_GQ^S$uzunyvZS*kAzSP6gY}!P^OYxv3=:mG+BZJQ`)3m$>eqJp>L1#O2_M%iVfRdC6/9j{zH.&zwP?6J2RH{,a[:x~>5iV[%4reZ4i(F9)szDL6;AsA{?]quD}irj]f:z834ik8TszH=~KXvMW@]u``nBn?1)9dw(]C|_k_gJHJAUu]gIIwTdtu/,w2]uY#hjMI$ssLIAmx8jNKlIcAv0v`]4,}_x9IwTB&ip(juaJ}T5?HBu-t)cG$a#;%g~X5Gdry>7fZ@fL)><FpwlTiGT|[sB7#_oy-J-j;5mei]nF2p|6@5;A>|Cf5kM+((,1r*]|d[X}YAbc~C=Yc%4Q]+YwpDF~3Ge/TpmYBWz#O}jtTiQ{|^nc:I9d5HHq4mz+sJDg~x%5%|vAHi4M>4M%=6xt1Z>0d4_MToZPLElH5$*vqI[NpAx6<vi8*D|nS^d.o$oDZgd?UoF&cTxGpy$}|<+1xfQ5r_u~#U4QS~$@+=9%P,?a+XYE[VP+?@p8+#.9A2&yfoVmLMV)d&Z&o,GS{KYI[8;BjFy]&SBbv];ZJ^GO8`-}-PSC.Esrh,m/9l1egx>a&HPlb3n@hT4];ySY:1Xv]?<[h[CMdku__>PyT6ms`o0s=KXv<N29N@-,)&0*@FkW,^7BW~u%{qTx^NUm1SU~F#[C1ju>Ep@vIEU^(Y*IEcz+]m[7StDSQ-VYNCS}-5z{#}fSJebh-^)e,xGa_n>2~/)vWWIcs=fTF;^:vfM.tyu]c]9%M^kHBr{mla>T{?19(xRaj|*Yl}o>C:=ULYD]keiN*siVwt2Xg2Bq(Em((U;(@9$JD/p}6A!BF%p-bSt#4r_c_pA,Q<hR8c+v?;Xfsz#I<j>d^UVj]-.*XKj&_(tr1jX|?ZgwsozO)3H9Um=*BPJ_(8Wy_7lAsI2PPQp3[T[3#$qI/OHq}GYr.[ZGj:3*VZ)I<`>S;dj,+$ZIW&5)>t!c_Cotnv[&lmffo&g}jnpR!:P9OJmm_GN^0075&Tr*OXfH7b*/opn<|UFl({@zpAt^AK4b[ZwE$`4o>Zsp[>*KeFNqWEm@cxl!T!_EODdm%YhXz6NzFK^+C6xCCi=/J/M>v6fXEpN_R|)}]1F?I2M6Yy81I_Di83k9e@qT,ZiP=$VhWQ>[r0l@-c;?Kmj2L=w*GEkJ!E6.yYg[}~,!VZ)#s4Vs)]tf~PJ^{}gNkm)r:0kfC`9;X!7x@,U5Yx<s9[QnNQZO^mgga1}(-by8]r<*|;9p(OGpTR#H36R1qTP!+enM)z7NzS&|Mq@<I^YyxId|1n@y5;WH1qdu;kBI(aSEl1|Ivn8dz].dpr&TR&EmKL?s2&oN4b@!YH5TN6-1vbD$kvTS`@C]t7RZ?.c`7J,)cSi|;I2`kh%>Yng;ZT0BAJXf;}>>MGKmL@sy4Z#v]IFuGpW0CrUL6lBz=IhM=yw{Ua/HH+s*nCxWBBhmph%#2wbv5.49P)FY^M0:w0*B>t$+(4ieL[;#89qyo4r?CQ`A%#.q-Uowm.YQd<8fHotA_^}NIyz|Fu*|-#}:mullC3lL9b[]tsp!%:SL)e39pd4Ma;VRV[IoAi}vTBBSqvjP$]_0<bQ*|oY*$.o${pz!u@?8bf>8)+&4Q9Fl.IV-Py(j,VFU+X}D8gb|z(YABxSPL?LG{UY%K1IaBnD`cw]1ct,rU#-cMi`BVOF(S3k.4q1f@NaN&&-v}T(czTE!ZJm8tj%Di*]`5+E83Vk45(Oa}ov358fvV$T8Z=r5)GP>H%K$c^4d}AYyS,(N%/k?SSZt~Jrb9+<)kK?OO=}QM!mh[>$~gxWs35,1-7PO9/He$SiS8h|XNho7O17/0k[#wpUeLn8gIB]8A^<~QyJ>pd6-c46{R:${&U4H.V$WBTL3B0py|>g;dh5XxxPfOeUET*Z)e5)CYj{Ps%/H&,(ZZNo#K)RUaX6/,cT=[#EGgNq[42&Nm>,mSnHTK3ximo-cUOG&fC;r@]LPC/#$![c4V=J~5@vyMV.*n?])k`:o_lXhDz)Jjl7?$Eg|kF3N5{lEuHptW`4*iU}@?&q@Z2#ClIa;l&SQ_V#?znUdM^8=R>$/H_GR%O&SY289D#^QEAQZEwZDTgAs.xp_GmkzpKu(e(Ve71m#hz|g]w(b]9r~I6zr>Hx]rVL>8mVn0>w<V]|nmS<PyY;zK-ekLw[FION#RZr5SYgo8f^S9+%d}BhoVyUDghI]2ZwVzgi>#1sHFsR]mxo`:LM3%lZSM>[vb^(i@O:9}$G>b|/iD$A0CIaWGFl*iO?5AVPq6f_or|Ps>q,WJmudplGa4>4wamG0n8-r`^q-i1[p`XlV%hz9@``g@Npni9gMCI|v2]UB?Q+X+%d42n3YD5ID:{/.dyDr4fRCw`sGQh1A)PSf]-K$AH@*sf]VM,7h,_,yvV8G~~4V5b986]?zU.`13GF+Qa8KS(3.I@E62(U#q!}~%0T&G>$hRSv=fHV?ZM-!p!fCS_t3c<Y,|rt`w~U&kk5kki2]owXJ)m*[^IyhkdBhuj~MeI_+s*[Yh)0bm1F8sUFMkssha[|q/:xr<p,JH|h4TM8*QxDMs{riG4}b6pd$$Q!5fUscv$<j)^_,+/;/}!}a1BONsj}^AZ%>Tm/;x<+CRg3>f5X9gJW0-w6rOgTY#L-S0!7I)$2xLku[Ev)6Q~E-tE/hY]$:.P:8FTO8nKQ6#V1rAtS<AezZmae>cEjG>>@y;i:13<+fo9tj2!Cs+NQ)S-rMBJ;;CO|Nen^xYx5&8*KDxnW=_C99fH;wG7proH+M*9-H=;ly6d=.]l//h[S0V-D|5qMQCq.T+N/}A$1d)6836,Po[1VZB#m8:keN)s7lB1;0i;jTgc55%C%[J-=xJbj#y~|0:H6fC@{[.c`lsPM7I)&?7t/m`5}@ba!(F0*d7L>}l4m!8>l?}N6?]:,/b{4jUEf_FY7*@Xx81swmpZiQckB+Pb?rYfE3)KMR7}F`NXLr7-3DSCzTL)_D{D$P_2A[6;T5`vxYUxWGJc^eq}VNkT7_c-ROE5t9&.Xw@bS:i![:pe*9|q9}0(=r!S_vz,p@SYJwB#0vp}V$OoqQW[W~O./G9`$}MBMtnlKa1q;^7W@fmzCJd&S`sO37IykTo5V!G<JD&y-IQY/_EB.rWX$fSj_+(k(mE_Q4Y@xX,r7~QlA~C;K4#MCyF/;t:T.FQWEd}uRtfTD@ZHKL`RTr#|v7aZSHb+I#FH9emNagho;arxa.Z~]LcG.T@|^L[j[uqw`x{8J(K%Gi?XPXgoOYgI/WBR,A2X>!^;$eOSH`zDaZQ<[TK[}OVx<L@q:&dQc]$Is{%=KjZ|PvV]PA-^:3]-`UNs;k;-fNMXrB%@z!;^j|6vs@RB$as+m>IAL]6}<IRELph_yi1e`^g~b4.C`7vG1cVpj@oue;p/R-a[u`W{_A=9v}rU$$:mP>kz+^+XXf+o/uH:pI>^u6<^-b`lU&w@UjvkQ6I(pup^dosLV2sXCs_-@z0Vy&j<?I4_B6X^O|!Cxxfo,Z#LY7NHoI5O}rxqX_XMWm(1=#;c1RkKa4-~Ti>][IlX,T2<RQOdJDI)=Ef]#;5>>X-$*3OLZj:/O4DUpzylTDa~*v]v&M2CSiqW:%d}L`[+rxmg@0gksfpU:=P;`EWGsqs7)ocL]pwUA%Xv3K6sNNbu1EL#*g*Z}fr~!}S50qiNX;11PS+.3=6t3_A^DU>+TlA.<0ldR`]-(KUxB@Cu^t&bi,n:f_x`k3e5v,]^E6s6pl3N4(w?>c*vKy[14|^(|bb%H/N.S}>l;fUWMe1!kNqKAS;QN?21ed7AjHae}KLq-)DY)Vwd7>0RVZbt0p5|bFJk&Wr:%!-bHZsCcn_5;syvUm>vb:i0,M%vg*8I+:vwji`CR4^u?J8Y>n&79,e,Ky3ohdY}}xVY@8),K%8Z;z9097<vtkp]Pg{?s9bQX1D`q_bx`i[v|:IB4s|}{Rp/V#hVmb)q|m,$HKH6ze>%D`DZFH;W^5^wE)bQPQxs9:}G>T:H8hz1B+aU6Y-.Y{&:TIO8n|d0BX4&AHTHrSdT_r/jix_R3!%+-@ZCs9]>&x!pfAtz{8??6Ic>|*FXQk:lB$In|^qlG2U!R%Z{v9S]8iM*Ru:S}:AeXjiC|fJEZd&lP3R4%)ayMoy4IG;#I#WXOA4C8.jkPBoigKdHmHw`v%NeCu?SM|yaWb3W_9njK*VSnTh?2<uP9e](;kVK/&bQU9ybqWA@;n/0QM?H,85n>kmW-0xMByjZ.P&H^7b}Tvo&UOsaVyR=`(d:Ig(gYc&V|8pRuDuBh(&~Vq6bzS:I%O3U~w)_6:$x^MSg2VueDTb).C2-,riwT#*h#ICQ44K&p~t}UMAy;,7^VI.sK6tQ.vVO|>ngT,n<-a&5)#h)+x~s.F!C)&Ya.SJy#EWImo8RMs(I%$==yH*)vnJhcy}xywem*!C4B~%<r~BcG`X8%yU)Bnxi2tWm(T8]3#;3I:Rcn||f<Vb9pO%ahM?l^|fj$ADt/~<wbJaP(v=>O,5Y?n_{b!2qA*rl3KVY#sRv~%PnZ%gzmmNb&)O|9i(Q`J}T+mXnD2aa1_G*PMwGNJr-DhsCI+eBb~jBNi.~O:c6e@K!>O^vJ`1`r){xc<(ET|.)T19jV1;3[<Bk5v6KY8s5A>#X7`S,l!4nG{t!r?xPv-4k:][TuqP1;u,uS%N6Z$r2&YP&:4jD3m1Ix<skKpG&fjWpUl4)?Uu7)]>LxKEE7IpNYfK:@4d8vrwKPC`CkJo2U}v(QB:(w.5m)]>dRG4H47jYbh8T=g}JE8C/Dv2#!P}b&B0B!EE0BO4Yz}q}RU#fThi2xO0o90{JiaFA$-BWWZI.7&tc.o[<v?JLkh3gp:V<b:mXyND}Won]H/#ho}gE)O8DSa_.IvDigg@k=*^J,)a|0zwx|.o{5MGUeL[8T40K#C2)$Q3!{OQ5>J*{tl}Unm[uW<p0>iE56y*R.0@Uh`3SdpRbM!6y5&C@VR^J#GY^:QIH>^KpDTZi&W~+jO#~cAl@[>`nAF@pbg4Onp3h]xc^o;=tF_LZ8nq3m{)<&HwLjLa_}i=M9Im`+!7<I73cOn:B[4L.dlK%=~u&upx|Sf#l-AH>o^!3BM~t-{{qvk}kBuKtt,NG[FpA@}2#3rg-[J*lnZqR0jbz6??x@,HNB,Dg%l`N()[d_oB3x6Qu<s<(fs&#RQAH-BCVL,xFh[QH<>_88`-o1W3fxTPcCqc.pslfod.h7ttVbzh6{%IBwD>>y5-$5[Hg/]J[g0?zPNaf]ZjJOV8j7Cg:cu#h6Dwc+7dfOuzuxBF/aw>wRE41V,eR2C;7/<;F@<Y,lIQo.c.XS@u:|nN~{J42@sCldk=d7M^^ZE+X3Tv|%sy#{%77hT1y((b+5[d%0(/CjVRJr?$Y_z?YVPH.^{5X;ME*VOnCNsQr`nSY+[PxAgC{>P!K<^Vw!5$3IvuK_dEDws^WqK)w>:)WEj9|au:37g8Z7pMa8v-zyQ<+buyyw<I@r0Aa6E(oY]fGH>l6qbHHT[(Bo9vFaSJ3-r_,`8+JAjPlwb6X}X$t-P#R:f{bI15TjE)crBxD&tjKRDVQ5h/W]6hdBa9<8MlNm+@Tp(d$Cw>[7x;`)M1Ky<bYwxY;FpO%MnA!e3~#&Wth<5$kX8*V_.TDYAY0fP9V>:?ri1h+y-l|%PTqi-{3gZ8l?d}Bx-Ih2s`AnNgW-PdV<D,B<>8bgy>,>DsCz,7>{(U4`dhZ,gr>vwd$OvvMa3G*B|p@W(b;or+Nn$gzlY),8}}c2h=quIAB7B9A*$7X6udH`$C3QN#,XnAmEr=yDScR?a!JSRt~9/&2HS*v6a}:Y=ME@b&4RAvzmCwWZ]2Sfvz{84Izo8@2w__i5(+Gu0S.g#`a;i?~GgmGT3zL];95hGQy_ot3s5rixV<%ZccZbS^A)6dG_5AWp[CBu=T<mWI,TFg.^$J_2>N<W.|Yx#MkLao::g2lOHsLsT`a4.i{M}*2>T}X0,w~~71V?#SY_G3v2sJ[i]/$Fa.BsF1TIOB/|hj@d)Jko+`8bg13O!7Re<FV8H|Z<J*qQFYKI@NwKg*caPSTtB?H^`^;g;}9@l-X7^-5bdQ!hlMGahiNw3UZcppEy{OY$OkP4kkB`*H!|z8zD4|x@gE!4<2yx#tUx}`A)#&g/[N<m+92d^*gyd%eAo)Fz_7@r@SM`w$T2Iv4BL+kQJWFO9(-ZLi@?p44%+=1?+qa7$Gtm,G?VybUWL9|@od,L`dw51{.=gNj.[sx7SdD^rW`4d{VNc%Nj3]UPw_Av<w7D]Vax3n!5vg[GXw4Ek63tbZmmyj=`yo7yc+j$Ridq;U(FgH)6eti,n?GLHpQ`_m7R(ZMvJo`FI5wcG0FKecekwY5.)Sh*?J!R(C5xoz,e0,G}H[ongu&fF^lzmwP-$#_h;2ss[Q9u_.d^Jkmyut#yuZ<U@r!wv2*)$BUny}WyaY98xd^hd<_wU~o;zl~K@u2%8Um-<Jhn=N;#N=.LflVqa#RYzdSy+&EI6zD@?K?`t8Q_?gh@RkK)H9:c8:xMlmBSZA82APthe%VR-.Rb1M@]nAb{b^YPs*+l[N8ZB:2WXFMil!5gggiPlxcF&=u_rDG}8{2Sh_GN`ogGu2k8Jk|LQ:W9R9_Hb@tX.0vuBpeC|c@$ppkqdBde67^82W{}J51j]6ZmUtl,YdH}ik-IqkPIHrR+dEiJFHx[G}T)d~;Rpq+-R+J6N-)5KG1Fjvk;o,sb2}cHF?@Dom;9D1=|;4.btbzCRqjN#~z|*`znhpXV:2^_qpX1ENtYPNqe<8+u:MR^Un3OeI5_/S;9pjxeebBFzhr[h#_{5j.&O)Z.(qtn&G2X!8ai0c`K(55;2y.8:Bf+ql^.sn!k&yyc3B^L&|OJ:VC>0&xGZDtA#)7jr(^i!&rRcsy6hsxB/.)EiGFG*,(j&zE/N)4.-E%$U(c>BZw+HcL<*58U}j$}$${4W6e:{PHY6uX5//:gC5e*?O*xAeqZH/ka_3`vrsbZ_AC;pG23&Y0S{FN?,6%<}hi%4H0NcA]/R<uCB#R&kqOtQ1:#o)=Xf15V-ec~;70;4Sa$><aD={xt^X(v9i=;~MJ=vkpYwR@f,{SRWh[_.$)qwjvAy.M7QW3SvOKsf{H_iqps/VY[X^Gg.1pD+7k(e9Hq/5#=_^&p~[9YrI8)tMS--g=f0^;Cv[<f&=.1u`U3i.)NHpc6~+Ge,V7K,uO@l!C<Ag/YOO?74KDZts9y?-VNb2%FF-VfU_@VkH@c!>xnWWII4=iq}jGr8W:tHznCXwc+YfeR-Nj=o}K%,o;rr@OW=_P(n2ViSoghLE8E?X@J-2Vb]<kbb%rJ7aV?~aL[^3o|CTI,9AVv{|wYUMIZ)bwM-iy_w)z;+2G(QV05%{42#@3(_~Z9rzGcOJTbt@[Qcb]s89?@bNPEzZ.QnkXY,9V-@-Yp?Qnpixug7J=]m|e0<(uu|Lki](?g#+u:dW;)$+./qv),CKml?@jNmLn<gs{ii1lU{LkFmck~=V!`7B_xj&h9-kRx4&Sw=mb=zo&n[L<9B@F?cUtH1=1oCH|h.<vA%&v$[;>Zk/I@a2mC6n*[h|eT;1p5=eV#-ahn~pjVIJy&xOihC`u>V_F?m|,6Hcd#,#!^{[)9UsHGbAzh*0EY1Dij~_;:KLFV8c*DnW*8s?V[R^{wB*br(lT>uq9]raC*m{il}FkU(bI:)L}$8jw<!a!82r?:m,fLo5MFEcJ38uth?wsM?wVTMbKD0ti$|Jn&`uVyZ`v|5OwaW*7&dB-:Tsxd,NoymWKZh1P^QMYeKw@-n&5*j>:pdflh,pg/i-@qmaqb=0QIO$r%26@vkrJ%XF!LMl=tGt7`nfg)>Z-{:Vq}r1Q|iAJ&hsawyb/7KEb);IYy;4yXaq>zs6u:@,_Pe>nG&T_:0y7.-+7]c.[8,jO]f|t-Y*+*,8^iU%y)$8t$4OsA8b<b:}q0$a$@m[L@4L:d0e){@6=CC/P-<SK<&jJcg_?IEHKf_#1p4jHzonFo^i{)J02mz4WTxa#6j-ql>BaR&H$WK_EuBkIe0#ZE>6tV3;:I;eKieE=mUmp/}//Ydvkp#RwCLn&KcMdIQ;e6_KE|^`Dz$l+RYvwWit*(oMw<e3q|@xM/yNVQP_:F$91TjYZHC!qb1u!Ggu&nlp//du+D2m;HnJRZkX{|Zn|SjaXQ(Y[)<s-^GK49jj%NhnTkvMX9bho)RwH<{xh)O,OV%tUR-b*zm0:hot8jMxWaI.)A_&VmUO&+}CU.sp<l6W726l_8Pm1@KIAk6YC7/.UV2M(O|[P)7z5/t>|Sm6--kxI?X#%-/lEj<w8L/Na/N47ve;eed2&R9o$7.;JiSkkg/{NJh@lxA1u&4BFEcz`aBsNzNgw*u*kv<kE%_FK3X;gnvMXa6Y7;CZ&O=V4n7e^dbiyNd{%psY=W%pp@g]ZM?#vn8c#iMNw1e+=lZ#Yz;wYGX$yHQ[#@77,U[a70De[D_}=zg;vta/j0X$rnYu.l?yIS7?daHK01&fsOOq[f@}?Z=VsrS,z{=;#HO@-WbEUrNdgkw.~:$FSI3hZla]WgYq078yFBstEzM-@%/+}+S,F5SH_!UqOvi&6vnwCY8Vu=6:aG4Yi+?i-,_TE?J6;l&?bZ-AC_AFyR[KKlb5=#+0BRsW>T|:lpa#D:F]~4D8paET13x`T*g8mAVXf35K`ABplDF*!&#SZYaA*eYbAsxf&Wu=K>g-9iRDYq,VQ&b{lB:;6yVWO5!`8SSm~wibA]:Rt>0<.AW#PKPAfu0{T](.WQ&&A-teQEfIZNCn{h|>#.zJW(gE[zwbTe@u?NE><)B*77$^.<e<;E%fBfE];~/B5=2&={:*=L-iQ;.m}sJm0noo}s0@=9-<8f9@^]ITclFBwuZg@m<#y,q%HkwdIoB$)a1@x#5fv$+|6&xb*:=*#}Lb#K4e`awaU<nNsHt!4MUWV@&h}sE0h,XXL[3+}|=<3eFNNvw93%AN8s9NaF#pZDf_!T#NB5hOO]68UOoI]mx^.hXemx<Y0U>C^#*7CtKE)*&#j)<pmFq=q#_ga(R_^?B04P0dA9xBqAK0{^F^hu:m7]BB^(c^AND6eO4rn.O7{J|lXEJ~lzqR?9H1QS_T1q&5Dv_h3ESP+R(n%wmW1qOLrM(pY:_+ICj59OwXWN6X87Gr.v}Y7{2gQ,oEba;w+p0G==dg`|V(:X`:^KbJ`mPb#Whcfkh64n>H(8;2g1^6&s(&ZGxzyXW6[Kew%hv,_#l!)^nl-hV,+./>I6oM@A^AJ_JqoURg@qxs#A]asjq,|xd@HFZLL^Bw+DMnPc8YcB!L(d(D3v&FoY{yp`ydIT^U].Z%A{4+Fe<.XffnF;Ik-cX*fbNn3IuU+@x,9enE3zgU)anu7FOtugqJ6C(N{sG&(y&(Y<isSZ?EMiMNA`%GvZ62]|+57D%)ouHPb!O?9ni(qx<a6-cP`tGL+(Ce}*R#C&]>:#[99P3v8h]X@?y-a9i|odT_b~.)gZiC4EX93(YK7]}#^ML+SU(_6k$]ld&.O,?e&XimOigU@5@WbJH^~)Cav;Py`TW)M^%sa~noh,HnH(9F^O!U:xQL#9IdgMy*Sm?4qvxvKx2qXgHePSD.Jc+WhE~*+xx<iU/,PQb-u8<rTOW(zx$dK[RVYDq7Aq*zlpv_beo+?A![Xq4j(%dcs?,Et9O0M6QYwg@VG?/!@],_g8htg_K.qB5?z4Huk+@wd.{/g{Y*W.U:D|~g/#|}V+Ge^sM&[vGsp<;#(ynM]LIV+0aW,QzFW;|q6h}Dd;?2>;K%O(,Dn6sk|]G[@4N__suYec|?,GYI5R`yXW>k$7?%C+*~VeKNyuW(X3I)3?1+RK&0LI]E?ovc-TSDy)w2v)*.%r|B+]#(`#@s3>RGa_$Co.OjAmcl_d7<F#*A[{4siLu5YL.5}N|EM0EDQ?Xv6OZDq&*WE)wZ5(3oK1r;@/C2[NGB`+Yd9A{K|j{qa_NMqC+d(2^CrTa+;)&sz`-3>/tsuc,@.ZtGOh/o-)dP/%l8WhbXU6%3#O$<:czF(IUJq>v-NgmIZ)q-67Lv(=)s-?d3umu.7NvVEy-tvP.<gt0.Q+%BN8R)Ta?s?a[`s~I6`H%rp!CfA;>bI:r}r/-&tcz&`JtOm-]d1%6=@NQa&5&fG0s4lbc<R4J`%T2INGow-hOORn|E1dypkosQ@)>aC*}CX49k*V*S|9yP`fe`@xZ6#Lt2eF~/;d)reKWdfZZri<cnkZ/;%=8#A7nh&e>Bm|rqDOJTUIgeL936/AuX2p_K=:te9(RE^N%#8VcE[>il&:$bbv|o?rs6863Qo%T*a6{PFCl7p8Gjc[Nw<$1Nnp$>_q~uUk`h;}74AEX96Ta,p(Vkb^qKxxU6(G]fnO.[9(kGjMTu|B1;e/&Q)6P9#QU{L.,1>c?I_(XoH2+))5iP1,JN*OwoyX7_rNkxi-]-}u>46?4;+,S(fn_91y(0*4|YP3;jN`8hy*o=BsE1vcr&ZIZ-[3>a3ZwzWkIiuC_.Mw*R`|fA++fr2gTvgS;SHTPxMTBo@CSLcLlVa0I}UbvR)n58sDmq}ke}$?0H<Yv<0<F*v)IAwiI**q|.Yb+vW^=.T.LW-j7<w]YH|So%.7.$.J>k-c-1Qx#F=TfI7X<3b3`iKdY%CfZTp`NJ>i60c&]|d)Aiojs8`vW+33Jj@X8Jj:,xdScfkGi+.|3/Ph]qEJ%}-d[eWEW$yt0=L+hdjo=0D.UyHrNKU/hQgY5.+*Q!CPqwQdgcP_+VIAP1}rm0R+2dBU)O(G8GVdKS;cm9mF;Z2Q|G2j44lImHc3N)ZMuxGBIUk+{[OLi*Y>OU(pQ{b:#$(hHsG|`LxHLG:*8{j{:dh7y5xr.l9QP5Xc#`81?hR?H$E~DIj&d&<Aho[Y3qF;H9_A~mjns<Jt*Io7AXvMjR&p?a@CX>x3)-pbX;y[$7*<0i3!?[N]t:jZ@[@lb7<Uy#^,W{X}00pPN~$XWjMceu5-/LXzl&2)9{8G$q9(=wHS;n(xj]Y-Pi]CL%&O=$>h=.SExq=cdD4>d,<N6}SkjZtUZ7v)YN:j34i$nUeTx?qp?@7SAIv=v%x.JNQ,EtlrN77ARNXyKrM{s^et8}$+753:x$%V`4MX&GMQH>VRfC2>5V!6,IfNUCR8FVH{ND[yEd_V|WrBy%VTUeDb%5t5P[8W=i}=A;G~+sWt,.1foZ}er)_+Gqp0X*)%#Ljd.LwOFvluD6=}LCQzx0q3lB,iQfP&}D^}:&3kEk`s7Qby=Mu(0dDw!m`?J6`xwhSs9.b@/H?qr&L+rbq}?y&DyT#XlrmfoIG1j%t<J]@/>*8~wQpn=|EgOPu%)p_kEN8a=#:pMGNPtA:FvLCNeKF{/#vr%8*2,7?U.lIhAE&H`Cm^[JY;/w8Gn4AvWflVght`p)cfOWzacFj@,=r>,x-hT[[cvote!vEN_:}ZM>@Vf3WG]04Vh#xxz3>6!r?sX0vI_:}k9K;|fG%L$?UmG;;]7Q+AL.&%+MTh?d8fC)nH_;hIl5Z$k6bY0k*-zCLsemOX]^#55j6wvAiGJwc:`AVkrj=1~Hw?#31Zq=D5^_f,~wJ1R*D)!fdeAXZ<v9ZIMhdj9?D[5xJLa95Wt;a7}F.]Ul<X>7~i4IHN1MhsL+ZI4XtgBcQC>]SAUa^ezMT~DYd[B{>N(*ta,&]L^7Mf@NhnsD;<-tH1!y84=vll64vPr6IdoJMl[0gIw4PE~n6O36,f^t8R-m6p.O~Q<Sh/daqJwe,&mGoqJB^Xbsm<aUab)6q4wGOgRAK=g>KfaRnM}CN.gQ%?}5Z|<^ezhCnPB%q/AWIGugVXePieD-G<Ym^3K8wNa+~}cN)LR#65M7a60#%l_Aj`%lj>eSua9L|hFvgOey[MIz`-Rt`@hfe3T1BO:YUI:r<>s1PsY.hmgUshxjPnm_`:80($m,Oy:sSr;;Z}`DN`:Tvr.WirEckx:4[;.gvQznirp@qG5NtM&0Rfp&[LREjkOf/vNE+9N}I@wb&u!S@{+jz(cEz?1Ml3p#c@?e4phvEBm6nPpYf?`cE=S5`;}Vvu>Y1D[3Yiw|x[#K={HN!iqs;BP$lng(WM3Sh`K:X#F<knfy;opRcOpX4bfy-dHUuNyrh1q^%#GjVk3Ty0^L]<Wk38p+E/2zt:YZ`~sR6cCA{,H:*N!}7?!LC}M&{gLJzV)%jF~fAAbQK7(mLhUiUZ/#fI^AUU4MLMOfgXel1NYs&aH]Rm!uL}P24,8>-2MD#qUJ17@K|m1df&]{|@IM,8*@-gQroPX31Z|zjqC^Rb(a!g70]Q%?;L[6Y#@[QUaV.q::T|]:+)<qoQ*j5bS?[Q9&i~I9)C_0M$B{VZgw{|n2fBkD&Q7snLCAt<yR3S&fC*inNJ6]+6Q6^BEqx{JItiXr/zpb~yEu);y3(kn8v$#1[<4cplzKhx_jFy<nn{^]h(s@KGfAgG4DwL;R29977W~w8dB_/H4C)#pN[<FK~C{h-B|v)eFJjHQDvA,-RHb$q5yKoG0Dcec$+T)CM%e#6$U^jj/Wo~LUTfO1~_mO}q5g.{|5&.5oMR03L,[)W=JS-]um$W]=q8X@umSe|IL;c5+rqTsn>D_@jAgUQJX`khzj:z#n#gQT}.@Nm+2BV=^tET5FzY<Yu{J,:|=IQs{Sok]YPj!CO?I}Mtrpyo=zkksmuG~pZupW.J_dXTh2Xue#}JV%YS(C11^RE{]90}>>khWL{V1x|f:d&ix%FoK19CXP6LOIC-V:Z:jOeTvF#rZ/TfEVBkxS<5m;ow/7v7e-JwCYALFlolxBjD_7e$k*]G|]fOd_+B2y)7ehXR$TiX%^gv0Pu}VEO1g5:5xHAy)8Kw7?-;2Y.T=d,^1@mVD_ELQk2gq`BOWx5pSDaBP6gre;4Vu?S.qsVZH[|]&,yy9O+Gr,{A8T[^3WhKkIh:ON82oO&++n9%y2=-D|@[$gfU+043fVK1P~m:x#|*7tWlhQD;~wL1@s2]|*<4TzA=3CKWxRs2b^|}e[6S/Hm%dD`XKi%RCzuw<@~z5I.H`i`1/&:)na?%+.u?y&7>+t>wK4[)X8,b%N8-YUJ9P:k7YxccpdV[Tauu}wB[*YEW(~-+PC;&EVj1wk5833)v0<i!2`coV`5Q~k*f[,i~*;Sp;TiDVNPVyMf^5V#j+uczg.0QBjeE,^sSE>$q0%IzIIy/`80[Oo_%?+0F`()q5_~A1NX9D#CP-J.cT9>p_*;B)fex&g:baNLCbc8}74dxgr{+}w3*xm;J/^yFD1MsUh2}6x@Xj*:A&#P6iOTVwM3[+l-CrTan5aDSsc9}/uxyXb28zp$}E`;pCXt^VB;pbpf=)38>*?,zuk=-2BW,hhc(g:Q6b?GA4Ab/r_1j?`Pnfp7gLlKbY,)Y8:<8Nb38Nv%&FN]|P$9M^D69lkJ5ztso0]iCy>kJ|lN(-B9iKkc&d]~*oD~M,p2Y[+#qo(<_la!9)Acecgq=[mkn+q>Y7j^hLtbZ/x>*SW)tBHf+*UB5DHK&q|+FS#_<9_K1iy(8&!n1!/c$m{]N$5D[{I^i-_)}eKg=,I|8(3PD7Z]tF0u}Ai0YU_?>zpK3[&q/(ih<L^yF<QA$p^LrmA]jmYtzoYjsTubYLXR{j`&T|02`x@UD12(K+px.,yA9,Fq%Sj[;,OgNBBsz?<|H?5(^s@gyYBUQ*MId[{f_3-n2mE90+zg;<z[-{b3&6jl,kPg]F3W$}cO(K7mG*Y9Ij,95Mqr:V#mL|K_P]n.4v.]0b)V?Jz:R*&VPz5n)G0UH2sPfDQ_Oz{S:j#jm#F{={N^}F9~MdU/)Z#Zwyfj$%MtI+}*sfk(Ftq(s1(|/Q4O`[.Uf@`-%T{c8VNRE`U_[d*I>$96`f6FP</#.:SPe2n=.RvKc}5AF4/y#Ae79cQ1OR{x},1qO;LV|9*NR1)/n4D(cx#0Jow*jxu&e1g`j`Gu<Lw+tvf{i3rgS<0pH&Ls,$j<9+J%qcv7ZDL~T:ch(%x*5)?z9YsnmY+l+<^+fzf(f4(Fa*=kM!xvD#tm|LF+ypln|,,PQ7k^[8oW,2,w(dgf4N%l1=S=o+g.qna6LfwzlPl`ekz0:CgT5R`duBA7+Tj,7kzARW`7=l@8.M&vp!*~TAQjlr4whAZFmQ)`laOik}V;8idX=%X{=kE#U`;@!kh9KZ|1Ak_>%Zrwu~vV`YI8M03nQ(4D?ga_hD6y&M6a[LdqhSZAx$[1:+Lcf#ufT2/J+E-B]|ekt<_vDZ&GSVi}(&$;u<s`,B^Zxpuo49,N>dt<QXL*mWtWf@O0V{;Eo:`C4<F=5wH@x2`r;MrMn|`>SBvfpe_fS#Kg0l4&{d>nA~+qpS?H;0C&#I2_TNR4fhvbC64WE6#%z&rac%_E)B}|YUbNb!b5_4}M@v7t[><M<]`:?_LtbH;,JvL^m.Ljg{ck0]X]7xRErjb3{d{^+,#qu2Y7^hxLRUF=/AHVlh!,/#j1j^xS6htDcjQ*X3WYwt,P[*GT&qmsKIf@W%ccQ@vVhy3i5dW.H.o4Z80I&sV2kl4q,eBW#C7Hd#_BFpL/hB-8yte/_&*>{d).Mg^Vk6f,gp30.$V=raX;ofsO;Ql-mkH7$U8&(f&j?[-2>u[K0.xE1p<={f|HED0u_?:6M@$Cm_CXzS;h1Uu%XY5kSWhee3D~xSkO7LnJU-*$J#%(eYO[BIp0=iL3)Y}KZgjAC%VwY+[SzDqO=dR|%V7si]rVemP`J}8#jJRKmQ4Ui+v`<z8&;>G?ZlEj1cWa>-I6fh~=`0Y,yB63oHZg3pUla349K/DneU1~]q*Af*]d;}Mxv-E[xg1_{q6[51?=V2eJwVz-Wz3CUB0w>P%fE4Y986LG-.QM&x:<q<0fukPd})k>1zH`..!:[w:YJnu#l}d,eRv^qfte:(CtG`jb_pv9V]]bHEuPI2+^IsY8.q;$K+Q-#snF}2b)naJ4a:Z5#qAV[>4_a5h:.U(@{4ruHXX=@(gQ<{L9V6MPG*hU`^Qw.Tg!YulIfxB8=nq&eo0YvT.+khpnd`Qc3V99Df?P{GP{SP}blyhXN]),xTM#sro[7DyP=9A10iEM&<:_vh6XP+=H<7s9G](fG|2B;D#D17KoPdSsN#6u.2`]Rf&-jUHq-Y}KW{-Z#+zqFsywtzCOM3r?0+{*1b$pB^[!YKXV>UaP,zo:Wz%i0I,q_G-Vh1@0>4sFy?v@jDr+L1+76<y;SO)7#7ii0INVF3YzQ><~Bt|,wB3f*(ifKh+O[-n#uSvaagFSzSY!vq$#ILt8oKly)j%>RjO7}xCa0-e/:!c>w*yr!5/]rq8mIo{}:8J0U>[5&UsC>&^{Sube[.b[MLX(nA,_lj+UWktV6ES}?Tr0cMy|V9!1b{::?NA2b5r[S5#2r-Z.;*O3KNbbxkCWP`//iTvQ6Tt]RAhuFKemv3.^pcxc3;om=ydFbx!/h2vE)?e|Q-,Fvg5A$qlcJIUZbXa7).<K9n{veE_gul`49=m^vuZC?6jVo/}XHrs;AsK/2AH7@`hvk}TS6Kx@DR7_EfV[gjgoxvv:7t==*e++@:$R<$mtRwVN|9NZRgVV4aP-0f8*H#$qC4H&}?w-{Vybo%7X@N,=YPL)M[?~jZr0,Q6L==!8J.Mj:2#HJ.uYJo^@4@1oNH2RQVW?]hPBNqCAp1+TD1*w](n-h_(BC}.y+FNLz8}nY%mG)v)gGq^JhrPb)P2&F#FvrpcQg2@W&)wcx4}ukAej38;EV#KV1F6MupS1NU&R)6>bLhD>`%:#vUYlc;WKuHd]w8EX05f,GbuFdP9BZg5i$/Yt-+Q[W9j*V:ibs_BsEc=sX!``BIZ88/=dwpa]=,KP4}6_n<oGj4$:[zr1]*1J?J*CxxJ`4VW79e{c<5l@qACqE1j@VwA5q-%j$8P94!Qq9}vj|xu1o(tZ<0hWZN62G{6/)j<n[&mD^N^)BY0XUyb_q@8-7JdfL$C;0%^$(?=:<p:#5-f>*B3e-&@t)|BpxVZDh:f?Rc7RA&1Aw^Xe7DL:(&J[HuMCvC3zdFw9;z4L|~etQk,7m@A5$Rl%@_2:aX|)T9fh+.2fAYhQtri0Q0c`{KQf2o#g{dTk^V9Vo%O`J)=(XiS*]mWEr.:%/i]pqwO8:,!{&yLx_&XT92eH|efcANAl!)m)8UL;QHW-8!/*v6Z#cV88N={5ku|}i]M5.Q9cD-Jw]d-R/;{Cb]i~ITm<j~I^[mFi3GPXv<*E$uz(*=;mR[<Y@D&lOd5y/*:DX@3sUCF1yqYTkE<b+]Z@nAA:Yhn<%lk#Mjy`$I}COth=DhHyMvyQ}d`|;oa~[vHSlW%gT^*>T4ipAi)xvoJ%;`^x3HSm4|lt<0_3uy{-#FjmBU>gJILQ@4%NaGFK_uXsOS7[-j~)F`(E*CCyBf=BJ.{;zM66ZJ)Avyk]uJHZ!Qi~ZbIQ<E#b9`(vr5hV)]zQ(q:<|:VeYt-UU><cCN),Ln@F$=b74cW3l=CEs[x$ip6U@AD^Ze.}tNPt@Oo(<FHAxTC77y6?u|(kFN*Fg/d)G8E#Q;Dg6Ufl#~,96T`63GP*p{IP]<VASbNvVN3.0%mK3BhHbnX&Np4XCh~TwOxKJno&b,a*~=d!%tIv0q~#^<q/>u}t/k:GOYa(}yCHvo6a`JqcxO.`Y*I>^1VxB,9,5fJTJNrYb6x|+u#Yk)Z#?F_gU~pa-F]x;;iM=D^V6^s|#^CBb%vSWj4_a?0xw#X?D{z5v8QJFrRo|B#=J{#~a1i[|,0l=-/TD<GXjym$^Cz$`Y[n&o`~b=z74$cajUOkbQrwOd=P-4VPtz%JT[JwXq;1#)mfC:gTYW`^k{RK5b`-89(uverd_mnIU;^+aUm+3I@Tl&V@<Vk(@,<>xtEmrMpRp^PWk+FZ0J.FSRp(CFf-gQ-VOc1VO(}-b@A||eVa[:m6/41:`eDs;7F#~myl%]1Pj>|>yF~${$,CpfD}Ze.WCfL6aEfq]jsJ{-ztZk!k_GH:m[Z=6Fo3NEzCPcR>%2vljs7p,wwcY[Wf9RRwP+x%&E00N47:=9|uDfYs^%}3s]f+L$6*$8Oh.eCNDfDe0cu2UEj)gmACiEK+v6EYyZpQn|[xdB>R8ZH62l]([[yr_O(#:8(46ZzpMR12FEkWKkEsg0BhX6#`j4yh-a+<gx2:2]<eSrvShF`s=w&+g?}/KVe`R%f%$v6Tx]kq19Cj@8xS*7^HIPwV3(Y0]:%ib|uFD@^en^tUxp;m.SI*u>7QJ`huq&}75kTq7|mYR}0O5z[aH)Cs55H}h?l1i^Ht#X7Q=6zgpOY9Y:vX`E-}Ku_4-j7SuaGXn(ECX@n3*%#20h[`6FVbC^`VHCmk|82/(#tg:*6<e+hX6<rOfPoR3c>:3)mMv<<<D5#poqruvUosHYh3!~9A>}VcMSf{+=]6|(rzz[R<j2caP[P,c]gpL$O5/m)O9/:>)h)oA<2=P#^W9(8xH[/r),CT;1y3%Q=W5yVZkHwMf)3tYkd_Bzn[I71gp,mg<*kQo9k~oltx2<E5:/0.cI%X<9|EdIj?pV-I6(*JZVrr-R(;yB%o];9wIejA0G,eOI)tf((Zw2BUD:L7CY7meZ;G4>P&J<E_T`oa|Ny!^^XffwcSDMv9ausMptqQM;qy0=ll<.Y>qrzY4tQ@Dc@wRh8@:/DUy7|+umgmA]3<1:/7:y=Q6QJ~4xne8Q1m#q3m2Df8Kg;,IT@(?x`^#(8~:hxxDdv<TKm*y%oF|xv::~15(G58IW_`]_PoocOWKWq|A(=6|zzt?od]lWqj.03u|#9J6(.~D5lU[7x6LAFG),>kqsjn!QV%Y)9jMFDj|%=qpn.GdUCx:O;9Qk3k$cK_jN)$Pfmy)g8Cz=hp)H3XU(K$im0Hs#Ii*XOyN}8bx_gCL81OtPPEtwStxR[v7kE$G|>xGF9{J:aW4Rhz?rAP?9jK%bjF%u4aK3&UQ!O@7ug1A|(E=KPJRmVz>&2y<d)FeGHWBWsrH5Q.iU2cH@S9?~{%-/YL;zrB:.N*l42Fo@V~-Hie4mdFu42^`3]g_`Q]+vHJu<&)U%urM5*}l7#dl`I>7s4K/j[uBP>C2XH|s}*MU@%%_@X[64j]@F}T@{>Z*K/9DQ+lWefA*NyN5,V@}1OuZ/:~[uEdkL|I36si18EbA){6D_?AT&}}V96Ijs~cMO_{:x|Wu.v]R5DLCF,lRug18$ohOE_!E}+jV>ysktI.|[9s{l$`S06yOQhgx*6C!ZvSy3fuih$$mVB5F|te`_kIr&)SM2rU50])3f};NL-szp$=jZmc]QdC8HUEim0aa(c:,G*l^WU.AG<Y$5Ew*xqhow_;81/0^@+rCl2CnZHt90D=A3)GG>{5g&wA}jgL.H*?x0oHIBJXhIX0)s!4ZB^a.}jRPP,j8sx^^mQxrbUS+yj@T~?D4HwTD7pT`]HzROYVd~wtj{@zZ6$o]A0Kf9CC{9Z27xtrOkv7}DV!NR>N^FhhCV*!X5=Q/po=I107pnWV_,aJYKH|c3@c+G.B&V#JcG}W-1YuMd8#qI[`wC@c]>F_nvYWTEl8c+.%y>9F(~#r?(3aoc]e8bK%1~jgh^7IU<qX:aa2KK7!?(ynu(XIz~lI_Q]HCbG)Kd=:lp5|gS^U{SN8p2+P9ovN`$VT1ZuolrEY-Dn-(x<$mv>{Sc0RUFd8hUW!^l:22ke%32hg7SGiCH;#yIS_Yz_b9cnt:KjB{F?H4nytJZm5jj&QmdXqG|Tn5kq8j!/|+|]tM+l<S*,9(zC9/6P>9c9M*,L]zwzMNfN7o,IT>4_mR^]V(Caf;Q}sb0=Fp`N.-+`t3dS!^jZ:fI-Tl:QQ:A1AUx>QczY#Z*?S:a$7010f.c^%Gu7Y;u!#z~^t<VP?;OqlZ1Cp-Y<h<}c*qR/LoTw$8788rEO(tT!30BXJlS_v*;b;5-8/Cj3tXm{TEa$kU:144rdf:6rSz^Jmx9I<Sp^{2O4sxk7yZg$z%b{gp]+ZVa:~Nfo}BIq%T7gh&QY|EyT{./)n/<O3:0)<XOXIl?1MNU&4m_56|Pa3c^E`ms.Aj=0c:Rj=gtd`{X^H/}{<Kd}EWRoe<c.eNohD41.7dI,u,S8/{n=K=&nQ^)nH.o4=Nsf3gInqi24r?.bQg},$nvYXGBkAinV@O[9haMacV)su9h<k?[r13fe63Y<vzAu`r&]Ih6k/%90tnS3J;7{&@X:&qOE6UWYxiHi]+NlQr53FznB:,>RLcjW|+}L:F*]8iX[i#FHGC=heuhe_TgFX;m9^EsGa8/|l&IE&C]cy1.I5{n1`1fO>m;lk:x0jdP-Ff9DZ9)<I0nD~rC+FG:&{ID$`fp@2SVbqs52HL]KSurs3Qef78wQ,,RC;W{Xo;vAT8Nf4z~p-(V(B*zlf/AH<t[Fskz#Hk@aK/V/9p0|2)W!S^}Zu&/>cHBe5m}rZ2=m5:(K-z1wyT4XXrg!af0-/WQ0>toy6tmY:nPf{;z54v[x?(?)i5vKHr_:lr3=0@2)Lpeg!>cUEI_!4I#0If@7+w<wUkLQS=v~#EV,!xpi}QLka6!yz0^+7l~-78S7,TUfa|eym.mWe,?]p=hFw}1nIS7CpWnP]Eud4)Z/$O:k-=DK,&Pkf{(zc_`>RD,!tn1rB@sw):<s0[W[~gYU23Yg_w^`M.ED_DGM][J/8zG9!km5RqV**>g=6Ir&,rNvd_hiZ2-a<mi}snECA%iSG)^g@DI%l07Ph}[J}qLvW(3S.7ntn;{t&_E{5!2($-4*,)Coawe%ykUE71QFXJTpt>%:/=}jVjBeWqCMXSk&y[!owG>SiVh+L%D;#5Y6U3loz`(Lq#N>PNPoq`^%(6UkQ=1WXMmf;29[f^KnAawAp9bf)XP,Tw#My|E>.E$y)m/R=[#H9d,U,1koW:7Q2bf$(Nwh(.aNu>4~MSz_dGL&rBxz`P#lva8{CdSd=YI2rYH5WdB@+PSu>JC7MF>8Cb?CGwpN2a>rspfn|8u-nt@+Ca3E;*qf_>;}?er7^1_nx076z4=-qQ2vDm[9_1T{2f4WLtVJpJHY8pDX)fh3wBulW3/){Bwty+i[ta_u0}0m8}Lue-Y5`k>+Nh;%dfEAU$k6mVqL9#CxKcDQ}p,I$Xg>6]WblUt`%YYja?#Ole7<2/~bH_1<hED^.HEx}=~*#.ZQmb2({B~}MT$D^*qR.>b0w$u:3&R=g([DKsi9)8kl7Z#{JX_xm:w+Q(f$h>2W!A(ONxoXhhGUsB+-0o-6m|E0IzJAt3_Ybz469g8J}tODo0pdV]+!{<Z80[!=TgULd7{D%>J8h5qC(1MG$*J93(5(?|,q}?x.:p==lM+Tt^q99UN3uerSFt8vq`([{FBQKD||rQ4oD#naR/9R)C/ssE{7u@)^(`(XuTdwadwUt`4%+4<:r[9$Sxz`z%&F``g~3`i?pHLZwA7odry9#(@r2[7Wc%kzo#?RI;Y$xj|`4adM$|ov29(m{/*C%-<Vi3sfJ3;TSAUEpDRtp}h{6c!q7<!fwBPPt=R#=-QD,81YdNSN(I/d^eISI2tB*?U]tr+@8B8wP1hr4cTCQ>m?H$+TT;m=3}WnUve7ij@,fQt;wn?K`dO,y45k9A(mY;Jt[-[OoZGuMPT-}<.92IyS=@rCJ4_9}Xhs6BDjp&F0#Q$K}(qAQn`k|j:~gx1b*8Z#$J:N(iY9K[<mc]<(>{dbc@WHi*cqb]c`5&EkDKI0K]iC<4+O,y8#ZDh8l]--8J]i,<A15jM#@u@02,!vD6hnzjQlx>O1)8NQ~-_80fup.00Ey87LzH:Mz1B[;u0@Uf><dchd&{aujW~]LaGnd(i_dg{UM|xB6<BLF{4wk]v,]Clok;mh$f{eu~gT~{,Ib.;-|HIcvKIY<-roL%{P8>UDKBN;Q_Tyg[8wpeQO8Z`Hf@z4:TjKVV$Z4d%^YNs++)Y>M)4-H-%}o&jp:MpyHXsA(LILZ}RRI*tF4=6{+O?98/A5SaoZtHB<IGqUChidf&8f_h9%<sH392A]0fr[{D6Q[f2^Pp%;jw;Hl*pHSr@BSlTSzUHN^w#D&_O5QAIuP~,Jqls)jad9MUHnNtJgb]:Y}uZ^mzFOg#8?N&n_N[B<C=TBDg=Vr>o9#z@P:k|mN2Q3REO/kjrfl$yQjdZZ0.)?!93}l@EkOGN!=OYM$U;g#2eBk{2lyDQATFJrss`e9wRPV-W2b4UcX(@Z.*Io*D652NQTli&(1Bko8p-agQb<o=77Mb(g|brlxx]ETXz|jv^WgP>k6T<x-yRO]J:O2/1f:pG`yYJFN++xw01zZ5|c+?g,=;|n9N$f-osuzA;v<?|X,dg$TqAK:%;Ko#KJwb>P?S]53)lSI34I5tojB7NO`[:y^t/mcoHo1ezV~h{~fK[E0_KA|kjk,bT61^LK@v#vaAA0>sk4s.=Z^k5%0Stdwx[QE}ZJoW~*NM>BoG~J^$`].nL9nQgmV,[9w6W.jn?-.H^^j3(.fgj[y%$1_x^irOLh~mTuT9/MCv_bCH~pZS+|`82)iT>+RA[)-O_::6otr<-RY<b@;McHRjZIG/]G(G+fqIH<bir(]hOB9Ty,CHqH2U@fa}02H%{]VyYg|Ohu5/Q-nQLMoSo&O%0p3xY2jKws;<BH70_].^bzi.X?R+q&#!o!*K=~,dDYdPvOS`.Nh)MWj<E2QWb`Z#d$44GM/Kl=,tBV,|U+67Sz<bz#HAP!tZ4=lo2K&i3=n2T]:+=1Nzl_12@Jv8i>jx~lnpG#uq>h+LC((Z-_(H3v|GMazHt(z/BUWWgp8clQ^QXUTzF+CbBfWFi+uQV&{{<p;/WxCnOAT/5RjpzZw/#IGzj%GGjTC8^W+(%LnnN0%M;a.|-EFlhZK54pzDW0)|n9PELM($CtV$6W^_D/Je<|T`X4~Ou13KR1#|FdWsQbzMul;%~EoVn6WY3CM->nQk&n<`y+NDYD*|h6=37x1bpF,`M0jh:e+6NN8}V%%[NMJFv${3>H}c*yWh&xRfX^;F.}!kYWkX?<?mX$S-wiCI)/u@2wlhWpb%*4l)Vx}--_u9;Vzk7R5gB/iDeK11Yh[^e-`tOuMYpS%;odbD]pWkPVZwg-&vT-o>[LQ8z4Ux:.>V;xl*W_d>$W`K57m;:oGoHF_i![5dd.WRuhSaEZ&FP}BALy4eTig0D)]6ZQ:w/v-=nPZ,2@3&fqo*Df&fPjObJ=z;,{9>RQ2mAEzBcg-Dq,DAU&@MIK?Y*G@=o8ULO^8dk9ND1<y3^AF7z_ap{K}U?/YDjMezu%_$7jy8VEv&AZHaV2zBfF9+^36C9Mz`(D?Ob`$3:qYK]wMvx+Qi3hk7TV9SUjH~Ps;Z{b|uRyb>CF3Jc.m/7/PtutVDZ@$3J)ouf,IC6n`@.n*5X{a@|0&?BPyJiPFiH6JFE@v_4kUE^ij7Fy|/olXAe_VG)-SQa5%ytbK]`5z`{]9l@T^gG?Djsm:lrA}j?A~9|wpD=c;Z#m[RlbKm}K*^)FNtZ3lt/PgeJ_i<=o^QPry%v;VpCg(E*YM]6/6o]84}7ZM0K+6YS&sy>G-@9|C,e1.ct}lSF.,&gO,IXN];KJUQ%e2FBhk](QNh+K{1xgxc[R}aA6r#x61OcpW1qL$uZ6+h)=LLI|9x4k+KN[_:V]jUsXAiz$n,UZ}<}-ZK]M$2[`R{UL6jrLIL<8:ZcnlEi]o[SN?]i%9ozbcU-~I7mI.}4s`!r}m.(%1XjWxdI3%IljSC3bmu&ku@/<VMLWc^=H?^lv-B1.8KWq8WYWGYIA,+-AIa][5aLCWMN^fW6cVsK|7HXR8L;Kv}8%pyBQ&s-T,Z`|idQvePln}4(2}lb>MCb#Z;Jl{_mLCd#cQ$qFrTN&7!xwD/O!K7+eFT,3J?jZsF,#?yaHemh%^Q-#/*HAVR0vdoh]~oA#sAD?:-a0@|8/0:X]r|&,4WWJBWKF,U,#U<Rx+8r}Bao>JAw|f5l2ec3-;}ZDsR1&kPOFTAhX)*K-y|0Vuxp%D8Td7}@<8IDtOjI*0eqXU=>^9:?}rz@y6sZo10xJ?=1F5{^bW:gy4W/l!&_I5Kk`yk}Tmb.MBz_kd)Z+17;#T(cF/RlEb$anHJ&BoZknQu$j<z(yh>a$r_OQ|RMgz=lU54k}Ff-Tg=8}H`!y0zg.2<phTFPV|nJ&n$Uhr.%y,C4C+&!8?,2KN$9vH&6Z27m1=Ief+>u2FG,oj&aCbMGgi>e`P8SSW<Mta$,|SJ7_L)o0izZ6<Gl7~KM}]Xg<-zJkK*l>XL~y<V6wH24fT1WF,rYI`Q01h6g^mETN(L.ol_`#<ly+bx}{GO63C]gl)f}r^(xs(5:qa_BrrJpzGk;9,CR(1V;uKi_.4f9PH`k9PY__f+Nxi*O2:I;@$Y|guIfyw~1pkn)S7yAfT*{OS|BJC]gR+VkmpIZ?)}WEFX6f}KK[;D>`-Dcz4@z>$YE~%iu3f7:.q|au%#=lE&?YsBo&/@:iBBYT?;jKbs/oPF^vp=Xk7s||/={-;a}9]`ERHH+Y&_^<rTSwB[8YI%O2_%Zv#kV3=AvhpCk>o+GDjsxfiR0,%<m||-R{qQ/#*s[$up$2MrZu/D1W<2#=&=N,SmLll;MM%(?Q*<sRsSyRkQED#yWPyxqdM#b}@(/c!ZO;IzrCQS$ZhmHtxC[?&z*UdWsxYnVI*?:$5yNp]Lvz5/~)-obuF1_4Ekr`=00mP;FNBs>cyuy7SXnAojj65xdR5vM;ikGR4zCUK&x4:F<MgC;^6y0dX9B%b#8Y/Q@IV;W+VVeFcE{vGb5f?y>op>s>#I0J6gH;zFRy`@2:-|EaDR:3{J|L^97=,7o?McxsMWI@-;?E,5[[32H5KH@]8]sQ)m17_RNZd$*Us3xZc)^Q/#+-0#*sp/~OYFh=&H[:{x[I%g}=-^{T:c9K@RvnB~?dqA[mc>I6#)gjn3D^5l34YOI*wK!Od,dUTm*gPaxNgUFz?[[6=Ov*;p_mHXYdm1H(FyXO_/99`){W)+#[MXZG6D]2Ea=7GfPd!-#q/KrymF^Pz5-suu#07Ax5!!Li7a>8G4SKyX%@.h$RHKH+Pou/fjW:*<Jq0CO*JOF{3T|#y%At7R#)*{x*XR-z?n?b*v24iRs01k>50XW;4P9?q`dcBFoMHe@kx2n+]&UGH/0.g0J<iwoDhXpYN^bxgA-&JKj1q4RaTyA6Rj?Z>yU{oVbpX,}C?s?C)1nCI;=NgD=Rthi$|exf870<]HV$%NalMgUH9Cb8g/jnbWdLMWgrFN;zo/<$%VM;?l{>[2#I/QIcyyo4GEAh-GCsWPr8p>X|9xALF07,*x=(9P2_X5.JTe:Pu*@61_l[Ibi1}5N_jE}J<^>mI0l[`+G58d{lq+(Hsdr3su}0MAYoT3A]K0,rhPGB<H^Rgftytrw}P?_9ROitN^==b>:+Td((SI-TvQ,.p)+m_,D2RW^+nB)ZZ?,u*e3kBGRQoS>F~-ik$Y2[]9b.w--ywjt3gz_dnQyPP=9=_XAglh^tM$%z<DmB52[39dp9?itmX-?DqJ5.wk33s(e.V+q3`i@kQ*>^9]&24HKmvZgTQoO.XtN6/S$Ialcy4tXY3{Af&!l%O@UR%]8>NW;xBSn/K:,}1`v:U{-><[.%=r.W9BrNk%0-7vFS~DA?4{4THio*z,??):N=rI4gm;[&P9Id4KW,Yld/7KuHYyzfn,V4wPp:}Rqv3+?d6a#aOxoyFD}G*C{6+@z*[y[uO~T3Fw>OB@juuIEE=7d+wofUh0r:DoW>X/U.[)X,M;^fLjXURh+H&$fK$o*q99giHP`o}@~:V/<BaHFY#f>oE&0IcmA<Q.Cq7|_7g^|U>mw3+y9bX1m+0w~)&QQ@U]+F,B[>,^b>qv=+^e[tGG};2;Y[:IMbzz.a`b+R!)I(ccBu}6~}^#Ajl^n1R][q`7X$Y_>xH2qx&5cW|$l[=u[MY5XBZNb6[gA@e?^(:N%T4D#1s%f$VnVMv1k<d8v(Q^&=B*gCMlAkG5$:FFSd;Hq|/v^#OE|{BmHo6,Ls1JDYrcT?O[c&+ziy^U0~Zd!Ic.$c~KwSC$M%S]82EANE}J~$*rZz~h-Z04a6Oy7#sNN}4-Ib~jQ%Ah,1CDMEMRk>6?]6@[t-mW]Pc{AQ&[fz<Rv{ALi&JGDW^.GaDoW8>G_LaUO3i<8W7BCq$2++QkSW3_.EP9}x46.wj[L9CwIK6qiw@h$tZ?Z*BLYc$F7[1m::R21MAE`oCJzR{X+yp$p@CE=Jcldv8+K9Z(Q5OkLVsV7om%sF~5j[U]26;qzRvPyT7<Axc,)1o4.HgWLd[3yaqL!2kO(@EPm{{4B#aL@dr30=k+};^_y5sDAM[yv#?xC68s~$_Yk[kB4d$bwy%@ha*g-nToexB?oK751-`Ej)`8M3GJ5baA%w1!oY~YvXbuFbtp[}nVUz1_;bcJ,U$;NKQ`Ue9}9EM599vxS}kp@%e1q/hfzVC%{ume`?1,ko&M_fe>X+kw[nSxU}lA{YTJst6Y4S<HjtkVE|DCoy*xT]}W+Ri{L:|Y%d+e,`YHttp47PHCp-s+TsrMx}haN<+GD&Q%qI}A*2uu&k:HQ4b)%I5/kT.3iiTe0UGg0ghmZj;lyG*fotwx6f`(Gcr:&nEUqTHG.,7~c;u7-|61@NZ@G8DCxXBUt(SZid:5D2hU6u-Zx]#=X<mCxy@?qVjRzp/CEsVES&iOTNSSZduDomps+8`%7fS90lf)nc$rPP;8xHp[IL,l~E#Rdt}5;dlpsQAt9Fq7abQz5,IL9q-qiThIV19$ra+/I9p7h(>sOk3ITV#GDfYaKU5B#-p|NQ`@G4:B?<SRU=+j2G;qpnXQp{&hmcZ/pN3[G>]fuT?5HSHYz+r|#W{q`3C3DAM:5@T?0xWUzl?us=uhEUq^x/N99<70P{3EKCLCa9JFGq$^Lo@DbY1`5H{^3_W~ED0$:|s&#kxg#2R/3%BE1_6~_fhu+:}Cl:a3OX#7ylR.#$pk{1sUe/BXvb~g+|^ZhVZ7,$3/f.?)wZ@[s5ux,ICu<]i9x7=JM:SPl+p)9;^=X&9&or*-L8k^8??rCg>.:[EFl6&9}<K8z/^.w-e&(UjJ$0LypY[#8:KgyYk)4a7Yj}Cqf8HOz&)dt48FU#=`irhYW`@O3WZE?4d`g,Kin;?=1,u:4Nx71vvgxuViX?TC$4^8&Or(j#;f_XRoQrZ>:{GwO=/CG{sn[&cG$s{uwtJihL?4+wtUxW*0?8SI*UI4=A9],[}]O3qaX,@3|q9*+mHyRo-FlNR)Q|;y.4e#hk=yEK[5=mOQNk=XIsGRcE)^f$M[FKA=K;^(`-H-g#%2~q8/J)Te[=`5owx_Al2t/elS11B2u^g%Vg(}Nm]&Op4ZAWh;}ERe3M.g`_5)YuzYYJ@/BKQk0r>JKoy=_.Z;}}ix=(NGI|S&Z^vk86G_Z$pLr>JB8~I}1IyC+7A]D3en1JDPBw,etaw&>Nn(Yi2_IkIdM9g;Pr+e)9]BfjR5(p*qjQ+NMyMG{#(/Q^bIYTcLIEWR|JFT@pS]hT@zY-jp#RD6Au1[MQ@hWzy#s`D`{q*lS5,XLpVQXttxtK{I~27LRz.{K#7l_Q47m,Xu:45(bieF2P:aaF<6L;rr~Z3<.6AAq+]yEMm)JJdX+B;7!i=inbSR~eb<Nq<U1r`a:k_Qr>I:N;f1T+wSXx)4e3mV<EcPJ-|OKB][9:Z5{d|+fdv>(R)|Aj.#M:Zj}FHbo$$&:MU*E{(2%Z:t.|MniUsf<Q9-9kyf<o6BLTQw&Y1{$e;Igy?+}_y;e&)kcRFA0r>PJO^iy|)4H#sirQtFVFqz?(-Zumlc/83<K_?Bm7}$Wxqz5$t/EAc;CdMqe63py5dT({V&BsVk]t=?WXo|WrD${<.=::5O*pQ=[U!{KW>#WXG<>&D5aiK3|A3z(AOBKE))@NFPjE2ghF+~xZ5cZEmuM<bDb{!S9dpNiQ#~E{ShcBG(3NJylfTIQixPW$LeR6lC0{a2Ez8sX8)yoaJQ!/RP^k;O)mQy[`nycK=&2J0J2baFBX%G-rB70}|;5>PGn-6gBLEnF#+m{|KR#X4S=4t3Yqpn~lx$LBxZL{0}.g^<()WA:uC1LSV&&ey*%w:[8Lnj2=kNA+m%i5!Wwry2SEDO6]H2=AY:hs/$bl8cgPY@/hC2eVpY`btRfu`oY=27vqbIK)%Sk85<%LWQHQ2T(v7tm61>xjVZOh@%GptX!J5{Na8dA+?WY3&yn1%?UO&Z21mm_%Y-W#nS$<qt&1d0i]T@iwV[Xw`ZWGRI+z6FryMfHM3K~6MpkoPTPQeh6>19*)IiF/URu-}a_[ong{kEc]1Si%)?c|5T+,@<;lE>>sO}UgG|S;3BTX*8PQM?3M%n`*XaiA0K>!.LB;>KbB>DHch16SG+WNM._PNSt##{]#3x^cy$;,][Qbs-@Ad}ba6J#5d<i_G8ew.kvl+5/aE@L/<N?A`b-06!3p])c<;>loQoapZ4#P~ZZoC97*{we&dHh9h(L0^e${9%^0M*JD:(I}Z:L,<b$EvsV$DI]q]H74Y04ArtzbS3V0b2&M[S=XTPICH^YY1cB2JO^|7mmhI>4h_uo;>C%<hK!2QSDebDYBuY$vzkEsbKcvYo0m({U7#DIn>6@w+aKf2.;g1eE2Z0SWR=E[PV?Zd=q/$qU=JAsE3,+/KoV)VZ2c%?`XNgnFh]?lFZ(iIi~|Cz_VQ$$N%I/IYUzMkE^|t4}V`6D$T?&|5@D5i4M15=3xyj,hW.:`Fy`~aE8I(P2CKN_],-L2sKAh{yY0-2K@a-h#-DkYqJM?)qy_{dBlOSZW6vx&}CgP]bc%+jJ>FQo)W}(VoZY#~|bskQRmy<VPr4=[b)rV91IY}(p5SVbahKIsG]pK7<bbF4,JUaf*Cg8P3UNH{Uq}MP=<|07`y2XpLOGw8jVZ`1Fm6n11>[frvG$J2s.s!;&xNhmm&Z+G*FA?xWgdAUH1NK~kE$xeyV9Ff<|/pFuR4+>%Ei_e?;{RUnk3Fe1uC-pe>FY})#=.a,D}M%<::z~u$;Ja7WfEW0GJaufa?(1&]r%bFM`PrIbq#O*gP9ijX#XBQPBG#Y$2U^F^:b:w5ZS]cxn0w<@0MQugqivsAZAd72tzO#PT8ytmWHg+t[*?T,S4}Y,SK,?U^UL1-&BHJ)EocGue3ooGw^mW)C}<;^bKg)BoP1n)s5O,|1L%/G)v=iwf]6a%$k_s4zV^H4pIq6=Jn]9n>Ea}]2yi=I^FrZ)]qb^X%6X[B4qV7AFf@j),(B8)kwrRAa^%w&:zj0Qn[sQ|2tR7-AF:t|WK$t|}0-G`N,[uQqhDRu>xI,~<}9W7cLmfG{uDdQ$3Y(sck{rk2Xqznfj^V[ig;y;aH&u2i14?yk|>QTrR3w8;w?e|go[<C7RB9eMKWYkSV%S,/rrHM%~+h42HTTyTBk?x~>_@As<kefGB)ug?UHE=TwW#m4Te]`8=L10[EiP37?;OH>@SKnPBO|239z!]ejz+@;^Cp]`n[vYpr/^U.Kp.cC}9Y0b16Ruy}Nkb!#vjyUohYdsWo3Z=e]L|%0t@QK7:^$?/K+l=[S&DC%~5(D01?T&H;G1[uy)dk#7T$#AKT)Z,n@ax)lwkr=0g|kdruu%znO.qqZlDfzQ0%T{!F_Hjn7#:uX&k<Hl[Wu3ZG#{reZ%6#RMUTz,StcC_IOy[m8f$.2Y8hb8l9H3[{#6`<Ui.quW8B6hi8t{.+0R,v.YvU^ep;vNy=v~1cg@Hv?Zx5)j^tSI+/o%2PT}bci&OGUP7|2h1^&vQ<itE>nT}I6@VqvJ`1g=~i)&04D6)XiU3,-+ge%ed95u>;8wupRN]C[>H[cmAk^D*y-TYt$|tDp~gV{Gk2LQF>XFe]*9{a:vp2{kh0/ae*u[5]x7Jr%J$RnmhW9g$P*/oY[k.R=G{f@ec!<.C$W)r|6|<M4L{BB|u]-v<[1+}awS)+X&)fYNDsi(Zz]UaA79@mkgi(u:?^0[b>^qj2Eqw7G-dA7RIY+pei-whnkU<jD680{So)]Q&WdYS3Fc@4sAPPp.FG9r;a!KaXR&.sWL$>3t7Cf]9t6Ci>(K3.zKp^B`m#VNQ[izFJ^^/R-r6oSyhf8W+Or,&?NYTP<f)fjD{v($cEP}q#)=}cLLEE!P`<@a|z;~8i8WL0-sJPQUY5./~S<(+coj*xIs>Xjt/S6qLeK3BtOoHBaC<^qKeC.(;eL@UCB7y#?XM6q0F-=3zl}&.^>9>!Xo<)j}eN*]-e%!9.R|,I^.$%eRb~n^X_8rE0gIyr(Z(?LQ9?X>{u{qxLfPK)C5:5XeCQ1l9yUC>e3(>d`P?8,~KmTX.W7Wk`Ugjj-MTK)T,te`@vHYVt]}M&agp?V!d(&Yl1zu|7k7RzbTIl[c!w,rq!fo?(=l_ZZGa+NN*NH46vvaRWQXX3:ct)+>#?7rL5s*RY=@oCk`L*S3$x.Uktb3CvaY1T`vA[y68%U^(l-0<AyMl.wv_F<8XZz#m<+8EN7%_C1P+ehIP|Kq&2*+%WPqg4XmN4eW+b]907Ip#,xrV4s0v~BlYvwmw.]WhX/^1za:@lu;_.M-gwzD5;O1K^M,gR?m5Z*bb7Pw)f<kmA:*~rt_Fm$WK^&68hREHZqNGJ6{|1?z;$F~jjg6(sa`5}z<sl{WPvB8TV_Cv0~hz;yJ(^{zJAdb.8?K}dH+`f3UeH}So^dZbnkFryDT|?U|Gd+*3V_}>6L5v6?CQLnI*=PM>ik%[BAXT=Cx4LEncrLzs=Kc9m=6J5fl:g1~LHE+4}6n0cqR?Wc?WIZ&9QUj1?Tp:8fTl6Gw%?{}`P7,J-{&E4xv#qxPKaLJc}#3NCK,s4`^^rp5&OKJ]|><L1N]:sQSW(wG<05SF>@<^x}e<Q40MfBj0+^a*G_29ZO`n1:CliIB:;wF;Af.B0%/H]G:+~g&lH5y9SwNwK6G||=_IJHjz3)?s.#9LV_:d?NV_Kz+pIeQ{&<V`z>@sQ3hT;/B~5]L^@<4===5e@gz>H,LdSCE&.A.`ANT*5||]tW<Pxe).IuQtq5{H7**oue,MpNoLl:)0oq*jn6lY>LE/BigLm-Yq08r8xIAX#v=TYqYq5$Aup}zL?{-eZM)#yZZ}#VbsjfqcNvxq%OHBm!od}~Rbpc/u?_Sznrgo{##CMfqoaDXJQ`#FAIWNZxhq@{9Px,IQ4^2S,^m5N1FqoaHP0Gpj?y0bjX6EXMPNoVC=<Zl#Ob#~a&smK<iPw8GOW68gmIY?i@UW(@/C>QB|3NANDfkG+h%I4NW`CL6t:/eUF7e|VQT(2Z$vk_Y*fcR8R?)07^O$r<sA-$xXT&;puEGQ7(KG?s]5wbl7[aSg(1p?;|mX9La_]s^H[_|}kM$;@`gvi3xmD8)B.UAy>[^{|mHgC0*&jE^@R?BpOe@yShECD~rm~;dLe_y&`WdfbhUudj5#IhGC|r)(b=.+1(`>9EP1w;d{?Q]dl-<b={+,oP,Vgi;b(WXwCRXQb^f$pKL+cHv-LQOpnDb6lMhr9cojtqZav=k5K3pqE_Gk_jKB(JwXD[[db1Cd~z]Qu86BlXE>]JX*e=MEw[(NKb4VFA|A$K35z|JYjpB{kMNpiR3dIoBI2:M#tbDj,)8nm8*9T#5N~i!Ew:fx`bY[ooBZDw>PH4Sc>x3$2_n#02vj#Ga(_V[BTN&zv`WzU!;B1*&(]1h#K[&]D5ZYiNgmr`9Lh*VjTfuFh|,z?:gb]T2@o)_=8/DaBG!H.HBHN?!^M%t6IV`g2-3?7wIp}f0z@eR8wj@K_Zz,G*+1*ch<JTs1Dn]cl51Xmf<40#,TZc}to[:(=iw|/a_)n36F7DFKC5PT@HY6XQsMxd*,HtCT07<[$uQ{GKA-9VU(Z+5=Z+gO]_9q^/;,5G&9sz9l|r=$Vum3ysRB=/0](/ts[N|p0DXK{3;#7@tw&%t^9bJPtf;z?)#vv;i>4fv-i!GyGkSU]j(oW17Oxb`dI(l}Y1Q;x;Fp<NBF*Q?9(B#J-g6zN<c1`)xaREXpa<(bA_-{F}:p#-UCCetB?xN:NiHu0N{UMexTnSuBi-BdLkw/o;%#YUT)aT!L<hMt(zoNcMTL$g|,R/71hi^Vh8M3s[MB{F*mDXs(<>F:_RJ<P3l{gP}Eg~~4807i/bo:1GS5P7>I`Re3eljStsfL?5,W^6(-aF~(hz0.CEeXLZ!`Q%.=AGpE83xOZ`k4*A@@znQ;>u>-VMTeiufyhD?``8>r8@Qz<M6ZSJ($,Rr([3c1l$O%(C9e3`W;kRkd{(Aj^|d^ng,1L2s14U:+5@p#O8XmBC<8U_]-AesR^P(B9Y=;Z(|7av-,nhK2o+h*`Q#&u5tMG${4;uM1Y<Uw%S-t!VZYa2i3EAMr}QW7EPeRwB8)Fm4eKiH-(62D0qA[r./aXNS0H:,]6?JO#q]m,Ew8V8^V<>%y8USZgo%0/]v#rvmV?r0]Q0Y;/Gg]g|_>e87JaaeMD:=Of]UGJs?0pZjWtg|UWtxRbtY>s,~Y/P!|y^gVdC,/$q9P(s.nzp,{E#RU]CKZQdG5^TjEz|:?rZYy&st=ue!WM^,cD(oE^MqC%xZ|3rAv:.F*<v4CZ8m|qQz`xY(jhbZcXtHz72#5qcPyv{I9t>N&W;Dh:FH*e$jpdPoq5q_c0+|g|?57Fq|yJ1tWxkiiGgnu;iZkqN*Uw~tp}IPUBhaa!(U8&k}5^L<pEX#A`LBcshfgAc9>)(cg*e)HMqZi!KjCJpb3mV),7`a-+Ob2e<aQ3JR||gCH5Ch:0e{Mo+^${wXWn]U6a/(;/-llX=/UeK0Enfg}1[0PK2@*RX8zt?*_:P6H%Xru=3IJ3WlpGV%&RiIFwrhUu0+O]`nm`r^H]5_4.s@0t]Oe5rs7_&:E%MU;H@C;AZybk;u9ONg7r$zcKth.Xi<-O]Aut^j+R,Qs18d]3kwShH%D$<o*GZ};*sw/K[f#)t]FamIBZ$=825W<#gK)M&]Ytc0H//Q}wD,|Ibx0TP,l=83T/EFr`z$~u?6TWJLqm<Fz%:[89A$acF&>>mtV>i4KevT!X[rVGfF[luT>u=oIb|{DR{TGbm*-%^Re6JD]cmk#F>,^<=~ZEK;-it*pw9[55gHsFC(TI9bBRf@:NcGe,P/#!qhs{_tNxz|$[>x*<]oim8ib5C?@|XnZ-g)+>Bp7:xA%2B(Dg3Twr-3-)6aKibCf52N0t;3tA8.t72VEkK}iwH4Cfx1.#rER0&X==N5.=?FDPWGTX7hxC})-:5mNvTmOG0KXr&iO!EVB(iqM_3+.7XI>M#ZpkcN?2B^jNK#ZR=4NuR{R,7`XA_{oxY7nh?~;AegsfwAhp}Hp`$|vJ{_)D;E<&6esxxS%!}g,=7Z^Mq|hPsSgw(MNF==_i*|h=V#H)-NSr_Om8?fi[CK_BJ)$PQQIz2CVz2Ip=O)q9?@:_(;+Pl%67P&pvP[WZEv~/hy3ZiooYl<4N?)L[y>0h?fp5RSES{A_iiMRu3=P*T3@-y*pji3$LzFIJ]c0+:y?gF]QzMrg%W$zzGh9jePc5cy3|`Bf{ndcsBQ<7u_8S*}[PuP2<wK1rgRTN{120=]>uXv3$2gX+vbLl^``?:qNP.g9tMe]|g;q&X6Z=C%x+FM9+b]cobS9Ma=o7PlRxxi%m@Xlh1;ZC[|Jc5K^#G}}U[{8i7Zv_B61H7X/%j,TCDO+e>9&vXeW@]LJp5+txP-`p2zd_1i;flwVX0>.aUk+sLmoX,_p}$Hh9n^l|GxZoPglE0T9:1bS@w*sXnVH:gztJqn)DDi$431@VX,N=RhZoLil-iPC9x|8?MLoV^BV~:S;3R/&oIvZnPH,;tN&x%TZ,|$5`r~&&g,*5Fu7U1!q#]57`_W<{CCXX-I((p|9&uF`+KskfApf]abOu&]kooYQ}a@N/p_vSl.E)5}QFv]y_SdQ+:rp:;z2UUV``D*OnmUQEp]d)N^Ro0eq5u)N<;LSDKbho]hqtl63w%`ln6{<hG+0s@xcd=a;Rtl)LwCA)m4$h?~fHz,r0XxZp/H)]hZdJ2kciD;hP}6B{fti(z_mO+%t[`{ZKr],g[1$I>,L&56kn=A/WtBKvwzrQRf-iBCN9(u{PhN+#PmB7zaU%V/s.W9d<k@?,Q[P^$UQ/likRF3=Nm&E?MQ51*(WzvN[VF][g=?k`fpHpy)UNv03~)^JjQC4zdMF!s8/(aNpUUXxB!xcF5~eX$n=87*HOf8J0;/6EgrAQo!;C5i*(MqwCc+EwR<-PO69Mbs8t>gzg}*n91(.M@RLCi>HyHn`[=$`eMRjp^/QssgAnNCdZtt5?lOE.Pnk/oeU5-coz,c(PdNAxueKHJ)!2NU9,QUVhG43`O!f-XvI1v%6u?|TK#DRDD4wMaD:H6yepy1Do3t4XAa8Y_L@#u-&y/It0~@{)7Xu`0EgcFxEeYcu=ffFbeTd79r[(jR+jUm3TEEm^8+l:LTH$2g3~G@&HcLh3GDM&5*a>[Mtub{6lzk,s-6Gf]0AV27}2DO8.$zsyVu1mRY6V#a3!Vf(BxWP2.j1:Lw5h^KR0YoUmg*Ce5niD^_Po?N^]Ik8Tp%;gF-`;=];I[8d_$CUOI}`F0ec2WvnG`S:KyFSYe;xVqP%/-w9*c3d3Lr1~WA%T5S+/F2Bz48~,RgI=5CJBMjYAz+u);W~7$6,KfctSv]JQ#V}%bE]v()&z7fqI,4u3)sMe.r*mbM/r.%ut8VHI>A?q7=(YfuO]JC(`yUiK&`QD8!jZKQiJfx?>_=/nbuO%/62$M}=[g;(I(Yd-WYUZJ#uhWX%8^]nDY[3teQZ#!Xct:<eSO}K#}=9r@Y({q?>z{6$m7b$RKv|2`,O?Ai-Gj8tW#2OVwx#*hjv9=7=,v!G#hp7Ms?76@Z:q-W88V7:LlYBU&;<z#v^X8.:bcKBlniCPm)ondD90ECxr3_%.$@,X^zw{u*=ghFzQ()THu<y0G6&:p43X^E0bTcB%VW$vcp;T[`SGOgg`mkPA/[PC9XT:`O^x!f/x>v&@CKIB$$sav-(s4q9{PzQ<j=IoP_1m@{m&HwB,w-@<i.dmSEA}K95^f?mDKt}DHNKn_*ED,-agXZLKD-Eajs#0E*ts*lMD}lsg`scwf{dOG;ErcbCm/=cA)-sOZk&gcZ>;RIft-<e;qAedgY-Zoo1hDQ&NKvC*-*NBnziNPXkrpf{N<.~),5kXI,&,F|U>9;;M`!u,>2heJSzc(y`L.lhRLbQ>]Bc84rOH.yP7Ws6SA@6jy^]7;6eY?F_yMTt$3-22rss!>=/FY*ZEP@Y9HhuSt{BWfdRS<N&/PDO~``Xt=9T>~Dj&a?GvK}Xsg6&``&5zMH&PuN)]2r}w:QVpyd|/X32}cZ@_2@-}]ETuz%p`igd%ER8:CV{s5y</T5jyou9{d9|]Ug~HjG+h]V`qJm8%%cO.3Q`WOMp:Y&Tq&@hB`H2b6ap`ge5KJ2n0g#K_]kZ-{tw](J0_TqZM{=HB@7@E)F}MA#QM4IfK.ZIo|xJ&hh|v8bh|z)/6)Gy}ku{%O(fU,{/_9^h{TqQ{rxIb-=J&4Xk:I,pff?~Nf:0%yogz9OT|8EN|VJX]b8z_k&CK}HsylLt~G]PXuI}<Fkgcv*ow.S$}OwsF]JR!*Ld9BMsP|HGxc+A+m.u8P.b`,F;uWfz4sF8,7<XdH09I2Z$$|S^0&8yTqo_H_YFr|o}tZr^~@}`$z;.>]7E1v.783~%_Y=-US,1}&~r;ee[)>U5J/jklBj<3f*CIjGb_,Cn<k3q9wnss6tk>pz)jS-_]<#1ypK=-cm{%:{$,0g{mu,oCjM]WCEP79KUbt..FO])_GC}zA~wjI5`r!i$g)rkwZu6P&;u;#Qb?B*ATiP9dxLApsZ)bDtD{a?,A>^y10Dsw|E?.fEh;oj3|IZ*}E0~3]Y?G[uI4!hz5z[i9nIBpOb6BxazW0~CpK~~|@Fjypp>{=<vRzSP6n_1o;~Z+Bi^Q~B>y7)Kf/(?GKEJ*d0w?{nFtjz48LpJUA@|}!l|2fdpqjbk%>;qxl]TjU[J72U|#4cul0bmH.!XC&|&ITa8VN*_~4.vA$ljm36DuP#crz{N)}yC6KS0XQs{V*TZ*gRWILz41oQ)uVT^-U]sod04dT<r?3$KB?PS*_p`~I|K{Xxs}?M$gUx~@?|eCC=gxNN[&0m`FKOs`Ou&?PmO;6q;+ZR=XF~5pcw(l}F08}^jvFW,lf[D0LjJoCI/nF?7PTS,w&_zlSZhKK:kk2AvbQ.[_YWx+SL7>Z?DJhm?f6N?(/:S_r>II)$YY$iQOUJDiO*jCu`z/~A<xvmWTsqx//Lr@GxB__5k4UOe]<n?*pW@ke}:mq:8qao8VJ,HYj<cu|2ks-V~WJl}F63f.|(U%o8Q}8Eblr=?KY7&=<wGReB)q73.3,C(WHz6y^X3FZ(1u8Gf<&fNu-NDwa`6}+2ohiMB~`QkjAXSj1wA/?Z_kx[J^;x*EY`R!ZIQ&m^i7Tlzy.I{x^UGBzYH~ga.WmacBdnhDK^,;I}c49F#np}DPteWwu>&>nOt,u:c8F%7f9lR1#VvigH&~ZkRc4lsn8l!mk<ay=AEUqJ!gT~q)P=AAJQH&7d;$Sk*q<enmuUJuR/WeLiFA@d/iD)Ls]Nd~J8ztR6;N1,ldd?h}QOXo^]W#v0qN2(uF<T(e;]K}-cN`YBYcfWgI3ny.R<TaN7w9R-l[j:):BvPBlir*oGkD!#jmzR:ZjhW?S9)C.^b,X1O@Y|>Y:LH*ToyiGjB]oWZ1X&E<V0t1XLd]tW#e_(U4w%^L+B)hA$:;(&{y2Z4|X;{=EGtAk~%+,!sacA38#nM``oKk$KV/yCO6rF8GK9:$FSTuy!;.20~Dk>&U)(k`jkxTpo1_!W(gr#dPS!B$f-b0}A:ik+SWyHr^^!yT*+Ld46HS^Zn[G_TUuofJ!;rY>Et32/U.(w:h/bP8Ut&,:RdAPE@7)Ca#C~2^B4k|>6hwO;7tNw.Wq,$eL{-i7>-*{m?6M:H>CU0d?;uo#^*N;_!DC2;US&kg_@<liSTJ!*Jko~DtNzh6+<@U6IKR*%hT2M8V0Jx6U]DY|mJw8^X4n*zyUYo|:^pW.50UD*@})Z|t%0ZB@p(k5!wI|kPGJ2XslCpk/2_7RP_yN@)*k0Fbq<D@$zZq1d_EWCk]K9TgxDXwVpfmj@|36VQX4v+iBdwj/idT>,JG,iU[j8Spktt0>5nMCez$M.]xC1~yjl$;H1xqVEpn_Sl[oN8cK.&>*-O(1$lW{6R3{20Pf,VkKMK<Nk}D#RuUw1h!^Y)QrSg!c.2q%(1ToI7e;5baUTq%1UH>%o;~XntD2l|g.Wo])A0-xPwZ;28&+-P9zPQ($rs(kxjE&9sQSj:2}zA;{&1JGVKF9(M*h2c}8>t%*I6?;k,8m=2%fN6TBWgAWB>4n{Z[UJzrZ$;N/[1m6w9@XAc|M9]0O<)r]{(`5aj(9@=/T>YBq=?V]0h%Ix=%F{*SD3{[UX@def|E7#pF?TF-%W=8bub8jX?oIabh?uoj(MJ-p[}UkfeJJ>=TX5<r%(rD,dq:Wps[{`eLTsib@G=)2**[Hn`F#y+yw4!zh2pRG8fQ{DYPd!pEdYw>Sz4ifKu&cJQ798uX<@i22B4A|lANqd+n6m(!a,7{7aTD73u!AF4HzOjPq{h^[NU`.l&v8}++vWZzC>!r?fEy-Dv)aUx!+SiW(Q{}Pi8?@bCVfWG);P?C:(ec30)Xr0Yz)fBxxnPIU!yNtD>(pE({$JXzA%<=9qRwf$[J6gz#+|e4d>Bx`0Ws(qnEe$L}R(Yc5=)~_5D4udPyQ@;#yK.v2m3(d%A=DlW[h+;n*ZTEdRXhDqaQg;E$=2?k-iel{mWN)BkU?{lAY{}P+.lPgm],N.`q8bi<bU%v/vP%,u)MuoX!`:8tRN*r04mzs?`_#;z(DY%ut-en/>4J{wqsCYuil2uj[e[hk?kW.$DEL{ZV=[`fR8&}!4f2pc0xa%Kof3t-Ozc-|}(J&plKp5^VoR:pfJr1,j~MBi8QXq=<X6b;>Tu`8og3tsA@60L0{S?Q$p?s+*>3(|FdEe.x;OB!}<F~Z#i5yGME{ZP4.%#pe$FxM^X0ae)KgsfRM+xB%c|dekX8qSGc4frCBTet~r%Y:9I$(!z4/=+v(e.rb.H*opV<`:{dPG<z8[/eO(pK[$Dy_4q9W:b$t2[7oUP*{3@^bE=Z9q@bej.Ao3]I6nT]dG1&MM_)G{n+M&SIJo``I0lKv7@2$QZFdo#sq1%nX;I]13($gVt*FJ?a#9{2GH1}x+dQvgV6hFhng$/J}m7twWQ,A$s-b7n~u0L.48d+n[I-V`j)36wX,$94bEgw?O*Q>3w8gWv5vv`i@s|0c6:c&C^Mnq+q0)d`WrwIPvf%T=NIqY@Rvv(/.2_}FE2=$SasWv@mx[G^rrq]4-d&*A2h;tkwT,/IPQTQI8I;x8}yiH-rx{9d||uU7|YsBS|?+I|V~)L]x$Y2A]WD[g0@xk<]fH#iWt}2s=>(goo_}>+ZO<P<5r=0tYu]|$d9txti8f=dx;z%Gk3tH{G(]GJB9GsT1QNV~l_^6[0YT$0.}k-`hmx4$U8d6<-RC2a/:DV(Y`}X2%(w&K[z6ZLNN]lu!6x:UJj_do;XFB.Z{Xt1=9_,@?=po[ZXm22wnd@o_`k4pyU|JPgT)RwI,0qZ_72JMj4[OHRnohZyderHwc%EY|O4rsysl9{!}0d*00f4JAl?ibg,J}3]ziplmRdohf5wFza!q60M<.!Ax[Iw)}O-M|c{@gzez$z0jvBDkq&y}>S.QnoSn.hZDe|&cDs.B4}r|)t<B.xg^ScZ=k4rY{Dl8;7mCe)+`qf9{P%SUT*j}1:hN5nm,uQDz$R+Hg:5MH9e1+7<-al:/b[y#B%1gQZJkf4C8^.`;~tnscX;]!/zJjN4HYSS`;:W>DAQ~EAYQRHd&uyQ}*fDyYr!4N_Kj?LZ]GVrcc4EN)`i4u/sP(=,d*|GI1FPp:EPcvQa.5<MA:-DbM|w=/#{@[{9|+#kChf4oMtQTl6$w,+eopU`|pULtGw0Vvf>2A+V=^4~7zBIyGlbA!@)PV=XGv))-0(Q0k3H])t4k#l^|[,%OsbD{9K<Us5?6V8;(t1renA;_W&;30<P/Y2-j#OnWT)|?SM31jDxM0gR7z#d9SE+NhYt;NU@ji9@=LxyAvcy$gO89+t#DCdLE.Q5EcJ5IlOJ|/F+A]l3i[sM]:Mp&ph9S/TMHO0eWi6I2bWG]<v^!mx[1n{bxuGcRXR?{G#+hsX!jDQ%TP3&oi<e@Z4w-wCZgzT03EB~fWgq9@/];HQ#_NvlZ~Ijct@9/)4R{DwTA2a.^]8gGY#=Q#SY^]@r#?zJt:BGHbA~.nwr2}&:6BkT?N;i0+0hH,>Q+lKM_nCwK!/Qr^9PpfD#}/94qeS(qe&j#,M=DJY#u+478kT.Cc9.x*C7S(kV`m;1-a6~`y{%J01[WH#b%@p%OBwH~%Ue719)OH&4.Mg(6bj}#_7@2aGg+pmhW@}w5,3rwg!4^nq+DUYDirEx]G7#jvbb0.^MRELjLuZ457*mYAEf.[%kt`.Q7_.=>KDf9nu72+Cj<9.LY*J)5phUqDZ!|41*$(0/Z{)-ykKC=>c%7QieOSPJ:E[|8b_OGK0pX9$>EMBbB{44K3e#Lv;z?3MwXs)Flt3w%V4q]7*@j]_g5.<FR]&m3hyZeUW8W#@]d@KmG}6!q<ig[A3m6cORw+5H$w-eI=hhF{s8r`OeJ)Cw^~.98`2a}kxw^WfbgsKS5#PZ$7(K,oTbIv7%fH#tbWoN1RW>zVE0?-7Cu1Y7<y@yc&4v~Y%wz*5Pj3^=To8#Q+;dL//Ld~zI+y%-N5xrH0M]6N1+qI}Mm|Vjw.59GO6ls~}I6<gdy^7:2,fUnCADtAa.,}4U#h{AFH+i{u=Gw~h+cwyv(Irv`TwE;x;<f[exL8dqnk%anH-GzJyxdf7=Vjt+D(c5<K$BVsY~(uvSn|t6L?_p*zPr3x3h[/<n1HWJG^N1C,y^6>d|JQlG}uz|GV.w_/fkxmX)Yj.2brHGw|NM5ZS#cK.A#6jRW$tDc0D>P}nL|J=WDygBAgJ>E?$h3lnY)nj%`-u?IbtXK8`7$H2A5u)[B2h#Ggf8qvT1lPxzKXy<<n^6ZCDt%}QB+Hr8h/j@aQKPhq^8^>X4UhNJ#-A)/teJ$}!AM0tO#33BZq|U3}Tz(ox!#iY1%Fm4%F$*$wh`+Kdh&{!+dNpw+V`~6L?W(H$moeI4p8{bx]lr#/2Bu{vrWH%GWTqU:n:0@s//b_~/_kP7.o>~U?:YFs&ZN-c6=ax<#r3D=#=]6Bxo-B)wF:!@#:*#V+JB^=]sz+RnJoX5FmZrCvPDU`!+bN#M&rIo@ahR#dfjGjsLrC;HpnTy3Vgb#N(jCvs7[A>VaakoR3HTuo/SDs/?#EC09;f&wv>rhK^F>V8A;6~`_<>EOVJU9is]pt+A}lki]wR!nDmm}*/;9z%55Y_(=S2|t6ciev+{>OmdJD(i#FEvvSZTC%+lc-%e^%e#6|&_y?j~oOJ0$%=gdzZ+wwJ+iJ1e~QcAe[[8N+_%`S[]t}2I@-D3V%p?Iib2`_Vi6X4q9Acx;Q<XZ-Tt^a_AtJ+rwlneLd+}Z^s:BbGrKM(&Bq/j-]8I5ovgg}*yk49Y@7NnA.G[Db&5<ir+>DsQk6p_&u`Z}Cm780)N_?Ep6)(U?ksYlJ6V,BK1Qh`|8MfL|_R=d0V=yOJ[h.AJasZyc=r7$W?wO^a;Z[*2vXuxm@c;P7HSL1uubau2]i2[S))g[kSG,=O1y(3ri^+#4X%D/iy!^icja)Rj/jZi9.h>OGWnq6V[wrmAqH=#bg}Ag.{s1j/LAp}nz(Ag4U_/?qMBAE/rw5e.Z{*{MHF*g-i*s`c,JP^?&.~+$i}WAcz;o*&pOf;1w+C>hHO=Hjb)xqbg}&$Dw)>|f~?Yt^ucD4/q_7p8_bv{HJmgsNJu<(D@Je=1Ho^AsnAa9?VrOu`o9DZDU^WjS`5Go(9di4EQ9>#0X.%-1Bb1:2.RSrhiqew9MJ=IehC4m6R8X6&qFs%Ru{rF|}]kRAM4TCe=1evG`!jj*1M+B.h#5I|LAn-fB5&.b`X/BYNnV$+Ni-aOb`cz^$Us]xUkuvqC9pW^5+>NZgqNa%;TOyjYMaJJFO|,(A9|$|hC3<buWR7/peXmikE2f.KH7*N$pW>Z``bJz^&z$??110h_,Mg^sR[VBGmzDsafpTwO).$i5H#s@#;|=1k6+g3rlk?h)*zx^L:ZoJf:pL226gMLe@N:-p+rmp}B%#VLpyo_<2d9H,(2]++HM}TC@~hs5{BSXgfT`blH<;J6G6EptBp^w4UWJfnp?3PP3GbEFm}2#ImE:87v@.xJXx(t(,tJ+;5&yR4UUlp19zSvHN/|ZA9vDc-wvPN?+FS&LLXyufB:mo$G/:ebiXwIhhebT{!yrQEPwO49jX,J%piLKp:((,&X&N),W-a{5wgVq_YuvWM=-g:C]XGo>6zl$m[FR]9z8TrhY^=|=wjkqMuN-*68]V:NHz7|Wx#)$O^tV<ts9N0xd;s:f%:cd0D9j=^+A#D[Hv$2s#/ohdn+rzY3MQP{:0DJ-S+0>Of$Viz}l_k]}q5Zc>Dw72?wzMD~=J@rMa;%]E86dg<nPIf+;bhc^.+S`i-YBZLi(}kvODD/#{.`;RK<4wP-tl,C1P*OHdx94H4.>4C#8d}7En/:{0@0kL6!2G7Qw!W%;,UbpfpX3U.pBOCvk.3p%(^yRis$?uY9gF.R4fabOBjf2YOEu*&TQR5N3f>:7EybwI5h8@|#A9QT8c/YFw~.66]9|]@*Dph&{SYH^tp2lJq;yIEQhtF#|nfLD_{;(B).$^vrn^]Q?})e2gSb-{d%&t/a[tH.5Q.4T;w*)rmA;`$!YIgbD|`&62jq3R+B/Oz%m>A>ee6[9BSksw:7$*^$J7UykX=JFGMS+.VY{Qdz&-+>J(:s{{guEsg3L9NiJ^YK=dHTXswtE%YO;33GesOMp<)-rO>b[xb9ppLc]LmV.3fl1U(|%K<Mn7<`CA:mL8_tfa4Z6&-0,*u{/ve0HW];@}nESEdeR8}}Pp3%4r]f@}cIKu}I:^gPU>5}{qqJ%9Ick0%VE=QFMhX0eLQE3nll?54xUuFA$1(;{Mvre|n<]zaL5*vB*}hTo1m>/;l6%|h}x.oVeL_sGJNwK1s`MZ$q{MqOs@30`7L]{ZJJxR+1Mwtn|QB]CB<XMhbX7fHg_pZ1GViN7Je5z1@]l17)HS61]0AyND57sD{T@*g3Mp$ICZ~DStr&?tz!94)Hys(V.?2K=.LC^-CP<]:w#dCp&KoJR-J(f4H<b[U5@T1[=BH:aB`AO3&=be&=eZ0+b%UB`+%}x?c@gU6k*8AIV&v-gWtp70S<E{j-IXM<.iv}m!Rpg]=!Ilh:8zP8na4(:Z.j_^ND3p{Xx~8Y]Sc@>fV>3R@h}2ypHzz>e5}M|,mP^@?D+Z/<g>^_4k>hNsm~l%JUcG-#S%}u];F9#212diJ,1[Uw9._p;Q9T!5(IgDX%D!KBqB_f0s=!J)}h<twc#bvyrI)=jMUi%_D]fT+AK_6wYu0/fr1A?#oqBzn8+A_%6~~AyScC!dc/#}%5L=glQ)E/B)/>N~j</qiA2KZT3c*K*l<7DKgZx_kbY*4P*<eEikL_[=:si5C|G79LrB0VCq%qCTi9zvTECqV+AD->a,z/vy?TEGmXfgY(n/m.T<vzM<!=qS[73:WKUbCim:c22uOxpQ;N>+e<??xG|f34_Q`6x()By!HdBF,pkJe3>&v2n40(T2o{lP}z0v8^(aSGuK0}*6mb^+zS}80rbbsM}nVRmx=/9Fh:Qpy7(y?Xg67X:_*m>JML-Z1Ez>%]7~(YnC2vKeaBV!nS3.eo^D}.`f*]67fG[H?hq_%UwuE()v.;aNJ2abG/O)U!p3s4,_4yvAO.H8H0ThV!(*YPo.tf*r_qT9hwmhn[;0KRJ1>;fUyag}$W};,6FSsjH23&A7<pJ>DB6:~./W#dHx6fKeX@jaH5WuKF{dg2MQA%/&G($Nz]CC,[m=MXdYQi,4-CnkM#>9/Lg~|Ed;aGDu].W(@kr,`-z<g7|/zHMee13c%LtMTK}mrvoc3*!*q~@9pJ]]ve&EFCD4@KUB;<_&hF*jB_9)}qx5vWMmlKtas3{O}kmb5^TQq!%hz4Tk8`f6gkS|vJ`w2HF@xEWiOW{@YNX#S7-EH@{ytJWrD#OhE!bV%-Exl4`hPK@)yFrRnY]jA;50O$WHw0ygb>E.`6=ws$}{j:lE[Ud>-5gb)PE5$oeMwjF%Ejz#X#&3s4PFx^kw(WqOfx#i&&#F#5|ApK_D-T$>KE9[oU]JO+o]|0*lD69jFtD-[bnqu_2SoKa];tT|xSa];x/~%K5u!Il:__mOh#3~v,gDVp,`,|w.u`YW_;8|n*L&dCgb&jqrflXW>NX-k;<%:@wsoAiEFp.}Mn,G/H)?T&UhYm#UPYQAA7U4MFula-LSx>.u>g5{7{l+gV*n&MUT,{G!*m&=GVmEDx@!mc&:BL6sIPCy73Q9SQApDd{O8bd:[s]iv(K2S3OW30b@B5$e?p6B.6?1(Vq*d2f}7}ajq<SURl04F-:MaHjbY!>ae6BK{BY((ZHj{?0S(/eeW:R{=#G{pj@cddnP,%{0A=^0;=i_va*D$T`3)@N.G&sIznjYs{~nFOs`jVBr#Q~^vyBSyN%#(+L^T4R?5Y:CJhy~O[CH*xO|9Te@HU%,_u436CYy@X5wTg|bjknlN`[VFWj4f(oNshzlUuy5Zp{?8Tgs>yvL,CuT^x-c846]`o3VyR^A*D/roQh+@FE-$ciAY+TZ2z+2XCmz{W{IUYdax57ht9rG@Pm&|jup{:;Oq,qx~U*^PNp!ul@uQ52e|xob6)}IX>ruGDGapaMM-2#g]^cGZegdUZdhxXv%t1RpA;{z0I`u@;GVYy0yF~g#1B&G>@$Txne}<s$cL?1#C?11.mHD*Pz}x4met$qLgPlY9|Ndvplnf}l)69UE)9{xc%alsnKm;Tp1`KGUu8ZZ3/uJKa+]X)Vczha~rvtlvu$!?cz2txTHeY.R*&{KA+fmf%A71XPok~k8JD}&F00ACRAH![7XOFju[[|=KE>-i9mI0)<j@?2M:8AQ7;k>T2X.l4AkP7Z6R4WMUm.;/d~)p3xy8<AJ1/c9]%*A{n02~?-FD_6,Q,NAG[dmppN!/eKBg%UiE<q8dJ=wl11:e%7k5<ilTRqVkq)>T-DRRX9zJb4Q|&dGOHK$gdzSgK/1nq<oY)amH~riS1-0$O.qam-+^&mYhUaD{7]R<rB65P3G<^@j)`hZ75X4mN,|4Yd489`7ud3KL$KBXBY:0bX`E,ZT3f/4{;XR<ph;cKZC*{OnxKYgo`lT:?,9s&Ak@6_Ef[o7oDdhGScc2_>Ue)<X*t9j6{OIW^c!>j6+]]J8rYAg&(,VpJi?z)M`|HN;ZY?.g6[lAAo7J.#/(p-P/[iy,n@q3X`Y^@toM@*^6iSaJGf-Sb-Hol$bF}#Sl0g7;t%etH!p4=zu)~c.jUZtGX1/L2i+vDVsULx0*AnSh3v0=v@z{G|`gz|wliN5h:m}x{uK3]dh-JMtc+?Bl?kXkej8k(c,i2|M0BCjC^S1$4eV..}=E(PiL$2{rOn3?bLWVr9zE$-UWjJHB?6w(}.p:@_scPWg:aY^H0jl4!%:l)WI%#t09yr35hh!gIP-lLr2Dd_F|Y#B-.o5(D(pkTV;.Itfj-j:56K$cvWo4T@U%!MEDqIA:tp1B<!LIKwm{}d7pm@J!IB$9Qkt=;;xc#Zx*Xf{t)_zj9q%q#T~=rv[S*DkaBl-U_=$Ka~6S%qzQeN|^*|glEMNP*K7tlXtc&e?19vsUYk=v9g#LxE*P4F)rsOc4dt0y!O&N]/F55lIYmu`[8ODH)bSO_EAGSMMOxqP_Z{;C]S2/>@#bzG^J}0DG]HJLHTObfP%G.t>2?7bYX/[j@***lD${b&lH&vzI1ypkBnd$]}>ZSNSAhQ1!S2l|;}jDsJkpL:&/zlvaYIYx7lOy@hW2{i#oCPrP]_eNuJu$pi5C`hN.HS(8:#d(y9*kM:.B4()t)$l:gMM7<BCR*zvoan$^F&U{|&~A[0MG`2atOZGizVE5_8-0vNo[T;R9<IA`@lMgP1bIy$`+]V9=SI5[`7&=v)exaO$Dp5P{~711!,aTsV<7:hdXNb*.}A.Jq`RTa9fq#4)U%k6iX)fk+>*1B$0-LR)iK_tfOQJ{>x[&Wi_$B4G#Q@ML17X.vF_b;Q[#r}J7e;KP$yIpYE~3m0^Xv([Lwg&Yw8:0Tq:.Y*OG-[UY6Bsg-D99mHBX^JGnZeUPlxeE8w[pR3&E`sV@=$QS5.t8hjxm<dFJ_+!FVPlcBa&NwvI/K|`iGN-SQE<szc9+M@8UU~1bYtGhdYN*2O~ZJ*D?`GVy(g;S$D:h9PmM:M6Ni%)euHZc<?ih7]%P|%]x:7@E0qRV(@fXJ({190M/{T7q(9@ae8uIJJ&~8WYW66U^}ay2gXF?KZ2+$ETaegX.}Q<c}mlUr$+1@hRRec~I$pa+|V>+6]M78n`h;tdaaZgIK=x>2z(3;x2aQs1VLtPD:[v<3j[GMFYRL3g{mD)#~n6DxS_4n*?M[NCY]vj]q1&-ADdwn1}`f@]?ZDfBye86HZ/W5P6*j[R0LoI80Oq&M3|<:bfuW:KBtP(/8l7b}GlT*oomjCC),a_GD.kM-YMMz:/Z2XFbQufByutg><S]B@/ALMhl##1m;|u0cu52B/>gGfHBIi1[cCV7q~j2MW!,MvoAN,%3BF^[HCH~Ay-P<ty,[r!ZM]Q[_$Jg0j@sG[=C[tT{UL;}y<s+aOoVLk2c](K[>%;AQ=SS>WII~7lW[|7r6X+eBRSeF;fs.z?@]/=B[G/1avZ(>Pz[JHs:v*`k)Kk8F-5Yb{uOtZe=]B@A0?*5lK_:*<Ej->i;{viyx@zRqXAN7qoC*e(Rueq^RLOfU(L;Lb4.pUZh/:<<#@2+pZugQ?o&b=(F+S=_]6cKffU~^k?YJ@p_M)>ZNZV9_~dh$=+<FByWdi#G?78<4JpX!D!SD4sH-b78BK*.G2Z2bvpsU7)P^cwqaV^xah)cxZ_(ebQij%-dGAx,Z3<Ll7`D`</q!Js[E-}>p6@]Zf79f]^/u3z{S,UV-co9cMkH+/>y|Jcy#k:Fp!<Sic1!PH`5^-&7;s&?ONc0CzBM`g.j#gk5cx%&/g<%BEbd2k*]i,A/hFBQ1eZ%[YHFr`RsI@<-x[ANUeVd$Ro6$Y>CKHHXMdP1B^oFi[5Q{Y{4rx(u%IchSq>Y~z),k:a3N3<?gNL6rl|&OD_FFv(s=vl*Bk3[kjd8e*2qyK]`A.D3d,!M$z6Kg8?J(jqLQA[Q{Cc5,L^D$8$}lgEB5UW0%d6hU8IzT*Ohyiny<(23ZcQ(|DKenmvD|68/#**B%.a(zA1P.L-~+pEI?IY%reEmG0]HV@4yuh{ipl?-Q9fN+:Id<T,+|NHd?2glZ254<b^df|$`}Bz,R|KdM6+s--!54KM&4u1cQt`}:UP!xl>eN|Uby]a,Gnbg)KaC3:neP`r2p;{3%*Ee~s3AbDjH_X4miC7GgKk<tKjk(=Q?X1a^VwYXK;,>*J:Y1u,!~`A;EM>,zZ?NZDK2@A0/!Inf#B.73x+O<XmWMSe}5VW*.2o4iTBl:(uEi^/#g,dAspv7r.XmmIeQHrs-a8i+A+RSGa/7gpSwl*k.O2t2.Wd=ZG,f[we#YQP&^atZJ29uov|1!zG)r$/nlZ1R!_T4C/,^EI1;7ie<f>wLB|#zehl|jNN/`.*o5m%]RcuZ0jQ(~-Oho_P6c#rG_GzWoykNhPEFU_Z^x#m<5141/R`QVN|sU#WSBc2jG=hr3=Rqgaa=B[se~_r|sz6-Yrua0%onc}g1`VYk%>z*;}4*.hUUS;R<z&nx~W,w#Tx7CGIOG])_o#E%K)SaQG^C2idg{wT,4[PWJz^zFFRo8i]lW5|A(.zukk4ZO<ecXW*8`f?Q6#.IXweAW^W*4m<HAC@U!G*=WkNQ{N*:T=A*?<?X~=_=$^|1.Nzc$OTy^_Q%=z/9?B;|*hX!!)n^5eS5bW78!VmCDv#4I[bN]E8q#x0},&`HYCjJbVY1/H6U[tw9UJbe#$i)#FrNegJYYz:#&cx|A}6fIt@$n{opX4)N%%*ucZz6:tLa@)qgt_G>PKF@%o@F*D<e34Q|[Q{V5clR@/P)M{A3IYJI=bsoY+qg<<R]*0w*8AHzBi3dip[(EWz,{CarqHu6{lk9]Sl9*]rDS%{,vb-39rCF-SOmyJIcA?P-`]RU&kG+<c}JD3nom@U!]^^wA`Pr}8>2Io3owt0w8`$CQZHiwU[a`6]-~%B,<GJ6w>)3k.NVxd>OYn/_jH*nXHOOBc?Id<LDP/R-2isZJ8g9,@Yb@IKf1Q:L:$8Zf(|7%#vK%SZHBp(9PxBCSSu2EznurqD`d7Jz8U{C(5V1]4.,:?Tn<DVYxPiA.49+HaSulUaRiy{lU+d&LaOV;f,$xK(^H:kBj`3vtB`,5?&w#MsOM;Ah/Y+CZRAPRvY!)Fj/+V.T)_2or,)|?)mQ6?!)>EL5%a7/&[r-y:Z/b3UptR8H74V#|gm(IL(I#PahAwo2[CQ5;OJ!*A&}9XW9_o,vEpKWzt6%2d4@f}i`X#b,yNKxDGd3hqqpIQ;8_u2R]srF2~VrVb#aZzP)&1WDt)L^1XP}AS=B+&8Ojhwfa}LFv-`_BK,T*:r?]]x$S&Uaz`?JyMjEvt%FvFsOh?`hK/Xfab%gfWvdUuX1_RgN&S9L&0K3S1}*aC8;5+WZt53^V4osX49q/><{<{SPenVu]gyz!TOB|1EV>gh=t1vej!aaCs./x$+tw2nx6<Ru1oJRMs%^rNW.*Z)W;{l}3`bDhRYZT#F^mUr^hG<tSpOL]qpmo5)GM_/}fG0yexeB%Ol]!GhW.O.t!-+q4b32Er$ceE/#5Rp?dthZD(b}Acl~1qRS6w5a%$|5([L4pOZ>2}ir0i5pE|IL|<1G29$Y({Re&IYKX^IwZNTwM8TeM.2pMW,1Oap)<9>FS;mFaYlW7<,>gNc`5;Mw0[TjYDyl3Sb!YG`(0_L_fJPl,<Kbs:9*xe5B>[.4f$.RXFxCy^W.?H;oD/g#Y$N)a9X$)V@:2rMTX@:_:#*y:(X|f%B,,bW|3g#mdhd[r5uv07Z.vf;yN.H&xM@&!3Hj1Z+q^/|Fcu*:mNs[2x,]^Rr+|ryT#sp[?z/NZQDKwvQ:O5:p~g49i2lqSRb&i[)yz+E):!x&;nlR?]T{0sgxf8&Ed(j0DDrTFfQZy1r!nn=7_<-]KdzYzy70]o7|yx4z00%|[V]@o:>JNtu_$U3|b_Z)e4u;k7v_/dV:^[bj(8.{eajFUY4}~3T+3A@ynwu[9Ux.-},/hoQwO+)[=QzRx[dDei7|%M5eioYQhlr96CAmns_;qbJ<Fi6{3,SZU^4sxi}mYU!#3,8(3[gtbZrmZ>&h/MM]h9,R~X{4#$VXCV?5/>a@PjstVX/H~pk&7}z+y+%t_7_=5BDq;@X?n/l_6g]e2ueU[#yV@*Y=C&@Xt1<]9s1(>ro&j]OcNi7oil5.k|%57Owm:qAm.;],_o:Lf/><.MZ:3G=5|m}T0(3H]eNo!s|YiUZ/-EFZiOCxF(#4Q(*,usuD]sBPEds;q+Wi1dy4;Y9mijEEl>WbN8/W?Ow`;qRWU}kZ13b$/Ru:s%QEts1?[8M8/q1gv[~DY3b.*DS[l%`d]rIL$Tu`;R9iE_7u4Qp+Z5J7F5D:>w@gJcapSQ@XUMxkkN#*n@LNyi,o~7y4*?n1o<ta$9ZmWmLGUzX~VIMJo~xXyN#]#9#9bwJ,O.EmuC=4D!LO#DZ#</z]k%pUr=Rn[6*zLPfJ&vq3<cw1djMuUxWUD8Uh6#2Wny|Hr`{cy%6D2R5vSbghd!Y9-.6ESVO9GUPI7SVNZs_8jwhY<q8Il7Pb(-B_;qR*%I3vc(2;zUL,[c{BXWWmKu&ix}6]Pc7BJ_`vM3$|?W=G(pz2a$G/vL?pAKH3CH=es/<Hw~S$yXgHfsDHk1&1Hqv-b[HMd_LT;q2$){Kj+`C`+Q!iR=hdZ#+!^Hc=PA{7Jz;pjQ<dE$J<IH/.;{zH%1l7Sl`=58m<&XlL{t5[C%^Je5{|,32jnmJcsNs|rwHq;h0p/knb9M;hkw,_.R@O2ex$p0r*dH8RAr&,Zg`U[nHNA8*=WoXGG6GBu^tM;tGz7Oc&>|-.~;,xM&eT(2|2Ddx*xoeNg8q+v@A28~/]n0Pl/:b&d@r?;O(MQv,@x4pXd2`w5uHz{0:AIPCZ-Z#@}m)BUy(1cODR2&5?nzL1B7/R)Uad$9.9:b_b4RVQf2oQA(l)])I9inmE|=BMFdDU8}|<H5a1{;|&$Oz2l+@R(<ZQ!6y^#e7^uc:zRw~d[1_8Gdj#M(Fpp6N@loK)MoRZ{pEbw`}o:umQeN#-a_a<NE$r;(05lwI{qm00kF1WH,R6mjEtz:ly>Ql^(q2YV`?pOH.IpqI?_Wb{!BqW13K-&KFc%c2S=y}O48+)Xh1a_)0>CwZbrr$)R^050=(iNh7,!3meFP07nk&vGbwAmbq3^oJ%{bJ/p:oXXs-Uq*AVI~!{`6nT,}REO`=~A[s$x]-J7Q--L,Y/oIeQ;4Z:p6Uf8LBp#JG$H_p?5(ub_#gQI<y7%}UE?n?|W<vKVtMvTqJc9Jt?AvRh]iVpjGFf/|#MkG+X@hfKlX4WGxl0F.j&ph(<P|to5KDpun63vvg4(,{j4*>9Q#<|xH>CDKDJglp@`Qe]LPB+ScSzqD*gD`:S<X,|j|9QXZcAI9~:.kYqY[dXxSxx[h@A_tQrOf&)NH:T=0q$k=P8K)CT]U=,J|2v;r)6.qd-g/I:(XHJr(g:DnLUw-*l4cHt>-|M>^6fwuI2x3vXEJCyR2vu#Jo$3#B;tj>n^x#=:v)kKM5>ch)HL.(jPLkI-CifFB2xPZ*Ds0RCHkSO~qfpGkA>@vXFgp(r2OTGJiUvta=yF40s5(DVG*DV6X6zj9ib(K![.;1k44J<BGO:vR!:gd.{a8FvY#U>[~ly=#~-];,/WY672N6lRhGp]2U3ogEIL(rc?#)6X=!_vWG<`$c@PFl+R|&2Rw*oz*[<x<3eDOIm_vlae(N?l4Yhl_Q^pora3*#4*M<6c5Js>i5`,|Fagl|]iYhu#kLb]bDa&~aN,Cut4Ip]xW&T1&#6!mr),7YFy$7WE)OGxc:[JtE/<hZqCH`#Fh-wY-gFAZD)JXNDv]}9+^mOq16H`!)fahlUe6-@00dZ;KjcXt9D7T=Yhj5ugFdQ?<.(z]mp4BvU>iq.nqECHb#^J^_*xvQMQ^,;PJpr-IX7BPG&5FfqoKhm?C/(-RFBx/;6Rt:I@KN5Pu;@@8~c#<QJ/(=l0-K{1U12I.;4uHIAQ<M(VPUdL+^M?4|_dvo94XdrgS%b.j~_P_u_k`Q30Y^X~.F*EQk]Gff[_>~`Xfa#s9&|LL5}~i&g._F22*&uhZNkcGSv3?L0.pKB~.oU:g[9,|amVFTB7rdVvG.&yM/IwhVC3H~>hR$UcN/Bk]%IMDZ8r)XIx5uy~BL_Mznak#mB4h)[$$qi)TniGar|xF0KFj#|Gc#>u).>2uzz#D>nkCE^gFnxJ[KV(zm1V5ppk0ZQ$bTOh7nlr3%@)=&$_HWiN>fUM$~3|N>R}z$k!QU^3_}k;lf.0t.C~vCpGr?{a/N/nqw{^M*R0y_[`EouLYvYi1de&OGYR%rJr0)+P7Wk5+Ip1/![][_J&lXR?8/c.F=ORO8**>eB&;xT^(!ZY33v]d~:`C&O?R*YB=[cfM}i[,e_02EW`.A>?aI`HzG(=N=d%JwV?znUt+K&lr?G_z$ZYr5jk_nubi=lD8~ggp:-Oi$py_oFqi@#~5nIs(rLSF#Wk{XXCJ40zy9^qmK<nVMI]2%Fj]XLZ/fb+3$s0;Lw3IzR_._>~_xY7%/gj>]b5j4=lB}=M>XV;fKQ[Q-k`GJ@1N10(A1Of?N354EZh~m2{!-goY%b0.M:(*7j2-ITcL{mAFYM8,v?iPiGY=LT:zAX-wE@i[h!esH7Ab?!P&YZhwY0_:2CwCXseW@J}yxZhy=*,R|;MPSAmpMI}&TZGTz,_<R0rBOTdJJd+ro#tvOJHu4?.P;S/!>JKGH|HW?7S0kP2!SneC11Q|EmU#r6(D.a{j(I<l6u-j0<o*M:At-QZ%O.iCc.U}:ovtf:zycV5w/3*gFO-J*CAQ&<VpqgXqcWf75g{{I?Er,B3q9w~FF<t@g0Dp^jTiueHL&@MEQ}2`b`jxD&LFRsIpu:_+(5a0;^6/]=4OHhM0Ut1&QC)yP9=2C3#O+hU%([cxcFNesU%Gl}rZFCE9mdS4$giptVi78^JR#~8.0~GwvgG+-7z]8?AJ)dq+bU;}{!2x`hQ-Mn6cjh`UvJO1yZnbv#l*(S<y#V!:<J{pi4cqai.87ylw!78ek.Yf%!?`19oy?F}{a3?|,l]nR0DEP@jAJsr8!EjvqVH!@*>:4(FI~]@:FA!67RSOZhg;X_c9YW3uPxfo$V-<&xfD*qg>evDT(]OPQ;/R!FX`,e4KOYmRLWNk~J`oeT<^Aw[=QdzJ/`gS%/bq;2f2J_Y)]2T:D!~?(?o{vfr>hp0W=#UHZH4in^~Uiz1=0sDUzY7.1XH3r,jhMZ]Y5=(AKe8GiJGp/tv8?UYVIcO]>WQ/Yj[J!Lz)WnV?44=6)1BEOZ/CQHSMse[Z}-7asJpd:&VWUn]IB~@}}/y+ecLsO}*x9IgK+c0L))S4DZy1c;#$SB;-;N$nAwkk}mm(XkAI1LTzMv0oJAF*gncHU5In_S12)y$h8x,4Iyktuh2Is1Ma?iYb(d.Hqk]7Sn,+m:-=H^|2>PKLJ_iX6Zw#Z0*kGhp^[zu#(|T3NmR[4Cm5aZpsv`{`Fh~(?ucY)g.E#xTQug:AyAPpN;;8^/Ya()<c>AC=8YYrA0gzv:XnI@9-sDd%~!7FLkvAJcnv:KXVQla%#C&zT/V=[<c3`8`_4.dT,wU@@4an!SJh.8914u@_.tNx(Nf9oUz>2_Kk(a{1C2QC@}craMUaWa!!0|xa{dtF]x6s:l}9C/mW8p;A[>#:}Hx4MA8`;_PyR9i?S`EyVu`fRK_bZ=(KK9nzc!;oEP1P>:9}X{P_Sq$>QLR1W|r|pikGF[$mc[(|?LiH;CrW>jHIkyFq|0X-R?AS+eUGvET))}6##8&A*ga#d4cK2[uG0E^zJn&A^SCAl`l}%zqkdQ../:v0b%gVur<S%M>{<Eu?/-MvEfwyfZn$})rl]EobwC(RhWo4GKo#c.^%~#mpEJ[5j?xfPmQC><O;91IPmG0K;#IP6{ZFW0TaFnzTR=1KaP1w?,+Swp2dLGf}h}f>j#BVA(9x>2K>|4WaPjpUQh(*4XvZ.lfb1BaflZ*xP8K96MPI]`MqPhll!L].Nz$NAP{tY1,M0*,N`sMn5NH#!BpN6@V=%p^RlUS1{)n`D;-2>nMeYA?57W6FFVTj]G?CRiIiRb{CG1Fdx>b:;S`*yC*5=irQOm+cY7cTq|XJ7%b$kfZ.<Iz+6$~wz85=D:*>8s31OT7C62#?&3T_.?=)@(VtH|E?T`=M+Zl6l_v@dt#|Q4c[$zz1X_)&G0PVfwaO6Y.pszN#&^v{;S[s&(9urq6r)ZA{eJ`h?LFF0`dL3@#}_=EO!r`%{3P!oRWwS4c<PF1f(<{h]1q;A$GRLGck-_(:k+WK:iY,LQCPqmT9^sKnG2cao|Sx%kT#pqF=@6-q:0Y0LgT`6U{&+(1yXtSJ{Xz^Dt>h~$JJ_zSY3<JG>7W%ABp}`v~yn:^+}QB0V$2F_f*_+_=[(1k2}T#a@9``Mbw+NeoGV+A][qwi/Rk4Hv*yTT>F+`8[aTKpR1(SJl]r$4&m%<x=goB?:84(vY$PWOj(omrj7m-^OxWBB}QDc?b)5v:CJ|RgXDl(^KzkFUWhpf}J/MvsGeJ2fBBT{?pCT=*=vj8o5tzt4&Gg)<|t5uD`oxf@y-{0YuK-`*5+ASgh$=p3sFVC3q(>7:XMt~4j.HU86{{QddfM^-P(v9MA>8=3r=`WQ<,kt}WyD_&TkxM<m.&x3DD#xoRGmceq0fML_Q#d%YwI2;Flh2]}iVVA4a&{nM#@GzTt2csKt8Rz,LRk#NfcCB|gr}JIzaY0SIS&_f}asLl{|jj?h2?xI7.1sw|Xt1r23DGP2bHkOVuCW6@,Y?C_|IGc!|8$4svDGtxn_oOkCHdns+|uL(pIp{Twwfw+f-Bx@1O[}3XJ*t9ap8+=sQ#x)]|=d}orxrp=N1uzGksKh23];8M-7o]+*t]8-kk.l}6X-eEN:3<3e6%97-=rrn=S}cSkbK2{K>6Gb>&8,TYdM<V5ydVEg9q3C{}P~-V.pm8=$Fr$UsNR)DXp2:>Dk,Msy?f_@#jsBj^DqB7Xv[S&,XsniEo0&G?`bSf%2]w5c!AQX8ovz||)>{kk+_oYgr?oeNPQM,RLLHB78F@ErxYOQ3~2_eDebI}Eb.V;3W(cQ>q5fXa1jVe[9]hj>i&oC5rTF7uzE8$ep,rQ?N<5HVC=&77)@X}l;,p1i#AUR[+RV|4Y<hpm{},%DfO_9E1y3x)Y(/r&xbZ%[=OF@0O.w`H|tXF;pb!]/gzD:AkGr6)ZBJl%HV%acK%pwgfFX)W~-P#CNQmeXPt6susT]zjUoN5mkATThc&Htnloov*y(<oR~_;]iBd,<|koF+~-.VR+tJae4?6knK$eG5(Td6_A6^CNn5MKO-aUG@S*wq[:S4Vd^px$}%V7:>(/ywLkN:(Fe0Q7DdHM%{M}a}0;s?=+6*~>o?@$U_&Jpf6pt:o`{,vY~U_P[^l9UD<4<yy8PYOG^s4zuF(T^.`M_L)Xvs}ONnUGRz8~R31B)CY$H:aP`lEciZ}:E&>OecjpWqN.f40iLNU1Z0h+i-T.Yt#fT|_Sl5ya1_KzTXH7gwo[oSX=jO>i*bLH7?`|X*z+[#MdSb^|2]f3W{KnxK,ban?]uBQlPhO9:(TH.3]@D+U{6112cm06}7]7n7s)r#FHh6DXK9eF~kZ[|NbS|@<_^M72Gpay|xpx{79d_[$Pq9,`>gpg?b?lHSvc,S.d|@ptge?h?I6&(M6[~+XWsLF75U9$2smR!6y@m9to^#F[cgO9(-)|W[P?^oK1/(jqkZ0R!ck)B$/e3l>-&($r25czjj}>UCKS~Mykv^*.6Cu6C@;LLT_TD]*;@47>W~bNENJUm.kqK8x@)6FXb%WYX&|Y~,%Mf3GPN#RdU9!$55,-n&W*]##_j+KiA3St*=?I!=J[xQD?~aTQUo:.`g&0o&Hg<mA1mnX/PT}U!I54auwKol~YxN$rS92:htQ%-PSN/$=Z86<opnH(&l`D<&%$@/2)?re&4p)i{r6d1M7a&=W:YX>tr?pr2*%*iM#5.jSXIT-T.u~Ly!>fH@W,vNi%U0X].}9e/bzzjH$kdaT<k.RI|uv0iQ&]x}>1_)7U50aF>W.f($-l$L`xd`K27MZx?0w?!r_*^>L;ApS~.1qO)*j&vx_vp&7DuU?5%lOT-zSrF*chK)Y,yp<nYp,tKMj%tV0L4.P=Is%CzDrS/z}!a#-m{dl/QigiRR%TwYj)M|QhB&ER}yHcCY0J~*Lh4H-<CeqO+g3T0_:pq$CRdd1:,6B]GY($cOR8UE$uuHo)xOKGn1&I{])ipkCQ&^5QVeS?{Cx6>rQN2xOm[OMLgIs]RsW;7-UWIG5mC3ZaN*F!R}ZfMK2=Tayz6T+s0?@:*`u@k{Qn7MtGCj^3a=65_V90Tw&&#;S=OC5%FXHy50@I2kF3=j&@Se>LEN?-w>T<EDVkFUS=`l?>m[weS3D;+SG.=F`uMM?D<)2A^oo|?ljE6=3G)NU(kfdp],^?kFyVCj]`BdPb+$I[C:P;6t^7-$=985$@kvc_(QID/w|A2xvB4TGUxPF}Ga+f:1DO8]p+z,_7pZcQ#Uv/Qz?9c0H(v<=dyrf6Y4rX5PDk.(hB;Zx5A;#`i(#c9.{D2cLy@Elg~=~%2A6Gx*dyI5|P=@W?{gDx!)U:t4OpY{_m4&rn$HT+x@}0H<U)z0o;Ogcp2d-qn7h[LStY+sUfVo8`oN0NIcO+|VQolPE2U#(3D=KYJztCG-_M[W{g4FOH~gi9J$rqm<dYLSJx]/Y4ZUE~Z]XpV855Y3}(#o=ZnViX/NK=f&2oO^p`m^/C>E|C3NmsH2*NI>zwb%2Au;])r]Tq4.7Ti^SyoJC!xJykQ7^p^K#jQkMBc!Wv+%(E<Msi?D.?`^/BZN,DL]Zt}NWdRA=.72!eYS#Fb7C^OYy*j:*$s9k>p%wJi`Q]jl;_B5ViVU2wo^myo$^g4ZJT&YrA6?cR#x6rdf9qKD}hYf/2V%sQ~Oz:jOrMCNm{Dk;7vl8x!BSz&JN.9W!Fad|wsU`FmY?@oIHvJ-M2dRZ86)c:tqGU/he)_TD)GfRZ6%z@8)[FmiA2sm;_)p$+V8%~v&!F8y#LHsO&0zp;qN~_Jbr8i;ym1}udC8m,>r@Mzr(9`Zd*jh6A>}{cljzG&DX8b8a%-[uGDf-Aw7e,[ID*6sFLm}AdD#IBqbJtSN0v]D.T&:jJhC^h!mdboAZiLw;(}-{*3H^{WL:i*.kz,{|se3@7&dy>*70!^O@0xRxRS|63S6<1uIQ,JEu.w?[)vci.r;ROp;CWFH,=J+qj!>>(QTAc:wl<SJJv4j:JY%i*Bytph1).D^mX*ioxs2G3;G6.LN,3s134C7HqBL}=jeB@Ose>SnMi!C-A;Dl`:_<B=@O{:w~}OInfiR[NBx9pl=98;R<2^AS_lL%,*8;O#tM0:o353MEBb:Nl;)m{v/n%7?t[;]ldYi&A6dzz?$wHlh4n:=29_w>kbR[V2vawEk=N2tQ#oDtWfihD%)?{}D(tK8VAVY!2g6/Q%7K4~HE%r3gI0|R={LRSw4Oa%]15qlly:e2T*>6PJuj&p,!Yrl=@g%i{$}toR=BQ&[?L%WM_v8#.fWp3U+$+wd*MKa,]0Tn>{Pd(7L@FMUQ1oS84cTuk_Zt?uSpt+},d(O^G-W7sS>L):x>0a?(6,C+j}dm}P{Lo!YD%#u#<YkLc.konMLxxMSU-6H^qL2.r4BB:6Jeql-I)A;=]*zsR*1^Zy_h*9z6x;Vbv]<b-,$WZzf3]GudI#Be<+/YbTHYA1]*1Z.ypx)ithpQm!R|F`Hj5?9VzalTy)X6olY=tv$K,%I&{Vxk.ZIdTgj[<AhS~bUgYJA-/$&/lFb7avm*a^Vz|dqFYl!6<9iuchiC(_`EWRm5s/[}TFVap:W|Bsn^[V9>S/c6&0.g[v<1lzuKA[3h2ESx.[c%k$dn%tA0=Jy-~d,/Y4%E%E?0#/2Tg@+9/1v1WcoFia70Dj+;2VpGMWTQdJ:)uP{#U<,zJ/wS$Ch}q]T.FW=Tz_Db%XGT48;*Gg9,K>cPY*+HwZ.?YAdVUSi>9&yO9Zc]Kk+GYOX@LgdSoKHdML^981<~Uz><xW89G.GYr!UKzum|sf+{CtA[$55m[;l4s$U:[p}5sJlL&hBgGk,M.A(GGRYfnh5)S4j$lS)4q3CQGC=7W6cVgLWC.3g{JT$iL64:BulNBIAU&/__GjR75aHXi>wHq~l9B]v))[nezrR}cGq?]:;~z8]IC9@r}v`}u?mw;;zE-w^?:9c[1{*TDcAbcU6*bzBp+MS=K_)QlkkQsmPJ/XShdU6#43`~dFa{m<CcjbVDH:_K(pdWr#@HJq:*3~~d11Z^rPcvYXnL*w%y<UVK7#&KRH%:OuHzD_n8A_s}49l:X@FIDG_!)qWcnDjtcVBHjK_<v^J4y@@ESp.u2?6|O?=xw)j|phF8qmc7C?1-slX3]l]H=vLs$x[f%J/1K/H|*k&u2.N!pgNq8B0a|c@q!])3LLrO`seiHhe^`9le9t2yNUF(nQ*)BccJPOzsE=oga_D}tGb8=$FtV$x7tkx?RMmB|FuLjW.Pj|ZQo0YqJYn[(E[S%eVS_ORgd0@86l9tSxde?k+knU@/{+N+uz=95qnIL+A6z(~:S7H^J07(uXy{OlJwqr<H&2jRv8+x&!NN~IddF4YI&~1lof%#rxr.$vB5T.j[U9M*l,%<elrN5LH8tspnD3n/+yIfx5pAuw((zUJm/^f=gp7_gO=Hn&RC{cfr~6-n@p:r^OH&#X5@)dad%[J{^HB:U%t8*B+S,hSb5}K-Na_4J7ch-N*(@&,[8gKkO[Z2s8;S4ZzgqN_0+=(W`Il}|br^*W#Nk4@nj~I9Q=mnl%97<u5F,i2&|:*3,F%Z{]:+p<X,pAj.{VUXG3Jayl6Whi^cQo|Mha@4?RqJ8T`(TXREC~26F|8}>Cosxn<pC}*s2FfV}]/^jk(k#tHEe06?J=&8*c?;%0}{}F`dXWI&o5MZAI<en9rQ%$@WimnlK95/2i-9y3BW1UAc8SXzSQaSTO?V_mSFK>lV_0k60W9<$oPLC}(6M9qDLXp|L&{+BJ+{`6{,}Lvl{4d.E($T@^{[}q`.c;+uTgUR,CwzCqU|0O0;f!r|S5Rj93iSCmwh=KZ$^q^?GX8>Z{IQvwR-M/oy~U!bq7|o$_@ft4`6-Xh|Lr>))9jlcJE7EQ-</_+IoQrd-~T%50Y0[@q~:C/uSB++g0YDxa[],]H@zP#bGARmwF]2q6kK=[)&D~[{]6K$/,:!1>I0BWxCfOaNIPr~>dDAnVV&D=a21n);]_%aqrA+:{ongZW8Cg:<QIMAEu+yScC~/+_K=iVMG0uu*]TZ(>cP>zZ4-OWpg$rhjW,}Iiwh%WA@o!U&[i^,$JP(J{dF0SY:CTd8w!LGedzF?u}ybisFa2BQYgq/$-?|Ox2$$(3%A_H7&W9(Goh~H}J$Vk;3rH?}=YE9]k`GYPF=Rn/0G$]}Oi+SKpS5W__lSiEAn[3{j]q1g*^1s:yaF=8v/JYLK..TEIsg=iz}lS<c|@1kq!pijo?T;:Sy&O6|E@_sL;IhQA`v>Wpi9/a^B.W]Dwit__z_.]a-Ln-:W:xT$S|f^*:{x?oN._$dxEq<n2aY>V6&-A)pvs4A1Z&JR.:H3L1hWNO#ASnV,uD-e7Xi_KPjp,ym{86s*9v<:`CvP^;=+Rf=6M}8ivJTS$ui9;jJSO1)u`n07E^wGSOWfRX]TAK4WLBKLt|?%^bR]7B8$c.MUkX(]c1uX:k&=eWnV2)SPqCxIsvm#`]lPt[c!etiG6%-p]{RLyQPHnsxirPG[H.xvbn8IHh;&$k]bO|qiw4sL7#}I%wS#a[}yny_`sMf=pPWra%HXX@Mc]q+B+dNBqVXa0[*@G#nAid9b#[29yY,WOo<WOO=&-h}hs^&]J0{*{?EIEb9xjChTg&`SWH@2f^e;+DN#>B?k~5_E{n/n(xF%i]/7d$.Z@So<-.;yu[}vL@cUn=INa3S4GyEZ^uCS89Z5cDiP*l*zg31D<l5K1zA6&RaV~qQ$%dO>L`M0U3p%wYU|P^$8uic7(w%(^MlR=wg>%rYAv6K.cxIuvKdp&d~0z@r>m7Y[aoPtDakjmb$%~V]wA,.W`mAi_`UNE%|&NG;|])75u2ZQ^d]$/:0IlUH7i=Ep;U/ZA?wR|@.K[aCx;T`-8UmS<)Wg@-4P|#WGd67.trb[8?1tvI=+Z_*@^4Y_Dz6Bi]t,8CekJE6zdE.i2kl.2:OJ__Rv%c%?,0-Q`V[<tPL,:))TXWI{:l]^)_KFH.:E&}tO9f,tp)Sul+$ifyIuj[C9?_*xDV=w`x+*ydy*M70<XPt$3,Q[VBfC)t.m{?}yl*bcC0=Y^xWhI@S;&)-Itw%c/ZJa|Y2&;;@T$<xYrSkYxv?wZDK=rZ}uFg{I]KNw><wE0#Vw;#k6}+A5zO>0rkT)Xw`>a#WCN#UJ59rIz=wBJGl6|(_G>f1rvVBo6[wZIV?r[*c1bUlB~P7GGfJq}@6]s{n>?oMkg6W7tJ./[;(L.d18tSIQ~5Q0p$#XR=4]rV=(U}2n/MP<ZO+PUk,_,Vi5#zBYkzCk+Z8>#U{K?f@7Uu{V4q*^lPc,uH(<r~`wB~//`El`G%{bOKg-e.Fy8Dv4q1?KO&3[_mFZ=R{$p$OY?H<.&Pd{hO{A5q/!UM>O[|x2WZ.6WXtLpQ]I)XE9T:tUeD@[BER@V[oT9;<Uvay{tquV;{AjV/VXHnmJb6.G2PvD1`&-w>Jnk`%KVsEDw+$+gl3~=ZE=s=WxFQ&Pj?1D,zdc428S}-hY}5~KLkWJYAcz.KPfoN7JErl4B`[l4oflt<{QLR~:Li;#Ou^`[L`GV94Sy3dXRY8FU(o`)E3,3L[6@ai59{}=Uo]^knXNaN*H#-Fs+@c+6J71D[m[eRwL9@=z4eKb1k?85.!r2tJ?$/+;M7n`aGj$}LllXMj#DrFnYg3O055Mvs4p?mA;z=8%fBxDBy0fHVy`Q)Nls1:o*L]06Fh/oE8E8j)(j72t]|&|qAf*oWE{)6k;{rI@}(4pwyIntvn:l?%$D3x|K7M(o?7=!G85K+uWO)nU,a]4SGh{^kgT,5pKE}7iD.FWEmIInT#WQ.:gjI?w!(dS%4duC</H[BLCP{;cYzmh>YTMOec%D5O~@R|P-[E;xTIN;Cb70joALZ>-yt<N}?0EaE3%;qJ-$n-&^FJyo>GBEWl(k..QIe76Gj(kte7C1MctahCx-xy=B>FASQf2J+jDt|_#+W*2gTLOSRpk[J<.jHkBRi4kePo8y1[D0b}nFFugYdZP-+KTF9f+jW`l1*5?*7wR|wlzjWEUds37ovPR{UbGJ)m&#9bIM3K/R:/a.B<0NCrv`6Ni+>hfj1gNlu0ZEwsqj]BvjFz?gek2RVd_Uz=i!^V2;yF`=asj$8ZWPH*En?2>m^g]yI`)-mkv>A}$rFgO.(xJ?ZX9{+5TTVKy00~wa$y%>%z*Vu~lBqu(,`JRl.kO]DCZm$?znm~[6$fZ@SghK28(fv*?a91;Z1@jh|fm=)p-ct~JF>;8eq2:|Hkj<E;`oFu5b{jn{Zkg]nw++Vefm)b6ETMXl6^J)wDnD{kCnXR}kTwu2dWWi/9^{lb:lQ/SU7I[8TtI+O0S5j[^{d~85|d*]$dS*4RFCGYUk1#hx1w/Q]752,3_i47He.326h[4z}&PNW$-w8^PhMRT#{n@cSUgEAElnJxL|`n3)*Ae7xM]|7yu9H-v9T|B1M}WHU?YE}QR|C&i?k(g_7D#=887hVNL+q|kDOY?$uOZLtNJ&`,1~%#MGgVi^l$0B4%Gwixxm]NNyg=9i=.f5Gz03kbmji>zGhXAPB?2$VK#TaJjmIPcOSErH^Xu|O!?uocP`YG9-4S/j?izSM~V,<A__Y^}BS4H*Q%<u4CFUY.BTA^bE8$8aNZ(0ln+W3Z(mjde),F-fMpDJ5:IDlS:{Z8y:1M3W(NH%@~9?=EA4R]L2khxRWIEret{QhS?v{.,/YmyLtNq6f%9`Kxe>;}=d$txPP5O<o49.6k:1]~RCSor`.PLdoA%5.8:!uPbusw_A;t2)>_p{zK3|GPacgtstS#,6)r0zh^{)m{6vx:L>vG!.K{?!^Ha|FhRj>1K48cw4yeoxN9jo}5ZAwg-etcnC*avDr!,coD`Vp0vGd}+]1Z5/@O%rU}ghqdU<}dO2/pgpUxlLdUQZ9(^eWxZ>clfiL&z3:?Oi}V7fk?EWmu1ilJ)N!-,hx3#T(^a;P5KM`?p`)?hS9Uw>`NB.m=x^`x8]*ETBpDo7mL}D&0RG:BKQ/FV`x>g7>icIlPN|)ekO_ETbSx`wg6-Q|_cKf9;XY4WUTU1OjbxU*dT2Fs.v2f~I,Et$qz+~k^*_~[V85bu5:<}y/_<-0c{,5xpn=+$Oa[8[1),f%^zuYq,Qjsg3Czv0kA3.6#+Wd[;nUxPwy,krSgzFs(=/dZ5fbmm?~RiB5aB|6)Gg<J~.3/J$R1-e)W,hhg_3^S!f2FJuy|SqDp4w?BUx1[_8lgK,zhhN;jv3G/V966vCpr#`_f}Qku6uSN/vJfw|Ee.<Ff$mU-j[}DV*M,?O(>/#rNqznrsC*QbMA]g`&@z87arEuUL&M&[Dm.)HuW09aXa%[3G%}JjuF>A#=1#*`f4mLF9l>(tCw%ZYN9}vZM.4DXiaSznkw>-h^GI]g`[>K.s[Hak2FSN|>q8eQmjlPh*^CgTK}/ow&(,Y{W;N:2y;W6ul-PqkmUM*a(F/DR(A6qRivy*ptM&&M2qbj#EKOo(`(8/aAk{_(X+AVw#r{NL%O!gm.PUJ$03rb(XA[?q2oz^N)^0]M:n|,_CCw!^OY3(+OEmyzB9w:pi8O+4NAHMJPXx<4qFPUv>`g98C^<Hj-Q*jC)r3;E{,U>9Con(r<rPPb6HgX5)p%AGzisO*HW^BG!+3v<>tz]oS#nek,@~,j~DUxUt>4tHc2wPG6!/:B,Xa<bO$tO$JlBm&*el6D|;Du1}/H;|xYNW)S6O(G16_zH>Z=)<>2s&<68CS6>;mNJUiQ<)eFj7i_48y1?2E?|B5ZPX;b60Fv7OmheRbZ]O>.rHw;lkhH@r<m5/N:TodlcXG-1%Ja7ClnMUOOFhf`iuq}}=aEZ<&&[Ak-nV9:tk,Cz@@+tA{<oPoaZp35pj{0aJJPZux`P<zy/Vk8,mTjRS._y.`liRyp@SBkj=_PXs:2{|&eq7:uLO@bYk5JAvXZ|+Gddyc{!,N}a*xJ[;0AW)WSLj@0Cp]W>f%9<)lKE(oV%s=X1/56yO[UqqT2s1Ikg_L[by>V0JkQjw/s^_NT+vdUIhAIg|;6V{uNY,t8bqdV0P*^6)nJOAN19^89JHJ]QAJl9(nPn(Cnq[*I_LLV*6}zoL_HHQqYS_=sU6Au:n$2G#R^jEWKB}T.@mw##/!-7IK1x3Qd;PM9!?FLHOC6z:Z9YaENHaR+$Gdd#[)rLqV[1p,8t#TNe@3Hsqft}k3EiACkI4D2]IC_2M1?ry^OoC]5<GW}ovWXu[*dy.Yj8{*4u@eeH@Sy>/w<~3,T^1L&`Nq|%w0Sp*pd-Kedy,q7Sgt81b8@$]E^E^uJf^dg}NCF5(>d[}0n1m;bP_hqY_xeUK(_<E*+m0&(ou}3(![`w^pqg]/q79@dm-Owo+x4Xm&u:OJ6J0f@jBH3_)~B]_S98<.$iZIJMhBmSA)*gMEokv>:EP7Z@T4v7LDoc+YYL6s|[h<g5,=[V-zvZ5v/YPbXUGIz!:1?Ri-ofkABOwV9b3.n3P<xT2Zg@n#KnCXq_Eu/=-ZxZJz,,Dlp6:]O*tmca,(P~pqAM~MAjUOAAf}MN{gwr!J-*$*Gyz.JhH.s]v=PO>m4edaZ/>bdMwl%_yN*eO2Kj-ut%29onn6I%KJ{:dD`X-U<mm@61xP/H4|V(-/YHo[0cId-aGan`*`}d=<m,_T7Djk20RieYTTYZW|:/(e=s2XnhAs`Vp3>^fI=+Sm#id|ScbCJ=.9Na9<k>nYgy:lipY<}4PMg(TJ;A2?Zp{.&;d~=rYr|e-O0zXW4?Cc<G>B_.ieA%C|>KI_uzeY)wPUX^5%Bp*r=C8fnNqc];*)=,FC2K|8}{^Oe2.0U5>6:3cau!r:Q??z@zOq&yI?Zx;MKuHohrca_:rWA,R}gLj}nGf.9u--,{Y`W{&ECI5C>UkCR+lbV7ZgwcWe!k>k3E1&KrsGHq6F4D_rPUS2RUU$,owe/=*Trm/S`MyBLl&THeaJ/WDj1:H#Y%JUM~I#s)WyrLV[<v8>p$BfOW2[ml/gP2lBlZhDe%Q;.U{u{3$z[fE4tB5R$W}=(7@o>Fs&UTnYx4ac<ZH9am}[&VyuX81+z9gsbLmYtz)zUoX0|#K9*zLhQ,UeH8&Sm{9}wNuEPF-g{qY1S`U,c]B1S%=j-P1Z<O,%uir9Li<6|Tylps.!>_Nt}<,Qs?4>hQ)I!~,f2*OH4A=j|7G*Do+;=7fH<#7?;Xn*4G@jt.w7bQD0[69p9/+YU:|gX=c&zV#AXZ!g,F->{o3NUj;rrm-{q-9m0qkjN/[)/lq$X@0/9:{!$+cTmOsuILf)F=dz<+GFoJR%;Qu[kHgt`x/B+b};LeNa&63>#ifa4YW83:/*4N%}QKTL,LXJ-pZOwTY]T<0+vFAUwsDW./d^AbTUVOE*rGCym8g#<h-/,5kOsIj319uKf=t?)czCGY{hf)&h[-I//UG8=QbRx*+m}<mS$esk3ZjnEeTh#SO(M,((h#WW?BVAE$KziPPao(aX5TB,a9G%~ykM8m}tFt4-pk:!~xHTA!g#X1UY>JG5U)W&6Ql)^1;a-6|I%:>EZ|Ff-f>>$.Uu?Hz-Dmfeq=lP82lF6U9g[qg[F~;QwaMej-J0wG-Z2qA3@XYMgfWQUIogv5a^h&LJ*M}=75w-SveQTz:Eh~!K)T.K<iB^}|`Er{#nj7H;`Nq}1CQ7|{OaEZ`6zdh,2M4%73Ldn)?[=.#nOA/9ir#=3lw5iz!xzz3-VY~C+]gFEn8v>9|5KsYK-}h9i7fT}Z/Btmo>p.3}$Ce?S~AYbzJ2U5(@6pjP[]xEv.,RJu5-BhoZ-,$F3OS2/~ZzGI$z]L&v,;HjHMsA<Qdu0{4DlJ|2$k;1ViPE_{5PGn]R&#H-%cOv[clbV{_[UI*XsVS~I{;d<3#*[~/ycKs!/;}o7b5m?LsLhAb-GPcQ(qM9z-MsqZ#=PY9v>)btgg&iR/)1_^<|fpz8=:QDWZl+3f=Cp%kpg$!,#eRV_g)!o@DT$`sn*)c`>)P-54hSdskcm@HMgGt8*fKRA{#@gw]~1G;5y$.!FVC_RNXLFm0:1|CtaOJ~&G.FLS/x%`DF@9(XSy=U=O0/db:HB1g,2Fy!&)*so%QaA9Q&Z=jtav[d_~S9g1`~=T~K4o96}J=|.GMtpXRHacypQFN|>:T^AIeAa6Ww2?G@He+*;x(N$6&p^<Qy(XX0e=m&1;DN]~Qs}+ZF3S$PAId-npWpW<j.^Q@ymR>FQY4n$FFltsyPAhGs&;i]U6)!al=f$xYsJJnu{,MI9gV+Jb%_{Dxi/:3VY`~*R)7]Jac{y]xhH]:#Ju,bC~I>V_Tj-xSXyE80jo-3@uqhB[--zqJQ^@3<ZZhaG#bD=F:2x(*y_8ibJ{:keZ3t[/eoz8#YkGVxr<={2nWv>9^B?12lMgHR2G#zalz$;9`l!Mb(Yb7gfB[kgju|j6FA}<e$OESb|qfaIb66/U`8>e8=`Q3xNcdp(mY?V$h09}AWo|B%la7x1E(U36bTl*;Un_kY&tw5f`FdPQuo?S]-o!U~mDau?(B?_%).>B+!!BzZd=?iVRary-,,EWS@7tp[*WL(D!&umI}Wc-J4uDM^jz;lUl|&0M(&C<OX0q]`-Q0Ub<C?MQj;V(sk1_X~Dh9Z5.59d4igxEz`@.fV&.y6+;GVKmvy;=mXhSXA{VA]>>=oY,CXSD$AX2A-a<3sX8k#8c}i&>]1<2UJB$KpVY*llqxEpBgjax?m9W;+oVq|=JZrQEhRI]N=2N2$}gz-_aNQ~&`]/?-Bk)Ib7Dm5T;wjqkKJWcUE1h^u&CvcYXJvEJE][96hP*u4BF{nT-tnZB)_3J*g%a{2ke0*zL?EC9;<^2C)Q_G?#xG-b6)_;v-=Y.RhUX`wm*P2d0!%O<XQNT*]E]Mi<3,9pr%WTz_&9IjQ5~ha[BaMZ3LkbA/n]&Gd:W5ZY_yI[BPW--z=vfV8@2ZME$^:O>3fN}p=OjLd=P.i%Rt<XH)z]_bGc7m<0S1OR~R=~=E7[-$/9=R>0#t?X=ELak$^Td8vR/0Qt+<Z?B3o4#0[gakzSjZ^NWww>;{=uuha@p[5VY/v7xLfGBRKz,lR-$f|ifYOaFWzTxcSl0{&/a/Oc~rcD0[E(&>ru*j_n?x>jklE{SKPzg%Wz0:?FjDd`YE|(xV<K!ByAxI4m2ji;(Lnm2e`;LD#CuR=YMCGC&l#S${f#q@V$iS3s8z)&c$]Vo5n5n11]rIKQe^pkHD.98}%z?*CC,lYHUCB%ye3!?T0o*y%8Tz,Z^r~*i9&k$=G/xN{d.`&(EA&jM-LHC$cH2VmV}A`Pkv0G{]kRl__XsD3HOtB%cI|/9B)pDpQ|[&*gB.&GLqm1Ll;0qoMulX/}()l`j-$($+ks:FYc}y&-HN1qmojq@Cg:mSGo$%w6XRG?Ca`f3@*P+{,2LZoemyL?5uat&itR`Syek,!LKP.5lIJunC^]z6l`D<RR@W,H2yHbs88CfGu2lz8dz|hGrMP#f55SDAT:dHIaykZj!2+XzIzY5)#X41[=}3F0-_Bu~g}V>Go_Ffv$%|w2rK|q{loa1@*M1-#|wr1YDJiXEY$O!tRxH_$OPy&dMI6E,;3,AcXU-&qm{>Eo@_O2-QAmL%CWz2(#PU{bYG^@d;&;2u)g|{9X.V;,%aZu``2xAg$+:$qw0RU<1tC&xDNaDT74hjP+u6q$M1_FA!kWau-t}0L){bUo%,@|E=UOh1)>,7/|Ryr[?_@)5*:6!TPb[x*7K<4T:#kp@EJcvw9S/i$[W&xcQQmr7,I=7Tw.)O`FsTH1/}~5EFjO*{i(OBez0LUZ2qL-+]e;G?;[9{FUvEA/6rosVH.-D{jt6cuuuXK/qO]6>%N[Omiqx[f5-!P48(xgb,}l-fH8i(+r!}dDH/8[QU)S{[jn#F!uVLKnw1:$kCd^=COSativox_45McoVdp7L*Bam+3ZiqC<I[M0tUPtl%rYZ@ele3.6yE]Dp.,kgx,@9Sp/IEN-AMJfD:o&$t;@;5YEpT,Fmh{S<Nk|#@%4,1Lc?zm<`8%}{f/<x7,VwR;T1fClA/&0EO!N>Nr,a:5iY/5J%HP}>U8fleyB0XRyqPi?rZ(RoBE@gx3d<FtH/J2<_f8Hx(njTB8~Z>,2Ip:R>45dVl%?RYQf0qoC`2wY0XwOFRW/%?{hrt5+3q[`PO,-{dW~Y<v%p*Ik]{d|^&D3qcJ>a#>|iu^O@9l5`)#CppgA7.}+?[NItgJTuov2BucL~9&I3<4gn7IX/Z.#wRud5W.^6|]f51FJ7--a_sHOhJ[mM2@lcxe](8_?#3-Mf)`S#|,e{FxV-J#>aF(zwxonIe4PY]Kt[xR+<40EphQ|``]8;?r_*5r?,oVM`pe]sQ?Ug2B/#d&zcBk6J<w78/=6PNX{]i894,@YU$cW]NJD)7+HPSB$9?%}c{cY|@jDyTFwPJel$tG~_d=hA4~~Tkk$^^,aB[+LieNn;NBhL.sSVVQI.Q2a6H]?j}B=Xu<zfX?Xn*(pobvVUFtl#[lv{py*?2dKm=<?:P;*Q1/!~vx*i#63CR76~}J}tzg<w1rvgA9dJqb+*WgYVTPEkDPtphp{_&Y&.=P<2nf1;>OXWZ-OlHl:IaO?0,*_NM:tv4]<S!=(V*]g|nzrj9lrXz8Yc0w/D)D]V-z,tE=*84(UFR|!U]2q`%0evZnodp$%uAd-MX6UemMeX.F>!*>^WhguE.TxQoqxRJd4<;f]Mj&`*)>Rt8j-JT@siT9|!eH@VHex,c=_JIWV@_y8}0stgM?Ebd[P9{#8XcIprI(@gVeUP^:l4%y;n>`0TIsr}RfG3$e}&WG,p9#MyUKHUFZaVF|<QLeh%MzGo]$>d1*Pf`(X}o52GuM|tVDYU=wc]%;Idg#,/GJh-Y9O*wQ1J@p.F/kbaF.giz,74njz3Oo7IvozKb/?aTw^cv5bwV*^M5U&bT##d0=hn!x}9&&[WvhfPO*Z3UuNJ[X-t?;8gTfTOW#H%!nN^$=9P61v47(?~MkXS^sBa=7{QCnhzTSyHB~_bAF0}zKf_GXdL1_`B}elIS>/3Tl;Vb4&`*,77+XG:Ac!/Tu/G,#Ikj*jFED^-,O`X<+hj<h!?3k}+4H_rpHu%_s0n#vKj.4]N33ix;ol?]B>-AqeL?u-B@q(+#Kn*&Hq?4b(j#uVeZ/p@<t(=Srw/xeFmlVE$D~J,Wqjzh3Zj@ZQJcUB7aSvsYF#A>r/V12g^)A+7/#HZ%N7z|?{;q]/HhqZ@$LJy/:UFvX_I%3KQ7_jGoN7%J,;BWd,Hb7JjG1Sj/(lq:.e|P$G.>xYl0st!Knj.a;Z}Yx=X1TN3R>:Y]q/esJ5B1h%q~_Hjc<_IrOY+yc#^PNh+Q22#/Vuk&]19F(`#jp(JoMT$<jkJ(d8s^IS[9G(uc(CbRT|u=/{sW%}{]hnN_8^7ToN]6Z+V-E9|HM(7/W6vwjxH/+T&gnY.yg$m.mbM7vz0<&F1qiYk-;z~cTZIOng$wEo$*((63g+aKF)7M)]CEE%U:l1w:[kf)q]^_*l`7nPxb!>Vp}ZjD5^{),#GYKqC&OzNW==+h}(:+saALl)vdk[uNDG(gsHqP&mJ~oe}h06L%gOTm/nSsDQ(4q@3>x2{$1:b{NlNG?NH/DT`z@68(x-.C$1_|*0nT&.:gmgw`{mh/Kz9MLm{!vg]V:tWRe-)R*DyU:pCaZ{3!!xC-*@:?ZxKULb^FT%yILKOuPJP/of5*U:F*+>@9B?=p(ka&`bi*SzZC@R?m7RRr<#fDt_f}7r}Efif0LQM`.|Lu0MOm7eX7(~w5,s?~blO>RiuL@N`oY3{3nsiBS{vxGXJhE&G%$,:N5RAq4,kN{LohQ:q7|a9mW##=hGGpOs>e~Od0&HtRl6gI7a5@u>v-/y?jXF]syhG;X@q%pQ[BAnV6f)Ik*-X5Q:Z+cT(t)aeYx]2zFOW2YF53ta#(yOY7A]f_,_hd7|/WK3Z&f|}sEeF/AA1Ag|1uYu&*{QD0AAX}giOzVMyT;Okc%(5Jy?4SRc$Af:<&KDdYec$-iAe+<ztXfI+VKo.[B~*^vQ=XVOMs<0-<NV!Dc{:d:+c45r9M^{REvW@e<m.4st0;r1=%,CGemXlXjQIh|Ji,+mEpy^=|+)QS6??_hg<6D`o%$;-xRd+f>u$i[0Wt.Pnp2hzo?U&2$y1LzGc%z&-r3f}9|OXViwIkb&j35dpZ/aEuGLf2v]b_W1;NppbWA{(sJ(X<fahOh&+vkb*(;;;5XFeuD1fbrr#Vl+t[MD|idg$[j,6MWqs;53,7xvaS?r~/LNS/8A:|Njz6)$lBy_lA__gC&cHNevPn-Z_q+cG>!`LTsL#jz#W{fi`Ot:233=a:R0(DrJ{mkPKJD&j<!O8]N%zZo=|]4LQjO$p*iwJxA}dbslqZuw?~(kMt!|EAZ3Yu13u#gX.%tx{cdSH)7cix:raM%/?*i.6O$reR$e2Ln7FukWK$_9V^^x?J)|^ff4[uk!M^`u6@]>M8:0IXXuI.zje|@8@Hmi^}93xCUbk(VJQ9t?e7c+:#89oBJRa>!AM(#fz=sjr%UN?n]|S7b@f<6~+B,oXpd38(XwLG*DAA4@.Mqxg,~Gv>K3nwJ&(|,Hpn~(L:/]?2g*O93cJ2h^,Pj8#nnQ1},WHb2n,J2qH%}psiBY+^{pn)rXsspnkrBxNTc<eVbGq*ExciX,ec#SXOQ&0O!bb_xdiCOaPm$JYU~k?`4BQ}fki>`N36TN-E:O9=]AJ#@f%XZ^0LH2uL[9caYK46QDJCf~.#-f*h!U!1lFivF8#/hhxlGV~F3S9h,]sz>Qz([>x>J1jq&WUmF~)Af7K-pRk7!q~C!%zSFicV)Z-$)Q-!G,qeZvw}yVWOq`^Lr?~_rKlo!x%#a6642U,?fE5[!TE6T[>WzM>[2WG3$9p<+h|@EIOCXEX3G]a:/p4Jt:j-cSQTCHjiKBt#k4vE^RLJZbFxn]Ny$oeGLhR}+YBcPY:?JIb<4DPxwD$V1<^?Wt!hw(xYI8niCUJM<E?w*}Hi_N-}1?[R}V&Tk=rtZv0|wZ6u,a-SGrNAT:CTab7Y?/kvlq`TZbM:V]q0Es/t_TGcVC)j|Rv);>f:m,{WlY[oClRs0iOeXPKPR|VTnY[vqYwa^3.,I*t7)9pi_@HK17U3.jkqd2g@c`Xz:M,O7p#QQbaI=i*k$>a/iTfxw[+{f7/K>W<RKm,AktNTmWg}Jbg@ddd3!MswRzOSvwP*KuMq8GA7F&x}MJDu$j;ftptFu`0ZX_vuK2n=F:/xp^s*tRwN}TXg#%wC&Z5AG=1j-zMwe9M`+QUG@WAzUQ[L5-zb+.551mU{UG<3XCq(r$H;Vq(zb=Y>Ro*_Xe$43X0U)K%g.=}w4;Nd8ZW:8K-h6_ezY`Xz9ikuF0PWOoUp@DgRU^I5=]Kr4xYvbV%TRu~vG%sEZI/H?J)5~^x%$AVOoi#RNVB0Sc=imY4nT2m=>?DVr{L~6D2wGl>tAB[[4D(OKEU(&nBWxi$@@{[M0C=8ITI.Nm!vU_%]-23o@VuD2gxuu#V?V2:>B1(uf:w*]kc;<l`|UCk@<W0L.=4a$v0?3u<-q^nU|AxUXOGRtu#Swe,0Kyt<rr`5j(/Mb&8Q2uBAE=~,$yRDM4(lG{kho?DZ5n<Qy-FrV$ZUlVD6n<101Y2b8osOcI6d%P|u?7oI-XOwI@3H4G=v1<dp1|`8,Y4|ux2aOj}k/i.Ug`<[+8j8JA@PIae{S]<Vln+|]8i/6eoLHyVP><&W/0TnXk|/O5U-p/i2h?IT3a_%H$2A$gaOQ#qR8]NM6)N%2sW>Z^yd&PSzx5TU`%TOMLx8:)HQf=+V~u[NuAzuLC+68y.@[>||_6dx5&lL$z>^D&5vh_YH,#vB^*qf$5j?sK<1n[:VbvoIa5fAo9wB?x8h9[!F](zH$!|gBLcf=,p3n%})oEoT}t=nQ.:$V7Mtvk/aS>*t<,Qq#=47vLo}GKyZ5tTBu1w8X?T:d:`P5]E~k~sm<=@V/A6HuLzcbxAmmiA{PTJA55+h(K/t0J@.IUUC:nU&/5|nt}=BmRnejId>;;rQ0JhUkX!nDds2t4gcC^qH#a!P#@}<FJ{rG8fv*[-Y?C`e9ZFx1g!jBV,S?([yS4zJ00|hsQ#sUPmj+OpX~o,B1$aev:GU+S*A-NEl_s(nx<%U_ufjX::UR]VZzlczK$CUrj#Z,2PYegT|xqT2D4Dih.UK[=fs%GVL_0VA<Q(0O0id(_#=7b%gv~)N.<3A[eyA%,0b$=_VB;H_Q$QbE;Kwt{aU<@>Dxb,h![o`G(g9(umeGc[QKpQ?2NHKH=Rq7mm$W,T[7up&GUs+cURpkwIs3iV6QJn=AwdbXgA29_uS:8f~a*u/{uBYW1wpo`Qb#~v-{Fq,Z5BIAD.c*(Z;O7i2xM!z=|-.9;x:?hkBvKr@l#1,#m4XN2%cK$}.a(C(:`<9UNt/GbxQQb4{/eaO9d{fM_IG;Bwho]T1R%UEH0tz2G5P,Rv%~ymU_$6fP8W4y,*b3(O]xwCr{v}D;:s*mO%5}tW9x5B_E_$<Hb;nN7?tQF1yxPn~kDrfb&:s4sk]ks#&;VI{+g;%Su4CXcCys6(.Jl:!^*Az<*lXLe,M![@,+RSOb}t3/8&fd$oyv-0K$K)I5T!P2.y&6_Jww6j(b-,=dlYS_WI]Z5BPKY]3R1L>QLvuYRR@QsZxU5~q+EOMnX}Xeq=5n{j4b6?DY0K%P2nRowt/?XS[kgpW=SVBBJTN<0Pc,B[1XqT$#QlT{2RY8%aS+*.[GC?wfvj5mZs38*S#%g1`wt*Tbj)+9g3jy(Sx(+6%{Q/8MfiD35b.v3I04b-b,/6dc`r`Ahwu6r0h9Xu:Ad%xbb,baX>P`B[:J*>@~0q/ClM=j<<60qVlO;UT]~H&TWD!De7?Hy/3ovh?a&y!s;K;9C3_PEc0}ch1OjfM91rebZmL7H<;`U_Bd*yS%RF7F&<VeAHe-8$D0@cp:V`I%Z5O=qWdUn=nJgmD5JL>-;W`?1c{Vf5qF1Ktbu-eor9GEfe905cFMi[gm5bL~>!)cjx#DqeK<b=xF9(zVGMnxvjpA}xU#117u.N(R.]/N=R9G!DO.A<W->7o-_dd,pNAvXmgLg!e<YKXo6t*wR|#?m_GhDE>D<eR,`inrf*Lxow&m_&K]],/y]{{1;QDhuOj+59JXnjX<0;lp!,rxQkfo{Cenq[O#1Lh+}3ehgfXuTwg4:5t]]0VIS1mG5=tQdg@ei?6*Ot9e=rdC3&tqbgVTZAbl#,G3y.6CU7;K9$p,e<USS+tZ8ToOz5(<J)@8!6u*-)y4ADHL;N>.b^5khXikoDB*k6SJaSk0p|:J(=m4S4!EYTcL&ag}lGnIv@H@jS?>(KY!T1u,``cxt-1f[qRKK;u^Q_}/?I9[,yQB>(XtL^>[QX)g=>Eg/HWO+j*mwJrootn-/dN}wIkOBo#9Jgj7}NXSDH!7Ug7zZ.!TiX9MAs)r`r$l1_R^kG`PKivPC[M*(Z?Ytkoy-+$13BoTq{][JCF/96q<$p=Bi-zd]kkGe+X-(n&yyJZTJv#Y1@XA/6`J,2xD,WWG}E2:zgX5n{v^Y(@>KK*N;QV@`5V5:.|1A=qXQ<m/Bn;EDAMzYhsOnQV[XB@VYpL1^#EyV`In})zxGG_@h<h)vcU9N{r~jCh#(]L4}z#lN}{WP22bk2nox5(^W+?!_R/Z!KKGkPZ{=L=/pyz(Y&Gg3yFp}C<i8!c$g<8~s=ipDM(2!K1m@9tQ}D@QHZ>9SO6lP/KIp+@2&8s?/p.S+ex~HQR7i}5ImL^SysN;IA+4WDQMrw^#@.(3`SdYMvV]VkP/o<3bNtI|P3<=9G^*m*@a:/8PUB{o[h34`xWc16{@#DD2vT0rr40ke{#GhsdEt_,^N5`mz8U%s1<wl5WmX+A]2Tn.9_k:BoT{>c,Sw#?HF61QgpgLl#E!g1ABv9[nLf9jLn__uY(9ye[(p%S=BT=dR[^@nb#VsQ-Faf8cKs/~Yd?WeK?RLQqq}mCzmTA*.KTQH)TZ:i.bd#m.RieeJUO;~M9W=^<Sw1jgc1w`%BNcT8O7nE<GZ&EZjD2i_W{0k869zPQ|VGOA|`RdfNam_2yT}GvY[+bqW]*!A(?x<LeA/=o}O*aV??*_T^[i7XX0o`}ypGc!xj,?^0A3{tts3=WET5usUjHxtC(,m32d+>B`ADCXB2kUeN7Q=qqc:&}=-x5l`>U1/`tjK?5.}WS-]);d__YOlc:Ojk-uI|8M4lSl;~ktdnu.wyt#b9=]}$DIBaD2ljSo.Y$X]o(uVr&SN#*}l0%vZ0/$rxOwCVi_&wk>5hi0B:LQ]2k#UoWj6}4gZukAA/]hYH.Ojub[pn{7)#z%|P(i1Oa?dzV63&of]|GPn]%yun$e11c4H]RYrMz?Jm2H8l&|eg|Ed:pnqR90rR*Oh-|?]&&0,R;@P?=$lmMAkEH,;3t~G1Q;`jJm0mT3r-UXW9L*,>iMgBF&R_Gz`1J?EF!UrA|@%C4yw%DZs9cK|&}H5se9v;SFlm.5DW1fsmmL/gIVLM5AXjy{zhvv7B:)|/2C|]UjtNi_a{0uDTat^4)4g(/MuD-Q%_LaXp</Dp:;V>|b(q6eYi|pS#4Ie9ON@AXZObP>]:H!aab^9,:.!d<*IA1,)YE]rsWzBU,HYT>P|U|OZI)koQku-k{B%s09LzNpX_`iC#<RfuwN!Osax([wKI0BJ][q]KPWXLk~7u(y#Q_|2q7wc>m:gXb9#AO@3SF_1]W+)SOA:8)_.Qx1^10R|n^AKE,{cgyBc.zYqZ5E4CxvC}9E<mVJ0$~x/n&ubq^2%EN=(.Me7bo^B3KDwkJ#h*`+Rno|eqmndZ:O$FFsH?>yE5a^lX_OsPq&F6+aJ9[z:dQ:NNMw6~#bj3;rLZ15*lWb9=^$lQ{&05gT5}v2w%z$NB*|:<~WqU91K?UTQ3T#x,EF$UNKWveq3Qy1g%l2e~4h4<!Q6j:<KPo&0Z.7OFOsVL)zDr6yB=R!h^I|+%$fk^>V!Wlh$VQ(>id55Dw|jXJoFRx8NsF&Em&~ySYTiFzMlqA^&9Bauk-16m0@6vV^JbE:&C#WxY%$@pf$76MeQ@eQg7v(^>o@+#u4O,s=tt.!`pH~eBiq$u<2Aa>)7NRSutW;tweIS67HHTc85]d311]#060L$XCOm2p#jZh&fqai*6EJt)5b,(L+c~(b7#nIZU8k[qkM]$MW@oDR}/jq<)z]A?9$wf).Hp:q[(`6bU}u%vO&a+bn%ov%B#:B=~+fAF$p/sj.`u.wVon!z+ybG+:jl+T8.g:NqC.}zxy][5W3I(8>A!=&4K-U4c@{LT&gRlJhMr@K^T2TEeM^)[Ma};*jjeHLek/l0;h4n|Z0HjIu^Rk-|IFl^=wQ3<|T1V.>Y%Zho}=QVNsx/xB%yCc)Hz8_6B;AID.AeW+@PBHga,7z+crUtL)JPF=!NyB,I.c)e1r;}}Z,veT6>3B%%d75`~-jbx0r}(yMINF+[v2={ueA+v@r]WMgZm/rW,J+tO&RC0L7gK&Z$g8(0Cp^6gj0:)b?Bf(iouN(roS24lDoA)nf2Q4-~6t#1.?]jyvV^*]U~q{SVWXf(:=U%[`Hf9%@M0%g9I>GBjW<CpMt>k@5C*<n0{ds:J9H=-T^Y/ty)2coo[>+^mbg@JN7peNd>}SIk+te1C*e+-z{Ik!AkepI(fGS]^C/Q4N-)B*a[=_xpR}6o$4~!ypS#.tWtP0Tt4$dq747gK(eV{G2@zhf_v^Jdfd#t?%9~-}f`(wEav~:&%g|S3K/DCP?PvIt%`o+o`A`=T^<F~#b>I*}u;C3_i+5~d^m(5?h*KMi01TQ;rM}oFJv,An2nDpDAOgp/^=xZgvo`C@LJ{4|AjgV|>v<$cFsL!b0w+c@vDdo)5SGhpg(_[>[uvElia%$T@5GP%;emr4OKmMhKL@t],K[MguHJx(:&WD]-QdA4oi,:y=1[N8LK:oRK1_z3DwpdGrJxJ5JXY{k_u#i.P#]n!}wF/}u<3JU=*SnhM41YtQQJ&CQ|vox^ynl>uF8)w&bf6[n3!9|rEu)TwUh*jeI6:ns,Ugj*ykgE]5h]2OzjrST7P`zk$*{m^hjEd@3oRrFcgY<gV,36+hGTiU4G21&p#o4>FQlE4qY;O`t8*ay%v{drVkQ[aWw[m%n11+C5QENVYTu19q6zgk7(tu-@AyuVbp^n,+UFr_a|QcYKq0Y3l:Y^Ov%`bw(CGjP9b)?G/J)-a#=tBW.Rg$u~AkHEi[n]quq=h+Zka5zn+lE]>%@y8.r:{FE2u/(:gGM]q_m7-jhqJU]~N.eY[C8v:_/kk6gM[-DnS9H-qr(>ZX`bs0E>S,LCRyY&TN9r-e(@1]x_[:zX4ZqXRwW>:I_mxgf_~^A`|<b(,r*612oT-_Dc&JCB)F[R{U*RN]|s`Wf.d0t3B&]JwTJqtg:YPUC.`hcI=(IF4!4s)B{19B8Bs:x-GsLU{GGsbhZCU.Wrq-yKb}Vu6+Gc6=/{Vpo^,#ANb1(Gr0*nJyNL(h8E3.,,]fSJ<Y:xa658|]xWB7%V_+&gmuK{ow=sxA4JG{G5dI8`5`)N]F*.Y[n)tLn8]y1u@F::`,Vy,T)tVD(0qb4I/!@)G:Z_SCmbxQ0*?mc&{%|-Kw.TBj~AfGu7Vi}%@G0t3wq;[QY2U|Jc67D4-m;A=T2_rh&}-qumiJFVbNg5/S$/f`r@`!X#F7e%`(0P4}[<{P(um&_+W26TiIRQ-?tzoDMvgl;KTq/5u%=ESVhnXas!_z9Ti,f6Muz?!aGBr-2n#isNiefz^5oA];+d}J$D#@]Of=KEyRI|n9gg.i6Nj9aWHDFK0,9d1n:WN8i%@v~.4sVo,zI{znD06M{.XZu}U5^Tixr/8zxhLlVEko>;ZeInHKmWRe~A|JSJ56OCU$M4r-NfDglfn!K]Ba?n?KI%Cyo7R%6X+q1E]$*!iQ9r,,j#_a~|!kunOmf(yk9QHdhPI;BMUzJA@%|DMgEW(y{z;2&l6Od$no5D<O]v5V.(@x7lPf<Ph@nS:NSA!N<NZ5TSc))KxTFxYwD@#Rc__x#D>.?XcC-G@/__J-/hx!3LSE)QL^jkq^6OO<B;$nD<Ef4m`1{C(xIzb`E7,U}A8Lhs.e|TO60?2OfShibIf?vQ6AO*v*Z9B!FUZ*@f/aS~M,X`6k%wEm6dEY=cdai/fBp_98@sOe2SR:Uz-e6PVrj>.#>D#_]gBxB=TjS9H##cOM1T:Tu22Y8#N&Jjd{CZ,TwZe[?d(Rm*>}vlXNgjz6SiWfM(C5@v[[8TI,Q2^Wo0uGq`TI.ySfv0oJ*fDm$gg{<lD]Ii7/Esn|La*et1H>x/Muxh$xY+vNG^uTp,A@j:Hr~Z$_LXsp4]dZbJYf{(96*$8n(0i_NuG8|lyF(FrIRoLa{=f#%,^`h:%5qO~,3l~.}5/UPQ){)SY6&G[>~xOSm=2SuCxlPIS,p/lmi7Nk>Ch7iXp/a:Z:$*cvsqC$%27K)xK|n:tk_w1Fl6k%3222jAt;{)4wd3Ru2B$psP^g!1eYf)%tI28rQ2}/(;0R)+K/|AR?2_A7Y{)mX%0Y7#.iXqLVRgPbD8L)>>U.!5GrF>7Q7f}ZQ$EOY>g.8R#6VXt{$FCZH__dsb<}U4+{UkB{iht-8;(9ywoPuK3/F%cTn|plzTM1^e!Ms&7Af{89@`zk06}jbY1;[1L<cA8c[Ypb)2N=]5u%5i4lLfrvU|XkXguj?M.=!Gy+B[NHqG$?b:OX(a4b*4A{i7b%&2l8xu3&0c%b[9M=V9tR:wYMVP+Rs3MBaWtwZ@ao2%iuhL2Zokf7[N&Ua45|5g+[R*_r$+-R%0|k_wtdKJHs;06~Gj=2I(,5rm2>%Z?Y6{r_8R|+#%hmW/0d?s:/q_r9+MVbwT4GB,Lc6?@^qtPg$ELz`w1zHL##3hSx>nDdPVjHj2awSw}*w=azf1ZS=2@wSx1F$g+aFSmS:*^~905fw@!)ra)Z>L%p(4!l(VSV@,<HS[&H&jeManfObCuX.@F@(aIva,YTgCd,}[@S=.l!GHl9:XuQsEX:;Q?swrRGT]0G-d%gHBqY=@6B@ZJVUioeYtuVP[YG;sZ:b_YTGMe)r,wg*,*1xpR#ZA%@Nt%}xM?&8kCG9,I7V6U5-p^ZZ;KP!GiXE;_IBjc]a=MOm&z0|xBpgjDc|N87:]=PjF>fS)~!n)Rp!j-*iLIv^eC1Ehs{%Woe)f#P?xh%i1<%<PwOfC[g6Z.MJ]]@##<yhhtEcGq[qwh#%75,2o$_.Gb`8K)#vpL-p0yzZb#A=d)YzkkCPMpfd])BwpzI!zZf~}7-I_~K#49J]?Ap^zB<htiu573g.Bi4ti>XqJ7Db]0TcRdDcSBlT`,yMKpk7PWwJ6[.<i]6R(b*(Tu3/#:XAz$(;9eLPyt&%M.R?wQ`>[+v_fZV0Z`xj2^j5q*=a3S[J$^WZ^4TQj2t0+(^+^QOzq>S&veS*7Ujp1$liAWG@A@B3XCrZ$/S%*ty8FcTS`bneP$N!jQx_d:?HzEe`6ZF_If|[%I$zAh4ZLI:B<-SQ#yCY?^)+S}y,/2>&`oE4=LDuiZ)E|WuuHDh*N}])(DsoYB5c.R&@8*OBa8}t;5R:k!vb^geZQUywb@(&bj)Hyw+Z6~(}B~LQ,M/X3uA][8R/wmeww@]gJ!7j8r2)M{Dow=-+0H[~c!Gf3=fyn0eqk?fQ6{=aTne8fu&n$DE2_3:7x*3A{H^V,4SqhQS>_Q`eu9`!f7s-ChkYvV%t[S8OsHeZh([*AUu%hPh+z};xLfGo}4o,~(]%65&hnPlNGFdJaFgC.z52&9)`x5#pT]MoAv@R_|(@?k01jSwpjTE$PDmtlYv|jd95:1ay^!#Jg2C]34nW/i+rqgMb]af4(ph8I9ymSsC=NW{R,cXaBO(yWrbBww9p&wUqbQW2qt9Vc]Bk)ItdT2Y;7y++%QPhl:0VA.lB2kq*O1J1;]uC2rXZ3dPblz@R!^S8vjDPh]MCytt@Yh/tr[`r&H^~p*R8+v^fgi7q@XH)tkS>i(9Rj3c7^E$F+>5g7ngONDRdjO3L1Y(M.reqoI1!mq)+ZX4.Bg3;5b-K|F*1V8hF}j/XKG,vx55,i%it:{hI7,%kaeW4{d)E_R&,Rb-_]q7kSS;,],IEgb@!f}c8JyGnK.t_tq&Vyok&0s?;2bDhgf$ilAGXK7U-}$D;.*_y:bScScql+}33<gQ?e9=tqGf?ic((tnz?N%+^QWl1Tnj,VjDLOPpG8H<0r>Dr`nD}Q(wZ=odA/V[%0zI%TeYv:<dL4RhG]JItCTKhgWEuzIl1c,>m?*t8Z`0y$W@^bf`R9oOI_qe]5zCTDln<[Iw&2d+kCA4EVvs[n:I[$+$V2I!IfM1W61S@{SFQI7Zq/,9:8Mj8>1fBbO_e+Bi&1x,E-g8@^u3bx}qb+3<&^]&M}%XV6W;,;Z%2%I@K8w(C)?*$-<#{,@#8u9R3twoX)YjbkmL,:mwJgY&9J!WHFA0%A:7|kGP-)tmr-?oHQQ#9iwKa1PNU9tIZF-j(x~r4Q6AZP&M._(vF9r+S)%j$o(P&7PR+%[x#.tW&$cnztgI-}`wC)5CX+ZS0#uM(/Kx4K~=k!YBS/}RZ?J%qCl2a(whDunP8d2a@#Ba^u117%%`}J%m)i),!ga<*QCG~q2fQDj$;_W}ZW1/dj92Z%C0c$rnpzY*)eNKvq#6HT:WKoP)arX~,SsS)uFC880syaIow5ZyA%NI%Q=P5n!Y2&!fO:>A!a@%DG%j9(+`eJ%2^e))%$3&ZkLtRO82`z8n9BkzvzVYR*vhv.:>/Y{2)Q/$Y<D~OMZ@D1RD>eg_o-el64*qnK}4G@^}H?zgj$pTSQUWVuT=myQjxMC95d5Ta^^w,+!TVD9=Z>NejP|*C0VQ-oe-Yz@CyB%Z#b.3Wrz1zAyW&@b0MoO_8UJ&7`DypwAzT)(8m_s9IZmcv<ShPcikw(d+`0EmYa*p>4_!Ji=8?QlAyR}!>_2-r1KBA-OhEHs#6{[#J1I1U9~w,T)a`2t](&~eG*?z{e!|fwgj8/z)QUOUH<tyoH5$UkSc8bUMus`4l/j#p$5|l][6ZP6_cEu#]ydW8,c4xUA@gL7Eb;N49hd4U%}+[/)Lz37Ay5PMdC$RY^[cSi`T.eg_Ig6J7KPO2TsIzlFY]b+/o_;K~f3+5BhXhiFc_zSVpx2N6xNsdjrLd9|3-a6.7kcDQ6C6pLL]|YO+8^I(Bm/!#=xb@m4JT^tIdxJHcMT`|z/V0X.Y87|-8llXcY9:dY%W/bk_8x>aavyP@oYhcP;:Z%{N=`$8H2*[/QD>aX8rm;Lw7NN_=Ex/7d|HU%]&>*#:VhyOS[rQQS7rTse[sC4crd!1v#N3vd$_)^v1A]QkAD5fzsysM+^cqRjpAmafU/Db+hEG=,ry/bc{#1XuEcHAs`73L:EH}._MWK+:2kgqvaK6_Pe7<vO$EOJpMWB5V,UQCuz1O.B`XVZ5]^I7.*FRqpF|XTx}g;tTlw~7B&OjsJg!3McqU0U>0ol*/2cprH}057EWUcy_TYUx-#c~x:sMR[a)v&vwz&<[UD-l}#MgqM$~o+|4+2J[sr,]W#WQ<6OzpIAp:f9@0Ru7]6k_Z](ORNZY;F.+A2vQjn}QUm9/|ySmbFeBl6ju,SEO.K-GTHw,YO6ioEM6YymuNQY=z4(}G&.S8z%}}I_2m@h02$0IE(G!lEcoD/OH~_OM|(NBa[|o<BDNP@*@8h4)CG,}D%.k4veyrWz#5.Y)up3`&KF28/M$fUD`FV,(t2I{5@yY-=m&w(yHSbWl>1NX)]965Y:{aIf`2.$tlHDhP6?Ap&X-Q#XupXJ.Z/f:>czeSeU}p0T_JwkD(,vd72.dcOYE(_TK50.%^Q:Z7<S,mIRp2g`oat6KXRX}BIEHZF%udh_zEmb/3.8FT-mzx9lZ;S^!S[=aGi4A[Tdu-g.Li5m/eH31#Tuv%!Zg(iL[1Ngw66UQE#mhu58jm@gw>g]xxYd7NXPd9~QAl)T&5r<4|Nb)O#j1o=FvXHF)KG.DBF5rg1ZH_}%.(6apDJ3}dwfd/8]i,Myt3D!O]=pc9a[MmTT/[2Y`76~HV=j^`n!TE$sluJ]fz4WEDTc_i=%:uxpNcezxxJsuI`AL^lv)}`jk19`KIg7#Jj)9paHXXYqM%0u1HFXCzfaJyG3>u(CT&0$G1H`2Ri|hhw7fr1v5NQ<z_54+g/8BZ.nXn|^SFz#V4=uzC*]|3vMNz@(N@5aP}fz8V:|v=!biDzC3wrA^#R||+&7eW(y3GE:+[]T$3M|9fNq%!OuV@T?M>doLV}dvYhqboL+VLesm+&y1+YWpbbA{Qpv*{n!JD*d}oYZp~4*AhrD+$<*D!.X;=?WB<EWtUz:YPj52H8DMwYyRl_hXKB_x<2*GS]z9G4.7&O:Zs;9P|:XFaI-HH&X^#rTaJs;`:3Cb^<5qh0p;g7$E<}$}:yTW--Q{WO4e%qUo=@px108y^Py40(&P4<6iRvWrU)*HA3W`)59>]W8j})0jej9!kty(@YgbDB+E5kNJmKk(&hd;7KJ>urSAGR[f@Dz;[bkY#C`;a>>u=uUq)halhc~Rx@:;Yoz5FoypGUN}Z@orae,9.4Hq-/Gr[)pY]6Ze[/|DyL1mbTBX*>=T.L}}PJ-Mw+z}{(_h5z>3RZqsQwLJ=4*E+h4UMhfyvGB6(iPhHADWd@aTTM_>6Gf&%kd)Y+}eOl(pO?gXy<8BxkFux<:A;?LN{Z90CN_3p/_o~XSnxhmM3}t>0EuW/<12SRYUyJGvQR@6<?21Ebw[yS(A8x]W@REb,[A3(MUjs7Xs>P`WT35VVI~I|yrHMw={Rsm;vfkM/euEjDt0-f_5[R,%3jPYYbV0YhDgXi`F*1#c3`kp=TZa`f`Z>m:zI8o.]-4CC&`A$y*_%pd_z}a);XSwtQUrSm#[+>iGYv_wlvnOOfxSw5_(,`?-_n(Umu>4)$*}`X%ItPnl$Z`|~6Y:AUyC{hW0mN/-~vq$$8Hr>EnE1/I-ul8fZuWhwdh[P,Q[OGO,|FELf,B${+&D?V7e6`$y-PP&szxcoqg-T!Ry#N3o)tG.u;w3*OyRCo-:|ct%=Q?&L#@k?9K5S@gL~r?]6/1kxosIs&@AE#)Mx=3uSHIC_^8EBB_7h1tt?nH>4lHsuT>Oxf$*dW5__W@j)0?M*ijQkCVm?7E-fKo&BKh06nWcl{3Nl-?k/#BlI696`H],0VAm3s02s@Jyf;(duv:$^ayb=TX*juL]!bH6%RoU&mW4L/});gd%o:#)h6Dd@E^8qWiNFG4Q>lBK,*Q=4u!dLx!,FOPqn$_9EzzLsG}Thq*/x/En_lDAc#tIztTe:DPP[eXZfpfL.DB?!$$*$SPM+u-1b93+IgA?87xki?`ao(!)-:?RwNedk2[?/oJVGioVQO[s40OG&lIsHrRNBy,&g9J=]jjYInUMLnM>e%t6L2C9e5d?&}C>dbZR-nA[E,GffxZ?L(O+1k#yi+=%#GBAy8.?a{9Q`<?kcMeZQq&<yY{f2wQtYLKV*@VK!U|)..`.d<ZfE@1<|C,5Nb_IKi,JeOY*L,X0}cb[UV$n|,fdyxuD=NCUdQ((K+Z1S@vdRji#utUD7;V7NLAg<y+%?=yx=lB}#?dlaR)}8%t]/|#Tl)UN<#~IrF3A!X(tET$zC2w-0miCt4_H=>@11kcK1a1sgI7}..$m^Uf,`v`O]Gt5)F/GCQlPjUIkJe7a,HrE{z(*E/]aEMIFO,`-m85I3S>w,hEQjYm{k1urOKdD[x/NpUbNa[4?Q#od8MMBtByP{mQ}apvHsdc:z>7CRhR@&eZ7xYJ[RLB%o.{turcO8}+#K5/Q?8sl#Ye^gReGg>d`w~XkkCbYT6sGZ?~2kE9KZ62u9^j@Rd*eRiUtF$u;yIUJ;%m<U/S2CP1>d/I#]0i^%?&.jB3-Z<d`~)k9vMo,QyhNz6]L3Cyy*0:4ujpZ@i{Cpv=a[^n4%YgB=HYbkJ!vN:98sW;P3c|}!?<YK?5D&cC%BO}KMAv9JiRzQxxN%pL1*0M=)Z:J[!rUU9$2.MS+r[{mqlfoN7+KdBBfwBr__zsGW[{M|0<k~yu4[fjgUd2),Uy[xZ]xp/J0o-pgOZ|Y|S`C24|ERb_yajSI|MmRB:WMt3N0.Y0zg|>R?BIZ2=%YxHx7]/tFe5ZHHbUa>Q:(y#jJuqCLdLN+p99K|PP)fJ2tdcLSl1+g]GD880*a}BMmK|w9UAe.TDFS8*@~.}XURel+IzQ@8Lua9(k35{q/8d?Ht0z<n!A[d&3.8e8ANL/.hDGS0j8qLv4r|*/tge.pzKDq_E}lStL8O7jFk3n4ckNaARO}uxIttu5-<W#eos=Wvun=qm:e8S%&5_:NZknKjzse&}Ij.t&nV)u8iKIWCf]*ubX_:6R1:knbYUp=S6x_v:nN-}HO.{sntvYt_IS}Jl1gZBj$=Gj+SC$Vv6-/jPP2kV>mi{*?<IW]_qsU_/bJ[g`D!@v!z*ZqutTUF}}jQK&Q#e8E{*d*p.n8fbCj+~tMx5Z!FR.a}.+P-SV=:i%$ppN2tj8X61yLoL07O%2_N@r6zsMc[5qyX^:&[<i%`0x6aJ{IV@b[ljPhBYNk])Y1}+{N=3%%m&rWa2Ot/_%`K2^5N%]wX/j*>B_%{Yo-QGXCK`2ExV%NZR+6zdGXiZN$&2<JlyvBjS^:D+tHj,%9M4bTcHI[uGM^FKQQD1]sl?apidk?UQRZR,dtoP*vxIMeAi1`tGU&qz.bw#<+1K?329s7C7P1gwUGXK(s-^@X.{0nmk2a%EAs&m`a@4Fra5{AC5Mb?D=L]iWN_}h(G~XTF?X!(P5>D#%OtWz^m4Az*-143E9fP91Cc{RqDQfJ-.3mA^#UmWv-V,+7%fc2E(?iGfSgpx<cxxY6URuLK}Th&/x;jhWWLAq)JC/fcUNQ~lfqjk9q;Ma-a|/y(M]^YIEKZy<9NNbf?|DGuV9e)_G%QDqX%|pDDY1Y%]uCs0_UWUY|^JHzHuN$p3z,N)5|{azvDqV0}kx2sctaA6YW#~/O^)E8FG.|<?Ze(DYiZV!knh5PFATfTY8]?&v*k1cc=0#e{|XH2mZ0LK,{ZoS}&=)cExmYuuQ@<p#e)2O<XatCrvbokWq]}ew2ua`=jYDr%DB56+h_Lp;-f`f0!49@U.|fa$-60@8]XzaiATeGlZonM<ROXDr[x$]TRU+,j}0xp[[h[*zHSNJNAxPnW5b`tA/>kQ*C3uaN0w(?aRPrMExn(f<u@.l#[14v8rHL}g^ogsv`W$!BQfAE![[y5O7oieyx7:!F#9&%`in[zt)aWD+pl(T2bi+QT|Oh*u&ipzEF]C:_^m(@&Pt=mI|j4_om1D_;-370HC93Hxt8,O>$dFp{X;iYQCRCAv_jn^_0H<cFf96MWyn#Qg1XBO_4i5tm-:bp-L)Lu*bbB$;q)tuC/[Mr@?uMz@+-W.&mC:^B;.oHE5gdtt8-h::d|[5Gbx|E0(|_sfWy*t:bqu;,^X/.n6;iwZErI9l_6QQ@*Cv*h{EoV4O(}Bp#(VOA*~!gS_6L~&RiuAM}VV`Un9eOQkR,UC`#TtI,.4V<#Y+|4,G$4?HqhCV<4M|X=0_VQ-Qu3UnE*nhR>ZbZ4}W-ie,Q5v>fZ_MN.{r;_kv2FJ0JF1$3/|+U!edw/6:([/an,&[U9}AbOyKZEw*@0UpOlU}$$VGz+}39*f9{y2t-0AZtIG~Nvf*1R/k??i@?a[aQ#r!>+843|+}Mt7KHSS2m0U@<X0O|07QM_$[e=,fPdo(pt>z[@X,Hta`0]@%nN%VRV+LVtP7H#[`h6Q7pj0qLew=hAtR?u;aH`B(zEFs+1K:PQTI.=eh?|_)fiCfy<G9ftR*clRquFI=RQNvUjm|mHhg<mB/ExI%TlV}#~EPE.f3Oc,>De8af6P#4t#;L;+LEK=8M{KgVG]XpR[pP^!P?9j[lv%1W-[f-[9haGR-T&j=U=/fIOIobW:RHvf/Sd=3Q8_jbP/&@57u`Fj|c2ZeDj7B7xd_#}0!;+DVA)g4fSk<|i5JApMjmR86yu)<D9-iRTa~(mZs|_W|$_sE3c#:a*:x{OdfwV[Uf1I|c>5J0o3Unwm`Q}r#;4~v,XxsrjH$dIM-@?[Ca8eA?gcBPF(ptYA*d6,jW.2>3^uLsyGiUtgym#Q8}Rb(DR#XWr?jBnQp%_7/Je^IJ2[Rzq&Mg@`QG`:[?qt>:!W,!UI367.|QS0*8+K-SvSDzw/7&6?YUVz//]p&UB+B[AYC#M/+QX1r|>rf{USb7pS<0n]!A&1bzo<Q:S]D0iq(TJ`AHQs-&.`I(r8X./A%+8Wd:ep}x{ip^sqi$&w:AaA[WtUPF=bBN[ca;t&O0L{nI2u+K)@y,7rg)bb(/.@w.r0L7R1|kVaw44z<,)]v%)ELoIUab84@!|-ybv}G2it(pK`o5ypE(zp2Pz80y1g{FG}=tv^H`IshG(@!g#gp{>PAZ?ufG=3oO{Y3nuq`X%K6b4[;gJy$<sYP;H4C%|.FNk;<VacDP<eBH!b`Kq%`U/w!/vu::C*/pZBOfhFN//V{`@Sn/%aGlD}17BDCzcmm9nm.{#9b%%Adl?4m;Vr?(1Btc`^@|D3i=fNG2[R-E%|mIB3uI.DXn8(dNKNX5J>b^N/0jH;du/%b*qRq4,TP]QS@tG}#RnH@F0;D5Mm|Wc[mfGPKwFW+yS5:{T&4L4)[loQzpdzpQ5{A;,)t`Mc74IzP8@rHo1j^EHXrva$N>py>I4^ikDQ4FqMGLIX~X7i;;kx>},E^END+&-<Y<.$YBI?J:3-oA`]qO7jaWwY6P.?9PDvQAoKO*TqDJNEam[y91.L>5&^Kp~^_*Mq*U5F$Y,Za>eoTi2XDiHsnDA5XN.]xKb02dKgPG};MkC9js/UP/bK@=es{)Iv{/O$I}P?3-)Pi{rp>BWKU/,#j(CfF?Y020kyY3%cZ$xenL/1nM9O38Cs0-7;ZG8Xf_.8VHY0=sxt%D~FY[gEmnbqIJOMWl4(E]x]M/o)l)&innAd!N(Vc3#sSbj-E<A{N7_hvn_xi)|2ZWBo++XU+z-A$fl/NIK!{;m-Rr9P>t(hkk`h^J<lKwnzI1^YN0K{(pctX]dwy{/ig@g0DOIIyA3@YVQ4S2ZRSCa^A?CVvt:|+OP7V]@zmJ$Rx|.(7^E<SHY&lR8GewsvvB<;@gMA.8nclce&k,gt5Q{`nK$,x`K=TN;{]Stl6mF@U6E_{dLn8$ikL#_0N{i-_Av-kd_;L.Wq=@BKlS~y`.u|Jp(qN1Q(S;?o-}+wcXO-qK`5kZ@$:p``1JwX0WX*3*-Pd@bNT<T,__*Wy9D`eJL,NCOBMj.>fazU/iRMZay1+rU54MmA(@s_ppkZ)Y+j&UN0t+5;j5:iSv?l}Wf+{(WZQN#jwprk5voIAu_PC5_Ygxw0n{1IO.=T[_=^>nXK[)C1[[X8GOY8Ji.`DM;HJ*9AH(n#ah)0Qr3UlHDa<gk3sOYp+H8,RJxtB/uSW6C5=+X.s1{a}L|zX]^jTa%ESZ83nO.Ezt^lPYi/c_~%p)HTN$JEa&lmXHyM9kTjanm@wJy2I?>Kq?|/_L3cQ08BY)s:u3PV1}G#/I(n9vbqsHUl99q>VG:b8ybp@.59Vi~h^A!g1f=znRwFRM}D>o`##;iJ4q1}W7EvIk2CJm0Oh)jt(imIsu{C7y@^J`FUk#^m%.(44/4vs[&&/+v`h!#R{6DkaHr4_:veqH,QFj0TTBjemb,Z+DuVpc^$avA74{Gm*X9<q0m?Q|V]}l}g3n3c.|J|OWIPI{2vC_H=UV-1bmWHS/]i(`EYjGt9rgL9oXj|?7RZ6rdM_;V%jY~wcD6:Qo5+}8yU+&0nR2u}KKvscIu9jUQqc=KXI$1;R?CM04yRk*yqxS&P~<bAMO1C;ssW>a{]nF1RsSEso{cJ&~3%e5(1v[$SmD1]BJUS~r.L,AWt:-)ITpJ>xxn!EO)WlP,F=1BhI$j!dvcN8X2AYM^}-2zcrRNL!m=FhFy?OoGe8;SpPPVELTPky*m%1KoNFn/YGZh0$BW@sy)uePkJ>VB-nLGC*%1cY4=Nw6+m>zg^7.u}bIVK1erzTWFl%RxC*fjh2xtb?5;)OEv,SDBd<U,[Y8QFN)vY{2jg+a])MUrx2BJTy{qoYC3fx*2VRSeSe;6|6-h%p;4Md`{<<>Tu]9DQO3b[9rHOt_w4#>{2ytZG&nSqDeNqIV9&E;/%dh21sx@p;)zUKZuU}]M!#CDJiu>qiQ-R>RM&|TbYWe2ZM4^N=Yt&2TsucWxJuno)OzSHiJol6@K1/ykVa2U#kl+}t|PvPl9Ap`CN_b0+!fEP5bT)u#$$M_5E$/$iZH{K.Rt.;oBMA/}qqC$Sq6N2f]Ps=/x6RYJT}FM2<}Rxh~LkRIyoK^[W30&T[v^,jU07#6I1}.IFCc@Sb<O.q[@|]1jwpd2v7-6{doN0UFZ:d?s=i>ecgh{Ds/hfq)W/E&zk!P.b=gzK62^QE_*;sJu1b#+;)7>DdoQ6X$<FITOZ[u#77{2(l#nTlqPT[o}[/Iyok`O_A)C)xyjQOnfvZa@?DyhqEL3d)^]l3}3?N?I>oDf;8_{yZ}oswr[[ud?K`yEP^W?_2xZ79.;q;Xn99)%XR48Ds7h1hq|w]D2VmP2qHNWgS%&8`yk2Jb}i$%bJL{LZim`d!E|}1NYcUhkcwd6g~_e0CXy[D[!E_J7*9Vt8t0$[^]~`V%`R&#G+Trh6z+qs[oGjlL-/]m`onC7`-`A3OiiMJn4eyiO|;Bf(>fI[t^{eL#%<*7t^xk~jI8[TG[vpen1;@vaPx?X2&^vxAUNM(a)JzNP*E{8ytX;J<k]bRnisA1BCk<Oc7d]cB#r(6SA;}{tI=h==VGE6:5[nU^1(]8HAmkC7vANKwQ>>4hgU1{At_3@:F9x^[3mK/[u))nFxw&l&5@GNHK7{R/cR0Mh<cr/{zvS%FOQ%s+=1q2)x>/Y9Y;@qEF4W&ss8^-=]F|%GGyEUV^51e:^6s=^IhC_25wl@aB!=~yl@C:hI`Dh;{|+db1f^GgF@/sE[ecn`^-R^J9Hu<Z5).G?_q]Wo1-EtZ3mRi~wS6t9?9_>nxBO7]*F7:^g98dL%-o`D.Slg@Mi7)zuc5}=O2gy1Geg*X/gk#cxRnHsRwOQ57(Q$!1=3`G>o:N~BC_~%-wvGTJ0}J6pbj%w<4JwY$9f,A9HY=vtsof::S5G&Wo>71r7;5J16U(UNPHMfzrdGF%zc&p,MqxvW6H~;BX>}.hBb0U#u*ljHgSN5;/GDK|NGBJ=X-6A}tGa/qll7C[wVF4[/`fNtqs14yEr]3/;Sd>B8e;`EY`97iQy0afo|+.<Z.PW>zx~][dRK1;rjuv+%IkTQr-P8g2WBu*KQ|T<Am67h]$g}k`b:9%Ba(N=C8oa`V1{xj4eo>!haiY?bbU8J##[hWv8i4ZaO<pW%`4]%U6Au&nN:TNn(HL}LLl?]#qnl#QBP/=:&=Z)/K,/s@1kq!gVE.Om17nzlGpkEg5KD+$]P<LAe2xI#g5EKetyv50|35KR4&1Sx6999Fg{U_lky.96^Htf.wje0pW71j7<d$L,|2&w<R?CT/4i&m?0J6r5@|pA9~1^K38zJYx}R[e/JmpdG{2_Rth,rbN?B_y,{[YH&R,O6a:Z3SEi7[&*|n[`!QxU}:vYPanJ`{}9YDiGE^Y(D+bXngEA}EUdS{19un32}UY=i[KAF|&@C4**qh~+UK_o_#ZA2:%kGC(yeOR^];Jjm|EL/0w44;IO4C~VJMz[L|/]$Gj5X:S-IUQ_di]u]?6H5j/YcrHFiZ}Mb&iWH)&5b&nFa9Uhn#LfI!uM7[nCy!0}.jqlK72wcLU7L}Jx0{:!zc{#)SKPB;OF$wprM7|iJrK#2fy1!}6|X^oU_elTOJ;s&]L{U2_Yw*}l{5or^Xt-H&O>bKYp7#DYhk<-4-/NR$Dg=zUK@UTp7kD*j[bxD)UO@d7u.NiHY.LPlp^y<.l7y`AfM*$C)Bb_Jsk$AF0fadaATFoB$(w/<d`OTRrcmQ4d*>e$gf*U1D+{@d`9^BWc@yF*{4ViLag?TP@*|lG`L]Y0Ghqm:P^p0SY($,w].o$S?n7l9I[vKpI!mdlbrat!Zm4OA5smu:k|<Vt#mo*SsmnCW28w8N}bJM@T[SjNlr}cd(cg&fz*],x[TY@^{LO4pe+t/)8?,YnVI`2j-5XnU7UJKTbw}Z}hK-A:z}?R.l:h|qp9L@<|aW`E,z,hN=Q0Qubkx26@`BPs-b(ljJ+!^)/0i7H[bHBkcjW=kgK0P9zh5OM2ZUA^CHC.RU[&+N}OgEy2a#Wa~TvVw8zmBh.TuS_y}xwqmv9gX1spOIRc}{(yNIx.pg%z+}bTNF2C+{wD^*jEELS)B*.p;_>Y]}81<L>YFJ_7K%2s7mYfH@?Psr%+{m[N^,%FqpCq*!4h{x7VnR^iI&J&Wx!Ic2G.akXWnqn{yMj}O;o*{.pa^x,TR(88paL$ekAVSIIC8ans,($G`Pnq$>*d{Ug6lwXeTasR0vSXXDg}T[9ATeMWm^2Cql-#&IQwcf8~O&apc%gcwPk~DqF6#/svt-_;gOlv#wK6D{7t_9W=T#(Gx,N/K%m?}-/{2>e>DoBhHMq0Q}w|?CB.%dCXOK&03`(,Q(4HKX-(<^<:E:/Z/D[GPgh:s5rF&3F|~`o(BwDW/}-P1%:xi+82M&0D,%lz,rqFV*}t,c+SW-d7Q,Kx$bS_U}:mYtFHUuHa7b`r+*.Rs(QA=S+-$g27#60m31jQKOj687f!_XqPZ5T0DjNBhp,K(.>&s_G}!CO8(9k1*sJLUDoeA1=_7%?[~f@0TZ,:.;]F*]0N$i7hU&+6>Vp{yQj}[km82}O7#v_4Axzi;<Ki+3$|x?`,.9+eOrn:w0E1`f4cgmE`nH?2iCtrqM8[(Pg79(eR<Rm^RgHie<kt}~=W:k&BTx<{ccN5@|xj>h8x(yy~sl_vom>24_)#MU4:n^B{/p*EC!_;LYDI(lk:9mi05jq`tVT*|MYh&t8DtG#()0n2e&goJpYJDEM8Jo%G*]mVSU+{Ix:ZtFTC~VYrj)|l@|EHM`-D)+.|GhD-^QARJueY|{FUH#S6pc]|`@xF^]`6){?15=4Ys(j4Vo(PDFZY8RY8s]{om)oeLyk`i9KT]5Wh*Qy7E0sA3H8?LvxP8l&*}Sjh&gE[W_2nv{PiJGjMp>GF5*Yq%W*n0?JC#C+~Vy[)ZZRGI])@/erOmWjjRPZ#Htj^THp=e@J$.;Ul$|x@/{~w9I.iH?}Z=%fF9Du<KN.);z=z8f+%fI(Vr92(O2YW,GiXrxeWsvr.gs298WnQK]OqTr0k)!I`Y%YUUk@MNquI6d(R0Oz]+{(>V8b.}.EL)qPxUqp51rE[Yo2xk1uma|=U?Z@WHO%1}oDP_]3V>S>c(AK7J[q28u7#/RE_@>pm43<8)&U{(GKtn8O0hEt!j^j{?qgZ+IC;_rH5;QkTM+a>{pzp<M]*c9U7&3Hr+!I8f$%66[sn&$9YtHIMJ3`(<;SlN#yqQb=pL87Kq$&SCmApLS6wk|3MBVQRtvl_bLeCy0SsF_m0[#W{{^L|u_*a/!i{22h4!m,]S3W-_qR=bo>_4q.24Y5@>ec7O3[8PK}Y-h]V(4C;0DW2./2R(L&n|}yx)0/a4I,Opn;l(B9S6<a2K=9DB&T+c,xACJf&:DV64%a|K*:Am!Vu<XccknhQFo@p_KNum3>lwur0<BTO^*iTGms9cLUizkzV7vkL-VO!tY9*M#sXS8%Z&rtjL^%y#A2ghvCbzC:qUeZ!t-dQO6O`t$[GW!]9Sd:Nj_Eo(%2rG0oSNsXe3RId>H.vI)a32l*iQ1YAr7dYUg9X+JJ~(}THh[|Qbv%1Y]n)hn3w#(@N)ni;*Kjv(AfaTAfigcnT-.#NN!X=[{XQ`8&K0/&,AGjd8)B$Zhv5H;7W.Z)|0xJ:Z`j8r27O`uk7td`FIYHD~%kwX/]99]]0(,1U:5AqH?@QFu}9&88i%F@BJc_Y6-I>NMUV.=;Nk{x.>dL?T#2dhxr_=;N=`c:.A<Qc$Nw$Kh9^uks;?oy_1RgOTBI`kZI6Ht3w&F@Y@@G8d1D8l48^L$2yF&.q?fvCz~qw19@htr;N?a:NWpGCeLefdYl~_R?My[V`F!W)~P*vwp$dQes2D=_/%}PbZm>/ENhRJL|<83/k%~vK`m0Ue=Vnpb^wKVoMe@l?uG-|<Pn^9mod`v+49el$Y2fXm.r0S8Q#iJYKDf@K|`+@^lIm#KlBYi|_W+:h7>`<XT4F7FBQLVIBH3ov0@!BTK^i/V.uhK..>PCy>S-4g}9v`I%~TXX08MU%wzyl[Ao7uH#/xi?5j9b.v}&~W-&G#IF)&-arY!D;QYRz=k?vVW}hY5[f:AEmP9by=rq+^{#3gnx:ELf!_*AUvk$)%7A+a/njs?^FsnQ[Z0=ts)RGJ#`YGz63g+0=Xus^#b5W6qvz`aIR|Y}VWS5M|R[>_U7>O_Twp?_VG`AwMt?VN}C>rz{/SaC]j{c>HQ4,R*xg|bt)(ck)Nf|#@|_v`&~b^o4/ZBV0CcD4nJmnqWBW,#a;@+<bvq%)C~*c_^r05zLa{c/0#/>6n=1|<84ok=i|HdQ|-K(.sQNDE=}!,qN}AzVDMQGnIlBG/rA?.acK(B%W2<H10wyrIua2|km#rw}6Z-qc:[>wMZBqnD+8O3DI=E1*u=++uE3MOqEt2jkj4/e+wbX~IH[Z1F}3mAN;/0hr8{7%So:^bs]H8cvC?jOs~g$Lr|[?YiDVU:(/9ABt%+*i[4CMLsyB*>bBN<[>Oh;lUlpxmc;OmC%]bNlt9Z}=$fmwgV64czbsx/]55_NiTB]j*iDyde3!+<F_t9g;1a9b=j.6(kd@wC)j%T>Tb%Rm@nVU,:NV}+{%D^^(]8nOmqr^yPRo!&:d`)tOG52OFxUe24yLHOq_^lCd8i=2][znmqb6F9-q0Tq4,4aY<;01lySg}?Re,e]rUp|4xh$UF?c/LvY9._e@CniTHO`x@[U>E,U4y7%`UNF[^u|W6j0g1jLq_F8^^^gaxWu=sJfR.*~DBQpQl#?rS>+d;{%DeV](9.]Hq4}|`cRDTdZVA^,_%E3`Hya<Zo<44gXd|u)QJc=cpTx5wg.ALsPx-*1BGB|kzzM&YzT@YD7R%ZE@iVkOT=BLOyjW{wjNN<-ggx-205_T1{(u-&-cm*MSKDdO@tF_./I*C,Mb-7q~xEt8Ze31TZ=?3P&1ZliIF%onCU*BOr*B|>9,Q+U*;nk%-BdyI?A.#Tj/E-qM__40<Fb)^GfuK}3vHo=0k)#%PCJSDu9XZfBfHysHUl{2_s$obm_pmI7Y?Ry{BpM*.ZE?X<?I>Z_*{:mJ-y>ifzbCa$O3m+KX5veaXhCq5C^ET2|FPq6!B.VM8V|_NoR8PheVmM[zz%DzW/J@Zb#|8u;A[;9uwXuH)_q_LZlnC8Mvj-&!FfX:qe5|AD@XU?<KB=.AFl@Re2vt9:qNyBL)T+@(fAOqo.Nk3ok!L%KC.TV2=K&gnUEsd5/7_dxo>%]Vjr3!S];4n}-0)*e/@Xs1D<A=<AfkBmk#j76qU26mF.8_e=LLh.[Ghtg0bS3C_0<JCt}`]Nn-jThB*LkT`k@r[R{:#]v_6rxIB|K0%GnULwun9c/Q&)xGK!4)n#d3mv&s{/8Ubyj/A6Zecc.haM9c7aN6p`cl=!jp^WcK}EdaKYcy;d$VfUezD_v#ZgN(Ba>)]D{sxuGJSaEfN,i|SiRhm#c.0A&a_KZHA7jRW$d>ayR!iC0!<3cpY+p4ub<1mL@vki,0QxOIPSr@W)]B-8`yQd>8Z~YcOYO,e837T>P.qt_=Z*IL?P?mTWb#.Y{#|BG*U2@/<ubJ;>;d_t}BDCSljhjr8Mfb1?Ns[uU{^k+I[~$ec>:RK(J*sv*}`[((,|:B]j81X7yDDD0<ZR0;?Q5*@Cp@nq;znwEUomo]@EqvNro.hhvwXuiPS}jiR0z.SIkvnGbjXN7JqG--hb%&M&IQk}f@CUy$[$$!~!hwfV_c{~Fp*8}i_nXz#$S9O)W8}s?F4KOd=LruQ^^Ib3$lX;WchN7S(&j7;CXi}_Cb+*$fCh*qyt#y.Rf]J<#!Z}DX?2J3Gme~ceUu9^DH8dG}qHbtQWdcMkOt[LE@@JC)vb~_3L=a{u]>$-ah0vuSBbg8$tb#Tic1|(^@$>^7Ocq/p*0[6pQ~eHk51T_q9c[elAO[QNo|`s&kWKgRs3h?tKt0K7OIX$ZAZ|:u1gAnaf}l|fa#Ho6hUo%@KU.XETPY(OxvG];}/]$#T]9sR$>YS_;!<6cCdnG5IZ>dq%*iIj}yrg*v|8lj[S+qJ}mDs&yf?tX=UTuu%Lb3Q(yB/GWJMT+CTQhhFmrng;/zt4]Fxoo/@>,}<(?G@S+[GA7aT+#$s}2f/xmQIs4OgQ.}mW@}LLkpTu@2:KL:2NzvFbS_<iu_Uz%=)m47mcUuN7?{)UdtP^u8%?AYCo9|3L?h#!UTYLmI}|`|FxP`N,v=iX,+7xs8ahp}%;w1.(/,d`j1$TQe<w%v3!<tN<AYyR?8}{3Bq):gVT%`=9x0E~w92|S1@9i?m1dYOsRlD7?/f/K0/TA1L#,x{m|M3FH,JA|+(M`^,$Xh<>E~zV?Nz:?<@I+t5IS^#eyWJ]~85Mk(p|hA81#|SueR[iPt.2WJwoJfVPHD9/P5vX/r:$p7LVzxbQ7t.kTjyuWKWt:G:YB4hFHG|%gVQKl3/z[Ai&zPjHwh]`tL;BoC:A60g3:ZNu3kTc}?CbxrU6([D8U=a>,xVS`-4:r*CNHTr)!%$F[r~$n+qW,^dB>x^{5`*r~<2Rk%J{,!o$?Ib8;B.2p<n36BLmv/b,<1&viKu5JE1MvAQOSN&|WGiM0;KHLwQ5G=aSak=ES$=,BcLq]*|D0)tr<<TFdbr6XXH4gC9Y$e[yK>Y|OWXS<WeN.Csdcp-GJ(&>Bol#-fwYC3x~]ZFggPOY)1A/H-B|X8$t[<6pJN}!wNQBeG6NP>GRmK}&i@SPWW#/N;px.e>)_%gkDb%Vx^P&Xt,t+*&2oSNfB;V,0Raz2G$}8X+X9hM8-~VEeha)Ee;-[._!]ne357CZ`ooVbkU=mOJR(ix0<S23few{f>_(etj<+6B{w[!RQSAIevafgS6a^MCT2`T;7ZUwOM#ZW!g/Qi;>HS+HTG>1!NJ=U1qTn=9>vu-1;+%+qV_yJI{uA${!~ksU&)urOmvW4j1Du9JxoZQHjVyk)G=yjp.G|#?4V.W~G_Dfx3Lfc@.B.Pez4bNs9WbDfW<y1Jz,}D[ZB!`<`k6q7Qo>?X{Rcug<o0#}xP,Ka}nz^(m,tqZ:XzhiL<F8^H}?5|KQ%q#8TX`(ZD(+U}C-#$d=B6dQuV.k{3u}SJy&`8#)3~sp1zsOPnC(}|<=&iJ|o2n<5^QWRBmxbT-6P(9J2r5gO<Il()LX*M|e9o~7K4V4/g<;w:0EF,oJ&V.v<L$W^1etw<dxtHF{.cHB`wLnmw51p(}t[z4Nf$cm9iOx0r=#y9.{ep4i8#88%n}47p49+FKS4{6wxU?bGsOl|iWGdUVi8md?]9}{rB/,]2`j/Nea35zjMl1:yE@rf}9nxOV`7j*B8tOr9KNU{HW`,NuU3re_UmXh^$AN#Ik|IBy4)uv9+X=cVji!W!X^%C^/?#ChC@U=K7<V[*9;[Re6UBchn:C,W*Zts4rPe4(FbI}BrR]r)qWz9_^Ra[A>RT<_2Jc%gye~$8p+bwim`v3{#(XL#GRmc~d`Hg5_9bKP*%;,V-><8S.V$IT]$|uG1JBnIE$f&!G~`lnI_tQH>jy1|z^U8%Ei66mEeza719Gf24=P$$)%,FVtB@aTW.#cPS)yyL$r`XuW[1P~BHN5POZw,;6/Y8%9)j23U3vLzWb&.;cWv=e~6sqmbj/5}tjy=AJkc1=*}8@f&LA&x5q|wQ8|5QfU6Y@NR(u]+`xjAZA}GONr7-Rz`)O>w?FZ#wp)`lYy#.McW{R`];3:^0m/fq8.BTu7?o@.)**~cKJ51VCXgc<LAji5t>e%/;/)9iGnI8[x`P$>gg+I%nBttl41|uO,.{~FB:B.^$QB}t](gVA$5!RvF>)fsv@(O4A2S#!}0gkk;a7ny=:#u{l2{{F,QfU{HNk-qDBBKLfaxh7zfa;]^&c6@a9kU,aO}6P}j>Q:OloMdNUUXH2v=(=TdxBg7NNE`E~2XD>#9~DULs9vyU8+tu2nDRSgv&@@+KoSU{Gm9d~_:*bvp9wu;+,Fx.-YPL=mp-L69Pb.j#c`:_u^.6-3M14n#CG_|m7Oc[;IS14izZgK{;uih~/)w&7%iyMQ29V$KO[Xi4R/):u>fja(n`m^5R_L^vCmdP7?vT_E(s+cU03{rC?QT+5S%]:ITo@r:x|}oFK,{3k*uTIjns18#Knbtkefbv+o-8]Iej2X1=28GO_)X|n[{DuzO[TgQ51{?y@OhPs^*6iMYMl6ZXLgxAu`3A*_X/bDO:}n25@hk:)&Ny0@!ghDf@Df42Va:Mf(e/sabbS<<?2-4}CnMMLnh2MY@J3{)E6J,B0{;}a*vra+y{:(Cbg55el1+aY^6.obN4Va)yXl*Z>OE|w&qo}s{|`.mECvKEtUlt+J7A.i@Jd9Beh/[{`$*Dsxyr~cj!|@JACT6DS7Ytoa@y~`&lf>;bk<@0$(D>?A!oU]rn*flQL%AqVQE$wa6x5fkw/Qz>-HUAS<p`WV?!Kn=VVl>+)r#)^0@D&|lca&9P5a|yv>:rcht~vf+>ojgzxe0zLUYfs~0FDf)Y*)`J^HcHm~*|$kPB,KkZ:$@5LzWCi;]S;pgd*b_*-$Iv4en@JjRU%,L?Z>!n722GP$h=lr[@dD8FtwUzGLL%M@,tBQu@JlZ3AgjZcgnZ+R<5M4lp-uY0Fzm$ZV=[o6p%5hqZvW-gNOs^S2(7g&JRuUj~mBnKj^v^eQ,L?zsS/hmgz;gm=/dq<8qt3*CTH3N5N5Hq85GsN,W5mw.@I?F^Jp2Z%HQUSW8udhk,.M|k89G,fJqANnIdIj,zm,f`Y#dYMNG&>2w;2>WS2{m(a{dX9ap;^pi{qA?#_Zu(VaQtZPG%(F;d*)/?X*6!;Qn6zAp0nIQh:*uU3$t]w{rg|m(e|)*+bQ,|fd|Q^|Prf^9GE*;+NP_m%>?U9InnO#O)kOr@N1mqKiB!kvsfhn3`B*6dH?75I~hAy<,H{s9lV2^ghv5;|yiJAf%T:WVGoJ,:M;wClN%IzKf2~!wLm3I=@.wQXK6sRY{IW&iyC/{yE<~O0=ZrX_POn@0`%kI3N2OVRwxWc;*[#cB|H3{V6pMBvcLrmUgtt?w^6phTFJOz,/e#gb.o@s>FL&0[hO/mrlS]PlMY<u)*)/Pr-@HU}lLZIr^fRpXJIJS.><S]VY]SXLq=o~Fx(@?[=*%V$89mFaUDJ`o,Nc4|<mQ86OECv4~n|h_D^Ryr]H77cVZFnbHW3FQ1S|H@-vv)|-Q:u<SY<FjpqUTvuTUE{G`>g#4,!$8$5y~Fp@kA2K/gV}b`giSKvQ+k%s&uj9Z+4+|ttVP31fh+dLVsaS8]k^HZN1=GX{`Z]@NuO*Q3F&2VU[>.*Q:m|QUH?,LH$1HFrnkdZBGccMXIt5>s#lc_c}kdN.9X2UFld@U*j6|!PGH+h`kVN8xVjBr82<D}Uy/2tS?iytgC4w6!@HqG@h7qQ:qg{%)@(Yl<F&:3#}>HV9~GxFx7yUz~D#9!lN2E?k1Q2Ui10_E|EF[4a*Cmz%=+xr-#}e_&z#:<$V(>XA%mLzrCqC;N3|Qi,Jb~+7;0^lR/+6kUFJcdoGN.YBLH7z(&?0*Pf8Zh<iy[1pdL2jw/+=`~txD+Kk[+$j:42LO1%WBPVR-.V$Kl<2OHV|72t+^6U=]v9,|{6oS22e-9*k*>j38os4FO8o@D_V^Gxn:+h.z_K?LM/;ZM*=GDz8%lZ{;(Cl>~RhHZ7z=4r?0E?`8`1qh&b/%6e/):OF&OF+/,{AF4e4qGf(Wx=k^jKd*zG_X1A%s,{b?hszMg?Nl]@c@mZsl^{9Chsn=JksE?u6Z4B{0cN)P]F&a,}ZkXl_u/Px^<>LP6Eru2g1;!{?,n`+d0*P+C+U0m5c/##z:Ylw_mRzhhj<3OfQ>Bb#UkKXj;LNd&0@7ptrfWNeE#?`l+pi[vg0]U}j<@$6#fDpso<.oMVTOXFe]&OV.V>qgBW7M~3hIRa=qKXQB,>;[H.:l,,nFazprHa^8xuNcJ18j/j%kZSMxK|m-#A1f2yXY,#^iOZti(2yA;LbTw]okySM?giKA#Fg$O%+6/zBdn$hoPUSEau36>0x}SX^`^1i9f|RSrtD:#t/UT*rg@eJDL<B]XEpj_WAhGt3o]|uF)iUbyQHn)w,mSIgyS8e<c)ynPD;LC#gDPxZ3Q!@cPSOGm(L5@q:t=d-aWBHCl0Wld/[I-#}tc!5ckn*ePe]X}KF,d|svllS3=]P7<2sTto]Omc4+~5QC)RzU7Re76G1c610KVA__sua{ipra*+KiQ9Az{{E[KxjY}H)^De4IDqbo<]Y).%mIhw8aMH[u_$o|1k5l/j1${/U]a^v.m{2IUeTceV@m4mLh)PP}NNE#eSr^]UC`MM6<>>Ie^hdZwY1BmFes82W39wryW-!}[mA:6Y!f-{&qoWWDh[BKbqgCF}O.!os+*b8Mcqvrbzq[9E!H:.,.-!i2?QcL[Ymd-@Oyd0-9iB{{,dwKj};3|ijj>xBD~<fKH;:?hoUAiI9ms8^/+Fon&&++(Ot(^%|:9U-Sa3$D-95do[WI)KUUF=a4M`rQ/RN,Z}MH3w0Me@DRf#M6BM?<^^Gmn4j{T?Q(||nZ||tV<o0PaJ?5Qce08l)fQQeQtdeWm7f2U7_FwB8xiHMY^q.u)@u6?UO@zy%6oM-h9,Y!r7CAdp/Ly1LD),[*z/~5/4Ib7(<mk.@clFKn2Rn-D%kM1I#1K{0LQ|htW$IRrL:4*drcJjw^#(.-BX%m8|3h=$>2lQ-8G0n9|9xi_;,[k7b}Hu}3F=r)3qY`uE<y;22cn87V.i+Gf9~h+]E2av9b*;xBR]z1~Feadzs{CI}.A~3po[_wl02Rwr:o|=i=!lw10GAgdsy#kn[kYtIAl=IVoS3TrOPHkh@r{t!#?CL[AVnVBP-d3BnMRuM*QCK.h<Lvso?*k:~)fDPD4RoAtH>|P=nS|<yb,i!*Rj>3PhpQ}iD2~9N:e0~(2HfCIAb:A)YasaznNW(Z]j]}M/YRGVMm#;iD._vhh[cKFT;.v8LNTxv,T}GW_4@6mAyva%s2b72^|w@(J7></GTItcHAA^kS0O8[pd@T8TA`Z&y#-h2%>twMj8$uJvOy<A[Lj0xXKrP1o:Lxv2wI;}x2<wJIOyS<TLJG[*,K2TER@D#4?M$,Q=%g+<d2K<>&sIyjtAC%y59m(H;D-$<WQOmRdvSf#:aH_3fNJ_Qd@YjhOXLBBLlJWo+%0174J<GDu.(v9(}|W>$4R8@7Jt%6V8yRDnl-p=;s_Dz*Zy)TI@[.q^01Y<~vj3Kjs@S8;b]LoJi~>Hu|]fu$kVC]/3R[$.ej_kb[yDM@8Z:UHzzkypj=%x:KUS9->!DH5N4*bb`;Xuzbg6T1kN)u58g6kI|p$K(JrI~cZ^;N/4MD32ZQyymL<V&Nhp_(!qQD!Y,Dc^0JUB}n>2U>u/*QTJL<x`?z75JHb^0isV-SDoqxXejSg-l&@5T~rpd,M|L%?)D0r&L2RW;{IeVHPL^5`rQ;xZB?IdLl}{SfUj$>VtpZ7E|J5fP1y90r&+Ow!Cf1Xpj?cR5jLZb`B0.u:y;H?pHRRD7#UCM]JDE!m?p<zA3jmB4~g8Sxi:hzefDf8UlPE({wSs0%(enI;^}M!:$*Lqs@@x<,S>|z5dp9_~eZOUT+9|`Ygha,{BCa{J`]0kUx*F5oIc7DJ7it6SQ*zW;z&L#<bb1:(Y>x%rfQ7_-]d^O4G]g8PbIqpSpQ<ySy!ykYVOGx5[vEQeD,aiX=rf5k#}$5t?d=+EqOreI*8d6s<Zj%~ev<31r[!)4VB_FjjkkozxCp1MnFviv%}kBO^=+2;jrwHVU|q{!nR;u%=6v`pf%ymVr9STgOblehJMYL:_3X.-+bRwOdNWn1o@Vysp0hQ@7&OB/Q&)EP{eGMGi7tC&UwU8`0|`OnQ#Hgjas]mK55w<[{A5Jxm4:a:GsUWmm#jm{1^N(MX~SRhAh|xqQS82cZsBHC]VQT+h=;TGoe*(]fJ<U5$Pv>td9uf@FwF4LpR?`SG@y%i2T#NY-2hO@Wo|bunDOah%WfvCXwSJ9<Kd95cF}d}nfjJcXf,IfNsR`L}SnjVnkbZRs&~x<77q+tkqtx/3i@NOE}ww:r)4x`.q}7~9;t&%8J@eHQ3W}*48M/?^QecWDZibekY,e=H$7d6HqqeWoy&CrlWh;:qHC:886*dR1IC0jI$KIY.!Iv[)x+:NCP.V[,mJ?38}<X1#N*UxP5.o1SjJ&85ylK4]:5EJ?D>UK#Sys[$2t-g]=+Z`M:W*2rq#?vQnaf=[eB:.+r%G7Y~:IHnTXR-t5c_x.[t%sKktKsTn(@$g4_mA]q(gUGp`HkhL8*I1y}]`o.KZVhvp~p4Seo25w_s[Tqm!*eL)6Q:iO3#72_KGrf@TTdUqtb9J!D_{(4cl[j^$lF4z-,6#z^}%1.YRU]O=st6Yv6bKvR&g?tZqsrgw4o;lYY@,uS1bYW`Ne%ZfyFU_+EPtcty}IifDPN&2zVj5e;k|]5PMO1p)*BXLj`%{_kY7G[>Mv28u$6j4%WNr-TiKGX%s@3(>Hr>%DA0bo^YN#q`Q08=mL[7>v7&7>GX*b^Ai?oLHo.O)[=$=W}A5@[Bm`rqrj3TgrLJSBKUnh|r]*/7:B|6lL#Q<y.<bhUB:epI[R.yPEQf2N]NM9tD5+:);/M:+==N&et2WcL-iQ4o}_+EgXJR<)VNe,A7`)p4=Gh32+C0f7*.K?NA{,-fdUW^=BSeV=t^_vWx0/x!3KOh+4D^[z}oc&gW2hl2*4|p,M/Gi8+xPEUH3EfVn{ORGB$9)Vt}ifF:&Ed)L8%;9ux=k<I<m6bP7F=aGeCw/%mp@4(in),vs~Z]bV%A,Q@V3uVs8x>Crl&!r|ilVeK,lFc;uu$+6DAx^TY)t2@A95~fnJ,>{#lm&6agRPzU:=PCCyx0ezPbJKAatmGEE%G^3nnevi#imSL*EMd][d$eQ^S4V1M?HtAHP1tO>SPGQ?pJ>cnb]eAG%53r8;`(vMLZb7zm4J2:48)7Buxu$kqxS>P~pj9GMSz~np&|wqv6}_:e8bQR~%pDE%jU&7+g[F&&/p<>oTbN3%V=MSUqC?)C%<Yd^ka!QW&B)uCh>LV-Of]]N&p*Tj#U_v62xj6y}MitNm|qq>BK/vOU_yma>/8Cievhob!vVF.&JfKZ]^J<L0K{,%xZrvV^{,wzz@;ET|fpoVjOClJ0C<2eFHF3a[&N?2v${A:,PayITXQTCoi):6EgcHL,}s>6I7D:y.:_v|vhTac?7z]TNS/9Xha-dd:G5W8}6w>W+&?NDh|Qk<b*vz]hBZ4tDa=j9I_NzZB%^2rik[SPh5x=a5^?j2BAm}RV`[(I*gnV[mk{dsY%VGF&:xLU2rg[8cpmNJ<Qo~1QJFCL?_n4n2o@bZh.Jn/&UaAUn2&>@ou=~~A32b)8@.g1e94S.;9,{&/JT7I16wcBqXH$1*Fa<qIMk6R|=%Tsk6eA%#k{4gPqseSu2(<uGSS6kz%[LVzv$k:nWkcw~P<|cFXQQ.a}7EDibQ[fj#][#wzf.r?aKZm3*a~A[py0eOo4as/A`V_Qzfwq.u@,oG@a<V,}>o`5j>,vZz+yacnB+;fN;NpoZ:$&0X$]NkwO>9;).xAzB8-$Nk#b]A>?`MnD=&Q~HW2,=*O}o%*q?CeE607e~JqD?ueRkMojc{(Fec<G>SGRjUDy<Omhb.aXWMuDvlq5s+{%ZK]V4KNPBZv@.IU*Xl:?n^.ADgJ8xfZP7^&O5}t:RWkXPHit#g<hOYoE+yed)NBP-LJ9eM-_cifq$rcu,<R(Zj{1/}EqU`c_k61rfs]FL`>`0J+*k{l=F~8&Ul0J45[6v(}TQ?=Qt/PBjltpC2e>q$#3|,`wiu`!Ti:Ia/Ebb3S!;FjfXuQ,xlY@}TPq#ujC~?(!<PU,{{pTuHXFgs}43FIf@yuFG]/LhiHsPpfZA>#D1_8]}J3(fEa}h^.CP^HMeSofz2n*8sA3jKY*-$|u!_,im3L<)!$v9`e6a4E2+4jy0./@*,|IHA4{ad7S>(iL%*#jTX,]A:+.3@{kP*nML]Qhsf{XL)<}7,RnHF:m/NWY<1O9=ToH;^Hmc*UqV6YIeYDRw`CFnA+lAgPH:C|TSq+?u0Ezw|2y@:fybY1Smc.$Qo?{P<yTS7]IVQcn@Lp]})cTftXFs>L98S{GGypi+IM9YWu5xU%>6Xqnlwypkpu:4g^,?sYq]EUe9pRMK%pTw66Nfpp6^;kA^|cR9MJ%yHtP!)?}V:^&O.Sz>tG8<NxeL5SzL@]yTfcirihe$~Gp(M&apm/s/n2#HiO]f,i46]|*#Pp]K|D8>j%^4fVk)D@|YKBQt>l!p[S1|/siE@M6Ay4]/@F;#!~wNw:A>]F^mdX7*t={,(T5|9M<1U-ttyRE8Ut-!XrF}HtD@#QEg$,,%ad7FZB`imQft?k7^>9z=y6wR/+&=f{V;40>w4Uwvbzh=A*`d#MK{J4]&#XH<Z|rJHyy;-~Ip$}*/A1RFEtzzvlfNLyn1;Fn#x)?#M6G]o=$#J+ee/0VcG[Y<vfYcu[rKNo#RC*v;W4Rf5&TKs`b?dg>oT/^C<T@$V@7{xpe#;:[$qmWk0V#6Rw&4BRekD[4:m@bS[Kmg&kFo|m8C^;Vzo(.*n-.~feh(XqG,F;e0UY?H(<iNZH#d!Lk(3!LMv6UO`O?GDq-|DlSI0h,vAwl[d>H(y&VqE3*4YG$97F(3MWgPQ<R!W~l{__3R]kRgz54etCdCQ|Z4lrTx6704QweP3dx`0fhP-]JBnY+xSO-(We?;RGo[jKJ9y(fOKW,c%,5@&SdQ#z[?Aqw[ic?)q@g,>[y>)steIGIP#!2y&ZTj+b,4~bGBpc#hq0G-b.b<%r@<_2vUO,^*p3s$%-|y4x`+gu;(u]p@Hl{_[vv[EXq#N]7TaIKg?)foq)B*uJ@^^{<R0B6@8).w%1v4$IH)%i!ZQ)cS{YZHNKB?_KS/XtDF(0W`5K9UJ6iIJ-UJt_?oVo`13NM^B}M/Um3RU$w|KfS$7vRY&o6I[t*bXMm$*]I^VXhzIjY]<Rpc?j%x^kbcJr/lJHY>Vnc?YkM|sB0x_)8C)Rd=*)/=(suZWh_2G.-m0O2z,K`;:xQ?8V*I^VM?NH?a6igO?xC&&<ty`yMUsA}n$V[Z!B&xJY&xt]cpLn}4$Ni0k2>}|=6A*v@BeFF%NX)Njt0<Z#7:enlZs(zM_gI&j*UL/gG#LLkO60L97h~rbm;t6<X9SIP).|]G5SPXoqbTL:ckUJv?)kChaS}s4;g:vhNC.[:s{g_PwlW+ae%qHLRp@:X+)1m%RA@LlI%Q(WMBWJ7H.cCP]&KP0l51:jN<_E]&1ZTfzsPN<UoD&@3kgRw}ivO/*7zPvlST.>J`w%:%v!w$h2WY{}&G(*s_[o6J$pxd:ZT*PoFt)w||6;)mF|58WBh@-xYVNhTKHo`DiG=:=q$dfHJr{{k?%x[y)_#r8j:(Tp*GdEqn7vL.{*2&u7B[gr@{*Z<&5#U{9b8n*P|D4w82C+C}eSi8Z,JV75b-{B+SBa64.mp]rX}MvKc=IDTc@ui[{GiJZ)l]2ha=[<q[u|`5nwfv;:*yV44n(H`O/~}EY[k*Se(:k&@:;d}O3>|Z[jAsR;nd$[pc](+yI^Op~zfBt4T,1#)tk*nmR5eW<:No{gDjQ^{v9iJBY_,J5JMz~R*YKf7m94sd)F,V~<pzy&Crpo`/1m{bwdSf(+t2_nV%lVX%pLdlW?7nCu6>N)Qj$@dkbr_V&U+.n4lZhlWwlCWA&JBL0:@gx<kiGffE,4C.91;Bj,(?P=,^7+$IUHSX}u)W=-ft-UxHT<%g-`dxV3XT.=vI^2DH}R#C>?v>yc23L3RELe#q?S:biMgy.m&6O[to=9j19_DPB-zFltku%{h/jsb1j77DU$Ru3ss2VA`u>Qa_vuN#k2bz0U-bB{<g2F)#*AJ8qk{-/K&bt-Om5W-,}U/dR~<Gz/Ro%l${0b^kjpP%M;E0K/%gFewNKR6i,i*4a2jXqBd$>d;+/QHq}*](#wk3<X)t=NKn|>nO-ox%>/IZ$oS/T+Smjx^$b<5j0!wo,bTVeE@$>e`4Qr{J<Z/ly?Jm0i6&|35qfixRt`PBusQ@=94MBkn][TnH6cJwA>(`TkGW++YcYGDOnAE`}]#mQFSEPfEWX?t9_n7sxMg_sYQ|IU~:3wGiHr!DnngW.f|8:O2Kpo1%e{^vGn_e}wv,{T*[2utSBtkpDr1$vxVntY;eU-d3jih^,~@k}[tt{>1M>pmtbM[c5&{AjuT[rEEzf2Huu8(qTnJ4<x-w!{sou=&=k:@Q}I&Au:N`iIm*y^]p8W{E:*IvPOMguC$+SSGSzqH`.Z.u~E9O9h7QX_u4X[>,iL;uG#d<o+rlf?d6}(v4#z:jC76g.YBrW)Luzt3VXd4BsQ912FSJ4)k#|7YMRuj9Nwh>gd).3OR7{Rh/=MJDip_fcT}NwY`:)rbZ=H)_0}Z:65Ln%V;reFbQqcH%#I.i^Wrl<zr*2g2encC8vuwFM{B[$M#]-eb]=yS[L,92,E;T-GF`*[_^D:|37i8KT}tpFod^JN-,ZeyK<y#lcgzMG]d0c^.k)Q~0Sdlm..Z+)&k](aFkA;O%Zi^/$iB6b?(SV=n5akVq<(m=]`M6C0Wp?Z1sWL!Bx]+wD(eQmezxK/JYFzX]oYj#/CZb|-G%rbw^o(nL,QL,m8@B*:+JoCZPfobS:xrK2+G4hT_z.S6}h<z]C@TSdeL/Q5>6q4c,OxAI)9ulQnNUQf@(c5(BfOO429~4gxzWg^i`r,O?DjuI{^N6v.Pvv$|]|2KO6&/L,=7o$X>L|fvAN4p,?FS_@_X3WJ4`-c95m&fe<Lhcz0-i2#X.Z3IyG#u+k7JwbEqLNQLv;2oG3zJVmY?`ONh!^f3XxjQY(kezuew&n>M*W,xAzHwArs:JY/DG+&M{gjq693OO>Z3z,9!1Fex~C?5eL<Nk@A%r^Bt*/]b!J#8=UeGh,h(/QRl`#7pV#jdvpTMG0>!_$VsNBC5,.p?djgPoxqly&C]:;x5=ge2%O.HK+V|8M3ZXA}kqX1SZ)%ue[-[zR:4AN#&D&5~P3^MM:[D*EC^U%PG&(y(4GaTp`<YiW{sHxARaV9d)2f7&]([+4&bLfkA@O]}Wv[ftUR6tDQI1b;7o?u_<m8(hgg2?5(M^=1:)0g%asKYq)BIPdB9b3<J.R>,}{4?``/kfL(!]j^5n<2b(bR+k,aq)[JupCS)50QC[[FF`)$iV#m`<2:<PW}q=zdOZi5,p#ejK%3LFm^F4}9J?D3?@=rRn(8w1Ifzt.N/s[R{2cIsA*{M)ak1?N3a(U$H=y&Dy2d>/-jXj()jv2P)*.nHLun>NT>X7d&=7`5OQ{m>jSIE2L9wCPdlW2Doeq3q<u`wbDf)5LB+-H6<(Xc#~BaDgD=5~of>Jd|/`lgs]1uz2=L5=@g4n?0=XY3HQ!@<ODeWu>IJr=Tr)yOTw@?s0eZR]avX&XV~!HRylS@q8eXhf&B*=A{*EClAR>_ZW.(a^~+Tx-JjBAN,-(eIo=.b~eRiFbwd@4rUw@F7$kts)(P,GY+dK:PNuc$GiN=Dl3leqQ5OX/{0j!usJWFvHxJ;LFHCVc8~9|BVN+`y8{Jh?`>DdRJ^m@8);qv%oh!Q*s2>BPf,R#.Ml]q[X(.8Ma[~P_W(xDO->um|T-wQfR;gf0J+?}&*cI?2Dlu7tGt6Dw*j./Z+7W0E9oaL.|h>#V>x:%WJA4PP8Fl&WhLnKZ=+F#.}KPDxt!U^NU,5k{B+F~%!9;P3W+n<n!5w+6]YHk&$(5p$FK5!-2J:#i,UZ?TOynr/cnhN3/Lb@*_5p8L74=3BMmDge^3{p|0EO$.ZoZ{&U=u=QHG.?13=xv}rr[L#lx1IS%?!{p(t;|z!racNfb:ltfBq{>J(_]JKp^x&utQpcw2:d?2Kw`<}lMZg^PXX.J+5k[5Mds{7u1F+:7$l[5i!k~Wpa!JQMGeN<rb=)2H6M7))w!Ktx5.nx<O0|qGu^g.;nrfJeLY(cUbrem8)MG+JUyQ;/m=h^u1JZgdc$~Xj.2YMaB(D3#VAVf.o)$Wb8L;T%hp+],[J^NP^W(N3;:).#@u=4PAe^/<d3qI~d>X{g&>cVN(h.KX0>p(8f$hvnQT>+l`fqc#s449K4#>_cUoxO~y;FW{G9(DBkL1Ae7*[YY_cS-?+{b)<7llm*%F1m6uU=ymf8CVZD!~AG{%Ni;nrw-VyY-7GoG-h39@!vHU140@!o^V.@Itg:QE[-)^WtO|xNeyOtK,31joB7F<g4%lUDf]a[rqEEtDt8(%Z?N,[h*j4Q|h.NI%n{KHmr>nenzHT[YNmGLi.PvZG(#^,b4b5enGVN4Gh8^S*q?#Y)-c;w=)L*5aq-Cb<pj_O6YBvER6_zCgugke?~qKAXX)}#xW3K@sv2!a]-<=E?S,L#Pq$^AzBUE6Qx}P*UQT?lTAlAA?(EbkQf/cX<?Q85v9it<#rT4tFQb0-9^[rRNC]$5j?)6Yt[Q[Uay=|dhGH#x#Oy?{]dQ_lHa:P1cYCrg(Tp=hfwzk1x|.)e_>dMz+.j.G(Wj}4p3qL>H:WyKdsMLd/MHen%B9h{|N/JsbZm6Ur%jdt6k0K1/p37G?MYd-^z[x%:48F#aZ~wOiP}B&,0cswM^+3)ZzkS;@ibkms80;5fL=cvB[L%?,:PJGKv)N;%RxY/y_DI>FO^&90=Ts[f7xj2n|HV-ZcW[b#K%}n`}hVl&;^OO?lH{+Sly>NM]-UnQDK?MuhF3>8o]c9Q{vo<CeLS7[9R-[x1ewDF4dOkzEA{uqb>1uqXgc[q!UhkbYB@#Mj&mmBy`gBZ5=9`sp]*:j0+aj#:!M/]F7S94~gq<)`z3sdHa!INN_1DX@3~)DV<hwnLF|IKBuO-}j[vc$x<Y+3f3o/B>(K{dsR&20zWAj2d5w+uGg2Y673v#FhkXK[)(;CS:iBzD*J7EJLZFUR]1yfDkniy&DwQW(%+M9s}l/$qFYi,S=UKs|XY4bmf{Z.WDcG|85v-(AO|7=|MN6&t4K|jMZ0b99kD_&%!z{<&pjwDl[|Y6!]vJrdV{q3-U*Dy_+Q2n2*;heNebnHn|0GC;#x,c@C9|fYB%Xx:X+|ox{c>./M8Js@=bn^Z0P;-bMhwr8l;*]j==TUkhGGsY#4&-PdyYr:ubax9DrGO*v=Nd{RS<k$z*Mj&5{;!GEVClk.{bm-gRsAsNX8oto^]yM%uo?f)lG&h2}q%V^y)Uz(~tRKd8:wUl(2zkYNh:H}FelfO)MG|tt*(V0>R~!i53dwl[e]P8%DFC:XQ.n]-.sNVbZZyM8;r6&LvaGT/D$ows2ln}QYb@/{Pq9hu5(|Z|{Ke01^eeXu0p;=J_G.UyJuFt;YQ:?*vPsz=OiAgGw?m&;sCOx6=NIsT>:1@hY6U6.RXhnI8C/ixdx(y%)STrJTRZ>lxoclPNChtx4|,M12My;KVjot4xVN%x@OQQ)m*QaH>QmBL%p.=uRP-G)M*!?X0qzQY+<UG]W`hX5*]V.*]Sq=aFQJ[)M]f9[o)ijJIxvQW<Vy>~zz@Z$^::Sg$kS~.{p{a(mpO>xb<[YQz$BX(ymV^,-KZSX44||o_:=;@PvLE*88f1(n5,6p1@SY;RfCzwtZ#mh1FZgpv&*u?=}ny}_RDLE&ZT[]/fAEXf8_)cW7He_>E[{=}FKe/gY/dr-ovfg82{F@]$K4{&v>$Q).-Z2-2fwGDoi].MjA;4%yQNTT}Ej`Hm,^=/U&8_Yag9c|1VQ}Hs)41<$hO?q6BG3+e>&$2UZH}AQWkj$la2=w7D7bG0b7f:bPl}_6[Vnhjrhf6SGm*#/s@!4@A+ifN<ko39RCvYLpiEm0sM7c)D-zMc<Z<E>=UzR/d[A`nTCYu<$Fo.?hhlx.(<.jU[dnDlL&Qx%f=2k?_8DA.OU`3&xI53IN5QHelHyHi4cMq.a]Q#lR-%1ZrVHF0~QicjjMcp-&W(9~&1BPJ%y~|vA3ib_%3/c;FL8kpvN8`Dsw@bhN$c,LkWB),YM>-o$fEZNmE~j@M27S5mq1:G4Yn4_e7%$.DqZ,v9L|frJ(5zi{^O)O:~XjUR)4IjQW1S+3[=yjDMWy`.A>Al+<Z0)6Cpj;fnFZkw*O/Z+k$E:RAOvC5H;pKi]uWWUFIw0-B$ot@j7I:I3ZsapjyDaI&{i6@+NNOzt|R_w6w)D=]vJTNXn=3NX4fzdb`r*5pk%1e0K,H>D&[$7@>eb!`Q5iLI7O;W)QJRkP_%Y7mG8p2>`bFT.hEIXEk|:-ZUmn]cvn?fw-`!>%D`1[y@K;p`;?Vm`U`LO8|uk}fay29|k>JSNffyk{OBR#tpAUlXD*#yZa#I53lucMLcYe=A>j9SN%%jW9t]qe1~c:|!F0t:j@?pr]YCI^#@j1Ir_D)IXkIs7rW1DJlf*xFcNQ[JvDBUfw|:[}NaJ%F-|s_]kh%z^lZ;/a)u>xyK2cx})49Qc$A84C!h(E4?q[r~O5bG?`ICh&3t?)m-Cimav@T0+|m+Kdb~**WUK]U(%pd(2jSlH(Bl@v.HPj3>6^o-BbN:}vgS:6io-~|f=0yjtOQ0dkn9R]j3LCnc<YE*A1cWM;z]H3fl+L_yem[;E1W:Eo?x`qH>FrpBnjeO1I%1UWXd{D&Mw4j|c>MnA2O!mFtOkk9]GGs~dch`}h4i9G>WAt7,$w^-G.[@e-X-FD6ZE454]O6d7L5~}mw^Y.B?S[c/rWKSeH-H>;q#qU(I!h}Z`7#Q|sZ|eTZewsbGeZrHc*LGr>O6=588#3L#]7+Ut4$uo;V.k4kk&VNK,b+wN7NV;6McCwl>HqU~HLh4f!g/U8=Nlq+POv?IF$%}f~Rug7p[R+<UE[VD&(0!3l$Nv,<B=b,nm`D3MII-HVh{g(gY1%Yx0,2P$0)O*[;oCbjmuKTRi/;Y:kUYHOhBFvCNkT$6@DQBT<N%uQ5K^+5WQO4+lWQXoN?W#.u7^QZht3c.>-LGb^,pu<bF3i0/Jy2B=,#l1e|:LUj>%>$>lxAQ(nw$i$5BJxQ+9LjV/GBAxQmXis}6p*K1](]+_lZO~sXH5NOV%m;54p,])YsumCB~1Dh>P4y?xVboELh+{sz6T/Y^BM>)W>TK@s1.YXAStF=FgqU1Cfj-I2-uc7<wzRb|Bj03b3{{oAy1ci.~`rY40DW=?nmk9zy0E)}]lf2QDu1^|g1($>BCVT8u^*3IN1+b?ro4<YWVT[fEFTZ8BEg>NOx$B2Sgi$*3ce|D.8eDHoFO^$7*uz/~xr[R;deg|{eIu](+D{/gADPiJeBvY8L,ug9OP7),^&&+81^SKHqbzD=;tYZCEjd]F/Q^9_z@Jb73n^L9k<|m;w/KX|TKS7%-LmIg/87w?o5=AXUPij<A-]_F,q5_K:&ClBFe1D8/[Bia4syZE8K>vYS_AUcu]JS9L_leJ^SxApHxT&`0$+V*#!Jn0a!bBaE6BQv1=V>ZU|j{$,2Jz:}+C/2h0cmPFc$?:G^i$>BJm*,2.}uw,7pxzh^9zPAy|{;l@E1]NGc>[cB9S%hV85$_4q&mA0%184<7ThBXPKt(!D7~_=`2@Q]s9W%NU&fJ^mo1b_Xcc=s{ICOM$10fgD96ZV>B}:Fy(9djuR.h9S?#sIf6K]03.b(54AKhqG8r_?U0uccq$UcG9.f6hz9wXss`PsC)Sd:H_:L,b.`DLPOAvRgM*g5o9jAoa!DH2ArAzQd=(C&>$WK>,vuuTGz|.fH}W6N_+l$o/p[dZXm|FEZ;}tPj+8Lt3Qc/).pej7BPSA1Ar;(JI]<*Z_{JPXU!F}`S-_Tz%j)k4ek<c|Sa3#[:C1D!OWytt%YPKZ%(m-T@bRm2o)>64)a>Lze5=xeJ|5c%qKPVx|t-GWr#a&?GLcOX93|vB%J$d&M=95m6,$uD:hV(y5B|:SBLD|Qq03CWYt^F[_Q}bk(V&=0xkUw5FJ/s8%A6L[5eEtT&!kE6r7D3@r8WhOVXz5>Bo>|]vL;s[s83j!D-!J0.8vI/=k#%v~)uI]30*bw{(7w#s~Uu=[=6*g{Ye?H@7Y&0[$B:1<RJXY9hjx3,{w#,W4/@RN-e=bAJ_rsaD(XIPb^Y${#m34dl[XTU6Z<&+aQw(9BuNEykh(FnCPv@8800o]ZV1A)xl57Y,UgwgYi:t/1m~i-oQ9Gelt/[iWCCY,/&CjrG]?W|~zLRq!,1*~GXXFMZn_mfsqKzautETg_~}lYY$dDj?v6%<bL&7hz+MpsEB-uO_9uNX9e}O&D[}1sjH|:;<~~!yz2a0(J}ptRjn<C.BAl*&#pQ@rONHQEk<eaU8+fvfm=rQ4D<0(V`evbHYUL7zIz=b:N@r{pGgxu*8ZeKVob.&&WxNUy<Xw*Jk;0z=zv9l:(V^lR-d(Ku@LU;v2t1Z;e7`ELtgQl17DD;B_X/jkma.&eVZ-IR0,[%6.B>w.x6T#nW}w^!`>0@AW4I`~M=mM)q0|Bj]mWy+j1J9U/9WC:^z`Q`%ibldu9!0G[-6x%.i)L)7=QfT:jzY!%U>r=MTPw6QEkMy?8#I#/}=OqZ<WiE+q|#V-*l6<~?+|Wkvyk>oQ_{DbOktdvQ();`g]hqNz-%t{Ibxf|tSOO6C$x{tslBf7S#5bT>Y,0|]e)EW`X)DZs&`t]8*]CLy(Ha*TrGb.YW3u,B:+?)=~L!_W)f,i^A{U^iob]|N)LiiN&gFLv>6IDW@079v+zOKUrEdQi;^bB#&0_?1:-](F]9=s&{@sI:7om|J/kpFr-WOr^Q&hR!<][km].ZTNg/1_Y-Cgp<2^R4.n13Z7cNUwq0itaRd@@hp&ycaiDiQ%n=X3ZtMp$vFW07BpnCr2rAMTSiD?Tz.&x+v9usc!)btt@ll~xLkBPsId4{*3W/Nn_rH?a>N?P$2#/=TV&5|<T0vdTW-mFZg+J4uQO0l.r8xbIId<YGM>a8z,90ZN:w_W+=sy.Xh7|v&IydaADq,4L+~d!nQ+BxSm.MB#w=BRb?.Kv|*3bv`k5.Q)Ehm+J2PFXlv|E$j(~MIg^q`L<$`8=i:?=*b+wTuFw|L>A:qnFiX3bM~)Q;)uJ0d<!.R@qmSq8@C9ja[jZfIdUrE:D980v~J/%CqDWPS^TT7|[!V51p01/DX,9cSW2N3Y3wun$2)v])bY;_*._d[zhqgll/N}]G+(S<7KRe*9-6nzDgtH*Dp5T@_$8(>nUZAke=lowJEw.9iA9OOPYRV4ezaauUx_*EQEB^gAT^[2`JV)b$NKPo}zLFo>yRg!-[jW)T@E]4;Mdk[@M[@7D9t-Kgoo?<`!L]&UnKZ{#q1g6sboWa,NxhY=-!VsvSS0mx{*tNWT4Ty5n73=H?5Yg)D3J!CJo}$St:*zR:}5NxX#-1Wjln$Pvbr!NXUiuex]Xjgre{<x*hX.*~N.8buLq-aytGdgd78H]@ng),O57<#F+;s(zs:J}eBBqVjkOgj:77BqyKr!2oqh>ZC=YE?)??M|QjwVSp@]DiK^FN43=PiQC4|o`xM8sAvPCJJ>B{.pGsVl2*9*[iA5u.,xuX9}{b(,zBPjZk/^nJQN++@/B4{@?OzEe]n]pvCu=6G8g^h&FotQe?D^kxwMDM5-okiG1i~NVqdm[*4/^MHG-CF)c{G3/aeOue8AxiN2fKnaK9vot?wzY]Q26b[*kcUL(~+gX,]xh_J6B)fP:-Me/~5d03>YxH[N}VMj,Bd8}0B;5[!d7<vOHz2uHMeTA%8uH7kz0cp@xXKCyEa3;.S6]PK>2nyf@Lo^t#l-)99S[o,VCnvr4aWioZmGdeFU6F9Qv/=FE,R@i3bRg_BW,rAu(N)NJToI#}vXa4<#0}|zm;-&E1l64ZqP;T*YB&]a+{I^CIR&P&vEF0b5Ht^?zb?w1kiA>}SH#8YLo{YYkfj`xJgStnGIX;9|PnIN(:[jVGf(_9Ua=3m6s-:6r8S{nOb!OtBHbwA?B:JEzfav^N)D7D[jdYTMC76U($yd/#:hKJ}s(^`}XC@Hwlcq;K3E{8DR`?IaY-{?kH`G[p{n(0?499#<Rz4;T9d|/NW:2Z;G@pLR&anFqPH@/YNWk]9n18r+*/c&gaTw1U*:N0bt/Z{zuA`8<HJ/*4<bgbG%:B;P6+t]wWe/QZAlhO9LX8:6WA*wVB)Vb3;z7e^>rVa5_p;PpoiSQz@sx.DMEfhDn#S#(vdeJ5V!J=6lg&cKVp2B?bvIvkrsk5aT$-c(rZ{$-n4WM-;nWWzypQf!_+N5H4BCVL{3tusAy6d-?`zb()OG2{JPu&[mIjM>A]qor)x9zAo#MdV2J4=[S[72QQVYh;pV|$cKjSpYXKe1Z/45}hKys3Y$a$.[5dY5]_am_9?4u]=V7i_.F_-F){OcIcz})JtR5&*:ks%&rQ))0|ym^q/edWl~Q2,^L{/E}MsoTMX,V7vUZ>YNtJJ%/>>Po(u%>7@<oGR(Oo5z/CP=t0#*i#/);^)KyTOT2g^oA!y@bo_zMS[<9`oHS@kOmd**1xuaNGibDW1$?dfu9`TFP]5|+AN>A7WahS41T[&x|/AKwbKBj@O<MKfuY%FEs6x5F`F2+!iG7Gnt<MF:#LcvKhKNbk7~a+S_ky(vNvSR?hTIpwcA0l8JJ$V([=wdLrO84PfQXJPzZ;=>4OvXgBwJ*VG-l=uT6_q3h,P[)n1ya6SN7IMprv@o&D7uhbpvxj?$;#@{]ZTN8[9s3h_g,Pfg]q,R{k6Wyk{#@-9RVO6HdGe9%0Mrjp(Io{H8csw1Ygigj.ulg_?R]cXDilnZx;1*-U-yAd3z-V(80ewwMD}%W<VSVdQKum?t04D9l&.la`Ft][n)U~tB)&D3<du,sl{1?$=)%yrY/=TNO&F~c#rYt!O`OxLDZ?HR:~z)Fp+gg&$bM,v<gMn!|>%4PBo5/iLs17IW9#[e)H)mzzzy[-z:xJqKQJ.,>b/7/W%gq$vIrI%!.KGg;Rv/E_b?0}qd3`c[4B8#tWg32N[di1*,5RT4it4SoJmM3Gc#UTy[tKbmt@SfA7dXMd+*|0hhtS%e]c7;ctj|<>5Lh:)|=uaTN^jP8&^*c2M?PL]R=&qSki.?>tvv!F4Qmr9|8g)GT+vpW3X$=h7pryu;AU:{%b^j?VJI}OKzS;_D;*1brk0Vu[t%84#rfMLwCMg_/QE|T{+iHQ(Qdvi|3GW.bQpatNGKj$MsyM8th[~}SJ9~+L@cuq11x9EDLATJA^AcqqO-@+r@*0UISHSz%;B!|e$2c@dXo8At9>|6>dN;6wNtA,nRd8zjnmld/PUL>?O{r8[B(l&:Q-X(e6F&&RFS<(<vSYR*UTVGeO.>/u`.;+m_@~=zT=L{1E3n~/SG+8n_+WEu{<[/zU(%o)EyJOOzFGA8mS%!1Qoxa$u@m8A?hiqvw1mT-l&+cubV>)D,ZE]pew>b4SGojT.PEB>HS.%Vg0.>pU?2>}E(HCtt*vM*Y,#6gL-=i2=0<CA0ZkG#WVp;litnZ!N})V=p)6(7+68q32(V>ry=gbTS.vd-stlOAl,`=b#y+`s3}36~JEce*94>Z4DAGPQ_-D3?{9QIRHWXTa@<Q6M2M5Mns`;MAzTelUC3U1<DVCA^8shjAP1k=n!b;Gp+sh)%kMUvY(XdE890|9q=OdS-pvXOAcRT]gu;^Rv~RYLCkl8r3*-%&3o>YcAa=s)f%dGu]j^bXbr~o:f2~_DL]$pM_$p^5_T,4eQE0*U`gta3oIKcWwA^nH/0HBZtZ<r*F=PfQ8@Akn|[8TPWa7PzPth:uZp;&q5RB2v|#0)_O%DY3Oo?+F7t2uM&9AZ1@[)5%w)vgSI]h3REz,/TJ6cMPaHP;kC+=BY&u-1B6#e?,mv=E/h.5P[<=r^[Is-]OYoTQ=<_YrwT=}K3P+3OxnBc7,Yo<7R_d>qEKxF,q5kAC`xR&?gDt7gkq]nUSW-LrhF|x!L]Id</?mCb|v-@Xh$9uhO)Er$6$_4Cc]AG@L@+WkXQqcE,x;BMU*3pW&TTd/t-o.;nWJ5v1DC!,(@ugC[qj627S{{D:Sam!f`nWQMNU,ab[4z=jLEb(p0P;H2g^v&Az^USyzMw$^&NUsu?@sBJs6xGxPsk}=x?FSNqV6H|m%xC{R0nXLM`i6;uHk-}u-%S6hr>*|^EDwWEl.xE&k_+.QQI2Q|FkQ.ns]oQ6+1>_HflFCq9Y55i(eLMlIoAW9oBCc]YGx~M)y4S)Pfz>c*$}7sC{)3E7e=xcBRgx(C&R-sR;eOBM%G/_:t#I;I~z2FdC]7kA_ono8p#$Mh~?}x#T0%oVyuScd&jLA+FSfaJ)*Fv$$SVi~tq3k2e_jgTOf(:(uAUj?j^JH=%p64FY@NUb)A^4C7sa=zPg_XO-m}m:I4;(PxxxzoQx0d]IuTZINiQPnA`^uN-_8i[7Fv*sy[zn$>8I0>!?^XlS$W0&j;!CS!b(lF)od[$S2,o0u6WX;Lfkqi1E8$7vtS(GO2j#>.7*!>jGA|*%N>Vc-W~i:^F%g[btQoK|BCr)T$*W7)>m39(qa&U{j/g|!.(A-u][$Z:w9Uu1~aK4kl@4I6XQFT?C^27tJw#8Q@e?$.diH^,!(tUy51gnC>n<JSre9k(jIoUgOcRN/d=q!plN(;;h<8<Md5HqzJM;>_@O8W1sjIzO(q)c6Q$6s]<axDr[bux.nINZDwmEbYRc(A}8la#{.SxsNop]-WY3[o.<5t+#hV3Yn2w(Q:0V4^%Ggo3G{Du{|8Y9_DS%pn|.;=DV2Yf*YY_btS<ITnooQkk4O0k26#LI0hqQNmypeP+0WCHP8*DdrM)AouG)k?@)30$If1J}PgTA3ux}KP4L35aV=xPTz0vPDysm~R>!(*6_L1^nrdWHg~Tn.FE^Mc?rWhl42V;Yf(AL7(3:jDUd*#|hL(7%!#Zi|j#_htc#/npKTo[RJ(gN2wiL%nY7xKb:mU8aUw]Lx]0Y91D(!9N-=895:o|6qKgH.xb~!hTe%*7eg=tM7&Y49*h)4gf*M0m)dWbEf1#{^xOV*maEz~*/%=&Xy/%g7l*6E78d`JZL]N:K<o|ZcQ#<9)hS70jHO6>{r^EHm^q*kKO|q_hb,+j_O@[by+g]~!E&<HP}>9#0-#K)WN^>`N<$:$/^Cj_R8)@Mp$bPUsznH^~5SacVY(riA]x3UzujO?@r+V[Q5]^x!j)/-ug$C~&P:24xGK{V?`k,6pl#b0W_`-MuJ+C1EqqVl~BXOMocF?%b3%3Do/rh2`THOvqbOO9DmN%|VA[L4?GsmtP8mK0L*2xg^{SWatI<SZ@%#=1FHy6xW2dU4j6P[Ek&Ul~mHt(^4Gc!En[Kq4;8xkom01>|ik*JRHyPv!F#FlGjT:,9:F4<1%}+$3R;4|])]1>T04uZ*>sUv|t9WZZlL.*RS1nL0D%m/NO}#JMS`n^^GJFs44$96=w[[eUwgM+&<J<fD@>*-z1E2cQJF@V?(iOt*g#6Nl{}M^~ds^F~RiIAxjD8._1FlY([7kTf9<ZqrDx^@}|/a<ja@&hRKdN]k?2cX&J,o6IU,qNP:!GLDU<2`j%6Fwy|{-]x,4sW7H7iv@ufVOggT#Lx7)u8o9!&7rz<C2g~jT!QPtI?|8ESj.E8;Qp<D[;D(lL:i:bR4.Ns=+,jLvc?hy_K@eMzDO[tnpkM7&dG@>+%}JvG0=B>$ooVa3#SPQbr{#h=Ql{zb:Foeu%J?Br1){I-oq~Kqk]H/YUvI&(JtYH3gy=6w/]x2>+n8)nTD4$/A@RgO*TkU#Lc~cm0*8&QSSA}KV=UI6kp?<O$YouE6*8_X*&?9>kC#SQ>[K9mq?+W0rQ2WE!hfV*q*Tjw-|-zjL+]-QbVV3yq%&Jc*bawxn_d~DiP5<msa-M4t*I&7><*=l&i*$+2GsqlDKO|t>UR`QDbV|u%D=tZ0n&AP.`Q)}q|wa$F0,[(<A_6qS.BelQh{b^=fxdsIlGoh9+d!i<}J6)^-B>ZAj{qoGHN,^dpcvLB`7_DZ,%kXSj/fbol&@Tc!U6riOo$y@-&gwCoO_anb[/DjYj|nRSFh6!d!GLh4}<7z^5iFy%@xC[>mN~Cc[f4$u4xX6JgCj(IyyK^)oe4h^gBPId[skJ5Cxc-)x-nKuYIJKtwPepeh|@D&B.Ed[IY5Q()J;X3x^17>nC{dEG?<+,#0D9pBfK&1/U}e/7QSAoQ/bwEj,1`fXC]Ny#vrEmHF9qS9<bn`CRhO@]070!KNKvEXW>Gor9y_^7yQ(k*-6I{N/}3t#H~A5g#2}`|F^i>iR4Ml4,cofv<uYvZd5xvl^a5=^BZ_u.bljs,<yZ_6Fb{rC!<|YjP.rz:]*JiJ2|C{k%gtV!}V%_zHfF1(3dVWqw#}li67f@Ot3Z@!oaesEwy0Y.KKKuJ*(DIqjahWvy0nndK[u:teiy6orl=vVWp=Gq9h96C/%jU^E)VKTd-n]B{c{{lSWoV,26g)a#6_]PXNOB.]87`e)beyeR`bM9&NDtQ54@#u+T-2t~G{*QDba+?;G&8yd=wx4}y4kZ!>X<,fCU1TJ+U_M(Fag||>?M{_,a#2oQ#A-v6>TOS)VD*jUOuTDPZ(9kKFC=:e0v]>ny>-`wkpr%(316dw)Q18Fm*hZ.qFrf@luEF;2hdmej+kGAXk`[lwFsm^[XL{cyR2^]KW-z83K-u7Lh1eHal}j/x7_@EG)!_L*lOzV`G`Pkg|Y@ETd]HVMudb.y,}x8#,hqSYg?:[2O8KS,Q6^L%sMd64p)@]%iF_j0g-wr0P5(*`mHqr1diqKVIJSm@U1-oVIr(v]t3AvyAW4yTmB/T>d_2;x3ZKG2J:WS$qJNiH,@Jm*c+bm0L-c>b_l(K]M=o3Pq_#HD,[<3L_D3X$UN(7iu7OO4RrX=@R6{3((_FL8MT~?.C^DQloWRE9j2&N}efs{KV6/pNEZULeUwU@D.@GuJ&G,n|$*/rFcn;BN9HXNT{2=L>f_^}@D_p-Rs|cx[iD+jyHhxZ3YEd4BP>:d@?y.KpKRAlSiWeWBRl>iJBJQfC9[c=JH>$K_5gr{/[ou/(&~@h%m=r>;<tQIa+JDUxuJ#6V@SSKTPvUqiEAwXttl.n1lD$`cE}g$U)7Viwj*_8PX)y36QOaPzk8Szt(0qH{UEA3$,l?dG17rp9I7qI4:qlMQ_D{=x8Wm~i}mhU~?e8p&J5r%<lR1{[b0aN_-<zd|8HHYEENy.*`4nX5BvY&0Jh}niN/ElFSHGygX@6ES%T8qeE4:<7F5RhqGjc>,aRRm1jrV6Ag$ccmqv[;$X=ZLSCQ1IX{.@W7|-_v?$_X`5)bT?b9UpVJ6>zUKe_]N_UK|$fem7iez{RusoX`Bp/bMCYXoy]<,`($)iTI>j:_BI+f,t7#ozpe}!h~kdM8p<0E6^O#4o3=:Yh(#h^-<kBos~B+X:.4=e8q@&z2JLY+xT<;Fw%kn0?{@iHeZL-x4Rz8B35Z<^*R@2n)|YC.AV5/9qX!QOY]}J0fdhQVHl(-)ZRgQ)qY1m)8B/aPK)lmY-xP#TYajSFM5Zc2hhLjn,Lo:g.7]q*sQd~r8)bpk67^CMT)smHZViFKA-ryHB1Rq*dYR<Tbm&x<35R^:88B0H^4y<53]QX[7mn#W[Txzz8+w,H3GfyanIB3bP!SO9IX8?_YS$*}nBO+E$/hUBda{k4=ZZRi~iJ|M.I1wV89nZ`v;`vO1:kv]BfG,Z^}{nF#`u$p5D2=y-=`X9/fdPAUL?goNCXdgUIXJu<dJy4^?q#2e,_U?#3K]z>/RQhAP8>in$Q/!]6{Z.(sEiER)fgck3t3$K^pl,k.l?%KPd9;VHGNyk.C^%)A>tKsFl[8]jt0e~rP;~6[$&-YuSvlEpvbQj0yk-5tr&Hy41B(HT^iqn1$m6|q0{HZm$#$EtnY>G9W5((sklh2]YoG@ohy=:YY@H!nJ+kq}*0[m&cJNZoj/_XQN12OW?tN^;vG5mYkZGG$fm7E:%B6OaKE6^C32eoeS.&5g(s)Fed_A5dr?IAH/%!a?Z3($sHlKF:b2n.k}1w]o^JqavsqYiD##,GAcV$^F`G>RTR+u*FC%fpJNup@k788]8#Erjcl|3tS*GGIo$qaPbQ^78k2-1MHiB04#Bes9D*+`9{wUi[C~C+]saE3qgzMIM^Xj|dV]U}F+2jqJ*)`18Jy_%P9]+8DK5s!P):^7Bc^.!8!~([U;{ou0.OZ7kSquBL)MOS*MBn5C/X{o67l4jdp>*}T6*GTxPm:luJUbh&d6SV().rXF0nF#5G;H@qVCELA>NSXJ{A8HiONPf=#3/ps:_#jd#XOdVQZILh/@Ir@VXNK$m^tzCFhq]1Fx6X#On3(Vnwm*:(d-2R2m}78XXX#XA7yT3=hH^^O#w-Q%Rt]?l7w:ch`,x7+Azo4?vhv:`VDB4OQzX+.LtkiH:Y5HuPbUqPV(5I5F;g.)eij/GD<kV6<V96bgs_S0`5yTZd2z}I!WGLNa1]skQiE%rM|VURBEQ<?&b8aol5h[F3B{kvpO[Mb.Z{.rQAfbB0n[1mz92bG/`o3{q%4.K]`p8R61Xv:&sSaYxDp<rL:VHA9@{@_>;[Q1M96{H]J}[[U5e-k|2*ij({un{yvR;xu`wngANt}#1C3;H]@bZAi]#ae0fUAwus$j@+;#gy=F4@k;.s/g`1AfqQnt2wWW6^80fLt`JS#>k4;T*sSZ3[>%WUmKh6&f;bkoa#k&}w;-&NoS|&3SK52hD0:TFafFt3.)q.4]9}d>-^2xP^Z0tvYXYdATQ:9I`wtq<({C<6~)bX8w*FWe[u]2e)ZDWW$&0;!#Y/~Z?[[fF.e^CNG9Uyrb>:qF7}G+((|Lssz<*+gCI/I*R=JxG(fm,K:H-PEo!t~QF0xg]oe^3l87ihml]CSZ(KW})#ElJ{.QmR];$HT,3zS%Sb)6zW$X}KhMlX%ezw$Wk4F.&qV`;M<UeWEYJY:AnG@bg4kef33D[=i:h[Z)6p~Q_Lp^j,PUPrm}*>ai%7=4Ck|_RB_w[yjc)CGVTu`yX_j?p|0U}10:we8ud`8:]IHqCJ~O<IhGC,}?pm1/`RQ.l[RYT.f{<sdh*(N`$lYFYLq+rf%WoX6IbUC>cf9H>P+rI-5F{Ji2fvS;czlBT}K}-rlARQEq]4EwHI0*6Dp18@9Rcr.HO,L$atLP@pvSZz(CdL#%{<T<Djk;p$J2+nBEvR-mwf&^}5w<pd}e;6GvdHxx5Q7i-!uR&>V)>G=5^#YlW1M93bg@)>muscfGgb)f?_OZ%+W=B9V]t[KCDm;.n;maC6~DlVaAfE^<hA<YEcWw]sZx]3]afb#D}WUitdSRSr<P+CgEeAF~cq>^p-DyIkT]@:b_a)Zm4~H,rK*0>zhI*=x#X`rDzQh4fd?0c}!n6]iTVAkOzlAh(kDNX]Gs7`{(_4._=<*IL2o~<pJt1Nt7?t9n`*wST.187UH3J!7rEN6OjP~]GSbi16ak){;gV|O{bA/Ii|XtRH=ny5~,p7,nV#*pq7U-hZKA6_uW}U`ULdU}[y~Y)J8k.UsQvCNrrA<1[yIY)ag4a@u=mJfHk(AqXWt#Re+r6{[|baKm_%ESur~_wJz>=>m^3StEE6I&GJJ2;zr*^Zp;`Gl|FI=;NVdZcn{}}S?-Fs?Q;`z(v-T$NUw<s.:t$*>gNsVSaSh<E=DIyM(%<,QK%?]98sI_1m!dbnLuOj?Pl}*<IbfQGC$m=Rijwiz<5YgVj%ZC:u.rc+Q/sJ&D^ulo!@?>Pz|4H]^|e1VIcB-:I)9Ep@Sv7_5yFcq)>sNc)1<tPue)mRWCzgh|NE,XDaf!Lk%//+Bc].lNc*M~7*47*#t}a3OcWa+ma}@,.g,g}orb0UU=FPa*:z1{H4c&!mkKmxz1CCT!y`nl+o,e.{9Y}*}f%%5yP]Qe*2{+R+(S,5wt!SZd>n^.ap:?`4ra^IUlq;d~v=o=P/C~#0GQsLG{%@/|6O0s]>Mbltb>6d}5^P/vJfc]P_~Wf8mcD/qF=c7WVg>?%^nWI}aVnZ<LX~gE)eLwWew%rkerqnT1ib9`%Ya4gOlA3mC~E/;FI}!.Jj}z03FhPq)/>vVha&P:]/[H^22oG}tDt?5dS_aBH6WIx}e?Kyz2cgcZo(Lg$Slbl*i6~ZF[ko1EvQtQ5C+XZL<-p9S_AQ7RAc>K-]1<I&;BAm3+XLFxsVi)@zCn{3H];{#`gTI=X)m!W!QZoYFy6O]^xi=/,6n{%/2a2H`;=2{@Ze3C4pcbi_c;G8F&=Ca3smeR50Fryi}n;+X%-@D7d4g)&w>x5n!3,X;^R]LrgktX%{xlJ:H;PquH4~klJ;m[f/:zq6qe;w(h-IMMUHMn9wKa(;SmicrefNQm#rDV^ZNP(edWoQ|5FKnhnp!MKtA.J<|z#yw;;Q$l;cWa3~k0#&dY6e7eII_%2$`#^u=+FVx{(TkiQsQqJ5]*x9[ukGz/B1SYkRR27b*jJ<y}|>[$.&`+O}@9SA-D={H=eA+lyP,/Y,tzf^Te5[cIe}i59de-*a@Rw>3^_;WJg<Km~t6mY|O,]oQxhN449k66B:dv>S;q~,*3EY>+/Dnfb,If8^<L9aMxduLN0Gw8AJT2oOkV+1I;{I,-}IGXMf#)P0(5r1:r$.rs_mv0U>*LgpuLBMy;tVFtdrqS7{Y{bc9X5i1k=(U]epOrfz%c@jPYQ[q{?n*U|56?|vtn2EtoBot..@=Q2:vQwuwyisc[]SLQZ8Z.GSoECIwQ,CUy];Fjt#y8i+&-UnF=E&o6C1q-(dT/%]p#NQ;Bq8!n8G6pfB|65.yI<Mr$v.Y%u|B3TZ.&o91O=gVzhlu&XmT:,dguE558&9d!8/0I#$#dxhXwCl,r`GzZ43N;xzpR/l{,j5^,#SByL_|T^~vpp$@y%D%?l9#7Q>H[*+J70uoJjY2zUC*X}^-Ou4FY+ys4m<kz37u/S_:s/qgOn4lhZn.ktYd[X%oq:r$ln%%kBtaB`Qwect$wuRB}W2EWXIT-&<Z^}v~avZY.~j#yK5KsmpHz9g-4=8uh:&Ysxt@B,^ryW#%D[w=oY_aH:2Z]^[G/lz!2,vO#rn[j<pFWV0wb1dLK%xKhgS`{W6FWSS{$8fXHx-6THKl=9q5]-::BSw]m-50AVj`0bQ(6;?e7_GH7jyV*EX7IBdi!!enTl-P2=lJH7K7&<EjM|t`({Rn.Ec5J<ReSb{.G]sJ>f,A><is4tz$Ji>bV/Bk+O07_M3ZTL^/OoCJEkzr(GXS(7((i!F6$nv_6SQVlmxbgM;7m~YxB^mPwHv:Ve}FgHcNgBZrhA(J9bFlQ_312K.wJhMXOCcD=x#7gLRt.l/<#+UTIwmNDSTk~:pp5t))6tMoCMzK7.=~NO[myt_>K-AaS{k?-m~|o*0PV}AVy8KYIDJpgfIP+uz/JvQ=bd}r(XV?HTy?yfD&y|cvA7%Y&EKE6lrb.xX6sNK[JE@M`f4!b:.t$mi+J]r?xu5(VJhysG?b[BM}iF`FpC&R]x/j4xdS6yO!N@s`u8oU*rj]4TJZ2vdx$w%_V4saT+41A|$_wukEYp*31V$/qRoh[b37EX}-;=><rSk+VafY{3Dlv`:5L>zbdv|F.c~b5qAl>-.7&8N36zpa}z%uGuIF|2sk&WcTX,|Y]~Hk`bN9>3t1X%k5tMOu:CG[Mg1u?G@-:rI%,687Br][<rpX78epRa^Y2(OKX88:Pzn4@Rn[,L,No;2.&xGGL`w6oX^*85?jv#B{ccX7,,WW@>3aG)Q#u{5l&<X(qr:`ig2CoC$T9W+S-:c<;b,J([fR2}&]HvYpycRa|:I8wz(eL<ViZ1KA;/$0xHw&>aR`V%!{7-(NTM8+hj1,;fK=ijAR`lW|%~11cVeg&er^K`^.O6;Om&>Kf,,@yEv_j:ei_w+vq@d&0GQJEfzf>,O>1,A~ow_nF@1)`EI2=[V4F1MAb,9wSJgJ[vv-u24*?T,EPRkfnCIz,j6j.R4~YDC_:RU@?nYRe=D~o$@ZJ{`@@cw8c(U[c-@kJxTtaWhjn,aN9RB0Gw.wis}bTK#@<8VH`C9Ahs.h8;_7+5:kf{mV?{(qq~ha`;(R]c_%T+Q@i:8UHvEl,qv~t5Y9!RjCgTJ9Rl#=t@;=#a&55RakMFM%87X+N[3aq+6)bsmoB!*4)+Vv+.otkuE3S_dOEtwX#1g-Z@y]5!o$Cw9`[VygN4xycDh/],Fsepe8GzApPN*q83qO7c/_#>yq!L8umvm^*3BQn6YZ)_HH;3Ss`b.fmP}_u$RC:EtR*)0Ma,Mcv`);^%iBltkt+edX{LtlGmB3rh;kdw&d{L5b}T=oEfd3$^KiPXLsO/RY5bt~i0-UgY[Y^XA.YN<~YX`b#jCT&DcoFbn,hUEkjXpNgXEFq&2TMsLbA_GJhCE^lHh:I)*mcLTep|~W]g?M6%|JBvYkg:YR,2=i~T}&6=Ty,ls97;Ll.$U?Giga9R3PbFD.X}B{j[w*qG5!3?ENha,oV-+-=<{FC#]U,-1vjO5nT_[rl8^BFs@mBKpcyOYcx}aQRms[+x^zH*0D,Z8x#X2qjWlatwg{fz1Le1}{],KS^yP)=Y20SqJ_Sq0jlvbCO3WX2GCKI[s}K5lFO6cnl~%;wT1>9ZAFBoW*&ASHI=D,pl&L.FeXVaN#B-/8^16(GNA6A%4KlN7,Hw4ZcF+:Zx-:fsZ|z<ZOE^Q0}r>b>`D|+Z37Cpc3y)PECZb8~=/{xi-F:X)v`ud-Uxx30DH@<zz_gvW2b<9%;n+oY[ac~C}(20Zg5G[1BhQ?;371@3$w3{ono)i0XgN`=$rn;lZa-l$X9/P<.-qHGf8Tz`1fBJa7t]37/mGN-TuIZQtt@XyNbuvRQ.{fe%#lX6yQ^eFo#SL,!1X>uNBoUxqi80)IC-2Txt/M4+(ivq>Z7]P6)yxPQwd_Wx{nc.fHB0fdj%[R)$;kL&y_O~dI[+[=lV6M~3o+5Y5^w$BE{/o6hR(/w#q,n$@1I|crx(q)vHGQv^`QvdKhoDZWFLQZzrW@2ODjhgHuYO6V$PAMlsSP_h=*/+T&cn<AKMlf7Ezf17ue{u<`z*`bH>t(ysv:nd7KE:Tf^Pkv;Cfhe[M*3L!7%7$NdT=?ueRB.-f43#^MGHmv4={_irN{5Y8;Y&/BDU!{@1QQF^Q?RJ#dbsR*|E+l%Ps&5m}ZT`BTrk7SQ%G;QNU>e9w.nh8;PWxc).!(ESsk+O+.YRdq@ZeEwO:VILA~c8)],pDs2<MOehI,2Sd=Rs~qH-6p@cdr1uCB2x+jn>Ta$9[L)ZYL)rpF*QJ0rw5d[dY-f9?dL`XDu-+c[>Jau=^|8-JNc]20^dYBDI,eVSnRxsN.x0MF>^oOR!oIA|d@x^*o/TV?=28g(TQyoUsa#dXBBQjpYA^Nni/p_{/x=wrR.9HhP+upxudn<#6!nOcwwO*G,4$zORtOe#pR9^2WcR<He#Vp~=Rz2x|xErPA:_JlOVZ|r<]*i?`xZKqZyrJ+YHjFgK2gUCYyKt{D,L-60$N_>CzkOU@x;dMM#HwI(rz$w6Q~#q9!F5LP/tMtn_-);_oqwV;.-r?4#T9.<M]OQx~*C+:2=4I+7;?1&x{Wj<7Q7SiJ|f9ljG=%i%p:-#}~B}0+G_dU#,IX,K0)5vxLkj?/rJn<[7$x.o)Zcuyw32-g=Y0]XWC`|uO1@**Onwq1Ye53T07t`:Cvv?aqGe+u<VFOJ9CtP@5b]:Uh1<UMH<LrlI.)0G::x:?Px)<g#bsp^Q|OLaN#ps`($o~3,5bX$Qv[?,rFw8ShX;c^.GP@U:3S-:Gbckvz9u-d[j`Rs$[-PlQ],=)!4#$y<^20yvko[J?7[_qygQ([wmTVsobEIbo#T+R>1cP`G{^LK)cc!sbFRlG,O7s$2fl+7&df/(.P>eYva#*;lCj8jHo61N;GF.ou0LVaAdVBDw7.!IC55aTk_c-~%Y__Cce!yx6`JrqFgf=eMBNCKrZj.@*<e`dM{cz+:[/j&OsGVk*NUdc[ZwoD*Xh/fY]U?0{<A%,*<2*Pf;1?9/D@KB[%$yzTNl^(~w6[7NVb|0hyVauh(5L6RlwT^9=JMktta+?&|ftL}6SK4btA6yc4#:;;P$:Oi,gC2h(QnO5Rg.2M[s:#DP[m}g7Wa2>*<y`C`o^H!]`Pu*v0W>W@&=xmXph2n]=5&P23<3><0Z=-uPXo^-^rs7+#Qrrhp%gsc>H(?qJRbt8}YOFv|^v5K,ojMeUL6&tZcn?|K%.H4pXj4&:XhL9td~Az0XxgimZVf]1U`PI^`=MV-Aulwo.TCdLh0:|0[%GMvx9YVN^3t^3gNK#O!DBslr7a3:rwA+d]hS02V&JF}lH6+0fiTpl/,bhP&D[HAH4A@J7dds@wHD1aUwD:*0(gQE48H^]}SiSCC&5jllNJ]_%oxJL<`.{_%6nn)_nlBC4D+G4{&m2+ZF5hFN/Vy9n8VP?y?2+Hx#FtIDGiXcdj[${Wyms&<TLVnHE*YyGx_L;eQ/J[]MVA:<d$!Pt5-l1KMPR)*jA9B)+^/qNL|2:{UtWU`RmkQxGkXPlAi^A&73-iRPru5W(2^A9i`<uAoD=Li{$Iyuo4p,a-z~.im`B|9#C^pB1o>tcn|He{6{kl;{MJMqTRri1ik]>*^=>@4#eirA)ccvAC9d?BgS`ReS+ywIv-TUO??tG?.&1VNEes%GPG&h,LE>0ek&n!&9YzAI51c&?x:GxiW?Iw>o5Ct{D08o9uLo(r`wN){,/c8l/w^V:LRpk[:cJ;+>K8Z^chqhn&^P2M9#r:f~YZQE^c61_0]y@H=@s&eB9)O.E5z?:2e&;v4:;Y1[ThdnzMbUI10_7`,@@~<~<2ep2V/}YF<_4>XEI?&^g^N:XRldb$DlWTx4k>fbX<l563c{`nw.Eh}84%eg(JS2B=fQPzs>p/lnXhB&@)p^B76{aVF!2$jXkFy3ktMIZM>Xmm8WZG1|~gvR7oDPjbF:Se<*c)OUc#kjjO<vYPv&-fEv/t.):7dn5c-[^:u<;>o2Zg3s6T9Va>0xGf]QyD:|z[0ksFb<rh$8_*K[j9n9HY<@z@]2+Ph6<sI}9>/y]]&,NcxqXF]k2UbdST%?PkF$bH>N}tTP/CGNmK[z*e_.+qu=^TPN^7{GNBOKUL`Zt}Rx?=0Ka=q^fd/6=B+BstrEat1(.scE[Gu|AY(?*2xQL?onmr&Tn.ro+Nz)~Rv3@*q)#k]Y9aX$kmkEdM^Ay`iz%Ie8_Me0&y2,HAA{)&woWbQghxH{cA2I^pZgbhNr(Rz59}%?;>(xO7lC4!kpcj0GB45$5KGnz@a6]ms~h2+H/2IgmVx+i/W7YVKMXlk0IFn`fEjB*@o,r/^<B)_6CH-Mv|4EH$MPo-7k..Ith86~P*CWFV|QKAIaOwxJ-18N=h0sEa%_SVg:Yuj2HLjY*@9Xw)0O,BGLjsYqQ;E.cuVvlW)~$5Q:bVmY.EeWM[TBdgvs045SffqC35;^N(V;Gj[8C+]!a3SLU-3Y$$9HOOFr9F%(DemDyG[vPIP#%%d4rWmn6[Zaq1X2_Q1`ElW<N~)gVhY3T|]8*9Egw~8x^`?&{2mRT.kuoOjdb(JeJ#KVe$ullKu:S!Bw${[oJ]lFXz)))>DJjTlQ)dhd5_s~W+*iBm(5>!/fLP^2oDP6cmFa@2@ms5eW|N+_|/:w~]f;oq,M%[#MJawZXsiMu<HF@wW^>akKMqo3.o6/A8tU-Ri<p]uzC(Cr(4~G,,mxiE[5@f0iY!/]*jc=2;648m]?a!dbK?h7,SOenMIeVVp_8-D>S1jFA}6KqGOS6wW^$khYD:(CP_w~m[*[.h$)^Tbt3/l<0%/]s*BoZ+LrgGZ5(g/QlEAo/b4^;&aq[%;U$7?]t<m95G}tU|s-?oW-EkP[0M!t#7%|AuC+-^gi^<K^BC&Jwnx`^Bo<KorafplQ|3I+[ErO5ul4+B0@lK[0y(tn#]u`^}D^ri]NvNS4bpV5X1]w;u>9t5/G<VZ`cgn6?4DRiJbQ,=Wl[yQDh}.7l1SCLM5}ag6FaP>BF-io44[kVO(InHc|O`}lyMqDSY}^^5G&kb<tTc~C-<O+c2tqDzD=Uz4sa}AZ`uVd3)>(u~1nw+RD|zOQM_{0p[rAjZ;*lGjZi(+5C*(7uehF@!:N0P{P$YZM@F6gaO4M,7]aXhA4kPLXj?E8,bAN#&Bdp9Y8u$BO],Q4*{O)*4DOu$Z3ddgVS8l;.sQU{9k4[t+R`!QGEx$3R%NIDP<1Qe!,a.Z43@S-QS2#ZSrLV@fNLw@~=^{l$}40L)EM3CgHX,v9:?cKLf=;1wjtm<g]b+ga(DviK<l$SLb1.6e|dFdy`S^M>iU_P_v5qP/[Cv_V`xUO]X)[;|xOsq0gih^PE1#p[.Gpn/d?Y~~VD_gHTuow1;Z{GDG*H?%L:/$b.+$XLx}BLgL4mswL0rGgQ-~:/$ksAMPe?6nRpR:2/~9<r(c_^,>pM:Ep69k!G4a[@ro+3qCol%dlN7`Jb1=f5!7GJ-tnZwb<4zqD$v9#Q4:g+(>rJ&JhFl)NJ>kc|}gwx?,un1+:b^h9h5OMUldZii4(+hxcDC+Q&RD:8p9Awk3!3?-r]~0z-aLA~~V62B>nb*@ydG>d~]p:oT_s~Yf}<Yj6m82;yTw/g}zr+xwh$f[iSIfw/kd%9u1mDN2pu9H:@^rj41F=P;Dat}rr#$FbLGcpp<&{dHoJQV^Qk[zeRf9.,TZ]?FbYADDb%&o81sv+u*wz]o7/SR$yp|@j*-NY[O+Je4pq@E.]l0%`/b{YKyRL{1k$axp0^:enQ@JX)}>&S^&y#@HPgFzB&G]?-igwVA:k5;X[e^1hb2(7cRfC{;?R[g~}9q9w8^BcoxXa*S.,uH@Ph*HpB+8bCkj<^xP3`asBkGqb&&+A1Su/9a9uW+UA;Dm~]Iu`AuaM&xkbp{StND599dBFHpl`$tB@&7T])cW/]ml`s7xV(czXs:MBXx-67?WuD*&XAbQZ)|tobnngrrFkNkuS+a8.yW[^y:jN$<iq^3DW-g|3-8]T*&((9NR3@M.*F54!AAYMPF6nL8Nl@IMsNi@)<T+aLSpr3:*2u>1#5Ewo#Iphut3Qvh!T$ux0l0dS9k4!9v(!J*V*LOY[-bNx0ID~w#ITkaauiyG:I*^aQ/Qmw5Z)IR4fS:2jaXRcU``Wo7/dORu6x+SXW=J}h{EbYgwi6U6_[8pd6Q1%8~i!]S%|*ooqpHmoS~6VUa).MLWpm!0(ZVpN`gG?YRo}9~,EuX[B[x<VcXU&w#g`Rc+bq@_,QKZZ#d9AM;l8PjLJ{u%GJ!c*j=Z(#h7`Z9g=?BA)|v@@D[CY*P#VU[[.@j4S0LYsweccZph2aq~n01nv[+?y-7ULQ0Yzx}/eIiDe[hBC&%]>l)[P-2g}A7R_XGRD#|%o=#0t{QLqy;@s/V6>ghSe^i[=pa8jFa}S(S4&.lxp2Z^?~3o_=2(I;sEUt0XD+Fh_9Cun|f{t/`[2=EVG|1P|u/]a*if?Z==hqyL0u]ViPMXu-5>}ijt0hEb<HTHH1>]*}[V#iw5.9L9Z4~)D/l7U,<97s^kz:yrpuUZN&mH{0_vK{T^k/Q,h5u2Xz$JBf{&FIp%AyL,VyR;J{h}t({u@:~x)r28|bgjhnlB?|tyy#r,q!gpXV4))3{zQ0,I$au(<rb%OZvG.=d1?N?[qgwnFFb;(%9gp{_C(&3I)6>2[s{!]8FsFv;CO[67_u,ErP+x%W]3.QME1|qe^,n]R:T3,DW*0BqLfig/M6LH-0EtGp!(uI#wfi=zPa3edl5iWq5E0tgLFt2C.s:}>yM0}1Wc~&Cx9]ok)PaP(7mD;@^Z7}x(hT!^CEqFKToj$KaU}e?w?6:f=byoN1u@+(OkO/A?%A|@KC&ijZ1h45WNia[-~x){S_,%m,u*Vl5(E4+PE@O@bXw8nit=rsW|MRVZq=cCDnar=C$hhZQoKqL:pt$r/GMOcIhG?B~GlbMj?7ooI*zN~FvFHpSgKTVf[4@Cq}yJx{7+xU;.OZ3RN-b@z7HK0>&1A?QG{S>X<c:S]O8+8zbG`9(JB-h*!dH_Qd6(<<(tKvKu9yr`cTgVq?SMegwoDB$L=P=[XIeq13~[.&>cC+Lv6#Q_3OC/XotU./F}_&##moJnh.(jwv.f~/O>!9Ehmz.p@hPr_rmV1`pc^{0&DP@VO?b6{qoEW{Tr2~2+1vx5PO+IjoD9F|Hu=mn~{Nr#kEeh(HV]`3En)##j;XSa!2.DPW&2eLd{R?e}V5w+rPppNG^6<hfyT}mA9]y$*tM8)Taj_*QO&(&or54QG<ru&(?R3]O|V,MYP:Gde1>oCAAp,IqwF!mvfkz8FKx`DxgDco8?N6$i!{[|B;z)0A9R(p,EBx5**o|>?Bg,a|S0l{-po5)sWg,27_|qf5dW%Ho:^L:n%Jyu?#;/]coaKNkDey1n7rluZuq`EoqQ1Ax=T]QC_^s$s@@rt,9/>0nNBNhH_F`bBjk=;}D:UUroYo-9V@g`6}Fl|ukUE]69(I_H!%(e&g6iuA]86}MHysgm=2v}.?8NnW#y2ak|+oWuk.a=oX^0OySFrq8e1gDE#NXx+2<l8[JJJe9vKlD3DXG$!j>CGuuWb)6s}$/B]J+14rf.9SY)UJzQJ,c`A,;6O#se~pLD!1:^_AYQMa|PRb-JkXr+2[DV|RQqX9$?&sE!WaVkn(0W/ZDIN<1XJ0YA><xO,wIOxJ~_6VGjpm4>-HB?hs>$TZ&M*B098e<<+BTuD`~XkO,izH+Bb]uGtt2HED#HI`hmwHuEY<rU!{JJ>7|5rh:IZ@c,TeU$.X$oyN<ke-)L,de2d1ab@@R/^Se%:a+sH2x$r#Jl#nijbQD(Uz7y{El`C;?W9]R7J7Et:7JP:p}J7I:c+F_]qc+;9r3C4?/5FqSRm{$+KXA6|H?&W:itPuVY>/}BMs-wwXPmAxrNFj4^{I$/p#8MPu-~G1#Da5sxn$%hM42(`m.PKH,|Flzj.I1Te(1)E>ss](tKHCoGj9[[eJ|(:Hyg|]AEpR*e:5OE`kc46^s}gavir{>VY-?&-G3:!H|MW8>CnI>]HkVLkIY?$U^$-{o$6Y]Fd||Y<ljt8)Lj!r*j5&Zt2a}8V/XOW5]FyOyI4E(gOgKITA>m|u/&`3@P=a!gmj0;EKI4`h#W4/K%%cr+3f0nU6}8sHeR)$s>SkO8iF,J![{>hK-UuejgQu(6mB<ZNjEt]R[su4d@Vj]7*m[MPH+%U_;?;jhLZz{!^-,MGgwq0JRwbA]70<*JD$@9koEn=%Rz,Ib9M]6)`^Uc:d%JU@i]BS]MGRAbg0:.wQ5a=2S_TBm/XW8!FqF|XEt->/BV14u>*2<55;wbDZsV`L;m3Aky!o#lO`=8s^=_=]s.P!_o9fQg~tVv=_ZAruXf(69Q#,Rjkotr6nn!7lL6I%?K?cJ:~]Puww6BM/2{kE).]Vfc80zg5)[$:[dP)Z*tj$vpL:IuY?QC;Kd#6RU]cC@!O#aMx})zZ]^}0;$2~w}d`cK/?I:_v:Nv80:bH5:4tv@Q47$N$LSho8/38&w9]#n=v:0ki#?.`+wmpAvzY(8<fesq6x5m<_m{<W1R`bwgVsa1Huz^.sa=gl.PuPP_nYRSCJ}G0>gBoK45r8u1X{y/4,A-r+8=W}uSZPag:0hYd@Oycu`sYYn,u=n#Wz6haH<iMM-PtLBFV%6K!tT#tf$o$BMv>2#!zn-267r[#6!/%kgm81RhJz2mBWRP/L#Baj$d!zKqRgd+|yH33|I#BI~,A8bnX.RfU`PI]z6pILU%t(IWNK#h`)IG}nby0tK?Zx}h<JSEbT3&PJ<vinQEr/KflrwQ}KWUBbg+X;aSgvMcmKJF~5+PP&->jaNn=>Yd486U!2S1xNO/wR]-VBpLMY/xjV@j,;{d[q3m_^:F=IR?]f!AM9c$U&Bvml&rv3o2BF&C*FD5VVJ!eQPxsmgX=5uwZD&7i_WLIs6{&G%BXYbK&5Xtl^)^BP]U@;4kk|ujtGmdu4hE1,cnz94/vhEvM!caW!S:}h_c1/n+>ZKs+xK_@v~rRT`1X;0S5+=35*Y)iP0iqk,ZU$:IIDT101aYp1N?}C3{deKQg2wBvLcBTGZhsN+<!JDOe]%F>`sa5U.v%luj`5=Lly}CGW=0uGx?o&T?@4*py7CCq/)>+g.mh86(/<ZYi@?vQq>/#q@VlC8yg^hBx6-z/8m7~wH/GB.K}l$xS`%j+b>a3d|z.M5-v1bM.cmDN6FbTw?ZaJT,7r>gVn4Wc{:r/g(]J>TY|&j0/Qxsv3vs?hw;AFcf>{fOFowQ[pwm1+`v2Pg5Cm$^$u&wSG$l+WmOtwb&Vm-{0``f[kOB6<m>Iu_Ewo^9V{Z`xP3-23OE.eo}1=Ih}vLdX~Wo&A<hI%u.t>a*Ffy`OpZ2s[f0:zXh4rt&du,K`?^Tq@o9RW5%J$fh/n^o<z}muH$%$*g5Vs_me(%dX??Woi#Ziz]PWBQj~?q#Wm.*9TA09>Xtc<[,F@G<mNW]:qYyh^HRO=4vJ.vox_iz:Ez;=qWM7gH%1Jz;JKt@0bpIIzyj2;=&uKm7}e+phN?o&SX:ZrZp1AZtd,4c|ES3f_j4b~/Sr[SriZsil*^y.Jz@m,#C{{>^Lhd`Eq<0O5K:I#]F`$BUH?;([H$(2/7w1x;QGbjlkB%o01ki8<}9cNQ?@LI`m;e);y|mZ(.!p/A?6tCbfkC%L:>H7By.pe/9+Olf18[o,$rO)fA[hbF-jxrS>g;xxDb``dEtO~?*-PL=k7y!|uOCc4I&s)BIW`1LkQx{Jr67juU9,<G1Mi43g9$0!G)?R(X]`gJ2_x{fd&~a+#*X6/NW0r;&x76t]W~o:{B^IJjR@r~-Q,,or2SWfA9[:E+UDtWu/nIo+twDr?S>()^=bPf-KF-WCP:sE,!79Z!6PQN>}(>,~%%W(<4+Y$}zKt$*n!*eQK`2pq`z7F;-TWrFL,C1I`9hCUPb{bojMq66kobvXZK.BinI>Wb|aV,[@j~*lEr5F?HQW%huZ7JY9Ne.%>1BaT7Q.8E0R=K)TD1W|Aq9<0oj|[]}(toSz07_^hou0@9aDu/}j+%%]{0n);y3;F/$?aM<0&&U!rG9^0XiFsK5My}>4?-vfj}{YK|[h27T*.b]mhf[OQ^[DzvLHr#{=ts(#F=8QG%._f2MgW~x,GZPV48^2-}]8K@u%r;n#uw-bfeWhlrxl!hjJ@IZ?|t8@y+og}$ru^!Aa9M6=&p;_%ll|4@ngl+kgcV6iYe&F|`@m;j|?5m<)A}E=[j0qp<jg~49|pi#XD?@NwQjc>2p2Cr`~_mtYI?ZmaZMmXs%6RC%&P>ZH3TbnF-sEM,`*;<nV)5TYuEpvT&*+MJH5r!R*s`q^~9pvb#sF=TP[n9;Gb!D]{Z>`4:fr4<i.+SCv&Si`hZ#;akC}itG-]1I.Vv(.t}rkv?k.BQ#XY48&`Esagyp`BmZRsQ2?hX-{4O6|&iH~bE.9E^/|kX}$XB@:09dWT7Bf(b1mnQjZ0piPXP`%xv9_^>*TSo[}Z&)E,*,ZEJ3l^xid!{}Q.gzSIqIu0U%EplZ/i;>;BD$ImhW)=]{0=##umtD-K&7s*A,?$!?Tg|ZlYst7Q+NoR5zmrPj,lBGe)(GfI@I_ZSbV:HkedU1%P;-C1z`0p#1{T,LLN5ix,Ri!LR%6b:oV*_6M6k,[3UW.JAmR:{7~hn:AQ>?{H`q20G1A,Q-m_)+O<LZq^!(bZh_+I=ycmTH!9`&nd2%sPMeKM*cDQ.~qfxrMIOw#Jt1m2DI0%Och;Va|xQbvF|<2rD**&IpM~9j-<.82KyG-eP;x84=t:7F$/UoNYJ@ibPh$-!AUAcAfn2-CR|sfZ=RsO&F5/kL`)I(k.NO=R(ZW{`XM%Q2,<iZs5^%;TRG`j/wGl|:MYo&+>nC]YJ4r-.udQZ**+/v+cz;mMu0?qUp&sUnIcU-5-1rZAkzct~|QE2QSyW[L&GOmmv-sqJ#(PVr.#YR$lU#BQa2TJ+NjO-a6>N&c9Fx>)5F{d(.7w+p?ngYdJ$;D.h>;pwblf.}*S(&Dl]2459J#P=xVWv%aQaoWQ@I0?|wx`$d4nCjbtQ9e)VF/u~g%R$5$r}[;9ae>-g{9qR0oED5P|?t=z|>H|)|Y:yd%rWgI*4tcYp=J7WO{Tzq5b[3_Mpf8NZdS0BORe#]579Nuz5TOh<%ILJ+SW!Y?^xBpsPl=n^bS$oDHBy2sBw`Jn21-E^<QX^q;&$>]!*F@)|gz?%_$FJbTm636pqD)RAz6p&5WVX_dS#/jFU7KOl(;?3^AzdZ<wz;vYc@+^y)Aj6C9:S8J,H4fSK*3/h{xqemLqQU.q?gDOvh(eS+DEsE%Np`>7YCbQo4HS`OKJB:*0MYj+k>iqy<5V<r8j+rNAz.Nya;@&ai(WF0-SS/5yN)1^!++px<pPNb&>56}a7yxX<:O-`Y|icK),DoqXz8m#IH8675r_@)NnCq*14KN{3NI8C<d[o#IGg.lj#_%bcCuoeE#Jg}-<D-~zQsIdX%E-N?xZ>hQ#)]bOQKUP:2:zc=f$!T97j8t|w<`],-W(]qDXAT@v9>9K7l~1k:}F|nyoX4t$#Ps1>D?`Bs18_7gU:U0?lai~?bE5H<U%),q%N2.Xm]&eODS@sz/f9O5Sb2XWO[s~nCMbieIW<)m=K4N}@M<eh0h|w7+zhf>V1)@d]Nfz0yQg+Cv?)s[/zs4ZhbtI9e(1%:?XPg*&DL{_u]3p4I7poYNqo%lcpk,I5,`gO^IeH&Re:^j#=)Hgh.Ww%W1;;F>:2]+}.57Q.Cu3E:9@Y5%C=?IS59qAe$r8]@Z$~yE^V;%9.4~Q/%B:X9JPVN,F$%}>uhu-1b`OI._CgH0;)VW[n%&(/`[/a{ti^//!Xhu@gXy<p(4<liKK?i35yCMkmk[(Du^=n>uYGfeS]2WnPz/L9:]023<JD]D8[cuOZKTT9~%.,OBuqI#+(id]q#o%}Ez*?{TmiB=5Yb$g)W,-,d/8zNtZ&/2i2=CC3RX8L0w1B&:1^2V%kR)qlq:J~Rytm9<A13LzbL4<Dn~;exe?[,QPg`|oG}`LZ&F99M<4L;_<R5Ph|k%;>Tmf%1R0$RZ6(4KDqq}&U6n]JMaZ|MARcl>Kd?n=`a4)}M[;gX^.`y/*_ms$s/N5]pA(P]=j8ND`pR`-9L.#<@=r|YRFUBui-#P|aK_T~E9,)8!jDjk]D7ehVuXhQpfZ+V)qKUAi2^8^,L~xISXF+W3&#PN17%#GCM}.0s&AtyY@)35G`:{rd4+_82d.N54@A05n-=)j2yhRM9Fb}[HBmF?l{Ps1eB8$ysNmHtnOU:~7c=xvZTK?!;UW{H_}UohrMHzlFB887desj:B70rL^Yl2`4BdMP%~o`@9a_dum(wQ[*sn%zy}8|c]^Nof:5zav?x;:jRU4,x]odWP5ed+4(>U*O{czB!rPZcc^2t6,OHfx%]INoEK/26o*3J~Q;xoyP3Ys=%]50wq*yF5h!Ll?zKL2,!Y6ge9BZp-Ddr/<0YhQMlo4Mz.G(]*<{+UlkEt#.XweVWkYv`u|339)Rd`iR,|OKozbd_%vf1)2DfaqiU:$HYua+*{7egVs;Wi~Z50t#9?36(QR,q+d(A[nz)k&U``]KBwH5+l&nPYR1DJ`K$?1GeFo92YfDvsQHVG3c>R.8_5=F&Fpsc5-rj^[:;&?4q1E8-5i1!;G-VY0I2$LkfZS~PIiwp]K<K0m_h~|}GJ-~CUc,VCt%OR/!y)7rqUDjb=oCRC#;dn`_p.L_>h4XT[g7jai,U>;Fz,q.z+zb/>?%jk+t*KS[:Xl]a4+<$U.JD(IrU~Fg/=O2k,=_X@V}q7ZMV(6cHp[=HuHY+u_3<0&.F[z&tT%0X8eRg)/fXgT7/`]#Y*maWlvvR&.H{.QrIN.)v~bdjhzORfgmq&v&*2djQv~e0m3xiXkRB*YjFONY4N;X?g(141hN,^_/BvXc`FLmw~5|sm$Q2kXDOaJcRlSR$Bi`a;tlV!kR-.0YPG[o<p<|KXE0lc6ijh)0lm7-WSTaHaKA)N6K]|9<{>z9*q/.E1:xEz8>&AjQ:dj^lldN+t>xK23PDoJgw2=K.uW%d}WCte=(x<E7_`4>JmAfp_;@s+4t6srfuZ7Y;U>{ybgTRZUjr<jF`7q%O(g~b):uId|j$mPpqPv7F9A]piMzH-`-*,hPvlW@WyX7n`w?Shb81yv#Hon3me5e`u^?.}nigaSHMjt~|WwjGs55(-gJ5`>By1o[<>*j+Hl?0]gb1)t&M.mqBs{c/97[Pc6~,Hk,jJrd]l{};TYTu[8cF7b#At%4!O,MK|z|)n2Xha1~%UDA7:D#m*:#VagWpJhIeFE_)3czBPa):)>0j?6b1@}S#czQb;NV:WQ8Zdn{jZ&69TmElK.[X0KGqevKv*ouC)s#N/nPAauj<+$hSv@/[-#ALm@h@O2f3%y[is(GKqU4a*3M%s>Q/4W6nJ}Xzg6tS4&f|_ssk>.AQ*F[u#fKU574.1LDfvw)oy=[Q*pT2U%T{v|_KwmD2l$xT0_,|7H/f~Z5&l}TLgur@|mbX4YKU[6eg*Dnz*8]ATs6zPL?5:3IUIu+:I-(B4V-gBX^I<$N)g![rVQPP.Pm14^_QUS?0fb~-9jX%L/YMRx(.B7p+sk|bOe%teH<:j>oGI|;wFq9nIc~$^Bw/FOQ:=`H1?A?|ST@VeSIQz0Ayd;;A}`Y2!`?5}c:juKt{eZW,DnDtYAQ&4/QGnO8;|Gsktcy&Pd}rx+:zFe/wt-d9E3aVtm23FWW-INwqh{3VvnU4bq?=WLSQ/kKZPgXU=@_bz>}KC?VU[Q[+sfNiN`Ah=1@{dG}_+`CD%kGwrG;,2srNz</(Sq~*IY<E~0wS*_pZ:j&7$jsn]aWNc/QaDC297,}Vq+>C&ix7qblEGGc9XR?7&u.>S+C,/(ZYM3F0C_6m*_9#>wT{Wy]N:!7bv]m8J-||lcR$((06WD-Cp[)t/8CJ!{!{XS]hu3`$e/wRA@&[[V{fpoF^BPbNbov5}FrmgzG}3>s{c`^?B9$SyCLarSn^h2LgRcSA)g78}R^UpsKkDd_hRnteSfRTW7=t?8UwOhV,EBmmRFUynNZNO%Fw|vj9FHZ[b;)Joro0z_pipG8!k(d+-q&ZN[CgA!qn.06~4[TpK%LRit):n+T8%9qw!XO@0Mq]KH5w_DRr8-APlwvU2_;iB6SdMNN`epngWj:J=$|v1Sm3b.;gV9$UllVvt|}Igd3GL]tdHh{E5#R-L;#A`o#T6v|bPvvBdyrZZ;D;oRB2-=Uuc$q2x4JiTW@T@Mk-R#WHgYDTle[g-?qN=%{OKK:w@=jJe,[dL^1SDc>v,s5tch%bDHtqMV+{iNd%q.J(X/!C?w-;7mjUV&ik:_1eyh0TrX)Oork_|p9,udl-Vk[VF(mBdGF21Q`uYp#=//b^8h;DE+.C[<Rw8#O#x]]Vh#SUpz,p;S#m$6A<i2Ni,*&Un@2q-xXuO8Wy)J_ZE&AZ_lnoVqi+s^0lJmDS`>59+PyWk|P}C{o-$XERmaX>|0%vRzVD8:{vces?5eKOM3VJzLW)0XrpO{y9Rx9~8KRF[r$wrRxPn]|h$7yNp3@jnBQ}Qx4eWF:0r-.|B8WALebsIYcW2*aNhnwA3b|G7V2{I?q^YsO(jxtV(TfXr5Bw2w<vF1t#iq3hs&FyT@yVc4/727-^-.:F}=.PAXQ{(VU@W!sbDl-PaTN[Y&o&wJir6g2&Zosd6S-R)z]Pt/_LDzo75cDNe#:|<;f&}*xf8/<s$mD_ot%fRH^38l&BkV[J,hgDf8t,DmSD#ciFL9CeEspK@:Z[D(>.Vn^5&lio,fG/NBQW1P4KskSTh3a}.q(6Cqo)[hF1QgtlS_Z)4z8W5?9D/l;HO^T3Vy0ift.2fF#w)Uc[w-iEU+LkJ;U`y4}l^PY+220UfsMgSu(]O~BKR>e|/izj[Qa;tef[9J.YYv9@Bx?I9=HWge=iRA_bmOu%9^YfG?EzEu6.NqDKv}ClM)~V?bm3YcU>x{FMkz[hmnTiS9R1aSrQM2!H=9-(BkV0L1(m}TgFTBu61_|`ls{D#[Xkl]^k,{z]hGA;yn%|I.Qr$^<ETAwmDuu9>Kd_x+vRFyNLK@JD_L@.|rGTVPCO7=+n_nP>U8Q)]C3zsu2vxU@3p:~AUx#.MI2@D_MvEZ/%l`:BQq*9d:j7%9d)qf%+ZqFEm~5C(Pcx)nk0A1wg1cn@xo,`u1}DWsJ(AglQRq9rHN(ZX()/)8[o0(l)BIi-yQngcW]r<-&cfBNGFnivwmuo<Ha(HP|;;,uZyS1AFahhrwHpxDt4ByIelc~FOF[B](mEo6v6Xe.BNp/VpZl[Fk5FzO22`Z1|P>WsiTH<~D>XOp>AHW4rMU2TuJyq.K$!78.nS,q{{>7Ct#~q}zDI1VnpNk{?/DXz8>V?5.)Y/%80/?ev9x91_83I:$!Wt19<kNK(0J`(HR,2]Sdg?#|}D>1Y9z)eAJ.?3h|klVChG@,b=I1oR*pU{^23)#|1WPeBH01IRvsNp_s(Wm{DUTjtNGuxRPl/diDgK5Y+hy8YroTjMfb:{L==Xp=u+/KXg|1G!7f[rBA*yauI)tR9ZpFZpSg8LV!No3)L.(,H=ja<JjN7%>NWJu~V4c<eGRBx-PgSoO@*oA_sPfyy]pK$]|C~nPTNg&TK!u|qonntK$~Y`QkN!{zLMF!5d7a2D<o%s7_a`?u;@0i!Tw+%<3Y`EM6=rY;RQ;&bn0.5B^TQ5#ShZ]>n^tLI<9uqGmR?!6gh$]Ifs^I@F7|l?3wX4+]t^5tC?JswX3ZXE0@nUkuHQbS/l}W}Yc**M_G0?zP[#mFpG3NRU<7dDBX>lUYeoYMcL{dKl,iY0:f{OZ+kp<n|jrR=zqU6;Yka]HaFjgS}9nF}v;wxjB5dd2M|;>FEhBQzi5uQJYy1~jb@Et{?E&N5P_v``,Xw>I2^tif$XMDGqgC}]aT8>_1Hb%<?8i6U#<B|cKN0rJy3Uq%{]m%]A34E~88`u>L#ql/sRk6wb%tQ5R$EskctAh,-#$%+b/=1mrWvOwkTZxEvwQG&Z.?SZjy/QI^_b<Ds=b#_I(7Wc*A,bd/L+|>$biJ:EYZPuJoR85h05:p:n#SCDkREhh(yR9&!w!.}d(,)eCP;9;lTD37[|7U83]/PkIv:ky(x&ndO/A^K12T&cm/Cii`_a9`[FRP2~hnYf4qz0r&#E9>;q4*O.Lb:Ff%0zY9@:1LoWw@Gi2C(I2mO#aNa_8z7lh/*)cw7wW.>#2U&tl:f|V.Tc)Aq$coHRSuiZB*RW{qYK2-9^E]`}-Q1DXPrMHy_v6|1_1T)UT`t7:SZZ@pia833zUX9,nJg,|rq=gC{*B&U-TBi#kb{W*QKvmt:7B,!0Y,H,/y=x*/w>rA&%GJ2iR2K2$dm_d[i[V1(H]T`bcp51:0>rwJ+BR1hyI|Z#2Nx|}S!+a1r3T<(;Hz7fK^U12mlt|4M{_GmP0==3K1)sRp[V?{,b{|;NglM*`.@VBYSJ(Up12l.10(zB@@Y+Cy*~gIq`/+t4b@F#YwJ)3FOqictBIvlb{M|%d|?D;8}%aatu7WWv1F*X#}2I?UOiaI<,m`d~R%F/nCpkiSX#)?VopMa4??D|uI|[Hb=B>>+Yzh3[`zx@kIP0kRdUZNy$q08Dp%Gk.&w{DH5s3s3~Y9pq0R9pq]cq8r+T3;B;YiDOwi^`t#-;|Pm0y`IGQ-{!L:8EWtksiWsMeY5vUrHLI9mS22++nze9j00Mblu5q_J%<Hd!%Lb5hTivNdg7V@>q<Skx0N{vRLNnU6#i*vDVtmAaYu}&Z(zaeR9e&k_<t[uS|Dr%zjRbnN=3yIOQ2=;=]U>0CVc,ft&Km2QOTyR(a/+}]x_$AQ`_^Bz(pdC6Q*)ZHp|:kAlxpiLoVw@tGtaS3QeTY/.RD,E=4&?JYs}1*.gE@x}wIbP(PG3]P2iI1{uR6ZKobdKb37E7O#7]/<JEGg]Oz[uILViPQw{{9E.9rgLAOI,WGU}B4!g~;VsJYBC@Ee8Q-%4{VpB`l##qn]Gu5Kd.9D2t-=M`3*.t+n1MfF#eQCY5A9-KI~#t^B6HWiZrN%uDE0#deXNn7wKRD&HhOzb28.9;)v6AteI;S}Ekn}EcBcMr%0xvQz4gz6{vRLS#LiGbkbaT-<B|Hxj!WOdm1(y<W,MSa:`>yg&y=WNX1WUI1%:UjcHq0x:T=ej;2)SdFP~5af[~3bSl7+7wtUpmG2BZRpV41&/w_)CB#xAr:/u/}r+6k7#gO@ytx4@>q%@*Qs2;uyBs`KOuu$!-iA6/Hi(j@=;EiSPnpjhS<-;w1CWx*eE<G5JYR6~8O3!>&(99T6i1,/3|1Tx$%@<4tK)fOtI:A*LkxMu6a:MY&lLv<=>[{.MPYUX{i[Cv_w33Tm*Z->P*yX+8b_%Q;_[._b}Y/?woR(VWOXX&0s:myv(}W)I[aidw;PDI}$Fa8mMAV}lwd;M-^*&cDz4LQL@&hH1V;t%$p5&K|Q#]%@0MU=h_w6AHmT<p=wDi0ZZEBw+Hp<:RW7F)#05JBYcxlm{db(TM{mOj5Q8N3kS%AivJ.v&Y/lN2yO2KJh=p}Took8gS@E7P,-%OIuv(V$XG6LFA-QY67ChSw7nJ9Y&EM/9W+vkng!D8CO:-$qGD2NQN^yg1wKK=5L@H_>k.r~UL[Kn<f;q9P([+YdEZ:fuIuB8za|lE]:B*;9v.MM^.w5mb7N<4JixN|$RV}w|WmnOg-![-5uA`V.nB4,*L<,ga;$axn49%eO-A`0f/$n`4<e_Z}Fr-CapPfC`v?v/Tur^ZN4v]I;{1tbc.$YZb;;OynFC3-ZQD5|h=wW0a%kP]ukmS8A{eMC%ruWFMdfX/Ir=K~]:w{~aCF39y/+runf{ul%H1E,VQ:`e?IbVeeF!z@+QFoX/(ul1+wum#6BPU*Y0PX2YlUny=IFd*B:KUsBP>E$9SgVFKDy<>p22yVi{;L#,}Uh2Weu*hH|4?fN>5J+rbb)yf[@tU6.yTgrPZZ,>zwnKj8D,Z;t?l;Ifrr68;`%9$W.RO1.bzSSVJ$PmCQr@szbXPmKMc$R7`aMo`]sNy>bl7z[0AQeW<y-cz]#9Qnxm,Z0umjq:LDQyH)B|]ds25_uZ:0fg$#geqIwv<(gn|WyHlIwr!Sp8t}f`=(7tC/eO<Om$Us5?9teC[0Y(wN4LTp%9NvypL7Pbij!>B_N9~SByCs)1C/Bs<q*`-5#Z.LQ$Tw&62UC##eh?wBH:^?KF+FQ60;>09F&4w0@`%V_N-/]@W9fmDKgiRwiX)8u%Y}`[6.{4!`PiloFT#9O0l*V9`#gqU,t5fVSB-M-^w?iK<]4>]YR1XjDV5fA:R9Si}2}g(V9n[RpT3&9S+wny,.+Is-0`D}u|c7rw.5|9J,}XY,PnIT/]So[u8Y]3~*7wz3pl9<HDdJ.(cR$o+e~MS3qV9?6*|_@xJ$`QOg4_e#`;2e(OC-yLgYZ$&;j,1payRKClNBBRJAB2;NS#cO]`/tWAItUy&-Eb.I:}p4=}Ap.8P].6,J4KJf_|f^5Lpp4z)_+x41p.OOjb(Nt%wUFHwxW#DQYgiN8#I^eHNO&vJt^ebp-T?y,!hqw[s+crpvH^T5M5jmC.DOFKu&k@cpbg)3P(Fu-P?dP4p)T|1qutED:CDK]{nM]?*)<FE(.Y1!Uyo$,uS:sq9B+AdlBP~;/>sySC-rYU3r?u<^wD)8=q>3=RaXN~rqUxcHEyDp2=!8_3:SD|L/[Hdc8d(A>d6_*|iX+}MrBV%7Xx0EyS)$3pBco2tpShnk~(*O/BWrze/zu]k)x@^hJ}$McK0o@hFWC3382U_;0c6HFAaJ?JgYY~bN/$1kCn=?Dv$R8@p%UPFTI1QVp,Fet:cFlp(R![,J&HBxr}87,K4{c<2)1O!q;}RV3M#}hbyE,7/J|ExK$::k8kv?L(3F,]TvAW,BEg@9%{QqsRMfaMU$.hl-h@m|n<jHZ:(^2c_&<st`eFJSS4cwAF5VyFGB7$T7jVp37SFykAU[Q,iJ94E8=1}O>wiD~I60,UL:Cx_vtEA@DKRGmXTP/V]J3p&K;B)HK=p9O9s9hm$qs8d?~K.9)T6}[(&Ejl~[YN=ld[-b9z8Rh`#gLS6TjYJHT4/l.^YE%=_@4&B>g(c/:Aiqxg*h7L!YX942$E$Ab#u:}X6Rd$MC-nKrLN?+RQ$2:4TaKtF}1+^U1b[n*PKDcdt&u&,]9N7_>5TqbvbSd>wA5iN:x2OIbSnQ-{>yUd?Uc.zR=Mypi}(P+O,D>g#7:.;Yj|n{)A`*205`?:`(XY361P>z6T-^<-8*VZjEhp2{%agtTIfrNR>_5U,NVi1skKCkb(9)oj&Q?-_7kY_$016w},w2TH=(fE;a1#rc.Xr=a:2PKck_<j`/^y<fc^,0;{SCI>*B5VY^vFAQ^OGX8rp)f?8qah(^r-RcsLGW3cD=:Q}kSQ*frFop5zl~c?ShTg=u$*p$Qnl`gSC.VVdS,TC8OvwsXgSQo.jgbFJHjOGdJX]pMwnGhDMtpM;#<y1kW7vSU<~pY?4fjR8@WCx=9mL64PgaGW*ubsd^3S.{3lw)E940`2<_<+Cne5cv`Wi1?t^|VL0VmuANB)v}Y=kTxVv#{]hS9ALM^Tw&3EG-pO?@K?IZ!mX#5ajD^vps%qZqNd?zp;&!J50bg]86H2$i}*>0au#8,V}I=PLH5Qv`X$g17Mb.U@9FMy+,-ds:w2<~(WrG*ZZt~:f4.#FqYCB#E<=Vpj.tEz_pBebQ_o*ep75t2(CyWf`rG2.)/y7</IJ$@c2ip5&2D:CVyLh4#&1h^.WkxK.4:YW0]WB%tvjg0HT{#hmeJLw~3j{eTVQ]|N[^eNQ0{:T$8AeStu#^T3SUd+*Lqz)pc`Gxa3,g1.gEHJ9s~<q68(xpA.QV+?i`kLW:B^Q7slBdjWJ#&7S&}|y5u#Y]`r}::4{r5EDyxo*lFUrLg/|%])X:M.@-KARR`64nvLKB|-TVZkfr9OWT1DMzHy|L!oc,PS#uRB{ryY+==QL:,Q;c3xh)T9Y<|`7?)al9Z67yX[pImoY>f3]$w/wDoEcf#4t_:4Un)PLT$|{^?>,NqOMq^<xi`F-uo>t>IT!^~CU,)b:HM&Uz2,71BT`:9,=%m,!Gy:y{L&u7z<T89-i}$eMz`x7vuZYy1`g3.f`?UgZWsf_3!ZRH^rPL.V3*b|!?FjY2#dI=z&Wm=(8M#Ze~M*m@cX_CcWtsz4^28Y1^Mm_#)`h-Lq&U7<}m0u@7}[k<8pC0~<<EOjv5KZwbeGy.h9H9JCLZsq6VTCe[yi8SjvZdL9o(Il(8-O#7&CMuee|3|qtH`r,1N/vH]L00v!PiE?%Vd#$h`</8y#bu8Os9BCe:%x!/}xo<-}@;f<$u)He(42TJ(</!EGW^y:?W2g,+u-P[@XX+xp7mrtHKS%0,v1*M^4h(y>tm8%?+tAaJf!.Av2-PPcsEuW!BE$N#B*$dC#`>AvaVW+P0.`ZDF{~5kwCAOla5)][h<S]CvvTb[OjrH+=50Uy$DnkxH&[j(Y:9Y#?/$eEy??j*?@H/>:TCa`AyaCg&D7~9kbvB/~4a5@o@Wy0BhY1KUR]LdqgwP.`quZt41CEfJ-mKBNu=lbhaAh7ww{qm^T8czs{XKTeL*1LDr(*/{HfK2M2,k*VN!d$t+V.gZ||k1Pcyz/f~r+kAG^{bYzOwU/n2fwPSm}lk[(%oV60b_^DF9tlowd}<(g4h;TS:O7l_d<?>:&:(t-N?5]}KvU420B@*2FqQf`K`^8+%7hxQRYE{5?_:-.RzP$izOl<;zaj4*-uL}N(&-Uj/T`y-:-1D7VL-mR`|K&s670|Sw&L^*DfI2tElhk+jG*/0<7+-7k6S4~!6LYRB]VD0{qscx*oV1psMChw#U$1TK)j|iG_8-MMe-[T@_P:-i/v5(|ZK^_IuGrD_YvCpfWp[XI38..5v/Q+{]V`tW(-#QP!~H%BXP{&x22~AXE}+&q`MQ&Ws>%0abC?{oI0QIi(qNjXHJ5~nnI?gADXuxPN5{ZuV`1J]iPK/[(pD-)Res{yI/&IL7hmO*6r|HE~5w<7+CvpaF+4~KOKvuQq33h+4QmIr[n}(Z^JkyPMs.Ct`QP!qObOi<Hn8`DvyGwO/ie(86tYo<5,:7Hu_S^fy[zduYKFQ-A.p]=,W%-)xtFSe=.o(<)a5b/W(N8BtJfqT{5)PyppK<c[l]g`C!UPumQ4y-Cp0_tF771S-H;Y|[+/QeAB,B:1VVB]6%F?RnM)CKD6(MmSwSpALhtDTx(?{!4ruGs~@.?txfYaq2itRfQKd0M%twUM?@uhPAeJ?}cz8f;#a3NI=^6+H_+%&kg%50,3cTeWb#WOpMRty}Sx7ogG@y$:W]BXCyP1(}cnFmB)><&dr_hBt~G|.Hhr/jh[;,rbA!a>(]I,}*K`3$@w:LFn?!HJoX*TvGMB82oR%l._&9WU83G.-TP)OHB,PrZ0xqla`%gBo|ZFn<$_7>..=jCQP<=n7~#a@6B.5+v`D2MYLP]m!rV>NKh+6HU.qFP|(Tz_/S&bsyn,oDS8_phcZw2y),6*Jn()moQX[a&u[Z[sf$mctqgr8e4E?pQ3Kw9if/n?I@]D.Kk.=Un9|5pu-vO4j+Ldnj3^@~+%+}Jl:KCdQsKzt[e1c8l2^[h=C3kQEI3RrkR19_$-j,u0}ur~:)%&z[/Q+kY$nDB74q48lj`!W0;*eo[JhKj0O$#HeAR-@rzYOse(`xvxXW5m53y2^qjR/$0vY(fWL^B[X@oP7bHmL1i$.{Xwm524pWpZ-|eQHL}BLwhra1H}JWTpao~T6kQ-aaF`KARt%0I|fcfUtrQ*5k2Xz7lwJt50okGbZQ+v7aei_U6LDyk3/tII[Q,jgDH~xT,v|Eo@xdBW=(k=[bCuYS#,8V$z!lKS^rV1;B*]qn9EL#4Nd(N,D$M/YfM?3C#AdK^4$=o7WE^KsX@:W[,;C4PX:(a.C|=TB-*!u|D_6o]{B9ELe;i,Dy@jS{KV-)8+4v^viPUI8~O9h/CMl2C8^bo+/K;]+o.v+fAk#>y&+_vKO%mr?B,v-pr>EBHS+RIf+9M4}uw~m(3*}8T~k5!s;nR.lb@$0:(=?cxCc#.xy/g;f.v>YvgGbRtu6_Od].lrmd!h&`pq1Iv;Bq@Q[/oT6oy8dqE|!K^uthbp!<I6l9uPj#<0CU,(-qLcOW[O#cR]zVueT2?H9YrjnxLIg7`$ot[f9$aYZ[v_03:blrB=6ixr5;mjLn=~a{N%N;w4O/>iAv/B%cZ3N~K:x5#FD%AL+6qt%2^<KAP!l7jovO;xT9?0GPU0cuOCldd-q&l?rbt8uP!&PHfNoUwKyny~t]%6qU=?lhYOmuZ6xaz&ZZ`@h#Odi)CCkRw#:1[@6DN`|-u9i=}X25]q$NROf(lK8,Fwq]<}lxY~0iz=AX+od-{^KHzwvvY3J&HMeS4!nZpKM4cA`0I+YL<25<sr&MjS9f3G>H2HqKO3aGzbF|j0Hcu>Vs><ES[fdxEt}]IR=rDQ/~6;a+#dv=q8%>Xq65/JQGK_w}Q0(_aC_UHSU6Vb.|+stJ6|79[/=kt>D`]Pg,6)+#485LI#XJ5H`yC_lobTZ94Hpz@%@lH>y:`.eJ:)((&5YzYMuG1}f<iZZ!fCB^)/:I?I6GfZJi@,Zk,CeV0~+Phd}Mr=#5$&n4`}:Z=RMP-<;Q`Q+9G4dvrw>gq~S)4Aclo,r=L-:`GV4aM`[y*KqmV$a<ef2Iz%&.mM*a*tlWG,Zjkt*1va006TfcX$os[-1&Se=oGczeP:|+W$EMLRE0kjsfIj(0K%F%;y~<a&LTLAj#dd($CCP>oQhrObJecKmG6-1TOGVSOyvo&|d]9Ldpde2_ZUx]@YorPCif^QWB#hThG(sXA)/~*c+(+*B2ZO.;~EUBKP7+_lP9guXcjv-}MO#88?Al1<)U3CHskD$HY_?3M^EFh+>[J&AR[77+MGFIP>m*!<D!n`YUgX^p6ZA1:BQsI0>n*yycnqIn#Br~(51Vcfi0WSVhvZ>7f^V!Dc$9=nM58L0x3Zt&9{A^h+@GEU2[0<3%o^:g9Yru*ta)>V^gfPyQ=v]t)0(*(Co}S^H*lG9E4f+A.0~ScV(N&/,0i^EFad8)>jbTvt#xJFK3Q.=P,Q^GLf3MKOEK{Fb11yt>jz}WAL;%|Ke~R!7/S~c0ppN_LIrxKn@f&.rG+]tlw03,xGK.nG!]7V6!^,z3.&Z,Ah6s401%|mz2{tYsZQ]FmxNuDwzeUM<mH$Uvbe=[L=Hj~&XL6!X*jKxWOYrW=Rr]8@N8reJ9:omYCp@Hkr=T)ua|HC;_55#&hHrZd{M_q/QB:sUfD#nOdcKGHCp[92[Q-N2m`c*&}r1Z%$G(,]$}s`FB9pwLma5{m.?2mDE]Igf>a+Wt4RT/f9[=hHaT)kx#(1I=@gSa4gyF:ok9W]%$fD#eHyn,2Lrf}Lt>TI?Hq&Af}8g]r}g.HM[2{&D!3jI#U/]mCx6C7B.6/:@kOTcrhKit?<1Z]BD78AykfzuSTcnccZ]w{iWH*DeG37%kBIu60jIgYkWvMN$&oI3gUCA4JuiCi,:v$lb6cy$`UxH02Gn&J:~Yo94ni,Y}P$}QP^($Dyx,a@*;ctMnt&:0ss8{P{hv|US`h-38aHz?[FGY?{>:,[yDY?$`5bguUU:u=2wFYo1v0xwAx<pCfM[tPFd>ImWl*oZK.!wo6K^UJy9jjQAJf+r!s8bC0A_%u~-pjj@A^A7!Rdd}U,av~B^gbjm9,ij{Y,G4:SoO];yrn&t!AiOg}#WU)WPd|T(4<6yPy3hu?@km7QEKH0@I6Rf|,I2jv2Ly)i[`X100rG%Zj~//uWzM:vUTHypJ/,G0<pd2Ar+#q%61a,L7G5>g@5v8[}6|X(xZfs,H%NA>QjZF.ryuX&CrUyuc:biWJ|<)YmK3kyIdPN}#(O7aBd3F;Z$NYoO?KJ:v1_{%P)(eW#5GNMe1|}RpPzPUlz_I0*F>j?yEvziIdcqKi-*$`<jXiOD|Qg}SgJx3#M8<v=t4T[poj{yAn4j!t82]_dbP.p=+t2VcxLoc6P7#Gbbm(B@WA}<S0@YsydulwBV-NV(Nj2_RgGG`Rih[=e(_@<(1>:Fo89IBV51o@F%_DA2c!2H!vPjM0xQ2XZ=^Q%B)a4oprPBv31BI`N9HI>tS_h|.[=Q|oCyXt9{=A$}4=<F5RY%NX[@B)f~<&1)S{$Vdv(N9{sHMAr4I`kij6;?f?|m2_~6eb*%C6Ec#v{a2j0gnk<AvL@?QcO3e4n5i7fF5ptaZR$x3j}TI+3YJGI?>tVaxrXUjGFWFq+EG2v6[x]Y>#^A~/F^z3}^V(6&*}iZ:m3AGucF_`XdUK6PQD<;,Qu:[c56OTv6!@%QFFBB1jCTrfxrKTRV$iesO)S0]`xX]YiM+G9MjgPv0f@+cgj_ZlZS~f9UeU^+&N;,uxu#~)Pumm;CkFmd}u5oyo)0dm~G7//9|1QaBNKxV-&EbMbK:LgX0dQ9L`-,kAZcSkC+$3I!*@}InZEo`Fp0DlcBw>dX:gUt|ks+_!)%_y[<;w%JUy=gyU%AAyJt+>qTRe#;3>@?ueb<fP`0)J|@@Xqyw&a;fh29t@z8&1tk:4(JGF{En066oo[7HtDU$Q`<z@*AcU-qC1yz4=>9y4b+PC4ti4iA[2[WmBf=6V//PAvXY+(k+j*UB]VahaZ({U/E*)>`4qG=^6`x6y@;1nwDV[/|T!x1JI`hxOQuW*<8E}S]9]n5l*legtpc>fjK>oHr/SU7Y6!K!Cy=RZ^LqL6,,tt&ygO6}kW$I5S<l,~;SM0+CaH;IoytonYAbLl?qk2YKTO4-Rh:j]Ym_ri|k3V4bE|12+f$J`<A7_Q,;h~C*wu|&e5M~)1b56H/+p?{{s@HliwuF!Mioj=C~Sw7Io@-,M)F^{Z}Rgg#Kk1URBo}hqlm.D?;_y}j2l#E^7opu*W!M-Wg1I3N45NtS+;IohtApDfo!w%xu4yLN]h3;peyJHgv/ZY$)hi`V@GhTjZViEa$+t#fUt?mv;D(k}nZBrCEVc${7EQ;Q=u5aCuVgUTXkkX1_kL4Zu[sn[9=:H:}HlrS;DkL7m0h[;n<Y<%W3*vxX-{GWuA?HJ3Gn`g;r|kvNX)}(p+6{0FWL00lk_|5-m[;So4[7W?zDLCNG^HqA_2Zu]sM69JoGsO6L/veUqb7i[6v#O(uYP.X6`|e;es-O-yWd|bTW8TR^R4dV{7}!8eC-FI[xeVMD3|-KUR>xyqxv%VgW.Xf]<yWM2U6@|x;IeE!cX*zTF;[/(}e(q!,8P`UkmcOys@=Jj-<vof{U}nmpf7yfU$|P#2!A2oR.SXE|{k?|0H$6j6ucy%EXI^CLX:Pv_9u9z:*p+&6O2)o8_e?#ed}Z(s9opkRmGwLXyfF9^|`QqKxR*px2sfNyo*J%db6LL.JpL8#J,@{G[hwN@bZ:egQzyAyDVjiV.hYm@[17tJFbwgpqM~_~Ddq:`l%M@B}W=&,bYVV?p$DZR(,G+La-XC_s+>a;A~z7r9dnPF2{W6H5@/{(I*.=*Rxa)?0dos1VF,vk6AuaqIXQ6=rXS5;^^m$zr<bf0Zr2|d`|lLI.&d55Z!H]@q/=IsS<8ekJ%$x.oxaB;@V!81bKY~Z,daqw$UXx=$z2_>2/WAIt;[(!2vXSn8Wiz@$8.*to4b&tHUk-M%t.uq+!at`$$mW/*/N;86^6N4+I4t*FSSUNYjx1NT.fEOxvzeqt]Xe4@Afj2dJSYKx^Ot1+$u`a=7>>nn/S^ijMLhRfqAvtq@maM9oMd}LS&|8P>LpP|Czv*+)=}v6,m1ELTrcZp[;S!ut|nP(W:BB#]}_n]3K{sCY&i7&{{+W??G>Tn.k*{MK0kueox1@X2XJj=5=~X%kD|4j%5_?<HK8ul&5zC(By6,+MfIs0{mS.S9f3s1(8&0d<u!d/gsu<r&x?|iIXCJ#8/kkIAa^}Uf(fmi]0j3W{4)Acrp#C5t?Xb.iJ#SOP2Eno6ucfNR--a]}9F7fg%FGN1t,+/9N/[(g47Hfv6gX0@z:inzSO~+A-XS,TMX>x4)xu|bHid;QWY_i#.?DadVk2::xkL7em4<nhAMgJ^n]2Sy94T~xJsaDE$X6s[Z&`#%5#eJp!<KT!lo#5]f?g4^7GTh.~9zzM1!tX5}#>Q3hH^HcT$U,qad&M`M{7O3riZ9:0b6g%)eem+d`l[0r/|8bFQes@/}LM0kMNO^{ejhr=Nfy?:P^/MMUs|C@ph:u2bKJ:;KRPCXh]b])wuY68w4F29sP{CPZFS?Gl8N&$_2]D?FS^en[Lo~uEEB+SmllC|NDD)@DbT_,9.&#.$3?`pT{#lcMz0iL=_}ncGR_M6qP:@3[9;|_n.{__]duK+3$T8aJ>T)]+?UWuAGj81;R%*@#(=QHKXS)fB;=Q>5wg{t?.)2,-9H.(jm{0RvPcqBIr`fbvhOwFI6.-#5s,craxb9g*O.Hb(uOV$y=s4$HAc)L9BQ#JJUfs1W`lQ!xQ$*-K}t1QrcxY;;By6B4-H3%SZ(?,&*1LH!U-2_pS+U%}[>60_9tWlZd6|QwSBiszk$wt!:U|FK(?2O2WqA-Hv:1Et.)}&/N_*@(JaX.9jVdD$y4CFo;yw7Tb)#*9vcyeQ%5HEq(+s$?Sf$#Iw&Tx_eP}LFp=3Gh4<h|+:]oX]3dS4&H%V<[a+9!_S[yLv;qlZAEbZ,h&-O40&VZ8X$4uYDxo.o=PW?q~&D))?`W?q#E4*S+^KY7na?XN2tn-|_5ve,9=I<U4Cz*Ju#eivQ[M|=;i6wtGwfZi#a(Cc*FeM*;&M,`s;<DL+<>cnZh0TFts-z{b`@]7Eq,}P/hv0r65A]btIVKo%~Zu5Sr;g/ytk*Mzvx#Eqpi,_O={I*[ksa0F%O^YqLg,.9&3{LAMuv`ADapxR4MQ0)qj;t^S5[:1A<^:#7utyu*ynTCU|XJPDV;#|RL(>VrT$kqllrcpR&5*@D6$-}Vp*+.p.ja^/>u||joLv.z<*EhJ)d0Q;=gkY~Db~M|T;7tgvg+6eMY)-ksfgh2@6;#?gjbfO%7F(M;BE[8#zm<JEZp)h%fJ]3F`mNjKp<GgN3#{:zWD|OGs.15%<LDoGi*tFOMtfHhCNoX!-k-}bimQA=#p_+?;A@SVL0GKQ9}HE(RCX3y@g)_z[G8zfd4a<VimrL&~nC(VLXNOvsb>h]Q.70Psmc6wH`.z.Olix-6Y-I*?aFYx?Y&3P|_5+y<%TUno9Ez5)8_q/QUE`-a/yMm?,hk4i>.|p^s!3,Z_`{ECPgB<zg[xVe6P~?i0-5.rD;r]kib@[]Ob$B7TrKY=bcM=*X0XlA~gf-5Ae<qC]z^c^NxjG>O.Z~]c4SDBJ5iC&.YDR@z-;f]`px_6>bAtoVwHDv#Af1(x@_1[mY.DkFHH0$ZjI?IpUX4+j2b)m~SfNA&^4H?1d;{y-Yas0pKj;}fY1W9!Ub1>$n1@Y.h|n~TZG=Gz%p{=W2IOF].^z+lTN1&kJC;N}o5PDK`gY6FHF-5G*XJ=Uh6cA(j]>F`%+=-+i|OfK)$x|Vd#;7<b*aN^sj%y5RjrBB=8w]B]i4/mVDTlv$e~s/8/,|b$_Mg<_W@Te(AMS3ks-xUcHDv<`_}_.l&XX2AL%AoUR/lh`=bjgtbk[y(Y<yE,-nWiHJ_&fA[AqGrjyPC:3Qw<|6m{G}Tqb,?i;K_G/~=-4z,|D<Wp5|(srDE2RWIMOxuYK!!RH@`*^o<,E#I}C5(c*lC~;Zy@xKF.$u2O6P(kV/LT3cvu>>^C?}e48%A1H+4~9_b(p4C=BUv>v`W^nQy89E_Uc*zuFByke8__3&.~H_/:>#M[y)_]B,P?[AeR:$2[K%2JOmUDnnX.MkFZ^k?ERI0>5qVo:@^^laqfpRkm6A,gD=x7Qk.gAYg.VR@m+;+^_(9moz?sI~Ej:a/5IB7jQ<dx`lgxBu2*%aA>5D[/`QV^Af|]H(?eas@.6m;3r3qxG`)4rjwDVMBQTXC|YXiK|lmT@%e6myx_PH(tK^(4D_|d4Nard7+3krjS[b8VZssqml],trikR`!!EyViVf*1RE{w*U<VN7Uv9#N2(!j<ZC1rFJ^j@{Z3`)d-Zag)_ihi=pP0*S8A.[jhAw~j^p)(ES<,gLL9B(PJJ=_0Y$9y.&jNcqHrM=FZFa4:PW6U/4VTl%=V%~imKAMh<0g5tD_U8FLCVd$6W8b+0/5yM:pVi|CmX8O?@|yiDh-lv:rk%mhCYt:|OHOEd[>p:%9}DnhyhDEzF=#~+lglkO,8Naz#yr?YdiO/U=f*Uu2QMp,l-j@zMG>x3H2k#9{HuU_99{c;iP~LO~T=By9HADg3nh,jqn]4eIX0KYDXX/T}F7{DLpPV,wL:pR-fa8wO|dpw->G.$/BB1!nb@L%H5v,Pea9b9yKgH-Tms0TT+2`gRQ0XG!Udw[;DmJ70Cm|r}%-iM#sg-?e!lzdz*RIo7A1B]Zrq{b?2]([u<NUWqH#rU}7L~0tY<Odr6}gAovM6/;xm[K]V~I.uK;97z^<Mtdor)u>]-{P-@DYlZ%Y0qoeaoWBTg7i,zyTe0W4pv072Q{V+n-Ka(|]5%pc1zhw$gAv_}kS]+j}zcw+Q>RHh_)ipuhfNsg<U`/XC@_b^;14%M+L&?LC^qx]}yqCw;P#!vu5`AgZj<^05,I9vqAbdeVjsD7:!kV8]JTXf>t^Py7;p90EJtO<WsP-0KA{YT@Of)J~8QTmzIua3Fc$E@BXey<IE>]&#Idr2*;%HbsZ[VC0p?g8SOs}K[er2hR&N1-1A{{)Cp]k=)IsX;dkAE1F%h{Yml~[vhCx^$X6`p&(g=)`Evu585z-jQdegPL`F~<I7},7z&}k_J+2AQ+7:4Nmra`+T>GAb?q[W7d1oIo^}pzJ?)xtQ@V=M=5le:!,l%wDJuvoGXsK.xZyOz*;y*$:GYaRE=zKT0*T:r}&+Nf9BV<GRq>(ukO~y9.mE#+~g_2`HChFfU.yz:=s2$cgqQcp>Je*eZ6C=wFxMrZbXVLahc.[2T9>VdzJYXNkR`g8;$BiaBm4_?W#?fka=FcR^/E)>*f{:*M|w;Y,l{snWmcu^(ZI[W!WQvqbQ<sZ#h~l`.>7t#V&<@*.qbTl<=`BIMKK(sfS^-h(5E%nVq//u,Gf;V6E!Ad$q5d7rvH2Uly~X,g3)w@mUr&UtX$1[wc0NDIOnAPFT+`UoLU#?M-)SSEKKP4J&Mn</J>KZI^XL8OsX0Pv5ULo3u~!RZ}GXE:(+uUG3}3:)g70*xEfuuZG##V^>qyJlNE.W+IoO-;I&R:dC3.X5kXP1B.GJSAQdj^atTdW_~(A%oVW<e+d##c6_J5e.|&X4n#v2bm7=/e:W(A<ZtmAjBH@CwTY<K~l4uN}%d}15-^08d~j3_`o^PL6Xlzs-cC>3|&r)}VR)x!h]9n1kX2D!tv[Ks*)vh_h5]<}8`[%_VV_ea|adO4&{9cZ$D!Vn36+B1/gjR[C:g*z,U&$.D>ApP066y.1f{@S9[EtWQ&$W1|]siw0,^?O*DAen(^Te|#W-l[n8Cg$aa-]LxGs|Fh,|]zwFM[[g`HZFKuvv-I5)|3VD#~Unt)c5oS$odRt2waRcmYw}:z17N-{`ghWM`kfee|eS)AYqa5S;,ke&?0rdK>+QR#h$?Us%=YDqfggEq.lN}6cSvvub)ze.:SIXqI:tf}xAra4O{#S)4L9A(Q[U%D^umKke@b=oZTgiI>x&<M|U*Zz/,6S}YLX7W`HFwN[QH6f?wUDZ48l^B[r$7}V/iwc#6a(/1M,_(}Jy>1,+`Ur:sM*Mu7v_<sUozK!`D$qKd_w7I1`dK}3o9ic[.hxgy`<E.Yq{ra}|bjih)@rRs<,5l-%:}n-yKlQOH^DDvxgT?p3gEFL{wpE*dU]d{d9RgNip*N<oKe=Uee2^^zhU$)q3D*Xm#2PToYc(cgB4}v?DkUv%Do;UFOa_LduXTm*y[AYYa[yG~.C3V0GnEU(g7|<&g#1?Y[i9OSSfrRXw?:]Wl(F)q!epp7xM,?uc(#[}}W9]B,aRS@zVN68?F?MTle_]nv~-N:*x~<K1bp|aL0byKfV!d,4$%}TpnEzAbl5=`5wnF/6-9QSB;1yTB_rfHz^Y]#*v[V!&Xij&R],]wRm6Batg}q^SxY[p`I7jliRMo_P;IjT/=Ice7xqaEN/^V+DL]Y2Dc5ROO*7LD6f)jkUQ>q}[bi2?E/%T=@A{{Qmh}!9hPZQjMs[E[:-9O~&W=P4$9Xi,(k?%Zg]Ik$b*U<-6FT]e:}?Pf,Fa?X))sg}Ey#P/$*oi7%2F3Wo=S9Qskeky>fD,K]vx~Rf=sjfH>cR,Nr8p8E=;npA44DL3P99SBY6JBL[K$|LSQaZYwk=%qXT6(8Pm;)PL]Z#2OH2QF+y`+YNeDNq)qoZPi`-k[0yu-MH[B3Y$0<sw}s-=)B}tW$=JC&pPkv5btfIusz_q}iM#*nWMh{?Re^E/3feCPnS<T#vJEedhVQuT?p#&Y?1Ee$MNwz[]Ba=6MO[re*~K^5$-guZctZ&(9g&![R9uPqo#&hm$>T5P[<(VzjGNu0=0t!OFErE4D+1qGVuS<9j?D6$ShNUw3nky.3t7+XFE{4/Ny^h}lxHTS,|/Zv{u&QKJ6p,w}.F?r7D/v_Yc,/sKgj9JAiD}IM6096;Ms%Fu_~~N$fTatfHH%[uq=bD}XXOoP&i%Ziu&xO7(Rzr:;M@dKjskOc<4Q^tt@[IMmP0QLNOSi7|.u;yv?xyAk2tIrUT[J1bo8-yvwS1=g{;#LN,c/=8h=){f)&6d9u~/z]N:A[vufe2W81jIP?GZU#QTH@Cp_#{G|;c7&tU)[|fh[/dDWBz6)_<7WyjmL{7Q0~-#Zq$3>01MVr,9(ly$A&gjQo%Q`P{w*`k&I|K3>7{.s}Ml%M|W>V9;?mR#}xnm>l9q+L9*e@$=ma~lv=9Z*7YzhdO=pVoj<qh@Y[NFPDG{N(ndrNNMjcygEz}`vWZMu<PRS[#0f=s5=9F#`d$19%G4aDz0&!vrh.deMI@94RHqGQobWx:aVSWiA18`e6y<r@P.Se8V-U;Ad7ZSiT&x#l%nt}>5aw~E0^k#SccE{wB?.egMuo3sqJ:84R>dt=B<*?L]sc.+*)FMkQ+jr-Lb5|&i}yONpQvMO?)OS.~7iQUA{C`v;mno+*pRofH+mf+T-{Uem^0E^hGNCRK/QM&l63m6hFQYW`TJBA2n|v8AYld*BM2s52u0`FDzG9.ky*+cxvP(963mFOY+{UUat?V)N;E{Ov9Db?nMN!><D@.6s6MZZ,MFfKpPw66Zt_7A9O$u]L0}AbgMHUO=rsl,hxm0JYS}(1]U(Uhk4=5^1`li=/:OdJ@ai-8s.t]D9n2r_>?aoa6?xR4GM*Y`J36}>|slyIp83IYm8XpF+.%I-=*ESTxjpW{CRc;*%bL12M&#`e(@q<z6E90l_1?o;)`M.q_xd:{IQlB(`ZN`i>3rBF.xgJZe?A]JeVgogXX7wCa5na.vkBK/#$EJrX`_)ChS:D.O=,dG8o6SFzsNgQ@^qUF]NhPiBe84j,D]_:*/3(v:O=}v^oq8w`SReidz-omd/lQZm^CBp;F9AR)JO4`YwVQN@!<.I}{$Ou@+P6yM8EVOPrC%1u5@FN8a)8b%?PNvU2TKCj.s:c+9YR^AAIh+yj`1p[)*]##jCx#rk*mQ_-1F$ndjzpTpoWAXYZa6(u}z*WL><H>W|;MWy^8-Sajt0i,Twg,Hso1Nh)cJ?eqDYDeY6oD$eA|D?gG)cN,6d+/_?A&L:lkU>E>_,mudA-g*d1N3YP3[gtiIqp,Tw[E/W>Pk,z=NBo:m5CPY^Uk.ZTTq^x;LMD%3]j>lLA_31>:mfNj{dhP@GF}:Wcvs*-LOYkZRwG8hyjb?^6/.xhK-gxw1`s(P-muu1]KQ?[dqHjbwQSJRe<v~xU1L|OhUr+CR80tlcFeR8Xzvdane/;}CZ`=jd1s.;x=3NA`B:n*Z9E+>D1DBgN#usVnelzsk0pQP50yeAj.&MYXsjMGXi8J(r<&r{#?}G*Q6}LvJY6srdmHHZVCWvsm3*(L3IyeTT;keG~-?)4ji}LTL#u{g<?h(^Q@T*&W0u_k[s[R&-=6e}RVoi9_-)n*TiFGM9HZv>u}}CUYk$Zq/qwuCkQ)I-Tq]o!bB7LV&UcTDkF/Lan(^tZ8)=[1%lGIM`v$dir5@fvavHX[Y+JZ:Hm]F1:qSJm5&<9?{Z7>c_&JK(O1f.X5o>)`cK(PpXg+7`o{#/F]_,f0i^ZUlvyUO1*wycx?;?mB^@Pv6f4+Et-mx~tH^52o1lePcgd(Cmf:2876QgAk&HG8-BC&mVoi7%Q2ZcW?oSlB5@>]]a1BNA%9b{,Z~>|z|6<^K>$6oyke`,Z/[ezu9)Q>b;{m3)IDU@gV;;eR2t+7hgbX.=_46+Xbs`ksKx$%2c&$G21]3gSKQvzB$O7j8lx_3Z(rM<({Dwm_h<0f_3Nm7j%b,CRR+n/NSaP5U=lzmuEWJDGmu+oBqbVghAVh~|w1<CEYc3%EI64lF?UGoRedE+`3.J[oqC}vQ@7W%C:QSpzVJli^5%oi=216y5D]R<iwU[Y(AE#x}N;Y-)8P;+Kf6dPlX%NwubrHx/(P.ZlRcf08>WW~2^qf^R!qkO:YTaTFj<S@mWZw58xQG6wI4)Dpm&BsyZn7I|Pzg@{&:XLRhKaS8vM(88ELKOyISZ5xb;B?{?BG+MJt!kl}o..Khy+IU)8<l6]j{>DONX&UH0n`G$YS]Z:e~sdJ8iBmSCr7wbJZ%{_/28UYm<9Pvdj]7D.vD9i7bb+/g!_zUd>aha:X2?DM$^t+TlNPCH]j<vSD88{OV>f|/ABaq#Q^~duWmBrs*2gH>fbm`ez$|P~.3}{L$dI#{4oP<`)y:g^.Wi<2M&7&)nZ:ip[bXV+E:fTd/cQ[KujR;YK)C=Ez0*DL5*F3%=lE[{7/+Nba5g;5W4869Auvxk+.ALx>>t7vE:gB)bp]X(rHBF;l,%;9Dawn13JM;8w~Qyx]SP8xZLX!Y-E4#nQSb{>by&,M*eIO,uo,:Yrm<9j-,V(`71PxN]gF60eiY%r_b{Q}i$mNLA!@|;tN(#fZfnW|9,8t<m2#~ZDY@/r+?p-^!]9D4v=]qSH&#dzHoQ#1xg.nA7hTV[VTVc2)Yj`%R}L9L:GlZ(pq~=Ow}<2/5]#SOT$%gGiq50^CcUg9U0>7JL<943o$iIUN#=iFl,`2y#J>Hu{tdFJ&Y:K,*-jp*jUOhW8?:);hV2{iX*YnRbsKZ+P6M>2JYL!;5H,@p3QS;>dK7NRo{:HLZwbCvgMtg-/FF|2KkSH&XPRK_=MlpZ1xol|haKWYb/=BA!c&vBWcqi26w,3rBAXgy}AJ<wjei[7:y,LZP|vvVkB=W$Rh^vxK2p4tqWM0cAK|NKG[jb2dDck#e::ubYM|rJDFi+W1#=E:3W{!xyY<NuQn[^vxmr9@6fHfte8[@_|a8U]+X7^Q&dva=Rr|=i)(sT,2Yx&|T[C8~qp8s3&^Qq?MOuC3tH-v.V@[rR0s/Eb271R@F7g:8XV2k^=M^)Fi]S#AYK?JmSQe2-l@P->>R.VFxvarj3MWv53,Pd>afMuUl7ZgZK-G6m0hQ,PhUL%WS~3g(EA>niPqh`RZnA]vot5Max$UdxJ=Igie~>;34c^dWuwEpv79bvr^g[tp<uZI;>43)n_5ztIYV[6%Ium!F#}Dv@8YbwZukfz&]m7,+gOO^.I~_8G/VH)xW}/B)P8a%PUzP%v7Ed(1.Swyfl.+NE;mg##F0l>B@F;S(s;V_Pwc40Ul;#YcHQUHTzOgvA-Qf_!T>.n:w1Uruus*sZAP?COF165v32)t>My,cQ<Zk(`&B@%-mScfq{xa,x3VI!0/*.pY<;9(^wND0{D9s)&}IiDgiWcQLSJJJ3g@]*U*a8*BoC#CZ&q(@}LuRcP2$m#^o5?v,GBH%Y`)OvzJ=cmsV.kCbYZ|+kf,,1@mJXWYs@]0SU/m_Em}g}&bcv`%xAI&!<Y|/`4`x6waz)GoDTI8G-*~yIW(`;G82W/(&e=M*ChvW$i{CZFXrr.Lx7?B<}_{y6axKQXWvvvKr}c?/XQl)HM6Jf_lz*(?cE!06CzmkaUaT|,+~W[/T%5L#z79mwgW*q|hogP]CR5:paSB:O$RESNnc`gw&:q`Y?b)|ZZA-*sL8PhsUABX;iA88F]S}580jIxLC4Dt3~d%BbAcyAiz7w!lsl,}t*zmXbalwbfsT7iQI9f4KE)pke:P}Ew]jjZVHadv+@5FRG^,8u4tf@{6R0FIr3k.Vkoi}M^5bcnDGhpo2:;<a{k3Sp)VuiVKyF8%_v?:wXp)pm(RfJONy>l/!!G)<U~$f3Q}N##]69PIsA}b$.K3dboI/Epk/*:(tI|+*PAE~WGi<e1_y|TXH>@V`aW|SfBM/b&M|$qH&uRXNy(01f3i3Rb28aHSyje#-vMesvVYCZr?>:!u9Ik@:>2D@_GJD-vOGqf<$gcU>C&H7~Da,q9+/jbu9hxKc!:r9lt?AJXHW4tlH_@Qk51}@xBx:l#9r:eD}Rx|`*?-%^0lEwWBAwc,0d0at*P>B6?NFvVPR>B`lSMUrTX-I?RuLW1?k.|/YU:3[O%z|lt@2GRZo^<mB:j-~@^{f_u+WNK%:Y>dgy;<E|Yd`dL%L9]}ZilPq@6LUe*VZ6MX_QGJ./B9j`>.&L(W!ZOa!6~0.Zaty]gJJb]FAV4nLN+iLP43mi`L1Bi=Ed@2SvLZmj^6e&_Lu4Rfn/g#vkz/nXc8!bavNln{>cXtFZqOm_JhEr_HmGl`x!:-#F=uXKRvC:i?>(v5m9)7>a*7e+r#gZ7fw#_nm-5UOJCoi/8Qo1OqEqH6=>*T=HfJ`x=6~B&xy]j}DT}C6K~D0S_7`05(Uo@*?Bf.qQmtKqGUnf2EHG?C&F)su+Kx/#cy;/RYnlphN]M9&*L{Q%hlMXSz.xJ]JGGHx*51{^s;k[-#@_5>~;6>Zc:QU.7l$:`Nw(mbQDLBF#WJ,^mKl6YRO&(Hwlzqw*HA}Rn|[4aN^ykk-_&fyp2-/8WUcqb($t/D?Rw1Um5{N=$8`qaD!4VNRoXU1er,q-P81i&OUE0t0OeJh,Hu1Z!6&_;Hc`C{Y/bp;@B/=DQ%v+_{KSq^:^@&P&OBDxq,N!Bk2nv*dCTGM:Hazigaz=>[RW<{n;s_RanabVWOHrA_Ga-jCdw?^p]9NKKS{dQxu8{L1yD|Xa*(I6vq&Xyp)GKGV<RSX*93_8ii_>zsHT^*g=FnsJT?Fx[h9!!Bo)sQ{=hWs<|ONWKW8.!v$*7>l#cqxYw]bY%[o/r>0NutZsIw#u(dg^Ol[_>IP=LE4C3oN++{xzN29UIsejU{slM)[R.OM[{Fe3oIxrhw;yHttupEJtEU2b9v,n_T4;TzEf0Q`$tfmO.(>+yW2uLtov7R`Qp)xLo[mST[G.;I!Tqps-RZ-ZekvPQk/H~rYJg@$W6uc>v[YC16HD){Pwxz>3@6&jbeP7KEDdx>)a/_mvUxA9(4}mF3uE`}HOC;(i_zx5/Ou}r&D-6BzE<!ZVS/<+](oJZgz^J^o(~3:8WE(eCbuZs2y8qZA>gX>yWbmtw1Tr!#6/4r,8HERG|^:XqRTV)Ak`m-]q|MQTE-WTu9ytWf@2~1WBEz0/VS;]KX1]!JoDyV*Cjid=CzQQVJfmwK`wZW>+uuVTZXwfnwh`8f|EO8-pfmbN`!(`gZn_^G]l(XAgAgBBygFZ{67{;Y9VPmK`T0PokU3P1;:><OQQSuWC%MwuXuej:3%$a{jTVf!m/x|5JgdRlhGT38zYazNWIr]=z4zE~*b2O=Z]/&I<.U%z`cW-:xhM?8PGU/83#Qz`E0()SZCW^3egD5OPl5TKLe>jzOrOp$[[AOmo1clbSeHy8W8aIejQ6/YC73A]#,H6e{_aU|ffseA-fZ/5Yowx<4Gh@K4w/U.U|TDy0Z>Yt8K~5IK=XyDoQp6!g`%>L24w3utg){TR7sE37=1l0idjmRQY<@w+#WJBu=e[b<)U{CnE8D5H)`ug<}9E>@<wEsNnv}GZ4,:1:IX:oZ{h`$|@=k*@/5Z7Gynhh>/D=9*FlcM%%Q|G5TVINW7m}/5(+Lu_6g-gBFSO$^cd8}kuQ8A#DoYq([Z7gX:1j!J=n><@GyZ!2(+__E&e6H~8MGm(+s3o|uDvdb:h2198d>F8rVlI*h9PSG[Ped!`B]mBwmcs[Z,&SQDj|`il0j=J4L4J!zYq)D12;Kj%<aooyX~C|M4Io7t6JmFsWH&F^EPt<.^P`M_3&WDOdD4[&O-M<5FT@irW6puc)/#n+y9D0[nQ#{e?KI../|9rO-4aQ|/[cB1D~^GOW,;%(%0tRgi4+)4cSl9r`j{|TH~p`=Uo2ScLYMmh{H;I)Ts;Zht>!8/#;LX&>bvpe=X&:C8jeLyXKEil<GU[`$#6>ZbWD}9!)$R0E7i:S7elox1JzPQ^;bEX$l5fXP!9}>l6|yY0`kllgwY@f,0S-aFd5k8xp&YGsJ(HD,?(?O^6n>JKkE.&22q}Ja+S;mSK$pxlH>A?z[1$K~b[DUO4nLP6B>GG[4I<ua|^q|Ez<2eZ&%WMsZ_]`t&ozxi8c2g?nZ#nfib|JYm,c.XH#rG1@jBbm|Bh.L11HnSE|I3_n?J7ORS-.0a[S5^p:4VTG0p*>uBP^nHr1y`.[@;yIz]lAR$-8-46!,htzwyZKZA9{mex*b_it%C0hz=1QMHh9[=z4#y(o@!*ABT?BASh/(pz=FTRp9W(yZ]HpC4XdF[U]+:(Z3)l<Q)5SDE}6C:,^?z[1x/rV]m?{9@w3S%-XjK$_j9~h8V55k+wrDScxBa8YUApS/[(K>c=OXyclzz;]l%sqpWEbJ]Y]h|s}UWz]SxR4~PkNmK{(}tTE^A&Xs&d*HNv}0npk:mdYo3A%0Cle$yA/stpR@$j2ia|rBz<|6857%dU@J*C+/EUpDocUw!*SVYMIUxcYK:.!U)l7%Y62m*rhCXh$HE})T,Y<:[/#B$C2K~A<Q;<VzEMT|&DxO?({Pw$/cDRs8EC,9OD^3&Q&7AjqKSh%Lgizas:l|=^8y]G<198O3rIj%!f;[AdbY>]1#!iPMTypFL+q1=dJ6YkwrL)<S}Y,Pv)qp+kD!SYYmrtg|gN&>kisXNi#^8e++v]$ykO*{Rv*)>l[7X60<fw)_->}SU}}Gtc]p4|4%Nnv;+GrUnsl-?Fm.Lrdf{h1Fb9%mhW(gO-y$Pv)KDH`9|][^H5)hM?%W2z;[$Ak8`J#PE^?=Z@{0L%zS/RUIbhaR?HMHQ@/I}dFTW8suxS7>3v1hD#uJjR.@tbCbbtjHw.&o-[a*W@x;~TjTw}^8#jKS=P@pycwv,ASE]vaL$GPL3NoD:J?0Z]FY<<3<~@U28VSw<o)zmd/>U.9(J:+1~jx-vY{Q22K;A/V%rNG*1qRzJ1=I$B]B2JgL>%2A-a09yDl]iwa-KK*&VT6IUz`@%_54~7taFs5#)y/}~}-l^SgdMQhuIDEv|jh40WGiZz]qxC4CgWve*Az<oO%%FW*SLTBy]>hJR!srDT^lqW?O{fK=Q/V;|T#cN5zyB0CbiW@!Bgmvi|[.0(zTs[VHYy`;3OWzsc=dn:v>yP6m1CjQp$IQvyTuDGP}]$CfpU&o6TkD!ocx)(A,HBlu;/rw!5A]:Z)-^U<*kb>A-Pvu)y*~K&Zz-15#9ij4^?a3t&PytpguCww>kTXZXBkF#3Xx:q3o!Y=`Z@Q{oXdoLa[r4+]H1W{]@TkH=Q,zL/@Y`Pg&Ona{qj|7aiv38M6oWgYmd.37mN7O5CNb/A[isf1/krdIwl)qoA.B%9iylB]GmW8xGH*1d06sf)B&ndv>OAv{*8@>(*scqvk9wo4+uf%=SY~0mh.Y=LGcp}cXPwd*g{%08E+upai,+`:lY-4]o-9q&c(r>zI?,A@um5f%{;Q@m85<o+xz!7U3yz)JRTfL)uh6lS{7;?,e#psDOzLMQD46Srqp1%xaK|vS5WQsw4#h+PS/c&o3xNgZwYy1T$bm2`?jMq1d-J3)ISh>aAo|ouk{t+^n5}ywYNq1M^4p*N&rK,t,-EBtMU[]|N-9DQ>}E/`.rD,9v,;^Wi%NHfuo-aZ(JikP[O~e{I8x}0aiQUMx}%s68Xl3{q|WSO..fj*SvHeCA96hJVQV+MK=-Bx2VS$nj<xCfsmf_L.)xo8D~HrsA?(XHy1dnV:<ug5u{aw8J2YL4s41R],002~{Wx2wZvMvt0]_~)wkj0[{4TXNyJ_xlQ*HyvvJHnO):n[$l7n,I9U*z#yszyl[%#vv,Aa:]p(%X/Fo5Ro{&ofE+M9XdJN}s-@Y0({l[g*(}pI7GV[0IT}@v0,{x|>n$X<oj,Xyxpuq,LB^&;OXDXn1i*5(}tz{!r+#/-{=),[3>~KU-z,bC/dJzF~DZ3ZVFCXp50Wq!XX?kHz#ph1]K$w:_1X_EIMoX45_u3~2m?!nra<8RL%FH%e%>^`n<MQ?RrpK)5qW3LviFT4CP8qGpM|2/wUzH1MY}!)qVn%qa]DvYy~K/pjJ#M.%%A/RRz)KN,>+rAd[V`p0QNx%v}{;;/^z<xA3^}M>|9Roxpyh|NhCl[]T=&Eg6/ej[lQ%}Q:r`u_AuaGH,v*$i9L/{j6kOLh|wvhY.@//Xt+;$4t$&]Kw$rvkuKd~=<Kx6B:<Z?W*sUCtyqYLKauEU1k`ypYOi=[0rsOT{{LQl8Ia79od,@RopR_43|zVev]3~E:[vYYJ]k),TZ-]9tzD/SxJ>%$uvrg1_Q8P/-OpyndgXw?le)0F{Z#S#2j_(txej>mv?_:NVPH+:$3%uE}e,L*2QB>%Xyew(M<noWeQ1quiavmZ.K(A|<32J)d<du-3iLYe2Y/ll8ID,F81-L$k_5PAARh+]g98F!n8K`tU:f$_>|;Be9%id6~eC@mr;BK)A&3AART`w`XdOhUardq!ov6oV6S<wE^Q,mB?QwX;Iw4Zk{f;TZoVGNE+R[~YuHC9:OC=+_i@/l;C:z(%wKaMgz3x`{c$Y,>DE~#blsK8RJ9<cXXjh[w9<hr#W=$LmD`W`<!7vI*d`pC8Ew2;QO5b6A:xFE|b;-(LcpZ)vD0Iwee_G>VuI]jEbt-w~+I%bh|%]{e~sW7r%^^L`iwJ.CyG}LAM*fzDFCX>Gj~X$vXbcl.iq<*l{--%hme]z9oYR|C={vQOP)RLo1W?/H6e5S3%tw#c9O+(#O_ZtnhtD.s/tTbMhV{Yu0b7MU.#E0c>8M/M$.e>-T{n;>~2{Ikh-%f?-W3En&FUy<k%U0o79JcX0/DNdpU[1ILKA@/tM?_mZA&$.>t28z@)O*SOE@lg^,IfzpKp&N-+AYa4SYxhbVO9pbpx+ga]k]K<j;JQ5s6.pT^>KWRWjQXJ7h#1{:[);RVTBZrHfgAIETrsXp9hs8JDU{O|rQ%=c1!m:1CT`Z`^hQbgX5TtxdJ6t}.JC]Ma<+;ZAFf]2G6RV#alzhb+>RBkJG3d!IitMqQ-@`<7y_!>os_x?#,/3?BKax<QdYFnC7w1/901c%m^G+%sqVpNbf^U(UwBkLBPFg0q93d1l5gKLA^(x9h-P2Je3)iEGvtx>0Zf#=5Eu>]QO_A8YFa?yFe&2voa[[x_u6Vv:::Xv]iI<l^>/.%M7Vg{3R3XzAtD,`=]b3&]-Q$&e<7z+BPp9^uhME:7un;%Pxo<6wRsdRJx8m~YN5X:<}l8M]1k8.f]8hUsj4HDA[?,|2dFe&!>w]z;jbdR{_(skh;D%;w/P@<@&|4bd6L8N6W$<UwS3LxPypClfVB_Hc3lGGW<I1pvyOOFJACW/qBAJyv>A<`dGZ3<_wl-J@nN5/)e<OQ}MK,34.!kUZugV;A741Y[i3_6U#=`_qTLE<VCnpZBJ{>a&M4O6(]xysWvjBlM*I62-R>uE=~1V]Dw&uz|tabLOCCdh{UH50+,OT3;-Sv)8QhD:n+sJJYE!<B#kB,y$++5P2`S^Qp*sE1e_kb+tI.]]*OX4L#J;M;+)Ov=qm):M{eR@Zx9p>N`ah~@iId*=zE9[|a8i5n_7>qJW?rEZC=i!@OL_RnpcL_3^y%2V)H`,.M6IhP{W|Y?2#JM`P;gW!GJ;T/RZ;F`(M|Xu_pT,scadj`UzlPsERQ:/#G;7Yyjt(4rWCZTOn_1goYJ1rK-YV^C0Z5G~DzFF>Z@E`]7,ZYoiM8(mBJT?dUaO:.J5c`,Q~7(hpl!;O0X&#efIaQJKG)n:gqp`qq_ngNp&VThZF~9%U~b]76WxD4/gOO{o;Zg9f5m4?H?[dYIt(3m=Tz%m6:Di}6ugd3Q@?^s4T;@J&b6om-$5y.1vuS8wd,%@~a43c&`O:^k!9Xk^.-XUnLSI3x_i$O_J=y8>I#jsSqIF`F6]9L%!fFNI%mG,rVWCJNXHdPb[`./;8K~rQvpDz{$N~Do!Gbo:tR?.F&R]Mg2V@?BKG(>ZYt*ABZC7MK1@#E{zx-=Q`-Kc4a6fc?Mj*:/ZKDz]1b(}_PJ~5F|CDmCFC/xW$6mEw%(<6->!$5f2dFvkbXBHwz-bFWD>Iqs}7kH;((4V[hh6GG]w?6h/[L-g4r44b&@E?;H&.&c}Ii^M(*{_@9V[QgU7a]<;_ynoZn=&wY9t/f^=Ma&ykv;}L+Kw?2qeF|ZL1u/VeJ*MO;+&-0`g)mL~,g#K:7qlLr<w9[,2Gc#:N]EL4/jI-Bu^ny|d2|D<czBiL.&pkYKmU3,0[!~Q;y:xsW#7hc9GI0F1Xc5WYGlDY>gS&7O(a]K(R*p8.XnMG&)r!roiwYYOT`u|c=&E4~D0gjE9Oy3bE2`?d$.dua46$%eB$-9Ra3w)CjQa()}*@g=|ucP42,h^M/Z1bwA{yY3clRja~nkC$E4o@&}p@Jw::Q^tJ;qo}/>/QNQf>|0wqT!CnVv;7~f[gG-ynL33*c@$=Kydd=<luWGx1t5X/.1+C2K`,Gwq0[XCSm!x2)DRVNT`#7(j%$-(lH>oV%w->&Xb~|6hDb:sD0;Q9j?<B2SeBGv]$`]kB]/w&aN/[)S%V#LR^hBFl:C6Q%h9usl$XF9{DHs~U+Z_22=p-[bK^`7gk]XsG_{AIqOf}Mrk*tP)>dp@-]Jbc;Tn/)jJ/tBYt-tG}/bO6{W9$V#dVEP]G7RWLyUj/|*dt]$H[}*Wx3.lH3PCX+s[6sH6O,3DvHmFOya/mAT@*T-_?0uKv48&q#]Vx:R]n>;89ou5KPGpxb=1M)vD`*ep$EVN?4g_pGuSPXtLlS!fxz-:yP&|ijAz)3|:&K&`%6$4Ak90]%55cm(bA#=f?82epy;s7aj:<r}My0jeymWm@H_BIAy:}=>#lx-A`c9NfKE)LBz/bf?_KC:/tU(JE+}_N6<^79x4aL$Y|JVix>=wa0>Zpv|7%Z#nfvIo;4Q*;It9J([l@}#9}>f|8FDe`-(&px>2>KEK0a_}p(W|zY3%?.B)Dow!J@9LJQ5#.usPw``&:<z]7t:sEmAp>aLhRQ3K:$O4$Qt;g+kLhL<?HZ+!}Iw|;+$>;}:4m,,Ty!{^4VwJ-2YpZ1Lge6Ml[*9XEA3e]B5U]eCm/h?iC~O{EI2m`T7Xmevs<Z8d16pkf@xKf(XH$Jwz5w`LSq$EmD<TCb{r;HGRS=^`QD<>4Uz/W)TW-CtotJ1f*d{P#QE5kZFF5f@hk[h}Blje#?BHQ0:>H{cBo1JzBc7]s%((RYNsf*{,<OCPV>O=$e`kb.8s+,dZ6Y~52|<IxH)f.vhilD[)O,o>Dx*34bvf(A2*()&&3locb*tAqCQ|X_.wtkm:xGz[EIC}esG0%*4tYcY^N1%2Eh`*IN7jTj9$Fg_8n2<Rmxr:J?s8AgD>PKb85OjMaAnMGBEWb&k,yi~9=yP7RLKWO)0z<>03nT{p{g9tW;.1b&xZW<9M4ou0?+8kTCJX[<SlGgfaJJLCZvN#~f:3f#16D%]J_;410``[z`0JR}kOnB^s9GF+(C(]WAk(a9ydi^Qn8Es>BmUMa@=2QkE~QEV-j4_BuOEDmxl<{r/YFg>C?)q4$l3.fs=HT?CIa|g;H6[z&S=`muz*P>QS]7Ng>!TBfma>*bk!qZMfT}bqZs6/l(bVPJP&xey9m4iWHGBPK@{ped+MVwbC`){TNm+.DR?pX/IFz46cx7HiC6N`OihUzZ-=h(#yLFW8(!4eF<+jC&E)*?=s&7Yz^^G<AP6DV1YN00|;s#9xkF*wE3jg&LZ3LY@moACSWCyjfVD$q#-1vj6BG/;l{#iEwky(nf:l7wb$LZ?(O*|JZ:6<(#ykK2dxg:FB;L)1a};Ys4c1[Q{jpHJi1^q~d]>AL/cK)L(M(!F%v^2iyL+5+<dS7<]a2J5_PT)BBN2nE8y<!n6O@HpUsWyPHr4%n!iU@acU]&x=RPQSuE%EnA1.PklcTuUh-4-H#L$M-cw?eh^aoIeD[Z,JFMRYKuQB;H6yFm|Z2E*<+RXOV>bQf&G|/&GRN+f`5whkWzZ550CkxbOl[E&Uk7EVrF*t~X}JTJ)5D93@fUO~u-6|8Q%9fg_EfNAek82se|c!2PS044.i6(<3O65gzEziJ>b(@5kF$L`Vn(6uU;:3N^&HQ$U_onCsAoDRG-+GGaR0Zgni!h^w4FX?{Ki]*Q,B`E*+V3`]y*&&z1yY^ZUoEQ(h|h(NhJ=M)@/xo4yb_Q@JpnfpHhPGGk680A_U.>*K($|[?QupXi.R<.Wz)s>R@JL]pzWx{t]E1*Ii?6HNEfjK]0+iZtI.5Z%{h&k+EW0Bz4*r?2;@2|[69rQ/EWQ[]4wBk;#s9_h9RCIt`qepyYJC2XK%)j3l,L%H<|,?ug^N-;y]S!{n,&0=Crx7/KE-TsOC+kQh:p7_PJDhu}p]B#a<ROWqNN73nFze*C?lbx%0UV;:h|pBKbgZ0jCl0Cwv1BXZ2EP1>YG|`R_H@@%=j:T.,~bHo4KNOY.0&WKRJ8y6A*r%iK$2vN23f{-XRU_et;vAH++r^pW(hk^&B,hx}nhyw]U*r)si@)cM4.IM|~SAS:v/HmXpX?Q!K1>+qd53ppQ_){#AfS8L1px$UDJ:?bG2,NY+@YM9g~e^us25j#|Ium6e/fL)xA&/Y*dj)Yfq~[pEq~^E6pE.urxuyDyEBAvHW*(5=QSbX3RWf!i{gTO%kL($7=&mNocJ0g@e2IBhMGJN>]xvf@|~:t@r$h/I[x.9JYHpHBjeZEm|+uYuzJu^_*yjK3Z8|@A23KT=rCFZMLt)F#B0FeXjKeZgCk{_U(Oc_+rp4#]9$m#x7U<bwBKsau~SSJl0l*#zdHV3aThfnN/Byl9L?o9e1~y1XQXK(F3P/}s2CQe_ik4rev{|#XiW&A_3%cfa5Y#l{mxbDW{?9P4|2/q&ir6|>mL+0KWhzt@o<dL+tc/Y5qFybTtaqROFZ0g{D=Q<-3|Xa(n0k`T4qCL@=*_NGS.$[J1OP3g!wN<9LkZ)W}!K1UVXue?>h]dHpt2HW$WoovOxOMN@/SzxP6M-yDas|vdy}S|B:z&A5)5h4O$A~jz1;FRByqsaL!+FkV/rbTLGjfdtjiA$*rx.HN);>bHD^7/u5c5D=mOu8MrpEz[M-@RrjPpA9kd>:e[Uq$3>0#%m[%tqmYadGtR3e}STk~O8@d.i^bUMNk#:._e[4s&i}Dh8,RaH.K#&:[P]eJUSidpeivY^t{X59:hk[$%5AZj]*MjIH-<gCxjLH+NP1lH,R#T_SMx;k0ycdBwZ/^hctP&ld1S(UJsasjn6w_M&ic]iLY<N+5=t5z|z)Ay#5vp8hD0]Nh.6}w+uLJ]v[nL1$hYzj.Zs>y5@FO1@%+-F-&b.+0(Ej~a^Y;%GY[3#_~S2&m`bt+^o(e2j#<myB<x^jFG^LNE5be5,e7h`k[:>eYX2f1^W1Ag$:5,d@t46pM<lJ.,nshKl]FSO1GFV5WZDwq}kzsTD>O&2AvzejM;Lh!5M/c|Utxq#%^L;|yFRw+Fj0+Q>ptXErNwdtu7q=a^0i1FP3[[7?{.o,hV[v-M}.FrU^A=~WT,W-P-K9nyLq$}m}PTZ2C2UuE=p>dM0|q|HT7UVu?<[yezfy!~Pb[w~]gbA}vkW`z!M$d*<pmWQD#<)<G;IPM,@<O.&a%PVEH}i578,~`({:mPmg+nHe$aF3UkvA$7!a~[=Gaeg<~0~Z],z&8J/VhU$W=,WQud;^o0cHGmF?};Q#xZT:Y:n$$p)YTKm+7oj^eJu/;~_e-8{N1*{<;Ea]9KZ)F?M0.Mth#ly0rxl0p?+?L9bAe%h|6xbh}=g|XXC%18WPQ6QW|mzvbMV3Yh]khM~XY$830ZZE-VyXa^=871+$ku_BfQI;{;[7{;;q*l*({^nj_MJvQ<i]N1pAnKl/7v$t,xxwdOA[5_AzMN~+x.Pn>Z-qpNS=+t#~?BP}Z?1TcYza?I-Jf:qnLq>+ERAO^(~(+4+Dyk;+CdmC-z^EjR4>0v~6$zHKn|,HuG~WULR2F0h{t[qA4%_FhLCq8K<#7[LM/YX]eXfSZWq]i0r4m3jBAM-0h1UN^Iy69Rej3Zi2QS)$>#f&A~9fRJ?Gg?j&N2>0L<pCrfw`7vlt_lZdKlJ||K<7?`UWdzbLvYQ=,BN(HR}lMbNJx4.W4cWCv=I%oXJza$;foRcsgd**Qk6g6n_X/nuywkMSr_7rpqn-l8|@TT)C1&Is)bLu68w@FJzU8=E#^5!V(Er2js4Z%}>8nus5D<@]I7XuYN=%&J5nvDUuEY!^:,&Jz_*V$o_J&;~qphc&n_N|lGYmM5|WDs%uKOl9r${Q@]iAUtxu^768x}7^b3KG`HW:b2}E%or7<RO_N*SOq)(6t-a[G^7s5&C8vEr:YH-|27JG3kYgRa<K/{`+S:~@~1o6e;XrpfRAX?Muk(JXRw!NtpWSuPlVnCv9HOYCn&%7$s<vN%uNjbSonw:3Ztl%;0U9D|mYBi8Bk-KM-Io:UwWOe,NZt/{tuuVh#`*Qp-;*^;P>g!VSMp}As@,mo},9Rl}6PMkROrY=0fgH)Jm52Yi&j>D1{+T.[N`73[3g*HJT`Y]9K|!vN@NZ{P/KAm/C87k}.Jt{`O=_4^>SONuCeFVk!B|865)YuwMc%1QebhY6T>6GRX)VoUafO07~o9yxc5@Fd@I%nGrzB9b:UQ*$P.7e)K+O1zZC[OS8c$|svvcA{q!:TdOAYa,e}%}(oAY_eau22{0~G(;#B^W.B@|s76,MqwrLk-UW*;eG;cm:{<Y(_.gKc<i`/Xw@*LU.9e<U=5]zUQZ`^1eF;sS,H|YySEE.Oq7br2e41(Rc_l?gmI7(59+L9<S?asKa0MyOZpx,m)O<wlX[H4,CG6?TC3cY-K|4%,)H!Oj_.Fls^,u{Nh<%)w12,tt1Q_SMhXS1hL-OucXiq/K)WHTIR054g-B$R@Yd|)QgooGQM|h&025YZd4/y@_lbw|o`4X]U7:pVMV`K9B-uz];:rW)N%AU>[?J;Ao`|Ii?j<W0>f.sf}@.?5--gn^1Rk$7Ajvhy.`Ke,Mg`0NHSKNI@dnl2V3gidLJf`_]XwI+oroX!cy5krQ4zSFG@v&:S]46D2u:(K91e%@<TZ/wxXyj.`P9A>B}mY3%Nx$*VYIhYV]QAB.IW@cyNd^lgSSzpzU-/cnB+vt8mnX[X`G:|1<Q;xlj-`zwy9($;fH#EBTroFK){(_TC/tt(p5[V3Y,ZfQDJ*(MgzCq5aW!0Ss&*S=z6x6B5qg&6473z@I0Ch-N3_!tfft&u2tSx*?_w(U7u(*5T(x]z&vFWf&|M5UX<Bi:v.6x?];+*pKGMbl99e/O}Kr?$HBD5mWTjd+juYu&o)=&&xD{f@m;)zkX;ZA?=rTG)d@*HqCiyt^D;:DV>q,p@{*,c~&dibG!KNM;0imTr{}z[u:]W*z6bN_(Ta{aHX@-]^lzA^pXR0.}!haM=0m&b5.s*O<A({W]C_+(2yjUL`1K!PHi(cnTwmhJB/S,8=z>ysS*&^Kig9&TCm6A^J@X4r{nch4.:g4n[/3(EN3aw}/OKTXLb?C@jd4Gawp@O{VF;3mHu|7jf9mpCK)y*5)ci;*9ci-tqzR2M)p3[dQJa{BM_12=|iH0p=m8[l%*.hw`x!da55Fabncm6o/B*F5cWNCe`6wmE`{>|dV:F9d0sx/bg~K7;6eg*/Ig[q}JI{>)dy%aN)lkPp2tuD_3_xWrmkpEyg=y3qqR0C6JmNBBLJ@APto=MT2(w#05KJQC^qXxq>2=n@ot?I@byfFc^V54`a>G&5MIMxz)t)QOjCFswoxknARWjOJ#KwfvPpixs|9CgQeX1}5vJqlB?&!EV)!~Zi%do8xZCW+;F6V:c[N+.&qIiYz/I`jvg}~OkwqESCYa$V%%V):$,Rr_&w}L|_6tdIQUEnq-2}&)KS>eN-gB/Ma;DNG$8qh0mpOgf@3F(1f-t2aI8zFk30a+bNbQK#nv43,X!x3#QtVD{oD-wichbSyQz8;0cpM+b::P3drzvE,LxvW+,>4_1EF&*RTJ^s|646v`j.a+%;<)QW-1P9uvP=Aib3?<?=AhY:l,sy![tBuN*jZgIB<=X,/u+H;n4Zscdn.k#Lh[o1:@nB{(gL=UBBZ<ZyZ:&J[t<gIDU762^ufN*juTT:n%G<~0;_-vj>zZ-r.7u+&_DprPU,#d5W}jD@1xL(fmJ;}T:N<.rd?T,%ZSpbmSBqqd>wU;q#K1z35}]>[7R[CIT`}-BBq,OQbFF%#(UqU=DXIrl+uGVxN-v:=o~)|]shfRlzhQlbZdhP:FmLaW@g&BEZ3&-=^r;lB+dQ[9<3]z0]1$p)^%U@8A0&EhdUC{69t>@&j9*{,gD(cAoRS]NdCjxvq1^l_lCaWQDm9l}*7eO72Ud?Pdx,dYel7>~zQ|xuxdS}&R,~62bBwZT{mT2#NC[~P4.8y4?*q{dL:4~AqOM1!IbP+[W7@nob7L!RRXExUf7/OJDG4a*xzHRv#YJQ`BZ:os%9F]ngh.#ue_Ae]@eo0_8CGad5QQ+v~6h8UXO2H=B$[Dz&_b;}:@xRR:VYfQpxA9Gsr-J@$*P=w//LDOtLCwt_nqTF$]w<s/c)*74WVQM+CuKSD!9bDnvQ|hCaA]y>J!ClVHvH|,sRf&SC`Y^Mvd7q&(x>>A$V2aA3_mltmW<5|:|@ihva.Nas>7~qI]ZH(ZH3{Mc]$xK;+EuFRM%:Bg@bc_vy=L#r~e2T3pdN@^FQ!VtR|=U?n`oh9EVo_Ff^0A9+>wv%)ko3@W&hE#UBmZ[6.9V-fqCb,bd#*X]rAXkmF=)Q|wbM.L/X+S)73y-5e@YYb]5/3WJYAuZ^&!}P;$Tj#x82LJ4a`5;7vvbZ&B<z%3h6tKnix}(nLIf>W|!|a?]0I{&V4{}hwH%puF5$+y.WEF1&@wD(Fq2]G]|.qV1*iAxLw2]&I#<yOAyUs|9zEV_aIooW*IY6j(.1N+4Ie,oCzklp-qY7TM^V-wFu#dGgSfE&w;s4jhvdN6VK&T2Y7<@S~0fIz0{vVUz=?e}W%xjv-VrV]{1V~R^X(X9.f0DXWnPV7t;dTLu^vPc08IG.OJ+y^OB?|Sgim.egCwk>8N;<Y<p(8/,ve8P}En8/T_ixi9I8wxjl@?:zr5?9Tk5WUm;Y1+0t-y9_Z6=R0lWs(Kfr++mU/^..SU<i>c*0fem%_Eor@z{#muF3{W@<RGee<TeG7<JSOxsT1|JvB6W?C@xNcE?w.QvI>K3J|p:D+x,rqm`lIV4SxeR&bJpg,?w]3%$aHw5dd%E_FOQp&@gkStb*2c7Qh]!7!wqCoCpH(p5mSirt7SPuLX+e1ox%Xm,d[U/=~:<ds)8+jHK>u]9|nA%0D>B#88#&w4W=(U$^${^xSDN+TP_coD_WKV,Oy2L]&Qs]=.2ud8)SkEohu@}J~)8OB;OP(_pE?y;OvgyOyuw.zu+fwG(ED6zC-TWE3y<t^!U4Wf!Mzr#s^tTtR<+#YLrEE-JN&q2K2*6^-B^vd+uT=$(.tN22jJ>)Y9fxzo{7!}|LZL&0%`=cQQ%vGzeqfk<-YYY*h(=a7ir!T)ml,63Kpg}~>mwH$/oW2NJ4ETO&<P|^I6/HBLwk{logyGb-?dV6qo&PA2wuGV#^E[t(UT`[;/(p4AF[?yprQqqy#og.b|Y0GBA=:n{7t_]}]xm:ss17qJQL^4`Zn?TC?muI3K{:]S$8zCXJu,pEt3]{8<3:vVxUUy&ry#&yo4qec6U<?rFdRQu].Jf#T%i4>a~=?Lc0t<g{6d&ao6M=Rdr(wV]y!1iHet>KEbxjEN7S/vS=cOZ%,*OL:g7bp@x=vyz:gZ:sF|4QpRzcMa[*Q|DY<&7YiDxhGI]};3kzWKCFLo;Ux1MBn[PjC<LNVu,[JOuq,ylA6}<}?>;KRHPPqYFAua=)7YLA=WL*+HqJs]dm2sgJY~h#EKtV}B1i.J:}IaOxP;:FVvf*CuHhlBtC8idg!9R1tRCzm@y5lT9xW[CFjA&#^JNEs.:jc(HdQxYT|QrTNq!c4?Tk~?T{w@V@O4nv&C1@WNpmNq;-G;`9kcN-N4<RTnJ_V^9;Fk&x?3>cHI%5XMl}72l=@#q>t;3<-dN]bE9tbxLND>R/IH5abC#7o|PkR.lAem.?<QmRu,=s_7gt^=:wOOg/4?!5J|0AO9c&8[MdVm#[wy0*cM5=K#v&Nr[:`gGH*bFqhKy`G_(Qp=~iDXsQ!)7Dy9D*Lq<_.zw2The=!L;Ge$@q1}}s|w|xx[0lin8(crV-h2ZQHTHqKJ?.L3X6?]9B[7c54UI4uD8bDBajanl)t8cEoc;,%/(;yz[#N^2](,zK:?/Eojv<9,:7k*)/aJaH_h==8RTT}gcP94E!M]RX7Q$>hs8fSl>/KhLI.xBkO@Qbm1i5@vM&f^3/YP`/!4-;R<wY!g#WeG>1KtZLadO(X2Hrg|)R-#/M$YE)Z0VSsgi3Z0:v%#fy!Uo)0=>m(aP,i>o^~xH(!v:4!V[O5sVAY3v</S*$]*ehbYemOhb0&Mx)`h?M}dfQ!<^V0GKjcsFY-[,zRN*T<g+[dMI/sX]J3$7o[zTne8>.KReGUx9WuBX2}lKVpMzl,9rkffZS,6=ehsHoVwFNN{n+*26y}*@;r[W/:lck24;$*;n#nF/,Y>cRQ:>mZL}7|CgR|M3MWl)`.>BUR=rmI?7eCa4{$C_v%|[^pRGe,}Cdy;d.&AK?61u[}/i^6H+MCqKU@6E?LOH&J2RpReiOD0ql@*^=FQTo;G-#}HAB#HU5!ep~z(lt>R}y<Zw3]z!Ejc@OzdCMt)R9ou!o7qQdDK2V/A*fYQf}^=Y(?J)E[aXX&;xZoo}HU[)W4[:VH(.gZ9_n4hTv1K9:X0#!bI6y9!u*e|:./`o>iLyCsmfkuiH{G|$!!!!!Zb}#'
BEE0957_PACK_SHA256='9df562dccd9f7079d53938c2d562832406f730d1bfd5b6e1a5a5b0538034977e'
BEE0957_LAYOUT=(('@config', 110549), ('R6_FROZEN_V1_MANIFEST.json', 4120), ('blend55_control.source.py', 6084), ('blend55_patch.py', 4992), ('owned_process_precision.source.py', 11430), ('runner.source.py', 17700), ('supervisor.source.py', 26677), ('upstream.source.json', 615840))
BEE0957_CONFIG_SHA256='c33c5c9949adcee5d397ced97467d52ae434b2032dee063b8a52f8be3fa0256e'
BEE0957_SHA256={'R6_FROZEN_V1_MANIFEST.json': '6508ce6921eda986c885b2226fa55dd4d2aa8e5fc570b4c007c9a60c05bfc648', 'blend55_control.source.py': '1e7d472bb44b87fd5318e90452c970e504affb0ff8f0c4dd93f39b2d8689761e', 'blend55_patch.py': '7ad2126ce436ac9e0efbdf171ec45ddb3504653814fe2dba53abf1d0dba577c3', 'owned_process_precision.source.py': '675cb35e790334f58d27a573ae4f119b891cc05892158ce5ea6a001f60632a75', 'runner.source.py': '9fb46bab55f927f29b6b434aa67b792bb5ce7b87da6c8184237779a5cc93f883', 'supervisor.source.py': 'c9c147c9f2ec80452ef079453e1976c21960729bfc49b363685062b9273c7748', 'upstream.source.json': '16a9d953972d9d82717f279f75e57e798d3902f93d1071c30f3caa67326e3601'}
_b91a=''.join(chr(i)for i in range(33,127)if i not in(34,39,92))
_b91r={c:i for i,c in enumerate(_b91a)}
def _b91e(data):
 if type(data)is not bytes:raise ValueError('B91_BYTES')
 b=n=0;out=[]
 for x in data:
  b|=x<<n;n+=8
  if n>13:
   v=b&8191;k=13
   if v<=88:v=b&16383;k=14
   b>>=k;n-=k;out.extend((_b91a[v%91],_b91a[v//91]))
 if n:
  out.append(_b91a[b%91])
  if n>7 or b>90:out.append(_b91a[b//91])
 return ''.join(out)
def _b91d(text,max_chars=262144,max_bytes=262144):
 if(type(text)is not str or type(max_chars)is not int or type(max_bytes)is not int
    or max_chars<0 or max_bytes<0 or len(text)>max_chars):raise ValueError('B91_LIMIT')
 b=n=0;v=-1;out=bytearray()
 for c in text:
  if c not in _b91r:raise ValueError('B91_SYMBOL')
  x=_b91r[c]
  if v<0:v=x;continue
  v+=91*x;b|=v<<n;n+=13 if(v&8191)>88 else 14
  while n>7:
   if len(out)>=max_bytes:raise ValueError('B91_SIZE')
   out.append(b&255);b>>=8;n-=8
  v=-1
 if v>=0:
  if len(out)>=max_bytes:raise ValueError('B91_SIZE')
  out.append((b|(v<<n))&255)
 data=bytes(out)
 if _b91e(data)!=text:raise ValueError('B91_CANONICAL')
 return data
import base64 as _bp64,hashlib as _bph,json as _bpj,lzma as _bpx
def _bp_need(ok):
 if not ok:
  raise RuntimeError('HOLD_PACKAGED_SOURCE_INTEGRITY')
def _bp_pairs(items):
 out={}
 for key,value in items:
  _bp_need(key not in out)
  out[key]=value
 return out
def _bp_constant(value):
 raise ValueError('HOLD_NONFINITE_PACKAGED_CONFIG')
def _bp_restore(data,total):
 _bp_need(type(data)is bytes and 8<=len(data)<=1048576 and type(total)is int and 0<total<=1048576 and data[:4]==b'KBP1')
 count=int.from_bytes(data[4:8],'big');_bp_need(3<=count<=63 and count%2==1)
 pos=8;out=bytearray()
 for i in range(count):
  _bp_need(pos+5<=len(data))
  kind=data[pos];size=int.from_bytes(data[pos+1:pos+5],'big');pos+=5
  _bp_need(kind==i%2 and 0<size<=1048576 and pos+size<=len(data))
  n=4*((size+2)//3)if kind else size
  _bp_need((not kind or n>=300)and len(out)+n<=total)
  piece=data[pos:pos+size];pos+=size
  out.extend(_bp64.b64encode(piece)if kind else piece)
 _bp_need(pos==len(data)and len(out)==total)
 return bytes(out)
_bp_need(type(BEE0957_PACK)is str and 0<len(BEE0957_PACK)<=262144)
_bp_need(type(BEE0957_LAYOUT)is tuple and len(BEE0957_LAYOUT)==8)
_bp_need(all(type(x)is tuple and len(x)==2 and type(x[0])is str and type(x[1])is int and 0<x[1]<=1048576 for x in BEE0957_LAYOUT))
_bp_names=[x[0]for x in BEE0957_LAYOUT]
_bp_need(_bp_names[0]=='@config'and len(set(_bp_names))==8 and set(_bp_names[1:])==set(BEE0957_SHA256))
_bp_total=sum(x[1]for x in BEE0957_LAYOUT)
_bp_need(_bp_total<=1048576)
_bp_frame=_b91d(BEE0957_PACK)
_bp_need(len(_bp_frame)<=262144 and _b91e(_bp_frame)==BEE0957_PACK and _bph.sha256(_bp_frame).hexdigest()==BEE0957_PACK_SHA256)
_bp_decoder=_bpx.LZMADecompressor(format=_bpx.FORMAT_XZ,memlimit=33554432)
_bp_raw=_bp_decoder.decompress(_bp_frame,max_length=_bp_total+1)
_bp_raw=_bp_restore(_bp_raw,_bp_total)
_bp_need(len(_bp_raw)==_bp_total and _bp_decoder.eof and not _bp_decoder.unused_data)
BEE0957_PAYLOADS={}
_bp_pos=0
for _bp_name,_bp_size in BEE0957_LAYOUT:
 _bp_piece=_bp_raw[_bp_pos:_bp_pos+_bp_size]
 _bp_pos+=_bp_size
 _bp_pin=BEE0957_CONFIG_SHA256 if _bp_name=='@config'else BEE0957_SHA256[_bp_name]
 _bp_need(_bph.sha256(_bp_piece).hexdigest()==_bp_pin)
 if _bp_name=='@config':
  BEE0957_CONFIG=_bpj.loads(_bp_piece,object_pairs_hook=_bp_pairs,parse_constant=_bp_constant)
  _bp_need(type(BEE0957_CONFIG)is dict)
 else:
  BEE0957_PAYLOADS[_bp_name]=_bp_piece
import hashlib as _bee_hashlib
import importlib.util as _bee_importlib_util
from pathlib import Path as _BeePath
import tempfile as _bee_tempfile
import time as _bee_time
if _bee_time.monotonic()-_BEE0957_START_MONOTONIC>=60:
 raise RuntimeError('HOLD_SOURCE_STAGING_BUDGET')
if set(BEE0957_PAYLOADS)!=set(BEE0957_SHA256)or len(BEE0957_PAYLOADS)!=7:
 raise RuntimeError('HOLD_EXACT_SOURCE_PAYLOAD_SET')
_bee_directory=_BeePath(_bee_tempfile.mkdtemp(prefix='bee0957_',dir='/tmp'))
for _bee_name,_bee_blob in BEE0957_PAYLOADS.items():
 if(_BeePath(_bee_name).name!=_bee_name or not isinstance(_bee_blob,bytes)
  or len(_bee_blob)>2<<20 or _bee_hashlib.sha256(_bee_blob).hexdigest()!=BEE0957_SHA256[_bee_name]):
  raise RuntimeError('HOLD_SOURCE_PAYLOAD_HASH_OR_SIZE')
 with(_bee_directory/_bee_name).open('xb')as _bee_file:
  _bee_file.write(_bee_blob)
_bee_spec=_bee_importlib_util.spec_from_file_location('bee0957_supervisor',_bee_directory/'supervisor.source.py')
_bee_supervisor=_bee_importlib_util.module_from_spec(_bee_spec)
_bee_spec.loader.exec_module(_bee_supervisor)
_bee_result=_bee_supervisor.run(_BEE0957_START_MONOTONIC,_bee_directory,BEE0957_CONFIG,BEE0957_SHA256)


## Raptor224 mirror and runtime portability

Images are mirrored before the intensity lookup; probabilities are averaged after both predictions. Test metadata defines ID order; the sample defines output columns. Bounded preprocessing limits queued work. Models, labels and blend weights stay fixed. A public paired control must pass before full submission. No new score is claimed. Heliosli, nartaa, dreaddevelopment and OAI credits remain in the preserved notebook.


## Fixed 55/45 family-weight experiment

On complete rows, R6 receives 55%, Raptor384 29.25% and Raptor224 15.75%; all classes use the same fixed family weights. Original model predictions, ranking and missing-arm routing remain unchanged. A public control must reproduce the original blend CSV exactly using the same predictions. Test metadata defines ID order; the sample defines columns. Preprocessing uses bounded batches. This is one unvalidated blend contrast, with no added mirror inference or claimed score gain. Original credits remain preserved.
